# DecisionReady — Iteration 3 v0.5

## 1. Imports and environment


In [1]:
import json
import re
from typing import Literal, NotRequired, Required, TypedDict

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field

load_dotenv("../.env", override=True)


True

## 2. Progressive graph state


In [2]:
DecisionStage = Literal["pilot", "rollout", "scale", "investment", "unclear"]
ReadinessStatus = Literal["READY", "READY_WITH_CAVEATS", "NOT_READY"]
ClaimRole = Literal["prerequisite", "hypothesis", "unknown"]
RequirementStatus = Literal["required", "not_required", "unknown"]
DecisionHorizonRelevance = Literal[
    "current_decision", "future_decision", "unclear"
]
BlockerCode = Literal[
    "NO_DECISION_ASK",
    "NO_ACCOUNTABLE_OWNER",
    "NO_SUCCESS_METRIC",
    "AMBIGUOUS_DECISION_ASK",
    "UNCLEAR_ACCOUNTABILITY",
    "UNSUPPORTED_CENTRAL_CLAIM",
    "UNRESOLVED_CRITICAL_DEPENDENCY",
    "MATERIAL_DIMENSION_THRESHOLD",
]


class MaterialClaimState(TypedDict):
    claim_text: str
    claim_role: ClaimRole
    evidence_basis: list[str]


class DependencyReviewState(TypedDict):
    dependency_text: str
    requirement_status: RequirementStatus
    evidence_basis: list[str]


CaveatCode = Literal[
    "UNKNOWN_MATERIAL_CLAIM",
    "UNKNOWN_DEPENDENCY_REQUIREMENT",
    "WEAK_OR_INCOMPLETE_EVIDENCE",
    "ACKNOWLEDGED_NON_BLOCKING_UNCERTAINTY",
]


class MaterialCaveatState(TypedDict):
    caveat_code: CaveatCode
    rationale: str
    evidence: list[str]
    related_item_text: str | None
    decision_horizon_relevance: DecisionHorizonRelevance
    current_decision_materiality_evidence: list[str]
    source: Literal["semantic", "deterministic"]


class DimensionAssessmentState(TypedDict):
    score: int
    reason: str
    supporting_evidence: list[str]
    gaps: list[str]


class HardBlockerState(TypedDict):
    blocker_code: BlockerCode
    source: Literal["deterministic", "semantic"]
    rationale: str
    evidence: list[str]
    related_item_text: NotRequired[str | None]


class DecisionReadyState(TypedDict, total=False):
    proposal_text: Required[str]

    proposal_type: NotRequired[str]
    decision_stage: NotRequired[DecisionStage]
    decision_ask: NotRequired[str]
    accountable_owner: NotRequired[str | None]

    problem_summary: NotRequired[str]
    proposed_intervention: NotRequired[str]

    evidence_items: NotRequired[list[str]]
    material_claims: NotRequired[list[MaterialClaimState]]
    dependency_reviews: NotRequired[list[DependencyReviewState]]
    stated_assumptions: NotRequired[list[str]]
    inferred_assumptions: NotRequired[list[str]]
    success_metrics: NotRequired[list[str]]
    dependencies: NotRequired[list[str]]
    stated_risks: NotRequired[list[str]]
    inferred_risks: NotRequired[list[str]]

    dimension_assessments: NotRequired[dict[str, DimensionAssessmentState]]
    hard_blockers: NotRequired[list[HardBlockerState]]
    material_caveats: NotRequired[list[MaterialCaveatState]]
    future_decision_considerations: NotRequired[list[MaterialCaveatState]]

    readiness_status: NotRequired[ReadinessStatus]
    leadership_questions: NotRequired[list[str]]
    recommended_actions: NotRequired[list[str]]

    final_brief: NotRequired[str]


## 3. Structured-output schemas


In [3]:
class MaterialClaim(BaseModel):
    claim_text: str = Field(description="A material claim stated or strongly implied by the proposal.")
    claim_role: ClaimRole = Field(
        description=(
            "prerequisite only when current case evidence establishes that the claim must already be true "
            "before leadership can approve the requested decision; hypothesis when the proposed pilot is "
            "specifically designed to test the claim; unknown when the current case establishes neither."
        )
    )
    evidence_basis: list[str] = Field(
        description="Direct proposal evidence supporting the claim text and its assigned role; empty if absent."
    )


class DependencyReview(BaseModel):
    dependency_text: str = Field(description="A dependency, review, approval, validation, or prerequisite in the case.")
    requirement_status: RequirementStatus = Field(
        description=(
            "required only when direct case evidence establishes that the item must be satisfied for the "
            "requested decision or initiative; not_required when the case establishes it is optional or can "
            "occur later; unknown when the current evidence establishes neither."
        )
    )
    evidence_basis: list[str] = Field(
        description="Direct proposal evidence supporting the dependency text and requirement status; empty if absent."
    )


class ProposalExtraction(BaseModel):
    proposal_type: str = Field(description="Type of proposal, such as strategic initiative, pilot, rollout, or investment")
    decision_stage: DecisionStage = Field(description="Decision stage such as pilot, rollout, scale, investment, or unclear")
    decision_ask: str = Field(description="The explicit decision leadership is being asked to make")
    accountable_owner: str | None = Field(description="Named accountable owner, or None if not clearly stated")

    problem_summary: str = Field(description="Short summary of the problem or opportunity")
    proposed_intervention: str = Field(description="What the proposal is proposing to do")

    evidence_items: list[str] = Field(description="Evidence, facts, baselines, metrics, or supporting information stated in the proposal")
    material_claims: list[MaterialClaim] = Field(description="Material claims classified as prerequisites, hypotheses, or unknowns using only current case evidence.")
    dependency_reviews: list[DependencyReview] = Field(description="Dependencies and reviews classified as required, not required, or unknown using only current case evidence.")
    stated_assumptions: list[str] = Field(description="Assumptions explicitly stated in the proposal.")
    inferred_assumptions: list[str] = Field(description="Material assumptions strongly implied by the proposal but not explicitly labelled as assumptions. Only include assumptions directly supported by the proposal context.")
    success_metrics: list[str] = Field(description="Measurable success metrics or targets")
    dependencies: list[str] = Field(description="Dependencies, prerequisites, approvals, or external inputs")
    stated_risks: list[str] = Field(description="Risks or constraints explicitly stated in the proposal.")
    inferred_risks: list[str] = Field(description="Material risks strongly implied by the proposal. Only include risks directly connected to the proposal's scope, assumptions, dependencies, data, execution plan, or objectives. Do not list generic risks.")

class MaterialCaveat(BaseModel):
    caveat_code: CaveatCode
    rationale: str = Field(
        description="Why this uncertainty is material to the requested decision but not decision-stopping."
    )
    evidence: list[str] = Field(
        description="Direct case evidence supporting the caveat; do not include external facts or requirements."
    )
    related_item_text: str | None = Field(
        default=None,
        description=(
            "For an unknown claim or dependency caveat, copy the exact claim_text or dependency_text. "
            "Otherwise return the most directly related structured item text or None."
        )
    )
    decision_horizon_relevance: DecisionHorizonRelevance = Field(
        description=(
            "current_decision when the uncertainty materially affects the specific decision now; "
            "future_decision when it belongs to a later scale, rollout, or investment decision; "
            "unclear when temporal relevance cannot be established from the case."
        )
    )
    current_decision_materiality_evidence: list[str] = Field(
        default_factory=list,
        description=(
            "Direct case evidence showing how the uncertainty materially affects the current requested "
            "decision, current pilot design, execution, or interpretability. Empty for future-only items "
            "and when that current-decision connection is not established."
        ),
    )
    source: Literal["semantic", "deterministic"] = "semantic"


class MaterialCaveatAssessment(BaseModel):
    caveats: list[MaterialCaveat] = Field(
        description="Only material, non-blocking caveats. Return an empty list when none meet the threshold."
    )


class DimensionAssessment(BaseModel):
    score: int = Field(
        ge=0,
        le=3,
        description=(
            "Readiness score: "
            "0 = missing or materially inadequate; "
            "1 = weak with significant gaps; "
            "2 = adequate with caveats; "
            "3 = decision-ready for the requested decision stage."
        )
    )

    reason: str = Field(
        description="Concise explanation of why this score was assigned."
    )

    supporting_evidence: list[str] = Field(
        description="Specific proposal facts or statements supporting the assessment."
    )

    gaps: list[str] = Field(
        description="Material gaps relevant to this dimension. Empty list if no material gap exists."
    )

class ProposalReadinessAssessment(BaseModel):
    decision_clarity: DimensionAssessment
    problem_and_rationale: DimensionAssessment
    evidence_and_assumptions: DimensionAssessment
    business_and_economic_case: DimensionAssessment
    execution_readiness: DimensionAssessment
    dependencies: DimensionAssessment
    risk_and_governance: DimensionAssessment

class SemanticBlocker(BaseModel):
    blocker_code: Literal[
        "AMBIGUOUS_DECISION_ASK",
        "UNCLEAR_ACCOUNTABILITY",
        "UNSUPPORTED_CENTRAL_CLAIM",
        "UNRESOLVED_CRITICAL_DEPENDENCY"
    ]

    rationale: str = Field(
        description=(
            "Why this issue is severe enough to prevent leadership from "
            "reasonably making the requested decision."
        )
    )

    evidence: list[str] = Field(
        description="Specific proposal evidence supporting the blocker."
    )

    related_item_text: str | None = Field(
        default=None,
        description=(
            "For UNSUPPORTED_CENTRAL_CLAIM or UNRESOLVED_CRITICAL_DEPENDENCY, copy the exact "
            "claim_text or dependency_text that is the subject of the blocker. Otherwise return None."
        )
    )

class SemanticBlockerAssessment(BaseModel):
    blockers: list[SemanticBlocker] = Field(
        description=(
            "Only material hard blockers. Return an empty list if the issue "
            "is merely a caveat or improvement opportunity."
        )
    )

class LeadershipQuestionsOutput(BaseModel):
    questions: list[str] = Field(
        description=(
            "3 to 5 specific, decision-relevant leadership questions. "
            "Questions should target material gaps, assumptions, dependencies, "
            "risks, or evidence weaknesses. Avoid generic questions."
        )
    )

class RecommendedActionsOutput(BaseModel):
    actions: list[str] = Field(
        description=(
            "Specific pre-review actions appropriate to the readiness status. "
            "Actions should address material gaps, blockers, assumptions, "
            "dependencies, or evidence weaknesses. Avoid generic advice."
        )
    )

class FinalBriefOutput(BaseModel):
    final_brief: str = Field(
        description=(
            "Concise executive-ready Decision Readiness Brief summarizing "
            "the requested decision, readiness status, evidence, caveats, "
            "blockers if any, leadership questions, and recommended actions."
        )
    )


### Claim and dependency roles

- **Prerequisite vs hypothesis:** a prerequisite must already be true for the requested decision, while a hypothesis is what a bounded pilot is designed to test. An unproven hypothesis is therefore not blocking merely because prior proof is absent.
- **Required vs unknown:** a dependency or review is required only when the current case directly establishes that requirement. Missing evidence about whether it is required remains unknown.

These distinctions reduce unsupported blocker promotion by preventing proposal uncertainty—and generic policy, legal, or domain assumptions—from being converted into a mandatory precondition without case evidence.


### Material caveat layer and decision horizon

Hard blockers remain decision-stopping. Material caveats are non-blocking uncertainties that are materially relevant to the specific decision being requested now.

Each caveat candidate is classified as:

- `current_decision`: materially affects whether leadership can responsibly make the requested decision now, including whether a pilot can execute safely or produce interpretable evidence.
- `future_decision`: belongs to a later scale, rollout, or investment decision and therefore does not affect current readiness merely because it remains unresolved.
- `unclear`: temporal relevance is not established; the candidate is retained as a caveat only when direct materiality evidence connects it to the current decision.

Future-decision items can remain visible as future considerations, but they are excluded from `material_caveats` before readiness classification. This prevents later-stage uncertainty from being promoted into a current-decision caveat without weakening current pilot-design or evidence-quality caveats.


## 4. Base model and structured model wrappers


In [4]:
model = init_chat_model(
    "gpt-5-mini",
    model_provider="openai",
    reasoning_effort="minimal",
)

extraction_model = model.with_structured_output(ProposalExtraction)
assessment_model = model.with_structured_output(ProposalReadinessAssessment)
semantic_blocker_model = model.with_structured_output(SemanticBlockerAssessment)
material_caveat_model = model.with_structured_output(MaterialCaveatAssessment)
leadership_questions_model = model.with_structured_output(LeadershipQuestionsOutput)
recommended_actions_model = model.with_structured_output(RecommendedActionsOutput)
final_brief_model = model.with_structured_output(FinalBriefOutput)

print("Model and structured-output wrappers initialized successfully")


Model and structured-output wrappers initialized successfully


## 5. Prompts


In [5]:
EXTRACTION_SYSTEM_PROMPT = """
You are the proposal extraction component of DecisionReady,
an enterprise decision pre-flight system.

Your task is to convert a proposal into structured information.

Rules:
1. Extract information faithfully from the proposal.
2. Do not invent facts, owners, metrics, dependencies, approvals, or evidence.
3. Distinguish evidence from assumptions.
4. Distinguish explicitly stated assumptions from assumptions that are strongly implied.
5. Distinguish explicitly stated risks from material risks strongly implied by the proposal.
6. Inferred risks must be specific to this proposal. Do not generate generic business or AI risk lists.
7. Do not assess whether the proposal is good, bad, ready, or not ready.
8. Do not apply policies, precedents, or external knowledge that were not provided.
9. If information is genuinely absent, represent it as absent rather than filling the gap.

10. MATERIAL CLAIM ROLES
For each material claim, classify its role using only the current proposal:
- prerequisite: direct case evidence establishes that the claim must already be true before leadership can approve the requested decision,
- hypothesis: the proposed pilot is specifically designed to test whether the claim is true,
- unknown: the current evidence establishes neither prerequisite nor hypothesis.

Do not convert an assumption or uncertainty into a prerequisite using generic policy, legal, regulatory, governance, technical, or domain knowledge.

11. DEPENDENCY AND REVIEW STATUS
For each dependency, review, approval, validation, or similar item, classify requirement status using only the current proposal:
- required: direct case evidence establishes that the item must be satisfied for the requested decision or initiative,
- not_required: direct case evidence establishes that the item is optional or can occur later,
- unknown: the current evidence establishes neither.

Do not promote an unknown item to required using external knowledge.
"""


In [6]:
ASSESSMENT_SYSTEM_PROMPT = """
You are the readiness assessment component of DecisionReady,
an enterprise decision pre-flight system.

Your task is to assess whether the proposal is sufficiently prepared
for leadership to make THE SPECIFIC DECISION being requested.

You are NOT deciding whether leadership should approve the proposal.

CORE PRINCIPLES

1. DECISION PROPORTIONALITY
Assess readiness relative to:
- decision stage,
- size of commitment,
- reversibility,
- risk,
- scope.

Do not require evidence needed for a future scale or rollout decision
when leadership is currently being asked only to approve a bounded pilot.

A small reversible pilot can be decision-ready even when full ROI,
scale economics, detailed operating design, or long-term evidence
does not yet exist, provided the pilot is structured to generate
relevant learning.

2. MATERIALITY
Distinguish between:
A. a material gap that leadership reasonably needs before making
   the requested decision, and
B. a useful detail that would improve execution but does not prevent
   leadership from making the requested decision.

Only category A should materially reduce the readiness score.

3. UNCERTAINTY
Uncertainty is not automatically a readiness failure.

A proposal may be decision-ready when:
- uncertainty is acknowledged,
- exposure is bounded,
- measurable success criteria exist,
- and the pilot is designed to resolve important uncertainty.

Penalize unmanaged uncertainty, not uncertainty itself.

4. EVIDENCE VS CLAIMS
Treat stated claims as claims, not automatically as evidence.

A statement such as "productivity will improve by 20%" is not strong
evidence merely because it appears in the proposal.

However, do not require full proof of the benefit when the purpose of
the requested pilot is to test that benefit.

5. GROUNDING
Use only the original proposal and the structured extraction provided.

Do NOT introduce:
- company policies,
- regulations,
- mandatory approvals,
- required governance forums,
- technical dependencies,
- legal requirements,
- market facts,
- or operating requirements

unless they are supported by the proposal.

You may identify an uncertainty such as:
"The proposal does not state whether technical integration is required."

You may NOT assert:
"IT integration approval is required."

6. INFERRED RISKS
Consider inferred risks only when they follow directly from the
proposal's scope, assumptions, dependencies, objectives, or evidence.

Do not generate generic enterprise, AI, legal, cyber, privacy,
implementation, or governance checklists.

7. DIMENSION SCORING

DECISION CLARITY
Assess whether leadership can understand:
- what decision is requested,
- scope,
- commitment,
- decision stage,
- and what happens next.

PROBLEM AND RATIONALE
Assess whether the proposal establishes a sufficiently clear problem
or opportunity and rationale for THIS decision.
Do not require quantified enterprise-wide value for a bounded pilot
if a credible problem and testable hypothesis are established.

EVIDENCE AND ASSUMPTIONS
Assess whether material claims are supported appropriately for the
decision stage and whether key assumptions are visible.
For a pilot, evidence can be sufficient to justify testing rather than
sufficient to prove the final business case.

BUSINESS AND ECONOMIC CASE
For a pilot, assess:
- whether cost/exposure is clear and bounded,
- whether the expected value or learning objective is credible,
- and whether success can inform a later investment decision.

Do NOT require full rollout ROI, payback, NPV, or scale economics unless
the requested decision is itself a rollout, scale, or investment decision.

EXECUTION READINESS
Assess whether there is enough structure to begin the proposed initiative:
scope, owner, duration, participants/resources, measurable outcomes,
and major known execution requirements.

Do not penalize heavily for detailed project-management artifacts that
can reasonably be developed after approval.

DEPENDENCIES
Assess dependencies actually stated or strongly evidenced by the proposal.

Do not invent possible dependencies merely because similar projects
often have them.

RISK AND GOVERNANCE
Assess whether material risks visible FROM THE PROPOSAL are acknowledged
and reasonably bounded for the requested decision.

Do not infer specific mandatory governance requirements without policy
or other evidence.

8. SCORING SCALE

0 = Missing or materially inadequate for the requested decision
1 = Significant material gaps; leadership lacks important information
2 = Decisionable, but meaningful caveats should be surfaced
3 = Sufficiently prepared for the requested decision

9. ASSESS EACH DIMENSION INDEPENDENTLY.

10. Do NOT determine the final Ready / Ready with Caveats / Not Ready
classification. A separate component will do that.

11. STRUCTURED CLAIM AND DEPENDENCY ROLES
Use the extracted material-claim and dependency-review classifications as evidence-disciplined distinctions.

For a bounded pilot, lack of prior proof for a claim classified as a hypothesis is not by itself a readiness failure when the pilot is designed to test that hypothesis.

Treat claim_role = unknown and requirement_status = unknown as uncertainties or caveats unless other direct case evidence establishes a material gap. Do not convert unknown status into a mandatory prerequisite or requirement using external knowledge.
"""


In [7]:
SEMANTIC_BLOCKER_PROMPT = """
You are the hard-blocker assessment component of DecisionReady.

Your job is NOT to identify every weakness in a proposal.

A HARD BLOCKER is a material issue that means leadership does not yet
have a sufficiently defined or supported case to reasonably make the
specific decision requested.

Use a high threshold.

If an issue can reasonably be handled as:
- a caveat,
- a leadership question,
- a pre-review clarification,
- or an execution action after approval,

then it is NOT a hard blocker.

Assess only these blocker types:

1. AMBIGUOUS_DECISION_ASK

Use when a decision ask exists, but leadership cannot clearly determine
what it is actually being asked to authorize, commit to, fund, endorse,
or permit.

Do not use merely because:
- wording could be more polished,
- some implementation detail is missing,
- or a bounded pilot could be described more fully.

For a pilot, the ask is generally sufficiently clear when leadership can
understand the intended scope, duration, commitment, owner, and purpose.

2. UNCLEAR_ACCOUNTABILITY

Use only when leadership cannot identify who is ultimately accountable
for the initiative or requested decision.

A clearly accountable role such as:
- "Director, Growth Marketing"
- "VP Procurement Transformation"
- "Head of Commercial Transformation"

is sufficient.

Do NOT create this blocker merely because individual:
- sub-tasks,
- workstreams,
- approvals,
- measurements,
- implementation activities,
- or dependencies

do not each have separate named owners.

Missing task-level ownership may be an execution gap, but it is not
automatically a hard blocker.

3. UNSUPPORTED_CENTRAL_CLAIM

Use when a material claim is central to the rationale for the requested
decision and the proposal provides no meaningful basis for that claim.

Decision stage matters.

For a broad rollout, scale decision, or major investment:
an unsupported central value claim may be a hard blocker if leadership
cannot reasonably judge expected value or exposure.

For a bounded pilot:
an unproven target or benefit hypothesis is NOT a hard blocker merely
because prior evidence is absent, when:
- a relevant baseline exists,
- the target is measurable,
- exposure is bounded,
- and the purpose of the pilot is to test whether the target can be achieved.

Use this blocker for a pilot only when the proposal relies on a material
claim that must already be true for the pilot itself to be sensible or
executable, rather than a hypothesis the pilot is designed to test.

Example:

"20% productivity improvement is expected" with no baseline, evidence,
or test logic may be a blocker for a broad multi-region rollout.

But a measurable uplift target can be acceptable for a bounded pilot
whose explicit purpose is to test whether that uplift can be achieved.

Do not require:
- full ROI,
- statistically proven impact,
- prior pilot evidence,
- or scale economics

when leadership is only being asked to approve a small reversible pilot,
unless those items are materially required for that specific decision.

4. UNRESOLVED_CRITICAL_DEPENDENCY

Use when a dependency is clearly material to starting or executing the
requested initiative, remains unresolved, and no credible mitigation or
resolution path is provided.

Do NOT invent dependencies that are not supported by the proposal.

A review, approval, governance activity, or validation step mentioned in
the proposal is NOT automatically a hard blocker.

Treat it as a hard blocker only when the current case establishes that
completion is required before the requested decision or initiative can
proceed.

If the proposal says a review has not yet occurred but does not establish
whether it is mandatory before approval, surface it as a material
uncertainty or caveat rather than inventing a mandatory gate.

Do not infer:
- legal requirements,
- policy requirements,
- mandatory governance approvals,
- security approvals,
- compliance gates,
- or regulatory obligations

without evidence in the current case.

IMPORTANT CROSS-CUTTING RULES

BLOCKER THRESHOLD TEST FOR DEPENDENCIES

Before returning UNRESOLVED_CRITICAL_DEPENDENCY, you must be able to point
to evidence in the current case that establishes BOTH:

A. the dependency is necessary for the requested initiative to start,
continue, or produce a valid result; AND

B. the dependency is currently unresolved with no credible resolution
or mitigation path.

If condition A is not established, do NOT return a hard blocker.

The following is NOT sufficient by itself:
- "review has not yet been initiated"
- "approval is pending"
- "validation has not occurred"
- "the proposal assumes X is permitted"

These may be material caveats, but they become hard blockers only when
the current case also establishes that completion or approval is required
before proceeding.

Example:
"Data Governance review has not yet been initiated" does NOT by itself
prove that Data Governance clearance is mandatory before leadership can
approve a pilot.

Do not infer that requirement from general legal, privacy, compliance,
or governance knowledge.

5. DECISION PROPORTIONALITY

Assess blockers relative to:
- decision stage,
- size of commitment,
- reversibility,
- scope,
- and exposure.

Do not impose rollout-level readiness requirements on a bounded pilot.

6. HARD BLOCKER VS CAVEAT

Use a high threshold for blocker classification.

A material uncertainty may be important without being a blocker.

Ask:
"Can leadership still reasonably make the requested decision while
explicitly acknowledging this uncertainty?"

If yes, it is more likely a caveat than a hard blocker.

7. EVIDENCE DISCIPLINE

Use only:
- the original proposal,
- structured extraction,
- readiness assessment,
- and facts already present in the current case.

Do not introduce:
- external facts,
- company policy,
- precedent,
- legal interpretations,
- mandatory approvals,
- regulations,
- market benchmarks,
- or generic enterprise practices.

8. DO NOT PROMOTE UNCERTAINTY INTO REQUIREMENT

Never convert:
"this may matter"

into:

"this is mandatory"

without evidence in the current case.

Likewise, do not convert:
"this information would improve the proposal"

into:

"leadership cannot decide without it"

unless the issue is genuinely material to the requested decision.

9. PREFER RESTRAINT

Prefer no blocker over an unjustified blocker.

The purpose of this component is to identify true decision-stopping
conditions, not to produce a comprehensive review checklist.

10. OUTPUT

Return only confirmed hard blockers from the allowed blocker types.

If no issue meets the hard-blocker threshold, return an empty blocker list.

11.
For every hard blocker, the evidence list must include direct case evidence
supporting the blocking condition itself, not merely evidence that an
uncertainty exists.

12. STRUCTURED ELIGIBILITY GATES
These gates are mandatory and override any broader interpretation above.

UNSUPPORTED_CENTRAL_CLAIM may be returned only when the exact related material claim is classified as prerequisite in the structured case. It must not be returned for claim_role = hypothesis merely because prior proof is absent. For claim_role = unknown, prefer a caveat, leadership question, or action unless other direct evidence establishes a different allowed blocker.

UNRESOLVED_CRITICAL_DEPENDENCY may be returned only when the exact related dependency or review has requirement_status = required in the structured case. It must not be returned when requirement_status = unknown. Never promote a governance review, approval, validation, or similar item from unknown to required using generic legal, policy, regulatory, governance, technical, or domain knowledge.

For either gated blocker type, related_item_text must exactly copy the corresponding claim_text or dependency_text from the structured case.
"""


In [8]:
MATERIAL_CAVEAT_PROMPT = """
You are the material-caveat assessment component of DecisionReady.

Your task is to identify material uncertainties that leadership should see
but that do not prevent leadership from making the specific decision requested.

A MATERIAL CAVEAT may materially:
- change conditions attached to the decision,
- affect whether a pilot can generate decision-useful evidence,
- affect interpretation of success or failure,
- or expose an important uncertainty that should be acknowledged.

A caveat is NOT a hard blocker. Do not return issues already confirmed as hard blockers.
When hard blockers exist, still identify distinct material non-blocking uncertainties so they
remain available for synthesis; never duplicate or soften a confirmed blocker.

Use only the original proposal and the structured current case. Do not introduce
new policies, legal obligations, governance requirements, approvals, technical
requirements, market facts, or other external knowledge.

ELIGIBLE CAVEAT TYPES

1. UNKNOWN_MATERIAL_CLAIM
Use when claim_role = unknown and the claim is material to the requested decision
or to whether the proposed pilot can produce interpretable learning.

Unknown classification alone is not sufficient. Do not caveat low-materiality
administrative details, later-decision details, or ordinary refinements.

2. UNKNOWN_DEPENDENCY_REQUIREMENT
Use when requirement_status = unknown and resolving whether the item matters
could materially affect the requested decision, start conditions, exposure,
or validity of the proposed learning.

Do not convert unknown into required. Do not infer mandatory governance, legal,
privacy, regulatory, security, or policy gates from generic knowledge.

Do not return the leadership approval, authorization, budget approval, or funding
commitment that is itself being requested as an unresolved dependency caveat merely
because that requested decision has not already occurred.

3. WEAK_OR_INCOMPLETE_EVIDENCE
Use when important evidence is stale, weak, incomplete, informal, or lacks a
decision-relevant validation, but the weakness is not severe enough to stop
the requested decision.

This caveat must be anchored to an exact structured material claim with
claim_role = unknown or dependency with requirement_status = unknown.

For a bounded pilot, do not treat absence of prior proof for a hypothesis as a
caveat merely because the pilot is designed to test that hypothesis. A caveat
may still apply when weak evidence materially affects pilot design, feasibility,
or interpretation of its results.

4. ACKNOWLEDGED_NON_BLOCKING_UNCERTAINTY
Use for another important uncertainty that is grounded in the case, explicitly
acknowledged or directly evidenced, and should shape leadership conditions or
interpretation without stopping the decision.

This caveat must also be anchored to an exact structured material claim with
claim_role = unknown or dependency with requirement_status = unknown.

DECISION-HORIZON RELEVANCE

For every candidate, classify decision_horizon_relevance:

- current_decision: the uncertainty materially affects whether leadership can responsibly
  make the specific decision requested now, or whether the current pilot can execute safely
  and produce interpretable evidence.
- future_decision: the uncertainty belongs to a later scale, rollout, or investment decision
  and can legitimately remain unresolved for the current decision.
- unclear: the proposal does not establish when the uncertainty matters.

Do not treat a later rollout, scale, or investment decision as a current material caveat
merely because it remains unresolved. A dependency needed during the current pilot, or an
uncertainty that materially affects current pilot design, execution, or interpretability,
may still be current_decision even when its evidence will later inform a scale decision.

For current_decision, provide direct current_decision_materiality_evidence from the original
proposal. For unclear, provide that evidence only when the original case directly establishes
a material connection to the current requested decision. A structured unknown label, dimension
score, assessment gap, or absence of explicit confirmation is not by itself materiality evidence.
For a bounded pilot, ordinary confirmation of planned participants, event scheduling, or similar
pre-start execution details is not a caveat unless the original proposal directly indicates that
availability is doubtful or that the uncertainty changes authorized exposure, pilot validity,
or the ability to execute the requested decision. Leave the evidence empty for future-only items
or when no direct current-decision connection is established.

MATERIALITY AND PROPORTIONALITY

Use a high threshold. Do not convert every score-2 assessment gap, inferred risk,
execution detail, measurement refinement, budget administration item, named-owner
detail, or possible future rollout issue into a material caveat.

A bounded, reversible pilot with a clear owner, scope, exposure, measurable success
criteria, confirmed core dependencies, and stop conditions can be READY even though
ordinary execution details and unproven hypotheses remain.

Return an empty caveat list when no issue meets the material, non-blocking threshold.

For UNKNOWN_MATERIAL_CLAIM and UNKNOWN_DEPENDENCY_REQUIREMENT, related_item_text
must exactly copy the corresponding claim_text or dependency_text.
"""


In [9]:
LEADERSHIP_QUESTIONS_PROMPT = """
You are the leadership-question generation component of DecisionReady.

Your task is to generate 3 to 5 high-value questions that leadership
should ask before making the specific decision requested.

A leadership question should address an issue that could materially:
- change the decision,
- change the conditions attached to the decision,
- affect whether the proposed initiative can generate useful evidence,
- or affect leadership's understanding of the expected value or risk.

QUESTION SELECTION RULES

1. PRIORITIZE MATERIALITY
Do not convert every assessment gap into a leadership question.

Distinguish:
A. decision-critical or decision-shaping issues, and
B. execution details that the accountable team can reasonably resolve
   after approval.

Prioritize category A.

2. MATCH THE DECISION STAGE
For a bounded pilot, leadership questions should focus on whether:
- the hypothesis is worth testing,
- exposure is appropriately bounded,
- success/failure can be measured,
- major assumptions could invalidate the learning,
- and the pilot can inform the next decision.

Do not demand the same operating detail required for full rollout.

3. BE SPECIFIC
Avoid generic questions such as:
- What are the risks?
- What is the ROI?
- Who is the owner?

Each question must be specific to this proposal.

4. DO NOT INVENT REQUIREMENTS
Do not introduce policies, mandatory approvals, IT requirements,
legal requirements, governance forums, integrations, or other
dependencies unless they are supported by the proposal.

You may ask:
"The proposal does not explain whether technical integration is needed.
Could this materially affect the 8-week pilot?"

Do not assert:
"What IT support is required for the integration?"
when integration has not been established as a requirement.

5. READY PROPOSALS
If the proposal is READY, questions should improve the quality of the
leadership decision or clarify important conditions.
Do not manufacture reasons to delay approval.

6. NOT-READY PROPOSALS
If the proposal is NOT_READY, prioritize questions whose answers would
resolve the hard blockers or materially change readiness.

7. Prefer 3 strong questions over 5 weaker questions.

8. Do not introduce new findings that are absent from the current case.

9. QUESTION COUNT
For a READY proposal, normally return exactly 3 questions.
For READY_WITH_CAVEATS, return 3 to 4.
For NOT_READY, return up to 5 only when needed to resolve blockers.
"""


In [10]:
RECOMMENDED_ACTIONS_PROMPT = """
You are the pre-review action component of DecisionReady.

Your task is to recommend what the proposal owner should do next
before leadership review.

STATUS-SENSITIVE BEHAVIOR

1. READY
Recommend only light-touch actions that improve decision quality.
Do not manufacture additional approval gates or delay the review.
Usually return 1 to 3 actions.
- Do not ask for new documents, packs, checklists, detailed plans, or analyses unless essential.
- Prefer concise clarifications, confirmations, or validations.
- Actions should normally be achievable without delaying the leadership review.
- Do not convert downstream execution planning into pre-review work.


2. READY_WITH_CAVEATS
Recommend 2 to 4 actions focused on the few material caveats that
should be clarified, validated, or explicitly acknowledged.

3. NOT_READY
Recommend 2 to 5 concrete corrective actions needed to resolve
hard blockers or major material gaps before leadership review.

ACTION RULES

4. Every action must be tied to evidence already present in the case.

5. Distinguish between:
   - actions needed before leadership can reasonably decide, and
   - execution details that can be handled after approval.

Do not turn ordinary project-management detail into a pre-review gate.

6. Do not introduce new:
   - policies,
   - mandatory approvals,
   - legal requirements,
   - technical requirements,
   - governance forums,
   - dependencies,
   - external facts.

7. Make actions specific and executable.

Bad:
"Improve the business case."

Better:
"Define how commercial outcomes will be measured alongside the
30% preparation-time target so leadership can interpret pilot success."

8. For READY cases, phrase actions as refinements or review preparation,
not as conditions that imply the proposal is actually not ready.

9. Do not reassess or change the readiness status.

10. Do not prescribe deliverable formats such as "1-page plan", "checklist",
"budget note", or "analysis pack" unless the case itself makes that necessary.
Recommend the action, not a new artifact.

11. CONCISION
Each recommended action should normally be one concise sentence.
For READY cases, return no more than 3 actions and avoid detailed
sub-steps, examples, implementation instructions, or new requirements.

12. FOR READY CASES
Do not ask the proposal owner to create or share a new note, statement,
checklist, plan, pack, model, or document.

Phrase actions as direct clarifications or confirmations only.

Example:
Instead of "Prepare a note explaining how the sample is representative,"
say "Clarify how the selected sample is representative enough to inform
the later rollout decision."
"""


In [11]:
FINAL_BRIEF_PROMPT = """
You are the final synthesis component of DecisionReady.

Your task is to produce a concise executive-ready Decision Readiness Brief.

You are NOT deciding whether leadership should approve the proposal.
You are summarizing whether the proposal is sufficiently prepared
for leadership to make the requested decision.

Use only information already present in the case.

Do not introduce new:
- findings,
- risks,
- policies,
- requirements,
- evidence,
- assumptions,
- dependencies,
- recommendations.

STRUCTURE

Decision Requested
Readiness
Why This Status
Evidence & Assumptions
Material Gaps / Blockers
Dependencies & Risks
Leadership Challenge Questions
Recommended Pre-review Actions
Provenance & Traceability

RULES

1. Keep the brief concise and decision-oriented.
2. Do not repeat every detail from upstream analysis.
3. Prioritize material findings.
4. Clearly distinguish:
   - stated evidence,
   - assumptions,
   - inferred concerns,
   - confirmed hard blockers.
5. If there are no hard blockers, state that clearly.
6. Do not call ordinary caveats "blockers."
7. For READY cases, avoid language that makes the proposal sound
   effectively not ready.
8. For NOT_READY cases, clearly explain which blockers must be resolved.
9. Preserve the readiness status exactly as provided.
10. Do not change or reassess the status.
11. Preserve provenance labels: proposal facts, inferred state, retrieved evidence, and
model-generated questions/actions must not be described as though they came from the same source.
12. Surface important secondary non-blocking concerns even when a hard blocker exists.
13. Do not reproduce raw retrieved chunks. Cite concise source IDs and grounded interpretations only.
"""


## 6. Graph nodes


In [12]:
def extract_proposal(state: DecisionReadyState) -> dict:
    result = extraction_model.invoke(
        [
            SystemMessage(content=EXTRACTION_SYSTEM_PROMPT),
            HumanMessage(content=state["proposal_text"])
        ]
    )

    return result.model_dump()


In [13]:
def assess_readiness(state: DecisionReadyState) -> dict:

    assessment_context = {
        "proposal_type": state.get("proposal_type"),
        "decision_stage": state.get("decision_stage"),
        "decision_ask": state.get("decision_ask"),
        "accountable_owner": state.get("accountable_owner"),
        "problem_summary": state.get("problem_summary"),
        "proposed_intervention": state.get("proposed_intervention"),
        "evidence_items": state.get("evidence_items", []),
        "material_claims": state.get("material_claims", []),
        "dependency_reviews": state.get("dependency_reviews", []),
        "stated_assumptions": state.get("stated_assumptions", []),
        "inferred_assumptions": state.get("inferred_assumptions", []),
        "success_metrics": state.get("success_metrics", []),
        "dependencies": state.get("dependencies", []),
        "stated_risks": state.get("stated_risks", []),
        "inferred_risks": state.get("inferred_risks", [])
    }

    result = assessment_model.invoke(
        [
            SystemMessage(content=ASSESSMENT_SYSTEM_PROMPT),
            HumanMessage(
                content=(
                    "Assess the following proposal.\n\n"
                    "ORIGINAL PROPOSAL:\n"
                    f"{state['proposal_text']}\n\n"
                    "STRUCTURED EXTRACTION:\n"
                    f"{json.dumps(assessment_context, indent=2)}"
                )
            )
        ]
    )

    return {
        "dimension_assessments": result.model_dump()
    }


In [14]:
def check_hard_blockers(state: DecisionReadyState) -> dict:
    blockers = []

    # --------------------------------------------------------
    # A. DETERMINISTIC CHECKS
    # --------------------------------------------------------

    decision_ask = (state.get("decision_ask") or "").strip()
    accountable_owner = (state.get("accountable_owner") or "").strip()
    decision_stage = (state.get("decision_stage") or "").lower()
    success_metrics = state.get("success_metrics", [])

    if not decision_ask:
        blockers.append({
            "blocker_code": "NO_DECISION_ASK",
            "source": "deterministic",
            "rationale": "No decision ask was identified in the proposal.",
            "evidence": []
        })

    if not accountable_owner:
        blockers.append({
            "blocker_code": "NO_ACCOUNTABLE_OWNER",
            "source": "deterministic",
            "rationale": "No accountable owner was identified.",
            "evidence": []
        })

    if "pilot" in decision_stage and not success_metrics:
        blockers.append({
            "blocker_code": "NO_SUCCESS_METRIC",
            "source": "deterministic",
            "rationale": (
                "The requested decision is a pilot, but no measurable "
                "success metric was identified."
            ),
            "evidence": []
        })


    # Preserve the existing readiness score thresholds as an explicit,
    # deterministic hard blocker in the new blocker/caveat architecture.
    dimension_assessments = state.get("dimension_assessments", {})
    expected_dimensions = set(EXPECTED_ASSESSMENT_DIMENSIONS)
    valid_dimension_scores = (
        set(dimension_assessments) == expected_dimensions
        and all(
            isinstance(dimension_assessments.get(dimension), dict)
            and isinstance(dimension_assessments[dimension].get("score"), int)
            and not isinstance(dimension_assessments[dimension].get("score"), bool)
            and 0 <= dimension_assessments[dimension]["score"] <= 3
            for dimension in EXPECTED_ASSESSMENT_DIMENSIONS
        )
    )

    if valid_dimension_scores and "pilot" not in decision_stage:
        low_dimensions = [
            (dimension, dimension_assessments[dimension])
            for dimension in EXPECTED_ASSESSMENT_DIMENSIONS
            if dimension_assessments[dimension]["score"] <= 1
        ]
        count_0 = sum(assessment["score"] == 0 for _, assessment in low_dimensions)
        count_1 = sum(assessment["score"] == 1 for _, assessment in low_dimensions)

        if count_0 >= 1 or count_1 >= 2:
            blockers.append({
                "blocker_code": "MATERIAL_DIMENSION_THRESHOLD",
                "source": "deterministic",
                "rationale": (
                    "The existing readiness score threshold is met: at least one "
                    "dimension scored 0 or at least two dimensions scored 1."
                ),
                "evidence": [
                    f"{dimension}: score {assessment['score']} — {assessment['reason']}"
                    for dimension, assessment in low_dimensions
                ],
                "related_item_text": None,
            })

    # --------------------------------------------------------
    # B. SEMANTIC CHECKS
    # --------------------------------------------------------

    blocker_context = {
    "decision_stage": state.get("decision_stage"),
    "decision_ask": state.get("decision_ask"),
    "accountable_owner": state.get("accountable_owner"),

    "evidence_items": state.get(
        "evidence_items", []
    ),

    "material_claims": state.get(
        "material_claims", []
    ),

    "dependency_reviews": state.get(
        "dependency_reviews", []
    ),

    "stated_assumptions": state.get(
        "stated_assumptions", []
    ),

    "inferred_assumptions": state.get(
        "inferred_assumptions", []
    ),

    "success_metrics": state.get(
        "success_metrics", []
    ),

    "dependencies": state.get(
        "dependencies", []
    )
}

    semantic_result = semantic_blocker_model.invoke(
        [
            SystemMessage(content=SEMANTIC_BLOCKER_PROMPT),
            HumanMessage(
                content=(
                    "Assess hard blockers for this proposal.\n\n"
                    "ORIGINAL PROPOSAL:\n"
                    f"{state['proposal_text']}\n\n"
                    "CURRENT STRUCTURED CASE:\n"
                    f"{json.dumps(blocker_context, indent=2)}"
                )
            )
        ]
    )

    prerequisite_claims = {
        item["claim_text"].strip().casefold()
        for item in state.get("material_claims", [])
        if item.get("claim_role") == "prerequisite"
        and item.get("claim_text")
    }
    required_dependencies = {
        item["dependency_text"].strip().casefold()
        for item in state.get("dependency_reviews", [])
        if item.get("requirement_status") == "required"
        and item.get("dependency_text")
    }

    for blocker in semantic_result.blockers:
        blocker_dict = blocker.model_dump()
        related_item = (blocker_dict.get("related_item_text") or "").strip().casefold()

        if blocker.blocker_code == "AMBIGUOUS_DECISION_ASK":
            decision_clarity = dimension_assessments.get("decision_clarity", {})
            decision_clarity_score = decision_clarity.get("score")
            if (
                isinstance(decision_clarity_score, bool)
                or not isinstance(decision_clarity_score, int)
                or decision_clarity_score > 1
            ):
                continue

        if blocker.blocker_code == "UNSUPPORTED_CENTRAL_CLAIM":
            if related_item not in prerequisite_claims:
                continue

        if blocker.blocker_code == "UNRESOLVED_CRITICAL_DEPENDENCY":
            if related_item not in required_dependencies:
                continue

        blocker_dict["source"] = "semantic"
        blockers.append(blocker_dict)

    return {
        "hard_blockers": blockers
    }


In [15]:
def _deterministic_decision_horizon(
    state: DecisionReadyState,
    caveat: dict,
) -> DecisionHorizonRelevance:
    """Apply only clear timing overrides; otherwise retain semantic classification."""
    related_item = (caveat.get("related_item_text") or "").strip().casefold()
    decision_stage = state.get("decision_stage")

    current_pilot_markers = (
        "during the pilot",
        "before the pilot",
        "before pilot",
        "pilot start",
        "pilot launch",
        "pilot design",
        "pilot measurement",
        "pilot evidence",
        "pilot results",
    )
    future_decision_markers = (
        "broader rollout",
        "future rollout",
        "later rollout",
        "subsequent rollout",
        "rollout decision",
        "post-pilot decision",
        "scale decision",
        "scaling decision",
        "decision to scale",
        "decision whether to scale",
        "scale-up decision",
        "follow-on investment",
        "future investment",
        "later investment",
    )

    if decision_stage == "pilot":
        if any(marker in related_item for marker in current_pilot_markers):
            return "current_decision"
        if any(marker in related_item for marker in future_decision_markers):
            return "future_decision"

    return caveat["decision_horizon_relevance"]


def _direct_current_materiality_evidence(caveat: dict) -> list[str]:
    """Exclude evidence that only repeats model-produced labels or scores."""
    structured_only_markers = (
        "structured case",
        "dimension assessment",
        "requirement_status",
        "claim_role",
        "score of",
        "score =",
    )
    return [
        item
        for item in caveat.get("current_decision_materiality_evidence", [])
        if item.strip()
        and not any(marker in item.casefold() for marker in structured_only_markers)
    ]


def _explicit_future_decision_considerations(
    state: DecisionReadyState,
) -> list[dict]:
    """Capture explicit later-stage decisions independently of caveat-model recall."""
    proposal_text = state.get("proposal_text", "")
    sentences = [
        sentence.strip()
        for sentence in re.split(r"(?<=[.!?])\s+", proposal_text.replace("\n", " "))
        if sentence.strip()
    ]
    future_stage_terms = ("rollout", "scale", "scaling", "investment")
    decision_terms = ("leadership", "decision", "consider", "approve")
    future_timing_terms = (
        "if successful",
        "if the pilot",
        "after the pilot",
        "post-pilot",
        "later",
        "subsequent",
        "future",
    )
    decision_stage = (state.get("decision_stage") or "").casefold()
    results = []
    for sentence in sentences:
        normalized = sentence.casefold()
        if not any(term in normalized for term in future_stage_terms):
            continue
        if not any(term in normalized for term in decision_terms):
            continue
        if "request" in normalized:
            continue
        if decision_stage != "pilot" and not any(
            term in normalized for term in future_timing_terms
        ):
            continue
        results.append({
            "caveat_code": "ACKNOWLEDGED_NON_BLOCKING_UNCERTAINTY",
            "rationale": (
                "The proposal explicitly identifies a later-stage leadership decision. "
                "It remains visible for continuity but does not affect readiness for "
                "the current requested decision."
            ),
            "evidence": [sentence],
            "related_item_text": sentence,
            "decision_horizon_relevance": "future_decision",
            "current_decision_materiality_evidence": [],
            "source": "deterministic",
        })
    return results


def _materially_same_concern(left: str, right: str) -> bool:
    stopwords = {
        "that", "this", "with", "from", "into", "before", "after",
        "confirmation", "requirement", "status", "review", "current",
        "existing", "proposed", "use", "uses",
    }
    left_tokens = {
        token for token in re.findall(r"[a-z0-9]+", left.casefold())
        if len(token) >= 4 and token not in stopwords
    }
    right_tokens = {
        token for token in re.findall(r"[a-z0-9]+", right.casefold())
        if len(token) >= 4 and token not in stopwords
    }
    if not left_tokens or not right_tokens:
        return False
    overlap = len(left_tokens & right_tokens)
    return overlap >= max(2, min(len(left_tokens), len(right_tokens)) // 2)


def identify_material_caveats(state: DecisionReadyState) -> dict:
    simplified_assessments = {
        dimension: {
            "score": assessment["score"],
            "reason": assessment["reason"],
            "gaps": assessment.get("gaps", []),
        }
        for dimension, assessment in state.get("dimension_assessments", {}).items()
    }

    caveat_context = {
        "decision_stage": state.get("decision_stage"),
        "decision_ask": state.get("decision_ask"),
        "problem_summary": state.get("problem_summary"),
        "proposed_intervention": state.get("proposed_intervention"),
        "evidence_items": state.get("evidence_items", []),
        "material_claims": state.get("material_claims", []),
        "dependency_reviews": state.get("dependency_reviews", []),
        "dimension_assessments": simplified_assessments,
        "stated_assumptions": state.get("stated_assumptions", []),
        "inferred_assumptions": state.get("inferred_assumptions", []),
        "stated_risks": state.get("stated_risks", []),
        "inferred_risks": state.get("inferred_risks", []),
        "hard_blockers": state.get("hard_blockers", []),
        "grounded_requirements": _compact_grounded_for_synthesis(state),
    }

    result = material_caveat_model.invoke(
        [
            SystemMessage(content=MATERIAL_CAVEAT_PROMPT),
            HumanMessage(
                content=(
                    "Identify material, non-blocking caveats for this proposal.\n\n"
                    "ORIGINAL PROPOSAL:\n"
                    f"{state['proposal_text']}\n\n"
                    "CURRENT STRUCTURED CASE:\n"
                    f"{json.dumps(caveat_context, indent=2)}"
                )
            ),
        ]
    )

    unknown_claims = {
        item["claim_text"].strip().casefold()
        for item in state.get("material_claims", [])
        if item.get("claim_role") == "unknown" and item.get("claim_text")
    }
    unknown_dependencies = {
        item["dependency_text"].strip().casefold()
        for item in state.get("dependency_reviews", [])
        if item.get("requirement_status") == "unknown" and item.get("dependency_text")
    }

    caveats = []
    future_considerations = []
    seen = set()
    seen_future = set()
    decision_ask_text = (state.get("decision_ask") or "").strip().casefold()

    for caveat in result.caveats:
        caveat_dict = caveat.model_dump()
        related_item = (caveat_dict.get("related_item_text") or "").strip().casefold()

        requested_approval_item = (
            any(term in decision_ask_text for term in ("approv", "authoriz"))
            and any(term in related_item for term in ("approv", "authoriz"))
            and any(
                term in related_item
                for term in ("budget", "funding", "pilot", "rollout", "investment")
            )
        )
        if requested_approval_item:
            continue

        if caveat.caveat_code == "UNKNOWN_MATERIAL_CLAIM":
            if related_item not in unknown_claims:
                continue

        if caveat.caveat_code == "UNKNOWN_DEPENDENCY_REQUIREMENT":
            if related_item not in unknown_dependencies:
                continue

        if caveat.caveat_code in {
            "WEAK_OR_INCOMPLETE_EVIDENCE",
            "ACKNOWLEDGED_NON_BLOCKING_UNCERTAINTY",
        }:
            if related_item not in unknown_claims | unknown_dependencies:
                continue

        if not caveat_dict.get("evidence"):
            continue

        confirmed_concerns = [
            blocker.get("related_item_text", "")
            for blocker in state.get("hard_blockers", [])
            if blocker.get("related_item_text")
        ] + [
            grounded["requirement_text"]
            for grounded in state.get("grounded_requirements", [])
            if grounded.get("requirement_status") == "required"
        ]
        if any(
            _materially_same_concern(
                caveat_dict.get("related_item_text") or "",
                confirmed,
            )
            for confirmed in confirmed_concerns
        ):
            continue

        relevance = _deterministic_decision_horizon(state, caveat_dict)
        caveat_dict["decision_horizon_relevance"] = relevance
        caveat_dict["source"] = "semantic"
        identity = (caveat_dict["caveat_code"], related_item)

        if relevance == "future_decision":
            if identity not in seen_future:
                seen_future.add(identity)
                future_considerations.append(caveat_dict)
            continue

        direct_materiality_evidence = _direct_current_materiality_evidence(
            caveat_dict
        )
        if relevance in {"current_decision", "unclear"}:
            if not direct_materiality_evidence:
                continue
            caveat_dict[
                "current_decision_materiality_evidence"
            ] = direct_materiality_evidence

        if identity in seen:
            continue
        seen.add(identity)
        caveats.append(caveat_dict)

    for future_item in _explicit_future_decision_considerations(state):
        future_text = (future_item.get("related_item_text") or "").casefold()
        duplicate_future = any(
            future_text in (item.get("related_item_text") or "").casefold()
            or (item.get("related_item_text") or "").casefold() in future_text
            for item in future_considerations
            if item.get("related_item_text")
        )
        if not duplicate_future:
            future_considerations.append(future_item)

    return {
        "material_caveats": caveats,
        "future_decision_considerations": future_considerations,
    }


In [16]:
EXPECTED_ASSESSMENT_DIMENSIONS = tuple(
    ProposalReadinessAssessment.model_fields
)


def determine_readiness(state: DecisionReadyState) -> dict:
    hard_blockers = state.get("hard_blockers", [])
    material_caveats = state.get("material_caveats", [])
    dimension_assessments = state.get("dimension_assessments", {})

    if hard_blockers:
        return {
            "readiness_status": "NOT_READY"
        }

    # Preserve fail-closed behavior when upstream assessment state is absent
    # or malformed, independently of blocker and caveat classification.
    if set(dimension_assessments) != set(EXPECTED_ASSESSMENT_DIMENSIONS):
        return {
            "readiness_status": "NOT_READY"
        }

    for dimension in EXPECTED_ASSESSMENT_DIMENSIONS:
        assessment = dimension_assessments.get(dimension)
        if not isinstance(assessment, dict):
            return {
                "readiness_status": "NOT_READY"
            }

        score = assessment.get("score")
        if isinstance(score, bool) or not isinstance(score, int) or not 0 <= score <= 3:
            return {
                "readiness_status": "NOT_READY"
            }

    if material_caveats:
        return {
            "readiness_status": "READY_WITH_CAVEATS"
        }

    return {
        "readiness_status": "READY"
    }


In [17]:
def generate_leadership_questions(state: DecisionReadyState) -> dict:

    simplified_assessments = {}

    for dimension, assessment in state.get(
        "dimension_assessments", {}
    ).items():
        simplified_assessments[dimension] = {
            "score": assessment["score"],
            "reason": assessment["reason"]
        }

    question_context = {
        "decision_ask": state.get("decision_ask"),
        "decision_stage": state.get("decision_stage"),
        "readiness_status": state.get("readiness_status"),

        "dimension_assessments": simplified_assessments,

        "hard_blockers": state.get("hard_blockers", []),
        "material_caveats": state.get("material_caveats", []),

        "material_claims": state.get("material_claims", []),
        "dependency_reviews": state.get("dependency_reviews", []),

        "stated_assumptions": state.get(
            "stated_assumptions", []
        ),
        "inferred_assumptions": state.get(
            "inferred_assumptions", []
        ),

        "dependencies": state.get("dependencies", []),

        "stated_risks": state.get(
            "stated_risks", []
        ),
        "inferred_risks": state.get(
            "inferred_risks", []
        )
    }

    result = leadership_questions_model.invoke(
        [
            SystemMessage(
                content=LEADERSHIP_QUESTIONS_PROMPT
            ),
            HumanMessage(
                content=(
                    "Generate leadership questions for this case.\n\n"
                    "ORIGINAL PROPOSAL:\n"
                    f"{state['proposal_text']}\n\n"
                    "CURRENT CASE:\n"
                    f"{json.dumps(question_context, indent=2)}"
                )
            )
        ]
    )

    return {
        "leadership_questions": result.questions
    }


In [18]:
def generate_recommended_actions(state: DecisionReadyState) -> dict:

    simplified_assessments = {}

    for dimension, assessment in state.get(
        "dimension_assessments", {}
    ).items():
        simplified_assessments[dimension] = {
            "score": assessment["score"],
            "reason": assessment["reason"]
        }

    readiness_status = state.get("readiness_status")

    action_context = {
        "decision_ask": state.get("decision_ask"),
        "decision_stage": state.get("decision_stage"),
        "readiness_status": readiness_status,
        "dimension_assessments": simplified_assessments,
        "hard_blockers": state.get("hard_blockers", []),
        "material_caveats": state.get("material_caveats", []),
        "leadership_questions": state.get(
            "leadership_questions", []
        ),
        "material_claims": state.get("material_claims", []),
        "dependency_reviews": state.get("dependency_reviews", [])
    }

    # For cases that are not READY, provide additional diagnostic context.
    if readiness_status != "READY":
        action_context.update({
            "stated_assumptions": state.get(
                "stated_assumptions", []
            ),
            "inferred_assumptions": state.get(
                "inferred_assumptions", []
            ),
            "dependencies": state.get(
                "dependencies", []
            ),
            "stated_risks": state.get(
                "stated_risks", []
            ),
            "inferred_risks": state.get(
                "inferred_risks", []
            )
        })

    result = recommended_actions_model.invoke(
        [
            SystemMessage(
                content=RECOMMENDED_ACTIONS_PROMPT
            ),
            HumanMessage(
                content=(
                    "Recommend concise pre-review actions for this case.\n\n"
                    "ORIGINAL PROPOSAL:\n"
                    f"{state['proposal_text']}\n\n"
                    "CURRENT CASE:\n"
                    f"{json.dumps(action_context, indent=2)}"
                )
            )
        ]
    )

    return {
        "recommended_actions": result.actions
    }


In [19]:
def _sanitize_blockers_for_synthesis(blockers: list[dict]) -> list[dict]:
    """Keep retrieved Northstar text local while preserving source traceability."""
    sanitized = []
    for blocker in blockers:
        item = dict(blocker)
        evidence = []
        for value in blocker.get("evidence", []):
            if "::chunk-" in value and ":" in value:
                evidence.append(value.split(":", 1)[0])
            else:
                evidence.append(value)
        item["evidence"] = evidence
        sanitized.append(item)
    return sanitized


def _compact_grounded_for_synthesis(state: DecisionReadyState) -> list[dict]:
    return [
        {
            "requirement_text": item["requirement_text"],
            "requirement_status": item["requirement_status"],
            "supporting_source_ids": item.get("supporting_source_ids", []),
            "rationale": item["rationale"],
        }
        for item in state.get("grounded_requirements", [])
    ]


def build_provenance_context(state: DecisionReadyState) -> dict:
    """Label source categories so generated advice is not mistaken for evidence."""
    return {
        "proposal_fact": {
            "evidence_items": state.get("evidence_items", []),
            "stated_assumptions": state.get("stated_assumptions", []),
            "success_metrics": state.get("success_metrics", []),
        },
        "inferred_state": {
            "inferred_assumptions": state.get("inferred_assumptions", []),
            "inferred_risks": state.get("inferred_risks", []),
        },
        "retrieved_evidence": state.get("retrieval_provenance", []),
        "model_generated_question_action": {
            "leadership_questions": state.get("leadership_questions", []),
            "recommended_actions": state.get("recommended_actions", []),
        },
    }


def build_final_brief(state: DecisionReadyState) -> dict:
    simplified_assessments = {
        dimension: {
            "score": assessment["score"],
            "reason": assessment["reason"],
            "gaps": assessment.get("gaps", []),
        }
        for dimension, assessment in state.get("dimension_assessments", {}).items()
    }

    brief_context = {
        "decision_ask": state.get("decision_ask"),
        "decision_stage": state.get("decision_stage"),
        "accountable_owner": state.get("accountable_owner"),
        "readiness_status": state.get("readiness_status"),
        "problem_summary": state.get("problem_summary"),
        "proposed_intervention": state.get("proposed_intervention"),
        "evidence_items": state.get("evidence_items", []),
        "success_metrics": state.get("success_metrics", []),
        "material_claims": state.get("material_claims", []),
        "dependency_reviews": state.get("dependency_reviews", []),
        "dependencies": state.get("dependencies", []),
        "stated_assumptions": state.get("stated_assumptions", []),
        "inferred_assumptions": state.get("inferred_assumptions", []),
        "dimension_assessments": simplified_assessments,
        "hard_blockers": _sanitize_blockers_for_synthesis(
            state.get("hard_blockers", [])
        ),
        "material_caveats": state.get("material_caveats", []),
        "future_decision_considerations": state.get(
            "future_decision_considerations", []
        ),
        "grounded_requirements": _compact_grounded_for_synthesis(state),
        "stated_risks": state.get("stated_risks", []),
        "inferred_risks": state.get("inferred_risks", []),
        "leadership_questions": state.get("leadership_questions", []),
        "recommended_actions": state.get("recommended_actions", []),
        "provenance": build_provenance_context(state),
    }

    result = final_brief_model.invoke(
        [
            SystemMessage(content=FINAL_BRIEF_PROMPT),
            HumanMessage(
                content=(
                    "Build the final Decision Readiness Brief.\n\n"
                    "CURRENT CASE:\n"
                    f"{json.dumps(brief_context, indent=2)}"
                )
            ),
        ]
    )
    return {"final_brief": result.final_brief}


## 7. Final LangGraph


In [20]:
graph_builder = StateGraph(DecisionReadyState)

graph_builder.add_node("extract_proposal", extract_proposal)
graph_builder.add_node("assess_readiness", assess_readiness)
graph_builder.add_node("check_hard_blockers", check_hard_blockers)
graph_builder.add_node("identify_material_caveats", identify_material_caveats)
graph_builder.add_node("determine_readiness", determine_readiness)
graph_builder.add_node("generate_leadership_questions", generate_leadership_questions)
graph_builder.add_node("generate_recommended_actions", generate_recommended_actions)
graph_builder.add_node("build_final_brief", build_final_brief)

graph_builder.add_edge(START, "extract_proposal")
graph_builder.add_edge("extract_proposal", "assess_readiness")
graph_builder.add_edge("assess_readiness", "check_hard_blockers")
graph_builder.add_edge("check_hard_blockers", "identify_material_caveats")
graph_builder.add_edge("identify_material_caveats", "determine_readiness")
graph_builder.add_edge("determine_readiness", "generate_leadership_questions")
graph_builder.add_edge("generate_leadership_questions", "generate_recommended_actions")
graph_builder.add_edge("generate_recommended_actions", "build_final_brief")
graph_builder.add_edge("build_final_brief", END)

decisionready_iteration1 = graph_builder.compile()


## 8. Proposal fixtures


In [21]:
test_proposal = """
Leadership approval is requested for an 8-week pilot to test an AI-assisted
supplier negotiation preparation tool across two procurement categories.

The pilot will be led by the VP Procurement Transformation and will include
12 procurement managers across approximately 30 negotiation events.

Current preparation time averages 4.1 hours per negotiation. The pilot aims
to reduce preparation time by at least 30% while maintaining or improving
commercial outcomes.

The proposed budget is $22,000.

Key dependencies include access to historical negotiation notes and supplier
data. These data sources have been confirmed as available for the pilot.

The pilot will be considered successful if average preparation time is reduced
by at least 30%, user adoption exceeds 70%, and no material data-control issues
are identified.

If successful, leadership will consider a broader rollout. If the targets are
not achieved, the pilot will stop and the approach will be reassessed.
"""

proposal_b = """
Leadership support is requested for rollout of an AI-powered Sales Productivity
Copilot across multiple regions.

The solution is expected to improve seller productivity by approximately 20%
through faster account research, meeting preparation, and follow-up support.

The initiative will be owned by the Commercial Transformation team.

A formal current-state productivity baseline has not yet been established.
The exact rollout population and regional sequencing are still being finalized.

Estimated implementation cost is between $60,000 and $90,000.

Key dependencies include CRM integration, product-content connectivity,
regional access approvals, and seller adoption. Some of these dependencies
are still being worked through and do not yet have confirmed owners or dates.

The intention is to begin rollout during the coming quarter.

Success measures are expected to include productivity improvement and seller
adoption, although the measurement approach and baseline are still to be finalized.

Exit criteria or conditions for stopping, redesigning, or narrowing the rollout
have not yet been defined.
"""

proposal_c = """
Leadership approval is requested for a 6-week pilot to enrich identifiable
customer profiles using third-party data in order to improve targeting and
response rates for commercial campaigns.

The pilot will cover approximately 25,000 customer records and will be led by
the Director, Growth Marketing.

The current campaign response rate is 3.4%. The pilot aims to improve this to
at least 4.2%.

The estimated pilot cost is $38,000.

The proposal assumes that existing customer terms and current data-use practices
permit the use of third-party enrichment data for targeted offers.

The proposed data provider can supply demographic, firmographic, and behavioral
attributes that would be matched to existing customer records.

Data Governance review has not yet been initiated.

Success criteria include:
- campaign response rate of at least 4.2%
- acceptable match accuracy
- no material increase in customer opt-outs or complaints

The pilot will run for 6 weeks. If the expected improvement is not observed,
or if data-quality or customer-response issues are material, the pilot will stop
and the approach will be reassessed.
"""


## 9. Archived Iteration-1 execution examples

The code cells below are retained only as historical examples. Their saved outputs have been cleared and they are not part of the canonical Iteration-2 v0.5 execution path. Canonical current outputs appear under the explicitly labelled v0.5 benchmark section.


In [22]:
def print_result(result: DecisionReadyState) -> None:
    print("READINESS:", result["readiness_status"])
    print()
    print("HARD BLOCKERS:", result["hard_blockers"])
    print()
    print("MATERIAL CAVEATS:", result["material_caveats"])
    print()
    print(result["final_brief"])


### Proposal A — AI-assisted supplier negotiation pilot


In [ ]:
iteration1_result_A = decisionready_iteration1.invoke(
    {
        "proposal_text": test_proposal
    }
)

print_result(iteration1_result_A)


### Proposal B — Sales productivity copilot rollout


In [ ]:
iteration1_result_B = decisionready_iteration1.invoke(
    {
        "proposal_text": proposal_b
    }
)

print_result(iteration1_result_B)


### Proposal C — Customer data enrichment pilot


In [ ]:
iteration1_result_C = decisionready_iteration1.invoke(
    {
        "proposal_text": proposal_c
    }
)

print_result(iteration1_result_C)


### Proposal D — New supplier category expansion


In [26]:
proposal_d = """
Leadership approval is requested for a 10-week pilot to enter a new industrial
fasteners sourcing category and test customer demand, supplier economics, and
execution feasibility.

The pilot will be led by the Head of Category Expansion.

The proposed budget is $70,000.

The business case references a 2023 industry report that estimated annual
market growth of approximately 8-10% and noted fragmented pricing across the
category.

Current internal data does not provide a recent market-size or pricing benchmark
for this category. Customer demand has been discussed informally with several
account managers, but no structured customer validation has yet been completed.

The pilot aims to onboard at least 8 qualified suppliers within 4 weeks and
generate at least:
- 5 active buying customers
- $250,000 GMV
- gross margin of at least 6%
- OTIF of at least 90%

The pilot will run for 10 weeks.

If the pilot achieves the commercial and execution thresholds, leadership will
consider scaling the category. If demand, margins, supplier availability, or
service levels are materially below target, the initiative will stop or be
redesigned.
"""

### Proposal D — New supplier category expansion


In [ ]:
iteration1_result_D = decisionready_iteration1.invoke(
    {
        "proposal_text": proposal_d
    }
)

print_result(iteration1_result_D)

## A–D Benchmark / Eval Summary

The actual results below come from the definitive clean-kernel v0.3 execution.

| Proposal | Expected Iteration-1 Status | Actual Iteration-1 Status | Expected Hard Blockers | Actual Hard Blockers | Material Caveats | Main Failure / Learning |
|---|---|---|---|---|---|---|
| A | `READY` | `READY` | None | None | None | Strong bounded pilot remains ready; ordinary hypothesis-testing and execution refinements are not promoted into caveats. |
| B | `NOT_READY` | `NOT_READY` | Material readiness gaps and/or confirmed semantic blockers | `MATERIAL_DIMENSION_THRESHOLD` | None | Overall status is correct, but semantic blocker recall remains degraded versus v0.1; the aggregate non-pilot dimension threshold carries the result. |
| C | `READY_WITH_CAVEATS` | `READY_WITH_CAVEATS` | No confirmed hard blocker | None | `UNKNOWN_MATERIAL_CLAIM` (2); `UNKNOWN_DEPENDENCY_REQUIREMENT`; `WEAK_OR_INCOMPLETE_EVIDENCE` | Governance, permissibility, provider capability, and evidence uncertainty remain visible without being promoted into mandatory blockers. |
| D | `READY_WITH_CAVEATS` | `READY_WITH_CAVEATS` | No confirmed hard blocker | None | `UNKNOWN_MATERIAL_CLAIM` (2); `UNKNOWN_DEPENDENCY_REQUIREMENT` | Weak/stale market evidence and unresolved customer-validation uncertainty are surfaced as caveats rather than prerequisite blockers. |

Iteration-1 v0.3 now separates confirmed blockers, material non-blocking caveats, and ordinary pilot uncertainty while preserving the v0.2 prerequisite/hypothesis and required/unknown gates.


## Iteration 2 — Retrieval-Only Experiment

**Experiment question:** Can Proposal C reliably retrieve the authoritative Northstar governance evidence needed to resolve its uncertainty?

This section is intentionally isolated from the Iteration-1 graph. It does not update `requirement_status`, create blockers, change readiness, or feed retrieved evidence into any downstream node.


### 1. Load the controlled Northstar corpus


In [28]:
from collections import Counter
import math
from pathlib import Path
import re
from zipfile import ZipFile
import xml.etree.ElementTree as ET

import numpy as np
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import BaseModel, Field


NORTHSTAR_FILES = {
    "Approval Policy": Path("../northstar/02_Strategic_Initiative_Approval_Policy.docx"),
    "Pilot Governance Guidelines": Path("../northstar/03_Pilot_Governance_Guidelines.docx"),
    "Risk / Data Governance Guidelines": Path("../northstar/04_Risk_and_Data_Governance_Guidelines.docx"),
    "Prior Decision Precedents": Path("../northstar/05_Prior_Decision_Precedents.docx"),
}


def load_docx_paragraphs(path: Path) -> list[str]:
    """Read visible Word paragraphs using only Python's standard library."""
    word_namespace = "{http://schemas.openxmlformats.org/wordprocessingml/2006/main}"

    with ZipFile(path) as archive:
        document_xml = archive.read("word/document.xml")

    root = ET.fromstring(document_xml)
    paragraphs = []

    for paragraph in root.iter(f"{word_namespace}p"):
        text = "".join(
            node.text or ""
            for node in paragraph.iter(f"{word_namespace}t")
        ).strip()
        if text:
            paragraphs.append(text)

    return paragraphs


missing_files = [str(path) for path in NORTHSTAR_FILES.values() if not path.exists()]
if missing_files:
    raise FileNotFoundError(f"Missing controlled Northstar files: {missing_files}")

northstar_documents = []
for source_name, path in NORTHSTAR_FILES.items():
    paragraphs = load_docx_paragraphs(path)
    northstar_documents.append(
        Document(
            page_content="\n\n".join(paragraphs),
            metadata={
                "source": source_name,
                "path": str(path),
                "paragraph_count": len(paragraphs),
            },
        )
    )

for document in northstar_documents:
    print(
        f"{document.metadata['source']}: "
        f"{document.metadata['paragraph_count']} paragraphs, "
        f"{len(document.page_content):,} characters"
    )


Approval Policy: 47 paragraphs, 2,856 characters
Pilot Governance Guidelines: 44 paragraphs, 1,909 characters
Risk / Data Governance Guidelines: 28 paragraphs, 2,047 characters
Prior Decision Precedents: 41 paragraphs, 1,880 characters


### 2. Chunk the documents


In [29]:
# Transparent baseline: recursive character chunks, 900 characters each,
# with 150 characters of overlap to preserve context across boundaries.
CHUNK_SIZE = 900
CHUNK_OVERLAP = 150

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " "],
    length_function=len,
)

northstar_chunks = []
for document in northstar_documents:
    document_chunks = text_splitter.split_documents([document])
    for chunk_index, chunk in enumerate(document_chunks):
        chunk.metadata["chunk_index"] = chunk_index
        chunk.metadata["chunk_id"] = (
            f"{chunk.metadata['source']}::chunk-{chunk_index:02d}"
        )
    northstar_chunks.extend(document_chunks)

print(f"Created {len(northstar_chunks)} chunks")
for source_name, count in Counter(
    chunk.metadata["source"] for chunk in northstar_chunks
).items():
    print(f"- {source_name}: {count} chunks")


Created 13 chunks
- Approval Policy: 4 chunks
- Pilot Governance Guidelines: 3 chunks
- Risk / Data Governance Guidelines: 3 chunks
- Prior Decision Precedents: 3 chunks


### 3. Embed chunks and build an in-memory vector store


In [30]:
class LocalTfidfVectorStore:
    """Small, transparent, local-only TF-IDF cosine vector store."""

    def __init__(self):
        self.documents = []
        self.vocabulary = {}
        self.idf = None
        self.document_vectors = None

    @staticmethod
    def _tokenize(text: str) -> list[str]:
        return re.findall(r"[a-z0-9]+", text.casefold())

    def _embed_tokens(self, tokens: list[str]) -> np.ndarray:
        vector = np.zeros(len(self.vocabulary), dtype=float)
        token_counts = Counter(tokens)

        for token, count in token_counts.items():
            index = self.vocabulary.get(token)
            if index is not None:
                vector[index] = (1.0 + math.log(count)) * self.idf[index]

        norm = np.linalg.norm(vector)
        return vector / norm if norm else vector

    def add_documents(self, documents: list[Document]) -> None:
        self.documents = list(documents)
        tokenized_documents = [
            self._tokenize(document.page_content)
            for document in self.documents
        ]

        vocabulary_terms = sorted({
            token
            for tokens in tokenized_documents
            for token in tokens
        })
        self.vocabulary = {
            token: index
            for index, token in enumerate(vocabulary_terms)
        }

        document_frequency = Counter()
        for tokens in tokenized_documents:
            document_frequency.update(set(tokens))

        document_count = len(tokenized_documents)
        self.idf = np.array([
            math.log(
                (1 + document_count)
                / (1 + document_frequency[token])
            ) + 1
            for token in vocabulary_terms
        ])
        self.document_vectors = np.vstack([
            self._embed_tokens(tokens)
            for tokens in tokenized_documents
        ])

    def similarity_search_with_score(
        self,
        query: str,
        k: int = 3,
    ) -> list[tuple[Document, float]]:
        query_vector = self._embed_tokens(self._tokenize(query))
        scores = self.document_vectors @ query_vector
        top_indices = np.argsort(scores)[::-1][:k]
        return [
            (self.documents[index], float(scores[index]))
            for index in top_indices
        ]


vector_store = LocalTfidfVectorStore()
vector_store.add_documents(northstar_chunks)

print(f"Indexed {len(northstar_chunks)} chunks using local TF-IDF embeddings.")
print(f"Vocabulary size: {len(vector_store.vocabulary):,} terms")
print("Corpus text remains local; no policy text is sent to an embedding API.")


Indexed 13 chunks using local TF-IDF embeddings.
Vocabulary size: 490 terms
Corpus text remains local; no policy text is sent to an embedding API.


### 4. Define structured retrieval needs


In [31]:
class RetrievalNeed(BaseModel):
    query: str = Field(
        description="A concise, self-contained search query for the internal Northstar corpus."
    )
    why_needed: str = Field(
        description="Why this evidence is needed for the unresolved Proposal C uncertainty."
    )
    related_uncertainty: str = Field(
        description="The exact Proposal C uncertainty the query is intended to resolve."
    )


class RetrievalNeedsOutput(BaseModel):
    needs: list[RetrievalNeed] = Field(
        min_length=1,
        max_length=4,
        description="A small set of non-duplicative internal retrieval needs."
    )


### 5. Freeze the existing Proposal C retrieval input


In [32]:
# Retrieval-only snapshot copied from the existing Proposal C structured
# extraction. It is input to this isolated experiment and is never written
# back into the Iteration-1 graph or state.
proposal_c_retrieval_text = """
Leadership approval is requested for a 6-week pilot to enrich identifiable
customer profiles using third-party data in order to improve targeting and
response rates for commercial campaigns.

The pilot will cover approximately 25,000 customer records and will be led by
the Director, Growth Marketing.

The current campaign response rate is 3.4%. The pilot aims to improve this to
at least 4.2%.

The estimated pilot cost is $38,000.

The proposal assumes that existing customer terms and current data-use practices
permit the use of third-party enrichment data for targeted offers.

The proposed data provider can supply demographic, firmographic, and behavioral
attributes that would be matched to existing customer records.

Data Governance review has not yet been initiated.

Success criteria include:
- campaign response rate of at least 4.2%
- acceptable match accuracy
- no material increase in customer opt-outs or complaints

The pilot will run for 6 weeks. If the expected improvement is not observed,
or if data-quality or customer-response issues are material, the pilot will stop
and the approach will be reassessed.
"""

proposal_c_retrieval_state = {
    "decision_stage": "pilot",
    "decision_ask": "Approve a 6-week customer data enrichment pilot",
    "problem_summary": "Improve targeting and campaign response rates.",
    "proposed_intervention": (
        "Enrich identifiable customer profiles with third-party demographic, "
        "firmographic, and behavioral attributes for targeted offers."
    ),
    "material_claims": [
        {
            "claim_text": (
                "The proposed data provider can supply attributes that can be "
                "matched to existing customer records with acceptable accuracy."
            ),
            "claim_role": "unknown",
            "evidence_basis": [
                "The provider capability is stated, but no match-rate evidence or prior validation is provided."
            ],
        },
        {
            "claim_text": (
                "Existing customer terms and data-use practices permit third-party "
                "enrichment for targeted offers."
            ),
            "claim_role": "unknown",
            "evidence_basis": [
                "The proposal states this as an assumption without confirmation."
            ],
        },
    ],
    "dependency_reviews": [
        {
            "dependency_text": "Data Governance review",
            "requirement_status": "unknown",
            "evidence_basis": [
                "The review has not been initiated and the proposal does not state whether it is required before the pilot."
            ],
        },
        {
            "dependency_text": (
                "Confirmation that customer terms permit third-party data enrichment"
            ),
            "requirement_status": "unknown",
            "evidence_basis": [
                "Permissibility is assumed but has not been confirmed."
            ],
        },
        {
            "dependency_text": (
                "Verification that the provider can match enrichment data to customer records"
            ),
            "requirement_status": "unknown",
            "evidence_basis": [
                "No completed technical or sample-match validation is stated."
            ],
        },
    ],
}

print("Frozen Proposal C retrieval input:")
print(f"- unknown claims: {len(proposal_c_retrieval_state['material_claims'])}")
print(f"- unknown dependencies/reviews: {len(proposal_c_retrieval_state['dependency_reviews'])}")


Frozen Proposal C retrieval input:
- unknown claims: 2
- unknown dependencies/reviews: 3


### 6. Generate retrieval queries for Proposal C


In [33]:
def generate_internal_retrieval_needs(
    proposal_text: str,
    structured_state: dict,
) -> list[RetrievalNeed]:
    """Create typed queries from existing unknowns without calling a model API."""
    decision_stage = structured_state.get("decision_stage", "decision")
    intervention = structured_state.get("proposed_intervention", "the proposed initiative")
    needs = []

    unknown_dependencies = [
        item
        for item in structured_state.get("dependency_reviews", [])
        if item.get("requirement_status") == "unknown"
    ]
    for item in unknown_dependencies[:3]:
        dependency_text = item["dependency_text"]
        needs.append(
            RetrievalNeed(
                query=(
                    f"{dependency_text} policy requirements timing and controls "
                    f"for a {decision_stage}: {intervention}"
                ),
                why_needed=(
                    "Determine whether the internal corpus establishes this item "
                    "as required, optional, conditional, or unresolved for the pilot."
                ),
                related_uncertainty=dependency_text,
            )
        )

    unknown_claims = [
        item
        for item in structured_state.get("material_claims", [])
        if item.get("claim_role") == "unknown"
    ]
    if unknown_claims and len(needs) < 4:
        claim_text = unknown_claims[0]["claim_text"]
        needs.append(
            RetrievalNeed(
                query=(
                    f"Prior decision precedents for a {decision_stage} involving "
                    f"the uncertainty: {claim_text}"
                ),
                why_needed=(
                    "Find a comparable Northstar decision showing how leadership "
                    "handled the same uncertainty without inferring a new rule."
                ),
                related_uncertainty=claim_text,
            )
        )

    return RetrievalNeedsOutput(needs=needs).needs


proposal_c_retrieval_needs = generate_internal_retrieval_needs(
    proposal_c_retrieval_text,
    proposal_c_retrieval_state,
)

for index, need in enumerate(proposal_c_retrieval_needs, start=1):
    print(f"Query {index}: {need.query}")
    print(f"Why needed: {need.why_needed}")
    print(f"Related uncertainty: {need.related_uncertainty}")
    print()


Query 1: Data Governance review policy requirements timing and controls for a pilot: Enrich identifiable customer profiles with third-party demographic, firmographic, and behavioral attributes for targeted offers.
Why needed: Determine whether the internal corpus establishes this item as required, optional, conditional, or unresolved for the pilot.
Related uncertainty: Data Governance review

Query 2: Confirmation that customer terms permit third-party data enrichment policy requirements timing and controls for a pilot: Enrich identifiable customer profiles with third-party demographic, firmographic, and behavioral attributes for targeted offers.
Why needed: Determine whether the internal corpus establishes this item as required, optional, conditional, or unresolved for the pilot.
Related uncertainty: Confirmation that customer terms permit third-party data enrichment

Query 3: Verification that the provider can match enrichment data to customer records policy requirements timing and c

### 7. Retrieve the top chunks for each query


In [34]:
TOP_K = 3


def retrieve_internal_evidence(
    needs: list[RetrievalNeed],
    top_k: int = TOP_K,
) -> list[dict]:
    retrieval_results = []

    for need in needs:
        matches = vector_store.similarity_search_with_score(
            need.query,
            k=top_k,
        )

        retrieval_results.append({
            "need": need,
            "matches": [
                {
                    "document": document,
                    "score": score,
                }
                for document, score in matches
            ],
        })

    return retrieval_results


proposal_c_retrieval_results = retrieve_internal_evidence(
    proposal_c_retrieval_needs
)

print(
    f"Retrieved the top {TOP_K} chunks for each of "
    f"{len(proposal_c_retrieval_results)} queries."
)


Retrieved the top 3 chunks for each of 4 queries.


### 8. Inspect queries and retrieved evidence


In [35]:
for query_index, result in enumerate(proposal_c_retrieval_results, start=1):
    need = result["need"]
    print("=" * 100)
    print(f"QUERY {query_index}: {need.query}")
    print(f"WHY NEEDED: {need.why_needed}")
    print(f"RELATED UNCERTAINTY: {need.related_uncertainty}")

    for rank, match in enumerate(result["matches"], start=1):
        document = match["document"]
        print("-" * 100)
        print(
            f"RANK {rank} | SOURCE: {document.metadata['source']} | "
            f"CHUNK: {document.metadata['chunk_id']} | "
            f"COSINE SCORE: {match['score']:.4f}"
        )
        print(document.page_content)
        print()


QUERY 1: Data Governance review policy requirements timing and controls for a pilot: Enrich identifiable customer profiles with third-party demographic, firmographic, and behavioral attributes for targeted offers.
WHY NEEDED: Determine whether the internal corpus establishes this item as required, optional, conditional, or unresolved for the pilot.
RELATED UNCERTAINTY: Data Governance review
----------------------------------------------------------------------------------------------------
RANK 1 | SOURCE: Prior Decision Precedents | CHUNK: Prior Decision Precedents::chunk-02 | COSINE SCORE: 0.4302
Context

Marketing proposed combining customer transaction data with third-party demographic data for targeted offers.

Evidence

Strong revenue case, but third-party source provenance and customer-purpose compatibility were unclear.

Conditions / Follow-up

Data Governance review required before any identifiable data enrichment; anonymized aggregate market analysis permitted as an alternat

### 9. Retrieval diagnostic summary


In [36]:
retrieved_source_counts = Counter(
    match["document"].metadata["source"]
    for result in proposal_c_retrieval_results
    for match in result["matches"]
)

governance_hits = [
    (result["need"].query, rank, match["score"], match["document"])
    for result in proposal_c_retrieval_results
    for rank, match in enumerate(result["matches"], start=1)
    if match["document"].metadata["source"]
    == "Risk / Data Governance Guidelines"
]

print("Generated queries:")
for need in proposal_c_retrieval_needs:
    print(f"- {need.query}")

print("\nTop-k source frequency:")
for source_name, count in retrieved_source_counts.most_common():
    print(f"- {source_name}: {count}")

print("\nRisk / Data Governance Guidelines hits:")
if governance_hits:
    for query, rank, score, document in governance_hits:
        print(
            f"- rank {rank}, score {score:.4f}, "
            f"{document.metadata['chunk_id']} | query: {query}"
        )
else:
    print("- No top-k governance-policy chunk was retrieved.")

print(
    "\nInterpretation note: retrieval strength should be judged from rank, "
    "relative score, source authority, and whether the displayed text directly "
    "addresses the uncertainty—not from an arbitrary score cutoff."
)


Generated queries:
- Data Governance review policy requirements timing and controls for a pilot: Enrich identifiable customer profiles with third-party demographic, firmographic, and behavioral attributes for targeted offers.
- Confirmation that customer terms permit third-party data enrichment policy requirements timing and controls for a pilot: Enrich identifiable customer profiles with third-party demographic, firmographic, and behavioral attributes for targeted offers.
- Verification that the provider can match enrichment data to customer records policy requirements timing and controls for a pilot: Enrich identifiable customer profiles with third-party demographic, firmographic, and behavioral attributes for targeted offers.
- Prior decision precedents for a pilot involving the uncertainty: The proposed data provider can supply attributes that can be matched to existing customer records with acceptable accuracy.

Top-k source frequency:
- Prior Decision Precedents: 7
- Risk / Data 

### 10. Experiment conclusion

**Components used:** Python standard-library ZIP/XML parsing for DOCX, LangChain `Document` and `RecursiveCharacterTextSplitter`, Pydantic retrieval-need models, NumPy, and a small local TF-IDF cosine vector store.

**Chunking:** 900-character recursive chunks with 150-character overlap produced 13 chunks: four Approval Policy, three Pilot Governance, three Risk / Data Governance, and three Prior Decision Precedent chunks.

**Queries:** Four typed queries were generated locally from Proposal C's frozen unknown claims and dependency/review statuses: Data Governance review, customer-terms permissibility, provider match verification, and comparable prior precedents.

**Top retrieval:** `Prior Decision Precedents::chunk-02` ranked first for the three governance/permission/provider queries and directly describes third-party customer enrichment requiring Data Governance review. `Risk / Data Governance Guidelines::chunk-01` appeared in the top three for every query and directly states the purpose-limitation and identifiable-customer-enrichment controls.

**Answer to the experiment question:** Yes. The controlled corpus reliably retrieved authoritative governance evidence strongly enough to justify a next, separately controlled integration experiment. The authoritative governance chunk appeared for all four queries, and the closest precedent independently reinforced the same issue.

**Quality concerns:** This baseline is lexical rather than semantic; the analogous precedent often outranked the authoritative policy; cosine scores are corpus-relative and not calibrated confidence; simple DOCX extraction loses table structure; overlapping chunks can repeat context; and the retrieval input is a frozen Proposal C state snapshot. Retrieved text has not been used to change requirement status, blockers, caveats, readiness, or the final graph.


## Iteration 2 — Grounded Evidence Interpretation

This step interprets retrieved Northstar text for one unresolved Proposal C dependency: **Data Governance review**. It is deliberately local and read-only. The resulting `GroundedRequirement` is displayed for inspection but is not written into Proposal C state, readiness, blockers, caveats, questions, actions, or the final graph.


### 11. Define the grounded requirement schema


In [37]:
from typing import Literal


class GroundedRequirement(BaseModel):
    requirement_text: str
    requirement_status: Literal["required", "not_required", "unclear"]
    supporting_source_ids: list[str]
    evidence_excerpt: list[str]
    rationale: str


### 12. Assemble the unresolved dependency and retrieved evidence


In [38]:
SOURCE_TYPES = {
    "Approval Policy": "authoritative_policy",
    "Pilot Governance Guidelines": "authoritative_guideline",
    "Risk / Data Governance Guidelines": "authoritative_guideline",
    "Prior Decision Precedents": "analogous_precedent",
}

proposal_c_unresolved_review = next(
    item
    for item in proposal_c_retrieval_state["dependency_reviews"]
    if item["dependency_text"] == "Data Governance review"
)

data_governance_retrieval = next(
    result
    for result in proposal_c_retrieval_results
    if result["need"].related_uncertainty == "Data Governance review"
)

data_governance_evidence = [
    {
        "source_id": match["document"].metadata["chunk_id"],
        "source_document": match["document"].metadata["source"],
        "source_type": SOURCE_TYPES[match["document"].metadata["source"]],
        "retrieval_score": match["score"],
        "chunk_text": match["document"].page_content,
    }
    for match in data_governance_retrieval["matches"]
]

print("Unresolved dependency/review:")
print(proposal_c_unresolved_review)
print("\nRetrieved evidence selected for interpretation:")
for evidence in data_governance_evidence:
    print(
        f"- {evidence['source_id']} | {evidence['source_type']} | "
        f"score={evidence['retrieval_score']:.4f}"
    )


Unresolved dependency/review:
{'dependency_text': 'Data Governance review', 'requirement_status': 'unknown', 'evidence_basis': ['The review has not been initiated and the proposal does not state whether it is required before the pilot.']}

Retrieved evidence selected for interpretation:
- Prior Decision Precedents::chunk-02 | analogous_precedent | score=0.4302
- Prior Decision Precedents::chunk-01 | analogous_precedent | score=0.2756
- Risk / Data Governance Guidelines::chunk-01 | authoritative_guideline | score=0.2021


### 13. Interpret the requirement using retrieved text only


In [39]:
REQUIRED_SIGNALS = (
    "required",
    "must",
    "only when",
    "before any",
    "before launch",
    "before deployment",
)
NOT_REQUIRED_SIGNALS = (
    "not required",
    "no review required",
    "optional",
    "may proceed without",
)


def extract_grounding_excerpt(text: str) -> str:
    """Return the smallest displayed passage that carries the requirement."""
    paragraphs = [paragraph.strip() for paragraph in text.split("\n\n") if paragraph.strip()]
    relevant = [
        paragraph
        for paragraph in paragraphs
        if "data governance" in paragraph.casefold()
        or "customer data enrichment" in paragraph.casefold()
        or "identifiable data enrichment" in paragraph.casefold()
    ]
    return " ".join(relevant[:3]) if relevant else text.strip()


def interpret_grounded_requirement(
    unresolved_dependency: dict,
    retrieved_evidence: list[dict],
) -> GroundedRequirement:
    """
    Apply a conservative evidence hierarchy.

    Authoritative policy/guidelines may establish a requirement. A precedent
    may corroborate interpretation but cannot establish a requirement alone.
    """
    authoritative = [
        evidence
        for evidence in retrieved_evidence
        if evidence["source_type"] in {
            "authoritative_policy",
            "authoritative_guideline",
        }
    ]
    precedents = [
        evidence
        for evidence in retrieved_evidence
        if evidence["source_type"] == "analogous_precedent"
    ]

    def explicitly_supports_required(evidence: dict) -> bool:
        text = evidence["chunk_text"].casefold()
        return (
            "data governance" in text
            and any(signal in text for signal in REQUIRED_SIGNALS)
        )

    def explicitly_supports_not_required(evidence: dict) -> bool:
        text = evidence["chunk_text"].casefold()
        return (
            "data governance" in text
            and any(signal in text for signal in NOT_REQUIRED_SIGNALS)
        )

    required_sources = [
        evidence for evidence in authoritative
        if explicitly_supports_required(evidence)
    ]
    not_required_sources = [
        evidence for evidence in authoritative
        if explicitly_supports_not_required(evidence)
    ]

    if required_sources and not not_required_sources:
        status = "required"
        decisive_sources = required_sources
        rationale = (
            "Authoritative Risk / Data Governance guidance explicitly states "
            "that external enrichment of identifiable customer records is "
            "permitted only when Data Governance confirms the relevant controls. "
            "The analogous precedent independently states that Data Governance "
            "review was required before identifiable enrichment. The guideline, "
            "not the precedent alone, establishes the requirement."
        )
    elif not_required_sources and not required_sources:
        status = "not_required"
        decisive_sources = not_required_sources
        rationale = (
            "Authoritative retrieved text explicitly states that the review is "
            "not required for this activity."
        )
    else:
        status = "unclear"
        decisive_sources = authoritative
        rationale = (
            "The authoritative retrieved text is missing, conflicting, or does "
            "not clearly establish whether this review is required. Precedent "
            "alone is insufficient to promote an unknown requirement."
        )

    supporting_evidence = list(decisive_sources)
    if status == "required":
        supporting_evidence.extend(
            evidence
            for evidence in precedents
            if explicitly_supports_required(evidence)
        )

    return GroundedRequirement(
        requirement_text=unresolved_dependency["dependency_text"],
        requirement_status=status,
        supporting_source_ids=[
            evidence["source_id"]
            for evidence in supporting_evidence
        ],
        evidence_excerpt=[
            extract_grounding_excerpt(evidence["chunk_text"])
            for evidence in supporting_evidence
        ],
        rationale=rationale,
    )


proposal_c_grounded_requirement = interpret_grounded_requirement(
    proposal_c_unresolved_review,
    data_governance_evidence,
)


### 14. Inspect the grounded interpretation


In [40]:
print("Unresolved dependency/review:")
print(proposal_c_unresolved_review["dependency_text"])

print("\nRetrieved evidence used:")
for evidence in data_governance_evidence:
    print(
        f"- SOURCE ID: {evidence['source_id']}\n"
        f"  SOURCE TYPE: {evidence['source_type']}\n"
        f"  SOURCE DOCUMENT: {evidence['source_document']}\n"
        f"  RETRIEVAL SCORE: {evidence['retrieval_score']:.4f}\n"
        f"  CHUNK TEXT: {evidence['chunk_text']}\n"
    )

print("Resulting requirement status:")
print(proposal_c_grounded_requirement.requirement_status)
print("\nRationale:")
print(proposal_c_grounded_requirement.rationale)
print("\nSupporting source IDs:")
for source_id in proposal_c_grounded_requirement.supporting_source_ids:
    print(f"- {source_id}")
print("\nEvidence excerpts:")
for excerpt in proposal_c_grounded_requirement.evidence_excerpt:
    print(f"- {excerpt}")

print(
    "\nState-mutation check: this grounded result is inspection-only; "
    "Proposal C state and readiness were not changed."
)


Unresolved dependency/review:
Data Governance review

Retrieved evidence used:
- SOURCE ID: Prior Decision Precedents::chunk-02
  SOURCE TYPE: analogous_precedent
  SOURCE DOCUMENT: Prior Decision Precedents
  RETRIEVAL SCORE: 0.4302
  CHUNK TEXT: Context

Marketing proposed combining customer transaction data with third-party demographic data for targeted offers.

Evidence

Strong revenue case, but third-party source provenance and customer-purpose compatibility were unclear.

Conditions / Follow-up

Data Governance review required before any identifiable data enrichment; anonymized aggregate market analysis permitted as an alternative.

Decision lesson

Strong economics do not override mandatory data-governance requirements.

- SOURCE ID: Prior Decision Precedents::chunk-01
  SOURCE TYPE: analogous_precedent
  SOURCE DOCUMENT: Prior Decision Precedents
  RETRIEVAL SCORE: 0.2756
  CHUNK TEXT: Decision

Deferred / Not Ready

Context

Commercial team requested full rollout of an AI assi

### 15. Grounded interpretation conclusion

The retrieved evidence is sufficient to classify **Data Governance review** as `required` for Proposal C's proposed identifiable customer-data enrichment activity.

The decisive source is `Risk / Data Governance Guidelines::chunk-01`, an authoritative guideline stating that external enrichment of identifiable customer records is permitted only when the source is approved, the intended use is documented, and Data Governance confirms that consent, notice, contractual terms, and retention controls are adequate.

`Prior Decision Precedents::chunk-02` corroborates the interpretation with a closely analogous customer-enrichment decision requiring Data Governance review before identifiable enrichment. The precedent supports interpretation, but the authoritative guideline establishes the requirement.

This conclusion remains inspection-only. Proposal C's `requirement_status`, blockers, caveats, readiness, questions, actions, and final brief have not been changed.


## Iteration 2 — Canonical Retrieval Integration

This section integrates local retrieval and grounded requirement interpretation into a new canonical graph while preserving the inherited Iteration-1 v0.3 nodes and thresholds.

```text
extract_proposal
→ identify_retrieval_needs
→ retrieve_internal_evidence
→ interpret_grounded_requirements
→ assess_readiness
→ check_hard_blockers
→ identify_material_caveats
→ determine_readiness
→ downstream synthesis
```

Retrieved Northstar text remains local. Existing model-backed nodes continue to receive their original proposal/extraction contexts; grounded citations are evaluated in the hard-blocker wrapper and appended to final synthesis locally.


### 16. Add Iteration-2 retrieval and grounding fields to graph state


In [49]:
class RetrievalNeedState(TypedDict):
    query: str
    why_needed: str
    related_uncertainty: str


class RetrievedEvidenceState(TypedDict):
    source_id: str
    source_document: str
    source_type: str
    rank: int
    retrieval_score: float
    chunk_text: str


class RetrievalResultState(TypedDict):
    need: RetrievalNeedState
    matches: list[RetrievedEvidenceState]


class GroundedRequirementState(TypedDict):
    requirement_text: str
    requirement_status: Literal["required", "not_required", "unclear"]
    supporting_source_ids: list[str]
    evidence_excerpt: list[str]
    rationale: str


class RetrievalProvenanceState(TypedDict):
    related_uncertainty: str
    retrieval_query: str
    source_id: str
    source_type: str
    rank: int
    score: float
    grounded_interpretation: Literal["required", "not_required", "unclear"]


class DecisionReadyIteration2State(DecisionReadyState, total=False):
    retrieval_needs: NotRequired[list[RetrievalNeedState]]
    retrieved_internal_evidence: NotRequired[list[RetrievalResultState]]
    grounded_requirements: NotRequired[list[GroundedRequirementState]]
    retrieval_provenance: NotRequired[list[RetrievalProvenanceState]]


### 17. Identify retrieval needs from existing unknowns


In [50]:
def _is_requested_approval_item(
    dependency_text: str,
    decision_ask: str,
) -> bool:
    dependency = dependency_text.casefold()
    ask = decision_ask.casefold()
    return (
        any(term in dependency for term in ("approv", "authoriz"))
        and any(term in ask for term in ("approv", "authoriz"))
        and any(
            term in dependency
            for term in ("budget", "funding", "pilot", "rollout", "investment")
        )
    )


def identify_retrieval_needs(
    state: DecisionReadyIteration2State,
) -> dict:
    decision_stage = state.get("decision_stage", "decision")
    decision_ask = state.get("decision_ask", "")
    intervention = state.get("proposed_intervention", "the proposed initiative")
    needs = []

    unknown_dependencies = [
        item
        for item in state.get("dependency_reviews", [])
        if item.get("requirement_status") == "unknown"
        and not _is_requested_approval_item(
            item.get("dependency_text", ""),
            decision_ask,
        )
    ]

    for item in unknown_dependencies[:3]:
        dependency_text = item["dependency_text"]
        needs.append(
            RetrievalNeed(
                query=(
                    f"{dependency_text} policy requirements timing and controls "
                    f"for a {decision_stage}: {intervention}"
                ),
                why_needed=(
                    "Determine whether authoritative internal evidence establishes "
                    "this existing unknown as required, not required, or unclear."
                ),
                related_uncertainty=dependency_text,
            ).model_dump()
        )

    unknown_claims = [
        item
        for item in state.get("material_claims", [])
        if item.get("claim_role") == "unknown"
    ]
    if unknown_claims and len(needs) < 4:
        claim_text = unknown_claims[0]["claim_text"]
        needs.append(
            RetrievalNeed(
                query=(
                    f"Prior decision precedents for a {decision_stage} involving "
                    f"the uncertainty: {claim_text}"
                ),
                why_needed=(
                    "Find a comparable decision without treating precedent alone "
                    "as authoritative policy."
                ),
                related_uncertainty=claim_text,
            ).model_dump()
        )

    return {"retrieval_needs": needs}


### 18. Retrieve internal evidence locally


In [51]:
def retrieve_internal_evidence_node(
    state: DecisionReadyIteration2State,
) -> dict:
    retrieval_results = []

    for need in state.get("retrieval_needs", []):
        matches = vector_store.similarity_search_with_score(
            need["query"],
            k=TOP_K,
        )
        retrieval_results.append({
            "need": need,
            "matches": [
                {
                    "rank": rank,
                    "source_id": document.metadata["chunk_id"],
                    "source_document": document.metadata["source"],
                    "source_type": SOURCE_TYPES[document.metadata["source"]],
                    "retrieval_score": score,
                    "chunk_text": document.page_content,
                }
                for rank, (document, score) in enumerate(matches, start=1)
            ],
        })

    return {"retrieved_internal_evidence": retrieval_results}


### 19. Interpret grounded requirements conservatively


In [52]:
GROUNDING_STOPWORDS = {
    "that", "this", "with", "from", "into", "before", "after", "prior",
    "confirmation", "verification", "requirement", "status", "review",
}


def _subject_supported(
    dependency_text: str,
    evidence_text: str,
) -> bool:
    dependency = dependency_text.casefold()
    evidence = evidence_text.casefold()

    # Concept-level matches keep the rule general while avoiding a loose
    # bag-of-words promotion for unrelated requirements.
    if "data governance" in dependency:
        return "data governance" in evidence
    if "customer terms" in dependency:
        return (
            "data governance" in evidence
            and any(term in evidence for term in ("customer terms", "contractual terms"))
        )
    if any(term in dependency for term in ("match", "accuracy")):
        return any(term in evidence for term in ("match", "matching", "accuracy"))

    tokens = {
        token
        for token in re.findall(r"[a-z0-9]+", dependency)
        if len(token) >= 4 and token not in GROUNDING_STOPWORDS
    }
    overlap = sum(token in evidence for token in tokens)
    return bool(tokens) and overlap >= min(2, len(tokens))


def interpret_grounded_requirement_general(
    unresolved_dependency: dict,
    retrieved_evidence: list[dict],
    decision_stage: str,
) -> GroundedRequirement:
    dependency_text = unresolved_dependency["dependency_text"]
    dependency_lower = dependency_text.casefold()

    # A future scale requirement is not automatically a current pilot requirement.
    if "pilot" in decision_stage.casefold() and any(
        term in dependency_lower for term in ("scale", "scaling", "rollout")
    ):
        return GroundedRequirement(
            requirement_text=dependency_text,
            requirement_status="unclear",
            supporting_source_ids=[],
            evidence_excerpt=[],
            rationale=(
                "The dependency concerns a later scale or rollout decision; the "
                "retrieved text does not establish it as required for the current pilot."
            ),
        )

    authoritative = [
        evidence
        for evidence in retrieved_evidence
        if evidence["source_type"] in {
            "authoritative_policy",
            "authoritative_guideline",
        }
        and _subject_supported(dependency_text, evidence["chunk_text"])
    ]
    precedents = [
        evidence
        for evidence in retrieved_evidence
        if evidence["source_type"] == "analogous_precedent"
        and _subject_supported(dependency_text, evidence["chunk_text"])
    ]

    required_sources = [
        evidence
        for evidence in authoritative
        if any(signal in evidence["chunk_text"].casefold() for signal in REQUIRED_SIGNALS)
    ]
    not_required_sources = [
        evidence
        for evidence in authoritative
        if any(signal in evidence["chunk_text"].casefold() for signal in NOT_REQUIRED_SIGNALS)
    ]

    if required_sources and not not_required_sources:
        status = "required"
        decisive_sources = required_sources
        rationale = (
            "Retrieved authoritative policy or guideline text explicitly supports "
            "this requirement. Analogous precedent is used only as corroboration."
        )
    elif not_required_sources and not required_sources:
        status = "not_required"
        decisive_sources = not_required_sources
        rationale = (
            "Retrieved authoritative text explicitly states that this item is not required."
        )
    else:
        status = "unclear"
        decisive_sources = []
        rationale = (
            "Retrieved authoritative text is absent, conflicting, or does not "
            "explicitly establish this requirement. Precedent alone is insufficient."
        )

    supporting_evidence = list(decisive_sources)
    if status == "required":
        supporting_evidence.extend(
            evidence
            for evidence in precedents
            if any(signal in evidence["chunk_text"].casefold() for signal in REQUIRED_SIGNALS)
        )

    return GroundedRequirement(
        requirement_text=dependency_text,
        requirement_status=status,
        supporting_source_ids=[
            evidence["source_id"] for evidence in supporting_evidence
        ],
        evidence_excerpt=[
            extract_grounding_excerpt(evidence["chunk_text"])
            for evidence in supporting_evidence
        ],
        rationale=rationale,
    )


def interpret_grounded_requirements(
    state: DecisionReadyIteration2State,
) -> dict:
    results_by_uncertainty = {
        result["need"]["related_uncertainty"]: result["matches"]
        for result in state.get("retrieved_internal_evidence", [])
    }

    grounded_requirements = []
    for dependency in state.get("dependency_reviews", []):
        if dependency.get("requirement_status") != "unknown":
            continue

        evidence = results_by_uncertainty.get(dependency["dependency_text"])
        if not evidence:
            continue

        grounded = interpret_grounded_requirement_general(
            dependency,
            evidence,
            state.get("decision_stage", "decision"),
        )
        grounded_requirements.append(grounded.model_dump())

    grounded_by_text = {
        item["requirement_text"]: item
        for item in grounded_requirements
    }
    retrieval_provenance = []
    for result in state.get("retrieved_internal_evidence", []):
        uncertainty = result["need"]["related_uncertainty"]
        grounded = grounded_by_text.get(uncertainty)
        if not grounded:
            continue
        for match in result["matches"]:
            retrieval_provenance.append({
                "related_uncertainty": uncertainty,
                "retrieval_query": result["need"]["query"],
                "source_id": match["source_id"],
                "source_type": match["source_type"],
                "rank": match["rank"],
                "score": match["retrieval_score"],
                "grounded_interpretation": grounded["requirement_status"],
            })

    return {
        "grounded_requirements": grounded_requirements,
        "retrieval_provenance": retrieval_provenance,
    }


### 20. Let the hard-blocker layer evaluate grounded requirements


In [53]:
def _explicit_review_authority(requirement_text: str) -> str | None:
    """Return the named authority for an explicit review/approval dependency."""
    normalized = requirement_text.strip().lower()
    for marker in (" review", " approval"):
        if marker in normalized:
            authority = normalized.split(marker, 1)[0].strip()
            return authority or None
    return None


def check_hard_blockers_iteration2(
    state: DecisionReadyIteration2State,
) -> dict:
    # Existing Iteration-1 blocker logic runs unchanged first.
    blockers = list(check_hard_blockers(state)["hard_blockers"])

    original_dependencies = {
        item["dependency_text"]: item
        for item in state.get("dependency_reviews", [])
    }
    required_grounded = [
        grounded
        for grounded in state.get("grounded_requirements", [])
        if grounded["requirement_status"] == "required"
        and original_dependencies.get(
            grounded["requirement_text"], {}
        ).get("requirement_status") == "unknown"
    ]

    # A policy can state both an explicit review and the conditions that review
    # must verify. When both appear in the proposal's unknown dependency list,
    # keep one review/approval blocker and attach the condition as evidence.
    # This avoids double-counting one authoritative governance path.
    explicit_reviews = [
        (grounded, _explicit_review_authority(grounded["requirement_text"]))
        for grounded in required_grounded
        if _explicit_review_authority(grounded["requirement_text"])
    ]
    subordinate_to_review: dict[str, list[dict]] = {
        grounded["requirement_text"]: []
        for grounded, _ in explicit_reviews
    }
    subordinate_texts: set[str] = set()

    for grounded in required_grounded:
        if _explicit_review_authority(grounded["requirement_text"]):
            continue
        evidence_text = " ".join(grounded["evidence_excerpt"]).lower()
        for review, authority in explicit_reviews:
            if (
                authority
                and authority in evidence_text
                and any(term in evidence_text for term in ("review", "approval", "confirms"))
            ):
                subordinate_to_review[review["requirement_text"]].append(grounded)
                subordinate_texts.add(grounded["requirement_text"])
                break

    for grounded in required_grounded:
        if grounded["requirement_text"] in subordinate_texts:
            continue

        duplicate = any(
            blocker.get("blocker_code") == "UNRESOLVED_CRITICAL_DEPENDENCY"
            and blocker.get("related_item_text") == grounded["requirement_text"]
            for blocker in blockers
        )
        if duplicate:
            continue

        evidence_requirements = [grounded] + subordinate_to_review.get(
            grounded["requirement_text"], []
        )
        cited_evidence = [
            f"{source_id}: {excerpt}"
            for item in evidence_requirements
            for source_id, excerpt in zip(
                item["supporting_source_ids"],
                item["evidence_excerpt"],
            )
        ]
        blockers.append({
            "blocker_code": "UNRESOLVED_CRITICAL_DEPENDENCY",
            "source": "deterministic",
            "rationale": (
                "Grounded authoritative evidence establishes this dependency "
                "as required, while the proposal still shows it as unresolved. "
                f"{grounded['rationale']}"
            ),
            "evidence": cited_evidence,
            "related_item_text": grounded["requirement_text"],
        })

    return {"hard_blockers": blockers}


### 21. Preserve grounded citations in final synthesis locally


In [54]:
def _normalized_for_dedup(value: str) -> str:
    return " ".join(re.findall(r"[a-z0-9]+", value.casefold()))


def _item_materially_present(brief: str, item: str) -> bool:
    stopwords = {
        "that", "this", "with", "from", "into", "before", "after",
        "their", "there", "would", "could", "should", "pilot", "proposal",
    }
    item_tokens = {
        token
        for token in re.findall(r"[a-z0-9]+", item.casefold())
        if len(token) >= 4 and token not in stopwords
    }
    if not item_tokens:
        return True
    brief_tokens = set(re.findall(r"[a-z0-9]+", brief.casefold()))
    overlap = len(item_tokens & brief_tokens)
    return overlap >= max(2, (len(item_tokens) + 1) // 2)


def _append_structured_synthesis_coverage(
    state: DecisionReadyIteration2State,
    brief: str,
) -> str:
    gap_lines = []
    for dimension, assessment in state.get("dimension_assessments", {}).items():
        for gap in assessment.get("gaps", []):
            if not _item_materially_present(brief, gap):
                gap_lines.append(
                    f"- [inferred state — {dimension} gap] {gap}"
                )
            if len(gap_lines) >= 6:
                break
        if len(gap_lines) >= 6:
            break

    claim_lines = []
    for claim in state.get("material_claims", []):
        if claim.get("claim_role") not in {"hypothesis", "unknown"}:
            continue
        claim_text = claim.get("claim_text", "")
        if not claim_text or _item_materially_present(brief, claim_text):
            continue
        basis = "; ".join(claim.get("evidence_basis", [])) or "No stated basis"
        claim_lines.append(
            f"- [inferred state — {claim['claim_role']}] {claim_text} "
            f"| basis: {basis}"
        )
        if len(claim_lines) >= 5:
            break

    sections = []
    if gap_lines:
        sections.append(
            "Structured Evidence-Limitation Trace\n" + "\n".join(gap_lines)
        )
    if claim_lines:
        sections.append(
            "Material Claims / Hypotheses Trace\n" + "\n".join(claim_lines)
        )
    if not sections:
        return brief
    return brief + "\n\n" + "\n\n".join(sections)


def build_final_brief_iteration2(
    state: DecisionReadyIteration2State,
) -> dict:
    base_brief = build_final_brief(state)["final_brief"]
    base_normalized = _normalized_for_dedup(base_brief)
    local_sections = []

    future_lines = []
    for item in state.get("future_decision_considerations", []):
        related = item.get("related_item_text") or item["caveat_code"]
        normalized = _normalized_for_dedup(related)
        if normalized and normalized not in base_normalized:
            future_lines.append(f"- {related}: {item['rationale']}")
    if future_lines:
        local_sections.append(
            "Future-Decision Considerations (not current readiness caveats)\n"
            + "\n".join(future_lines)
        )

    supported_grounded = [
        item
        for item in state.get("grounded_requirements", [])
        if item.get("supporting_source_ids")
        and item["requirement_status"] in {"required", "not_required"}
    ]
    if supported_grounded:
        trace_lines = ["Retrieved Evidence Provenance"]
        provenance = state.get("retrieval_provenance", [])
        for grounded in supported_grounded:
            trace_lines.append(
                f"- {grounded['requirement_text']}: "
                f"{grounded['requirement_status'].upper()}"
            )
            for source_id in grounded["supporting_source_ids"]:
                matches = [
                    item
                    for item in provenance
                    if item["related_uncertainty"] == grounded["requirement_text"]
                    and item["source_id"] == source_id
                ]
                if matches:
                    match = matches[0]
                    trace_lines.append(
                        f"  {source_id} | {match['source_type']} | "
                        f"rank={match['rank']} | score={match['score']:.4f}"
                    )
                else:
                    trace_lines.append(f"  {source_id}")
        local_sections.append("\n".join(trace_lines))

    combined_brief = base_brief
    if local_sections:
        combined_brief += "\n\n" + "\n\n".join(local_sections)
    return {
        "final_brief": _append_structured_synthesis_coverage(
            state,
            combined_brief,
        )
    }


### 22. Build the canonical Iteration-2 graph


In [55]:
iteration2_graph_builder = StateGraph(DecisionReadyIteration2State)

iteration2_graph_builder.add_node("extract_proposal", extract_proposal)
iteration2_graph_builder.add_node("identify_retrieval_needs", identify_retrieval_needs)
iteration2_graph_builder.add_node("retrieve_internal_evidence", retrieve_internal_evidence_node)
iteration2_graph_builder.add_node("interpret_grounded_requirements", interpret_grounded_requirements)
iteration2_graph_builder.add_node("assess_readiness", assess_readiness)
iteration2_graph_builder.add_node("check_hard_blockers", check_hard_blockers_iteration2)
iteration2_graph_builder.add_node("identify_material_caveats", identify_material_caveats)
iteration2_graph_builder.add_node("determine_readiness", determine_readiness)
iteration2_graph_builder.add_node("generate_leadership_questions", generate_leadership_questions)
iteration2_graph_builder.add_node("generate_recommended_actions", generate_recommended_actions)
iteration2_graph_builder.add_node("build_final_brief", build_final_brief_iteration2)

iteration2_graph_builder.add_edge(START, "extract_proposal")
iteration2_graph_builder.add_edge("extract_proposal", "identify_retrieval_needs")
iteration2_graph_builder.add_edge("identify_retrieval_needs", "retrieve_internal_evidence")
iteration2_graph_builder.add_edge("retrieve_internal_evidence", "interpret_grounded_requirements")
iteration2_graph_builder.add_edge("interpret_grounded_requirements", "assess_readiness")
iteration2_graph_builder.add_edge("assess_readiness", "check_hard_blockers")
iteration2_graph_builder.add_edge("check_hard_blockers", "identify_material_caveats")
iteration2_graph_builder.add_edge("identify_material_caveats", "determine_readiness")
iteration2_graph_builder.add_edge("determine_readiness", "generate_leadership_questions")
iteration2_graph_builder.add_edge("generate_leadership_questions", "generate_recommended_actions")
iteration2_graph_builder.add_edge("generate_recommended_actions", "build_final_brief")
iteration2_graph_builder.add_edge("build_final_brief", END)

decisionready_iteration2 = iteration2_graph_builder.compile()


### 23. Canonical live end-to-end benchmark — Proposals A–D

These cells are the current Iteration-2 v0.5 live outputs. Each result persists the complete auditable state and final brief. Archived Iteration-1 cells above are excluded from this canonical path.


In [56]:
def print_iteration2_result(
    label: str,
    result: DecisionReadyIteration2State,
) -> None:
    print(f"========== CANONICAL ITERATION-2 v0.5 RESULT — PROPOSAL {label} ==========")
    print("READINESS STATUS:", result["readiness_status"])
    print("HARD BLOCKERS:", json.dumps(result.get("hard_blockers", []), indent=2))
    print("MATERIAL CAVEATS:", json.dumps(result.get("material_caveats", []), indent=2))
    print(
        "FUTURE DECISION CONSIDERATIONS:",
        json.dumps(result.get("future_decision_considerations", []), indent=2),
    )
    print(
        "GROUNDED REQUIREMENTS:",
        json.dumps(result.get("grounded_requirements", []), indent=2),
    )
    print("RETRIEVAL PROVENANCE:", json.dumps(result.get("retrieval_provenance", []), indent=2))
    print("EVIDENCE ITEMS [proposal fact]:", json.dumps(result.get("evidence_items", []), indent=2))
    print("STATED ASSUMPTIONS [proposal fact]:", json.dumps(result.get("stated_assumptions", []), indent=2))
    print("INFERRED ASSUMPTIONS [inferred state]:", json.dumps(result.get("inferred_assumptions", []), indent=2))
    print("INFERRED RISKS [inferred state]:", json.dumps(result.get("inferred_risks", []), indent=2))
    print("DIMENSION ASSESSMENTS [inferred state]:", json.dumps(result.get("dimension_assessments", {}), indent=2))
    print("LEADERSHIP QUESTIONS [model-generated question/action]:", json.dumps(result.get("leadership_questions", []), indent=2))
    print("RECOMMENDED ACTIONS [model-generated question/action]:", json.dumps(result.get("recommended_actions", []), indent=2))
    print("FINAL BRIEF:")
    print(result.get("final_brief", ""))
    print()


In [1]:
iteration2_result_A = decisionready_iteration2.invoke(
    {"proposal_text": test_proposal}
)
print_iteration2_result("A", iteration2_result_A)


========== CANONICAL ITERATION-2 v0.5 RESULT — PROPOSAL A ==========
READINESS STATUS: READY
HARD BLOCKERS: []
MATERIAL CAVEATS: []
FUTURE DECISION CONSIDERATIONS: [
  {
    "caveat_code": "ACKNOWLEDGED_NON_BLOCKING_UNCERTAINTY",
    "rationale": "The proposal explicitly identifies a later-stage leadership decision. It remains visible for continuity but does not affect readiness for the current requested decision.",
    "evidence": [
      "If successful, leadership will consider a broader rollout."
    ],
    "related_item_text": "If successful, leadership will consider a broader rollout.",
    "decision_horizon_relevance": "future_decision",
    "current_decision_materiality_evidence": [],
    "source": "deterministic"
  }
]
GROUNDED REQUIREMENTS: [
  {
    "requirement_text": "Leadership agreement to consider broader rollout if successful",
    "requirement_status": "unclear",
    "supporting_source_ids": [],
    "evidence_excerpt": [],
    "rationale": "The dependency concerns a la

In [2]:
iteration2_result_B = decisionready_iteration2.invoke(
    {"proposal_text": proposal_b}
)
print_iteration2_result("B", iteration2_result_B)


========== CANONICAL ITERATION-2 v0.5 RESULT — PROPOSAL B ==========
READINESS STATUS: NOT_READY
HARD BLOCKERS: [
  {
    "blocker_code": "MATERIAL_DIMENSION_THRESHOLD",
    "source": "deterministic",
    "rationale": "The existing readiness score threshold is met: at least one dimension scored 0 or at least two dimensions scored 1.",
    "evidence": [
      "evidence_and_assumptions: score 1 \u2014 Key material claims (20% productivity gain; readiness to roll out next quarter) lack supporting evidence, and important assumptions (integration timelines, region approvals, adoption) are unvalidated.",
      "execution_readiness: score 1 \u2014 Owner is named and there is an intention to start next quarter, but critical execution details (final population, sequencing, confirmed dependency owners/dates, measurement plan, and exit criteria) are missing, limiting ability to begin a rollout now.",
      "dependencies: score 1 \u2014 Dependencies are correctly identified, but their requirement 

In [3]:
iteration2_result_C = decisionready_iteration2.invoke(
    {"proposal_text": proposal_c}
)
print_iteration2_result("C", iteration2_result_C)


========== CANONICAL ITERATION-2 v0.5 RESULT — PROPOSAL C ==========
READINESS STATUS: NOT_READY
HARD BLOCKERS: [
  {
    "blocker_code": "UNRESOLVED_CRITICAL_DEPENDENCY",
    "source": "deterministic",
    "rationale": "Grounded authoritative evidence establishes this dependency as required, while the proposal still shows it as unresolved. Retrieved authoritative policy or guideline text explicitly supports this requirement. Analogous precedent is used only as corroboration.",
    "evidence": [
      "Approval Policy::chunk-02: Data Governance review required regardless of spend.",
      "Prior Decision Precedents::chunk-02: Data Governance review required before any identifiable data enrichment; anonymized aggregate market analysis permitted as an alternative.",
      "Risk / Data Governance Guidelines::chunk-01: Customer or employee data collected for one operational purpose must not be reused for a materially different purpose without documented legal/privacy basis and Data Governa

In [4]:
iteration2_result_D = decisionready_iteration2.invoke(
    {"proposal_text": proposal_d}
)
print_iteration2_result("D", iteration2_result_D)


========== CANONICAL ITERATION-2 v0.5 RESULT — PROPOSAL D ==========
READINESS STATUS: READY_WITH_CAVEATS
HARD BLOCKERS: []
MATERIAL CAVEATS: [
  {
    "caveat_code": "UNKNOWN_DEPENDENCY_REQUIREMENT",
    "rationale": "The feasibility and timelines for onboarding >=8 qualified suppliers within 4 weeks is marked unknown. Supplier onboarding is integral to achieving GMV, margins, and OTIF targets; if this dependency is harder than assumed the pilot may not generate interpretable evidence on customer demand or margins. Resolving whether supplier availability/pipeline exists or whether flexible timelines are acceptable would materially change pilot design (recruitment budget, staffing, contingency) but does not by itself prevent running a bounded test with mitigations.",
    "evidence": [
      "Dependency_reviews: \"Onboarding at least 8 qualified suppliers within 4 weeks\" has requirement_status = \"unknown\"",
      "Material_claim: \"It is feasible to onboard at least 8 qualified suppl

### 24. Proposal C evidence-to-status trace


In [5]:
proposal_c_grounded_trace = next(
    grounded
    for grounded in iteration2_result_C["grounded_requirements"]
    if grounded["requirement_status"] == "required"
)
trace_requirement_text = proposal_c_grounded_trace["requirement_text"]
proposal_c_original_uncertainty = next(
    item
    for item in iteration2_result_C["dependency_reviews"]
    if item["dependency_text"] == trace_requirement_text
)
proposal_c_retrieval_trace = next(
    result
    for result in iteration2_result_C["retrieved_internal_evidence"]
    if result["need"]["related_uncertainty"] == trace_requirement_text
)
proposal_c_blocker_trace = next(
    blocker
    for blocker in iteration2_result_C["hard_blockers"]
    if blocker["blocker_code"] == "UNRESOLVED_CRITICAL_DEPENDENCY"
    and blocker.get("related_item_text") == trace_requirement_text
)

authoritative_matches = [
    match
    for match in proposal_c_retrieval_trace["matches"]
    if match["source_type"] in {
        "authoritative_policy",
        "authoritative_guideline",
    }
]

print("ORIGINAL UNCERTAINTY:")
print(proposal_c_original_uncertainty)
print("\nRETRIEVAL QUERY:")
print(proposal_c_retrieval_trace["need"]["query"])
print("\nAUTHORITATIVE SOURCE RETRIEVED:")
for match in authoritative_matches:
    print(f"- {match['source_id']} | score={match['retrieval_score']:.4f}")
    print(match["chunk_text"])
print("\nGROUNDED REQUIREMENT STATUS:")
print(proposal_c_grounded_trace["requirement_status"])
print("Sources:", proposal_c_grounded_trace["supporting_source_ids"])
print("\nBLOCKER GENERATED:")
print(proposal_c_blocker_trace)
print("\nFINAL READINESS:")
print(iteration2_result_C["readiness_status"])


ORIGINAL UNCERTAINTY:
{'dependency_text': 'Data Governance review', 'requirement_status': 'unknown', 'evidence_basis': ['Proposal states Data Governance review has not yet been initiated but does not say whether it is required prior to pilot']}

RETRIEVAL QUERY:
Data Governance review policy requirements timing and controls for a pilot: Run a 6-week pilot enriching ~25,000 customer records with demographic, firmographic, and behavioral attributes from a proposed third-party data provider and use enriched profiles in commercial campaign targeting.

AUTHORITATIVE SOURCE RETRIEVED:
- Approval Policy::chunk-02 | score=0.2621
Not Ready

One or more material gaps prevent a meaningful decision or create unacceptable ambiguity/risk.

5. Hard Blockers

No identifiable decision ask.

No accountable owner.

No measurable success metric for a pilot, unless formally exempted.

A launch-critical dependency is unresolved and no mitigation or staged approach is defined.

A central economic/business cl

## Iteration-2 v0.5 — Auditability and Completeness Benchmark

The tables below are populated after the frozen-fixture regression and one clean live A–D benchmark. Canonical live outputs are the fully expanded cells immediately above.


### Canonical live and deterministic regression results

| Proposal | Target | Canonical live v0.5 | Frozen-fixture deterministic regression | Important change from v0.4 |
|---|---|---|---|---|
| A | `READY` | `READY` | `PASS` | Later broader rollout remains visible as a future consideration and does not affect current readiness. |
| B | `NOT_READY` | `NOT_READY` | `PASS` | The hard blocker still controls readiness; rollout-timing and seller-adoption concerns remain available as secondary caveats. |
| C | `NOT_READY` | `NOT_READY` | `PASS` | The policy-grounded Data Governance blocker remains decisive while provider matching remains visible as a distinct non-blocking concern. |
| D | `READY_WITH_CAVEATS` | `READY_WITH_CAVEATS` | `PASS` | Supplier-onboarding feasibility remains a caveat; stale evidence, demand limitations, hypotheses, and the future scale decision are preserved in synthesis. |

### Final-brief completeness matrix

| Proposal | Expected blocker coverage | Expected caveat coverage | Evidence limitations / assumptions | Future decision | Grounded requirement / citation coverage | Complete? |
|---|---|---|---|---|---|---|
| A | “No hard blockers” appears | No current material caveat, as expected | Measurement and data-control limitations appear with inferred-state provenance | Broader rollout appears as `future_decision` | Unclear/no-source later-rollout grounding is intentionally not appended; the future consideration carries the distinct synthesis purpose | Yes |
| B | `MATERIAL_DIMENSION_THRESHOLD` appears | Rollout timing and seller adoption remain secondary, non-blocking concerns | Baseline, sequencing, dependency, measurement, budget, and exit-criteria gaps appear | None, correctly | Three `unclear` no-source grounded items are not appended as a separate trace; dependency concerns remain in the narrative | Yes |
| C | `UNRESOLVED_CRITICAL_DEPENDENCY` and Data Governance appear | Provider match capability remains a non-blocking caveat; confirmed required concerns are deduplicated from caveats | Match-quality definition, provider evidence, budget/timeline support, and the 3.4%→4.2% commercial test appear | None required | Approval Policy and Risk/Data Governance source IDs, authority types, rank, score, and grounded `required` interpretation appear | Yes |
| D | No hard blockers appears | Supplier-onboarding feasibility appears without blocker promotion | 2023 evidence staleness, missing internal market/pricing benchmark, informal demand evidence, margin/GMV/OTIF hypotheses, and operating assumptions appear | Later category scaling appears and does not affect readiness | Two `unclear` no-source grounded items are omitted from the trace to avoid duplicating caveats/evidence gaps | Yes |

### Provenance conventions

- `proposal fact`: submitted evidence items, stated assumptions, and success metrics.
- `inferred state`: extracted/inferred assumptions, risks, dimension reasons, and gaps.
- `retrieved evidence`: compact query/source/type/rank/score plus grounded interpretation; full Northstar chunks remain local.
- `model-generated question/action`: leadership questions and recommended actions, explicitly separated from source evidence.

### Extraction variability observed

The canonical live v0.5 run preserved all target statuses. Compared with saved v0.4 outputs, the exact wording and count of semantic caveats changed: B surfaced two secondary concerns, C surfaced provider matching after blocker-aware caveat analysis, and D retained one primary supplier-onboarding caveat rather than two. Frozen fixtures now isolate future downstream regression from that extraction/model variability.

### Remaining limitations

- Frozen regression deterministically tests local retrieval/grounding, blocker precedence, horizon separation, status, provenance, and synthesis topic coverage. It does not make the model-generated prose itself deterministic.
- Completeness checks are topic-presence assertions, not semantic entailment scoring.
- Text-based links remain more brittle than stable claim/dependency IDs; introducing durable item IDs is a suitable Iteration-3 improvement.
- Model-generated questions/actions can still contain overly specific advice, but they are now explicitly provenance-labelled and cannot be mistaken for proposal facts or retrieved requirements.

Iteration 2 v0.5 is safe to freeze as the auditable experimental baseline before Iteration 3, subject to retaining the frozen fixtures and canonical persisted outputs.


## Frozen A–D Extraction and Downstream Fixtures

The extraction fields below are frozen from the single canonical live v0.5 run. Model-dependent downstream outputs are stored as an audit snapshot. The regression re-runs local retrieval and grounding, recomputes blocker-first readiness, validates decision-horizon separation, and checks final-brief completeness without calling an LLM.


In [ ]:
# Frozen from the single canonical v0.5 live extraction. These literals let
# downstream regression run without calling the extraction model.
FROZEN_EXTRACTION_FIXTURES = json.loads(r'''
{
  "A": {
    "proposal_text": "\nLeadership approval is requested for an 8-week pilot to test an AI-assisted\nsupplier negotiation preparation tool across two procurement categories.\n\nThe pilot will be led by the VP Procurement Transformation and will include\n12 procurement managers across approximately 30 negotiation events.\n\nCurrent preparation time averages 4.1 hours per negotiation. The pilot aims\nto reduce preparation time by at least 30% while maintaining or improving\ncommercial outcomes.\n\nThe proposed budget is $22,000.\n\nKey dependencies include access to historical negotiation notes and supplier\ndata. These data sources have been confirmed as available for the pilot.\n\nThe pilot will be considered successful if average preparation time is reduced\nby at least 30%, user adoption exceeds 70%, and no material data-control issues\nare identified.\n\nIf successful, leadership will consider a broader rollout. If the targets are\nnot achieved, the pilot will stop and the approach will be reassessed.\n",
    "proposal_type": "pilot",
    "decision_stage": "pilot",
    "decision_ask": "Approve an 8-week pilot of an AI-assisted supplier negotiation preparation tool across two procurement categories with a $22,000 budget.",
    "accountable_owner": "VP Procurement Transformation",
    "problem_summary": "Current negotiation preparation is time-consuming (average 4.1 hours); pilot seeks to reduce prep time while maintaining or improving commercial outcomes.",
    "proposed_intervention": "Run an 8-week pilot using an AI-assisted negotiation preparation tool with 12 procurement managers across ~30 negotiation events to reduce preparation time and assess adoption and data controls.",
    "evidence_items": [
      "Current preparation time averages 4.1 hours per negotiation.",
      "Pilot duration: 8 weeks.",
      "Participants: 12 procurement managers.",
      "Scope: approximately 30 negotiation events across two procurement categories.",
      "Target preparation time reduction: at least 30%.",
      "Proposed budget: $22,000.",
      "Required data: historical negotiation notes and supplier data.",
      "Those data sources have been confirmed as available for the pilot.",
      "Success criteria: >=30% reduction in average preparation time, user adoption >70%, and no material data-control issues."
    ],
    "material_claims": [
      {
        "claim_text": "The AI-assisted tool can reduce average preparation time by at least 30% while maintaining or improving commercial outcomes.",
        "claim_role": "hypothesis",
        "evidence_basis": [
          "The pilot aims to reduce preparation time by at least 30% while maintaining or improving commercial outcomes.",
          "Success criteria explicitly includes >=30% reduction in average preparation time."
        ]
      },
      {
        "claim_text": "Historical negotiation notes and supplier data required for the pilot are available.",
        "claim_role": "prerequisite",
        "evidence_basis": [
          "Key dependencies include access to historical negotiation notes and supplier data.",
          "These data sources have been confirmed as available for the pilot."
        ]
      },
      {
        "claim_text": "User adoption can exceed 70% during the 8-week pilot among the 12 procurement managers.",
        "claim_role": "hypothesis",
        "evidence_basis": [
          "Success criteria includes user adoption exceeds 70%."
        ]
      },
      {
        "claim_text": "If pilot targets are not achieved, stopping the pilot and reassessing the approach is the intended next step.",
        "claim_role": "unknown",
        "evidence_basis": [
          "Statement: If the targets are not achieved, the pilot will stop and the approach will be reassessed."
        ]
      }
    ],
    "dependency_reviews": [
      {
        "dependency_text": "Access to historical negotiation notes and supplier data",
        "requirement_status": "required",
        "evidence_basis": [
          "Key dependencies include access to historical negotiation notes and supplier data.",
          "These data sources have been confirmed as available for the pilot."
        ]
      },
      {
        "dependency_text": "Budget approval of $22,000",
        "requirement_status": "unknown",
        "evidence_basis": [
          "The proposed budget is $22,000."
        ]
      },
      {
        "dependency_text": "Leadership agreement to consider broader rollout if successful",
        "requirement_status": "unknown",
        "evidence_basis": [
          "If successful, leadership will consider a broader rollout."
        ]
      }
    ],
    "stated_assumptions": [
      "Historical negotiation notes and supplier data are sufficient and usable for the AI tool (implied by listing them as key dependencies and confirming availability).",
      "Commercial outcomes can be assessed and compared during the pilot period."
    ],
    "inferred_assumptions": [
      "The AI-assisted tool will be integrated into procurement managers' workflows for the duration of the pilot.",
      "Preparation time measurement methodology (to calculate average prep time and percentage reduction) is available or will be applied consistently."
    ],
    "success_metrics": [
      "Average preparation time reduced by at least 30% from 4.1 hours baseline.",
      "User adoption exceeds 70%.",
      "No material data-control issues identified."
    ],
    "dependencies": [
      "Access to historical negotiation notes",
      "Access to supplier data",
      "Approval/allocation of $22,000 budget",
      "Participation of 12 procurement managers across ~30 negotiation events"
    ],
    "stated_risks": [
      "Material data-control issues (explicitly a failure condition)."
    ],
    "inferred_risks": [
      "Insufficient data quality or formatting in historical negotiation notes and supplier data could prevent the AI tool from functioning effectively during the pilot.",
      "Low user engagement or operational disruption could prevent reaching >70% adoption within 8 weeks.",
      "Inconsistent or unreliable measurement of preparation time could make it impossible to determine whether a 30% reduction was achieved."
    ]
  },
  "B": {
    "proposal_text": "\nLeadership support is requested for rollout of an AI-powered Sales Productivity\nCopilot across multiple regions.\n\nThe solution is expected to improve seller productivity by approximately 20%\nthrough faster account research, meeting preparation, and follow-up support.\n\nThe initiative will be owned by the Commercial Transformation team.\n\nA formal current-state productivity baseline has not yet been established.\nThe exact rollout population and regional sequencing are still being finalized.\n\nEstimated implementation cost is between $60,000 and $90,000.\n\nKey dependencies include CRM integration, product-content connectivity,\nregional access approvals, and seller adoption. Some of these dependencies\nare still being worked through and do not yet have confirmed owners or dates.\n\nThe intention is to begin rollout during the coming quarter.\n\nSuccess measures are expected to include productivity improvement and seller\nadoption, although the measurement approach and baseline are still to be finalized.\n\nExit criteria or conditions for stopping, redesigning, or narrowing the rollout\nhave not yet been defined.\n",
    "proposal_type": "rollout",
    "decision_stage": "rollout",
    "decision_ask": "Leadership support for rollout of an AI-powered Sales Productivity Copilot across multiple regions",
    "accountable_owner": "Commercial Transformation team",
    "problem_summary": "Sellers need faster account research, meeting preparation, and follow-up support to improve productivity across regions.",
    "proposed_intervention": "Roll out an AI-powered Sales Productivity Copilot to sellers across multiple regions to accelerate account research, meeting prep, and follow-up, aiming to improve seller productivity by ~20%.",
    "evidence_items": [
      "Proposal states solution is expected to improve seller productivity by approximately 20% through faster account research, meeting preparation, and follow-up support.",
      "Initiative will be owned by the Commercial Transformation team.",
      "Estimated implementation cost is between $60,000 and $90,000.",
      "A formal current-state productivity baseline has not yet been established.",
      "Exact rollout population and regional sequencing are still being finalized.",
      "Key dependencies listed: CRM integration, product-content connectivity, regional access approvals, and seller adoption.",
      "Some dependencies do not yet have confirmed owners or dates.",
      "Intention to begin rollout during the coming quarter.",
      "Success measures expected to include productivity improvement and seller adoption, but measurement approach and baseline are to be finalized.",
      "Exit criteria or stopping/redesign/narrowing conditions have not yet been defined."
    ],
    "material_claims": [
      {
        "claim_text": "The solution will improve seller productivity by approximately 20%.",
        "claim_role": "hypothesis",
        "evidence_basis": [
          "Proposal states the solution is expected to improve seller productivity by approximately 20% through faster account research, meeting preparation, and follow-up support."
        ]
      },
      {
        "claim_text": "Commercial Transformation team will own the initiative.",
        "claim_role": "prerequisite",
        "evidence_basis": [
          "Proposal explicitly states the initiative will be owned by the Commercial Transformation team."
        ]
      },
      {
        "claim_text": "Rollout can begin during the coming quarter.",
        "claim_role": "unknown",
        "evidence_basis": [
          "Proposal expresses the intention to begin rollout during the coming quarter but notes rollout population and sequencing are still being finalized and dependencies unresolved."
        ]
      },
      {
        "claim_text": "Estimated implementation cost is between $60,000 and $90,000.",
        "claim_role": "unknown",
        "evidence_basis": [
          "Proposal provides the estimated cost range but does not state whether funding is secured or required approvals are in place."
        ]
      },
      {
        "claim_text": "Success will be measured by productivity improvement and seller adoption.",
        "claim_role": "unknown",
        "evidence_basis": [
          "Proposal states success measures are expected to include productivity improvement and seller adoption, but measurement approach and baseline are still to be finalized."
        ]
      }
    ],
    "dependency_reviews": [
      {
        "dependency_text": "CRM integration",
        "requirement_status": "unknown",
        "evidence_basis": [
          "Proposal lists CRM integration as a key dependency and notes some dependencies are still being worked through and lack confirmed owners or dates."
        ]
      },
      {
        "dependency_text": "Product-content connectivity",
        "requirement_status": "unknown",
        "evidence_basis": [
          "Proposal lists product-content connectivity as a key dependency and notes some dependencies are still being worked through and lack confirmed owners or dates."
        ]
      },
      {
        "dependency_text": "Regional access approvals",
        "requirement_status": "unknown",
        "evidence_basis": [
          "Proposal lists regional access approvals as a key dependency and notes some dependencies are still being worked through and lack confirmed owners or dates."
        ]
      },
      {
        "dependency_text": "Seller adoption",
        "requirement_status": "unknown",
        "evidence_basis": [
          "Proposal lists seller adoption as a key dependency; adoption is also referenced as a success measure but measurement approach is not finalized."
        ]
      },
      {
        "dependency_text": "Confirmed owners and dates for dependencies",
        "requirement_status": "unknown",
        "evidence_basis": [
          "Proposal states some dependencies do not yet have confirmed owners or dates."
        ]
      }
    ],
    "stated_assumptions": [
      "The solution will deliver faster account research, meeting preparation, and follow-up support (stated as expectation).",
      "Rollout population and regional sequencing can be finalized in time to begin rollout next quarter (implied by intention to begin rollout during coming quarter)."
    ],
    "inferred_assumptions": [
      "Adequate CRM and product-content technical connectivity can be implemented in the planned timeframe (implied by listing these as dependencies and intention to roll out next quarter).",
      "Regions will grant necessary access approvals in time to meet the proposed rollout schedule (implied by regional access approvals dependency and planned near-term rollout).",
      "Seller adoption will be sufficient to realize the projected productivity gains (implied by listing seller adoption as both a dependency and a success measure)."
    ],
    "success_metrics": [
      "Productivity improvement (target stated ~20% but baseline not yet established)",
      "Seller adoption (measurement approach TBD)"
    ],
    "dependencies": [
      "CRM integration",
      "Product-content connectivity",
      "Regional access approvals",
      "Seller adoption",
      "Confirmation of owners and dates for dependencies",
      "Finalized rollout population and regional sequencing",
      "Establishment of a current-state productivity baseline"
    ],
    "stated_risks": [
      "Some dependencies are still being worked through and do not yet have confirmed owners or dates.",
      "A formal current-state productivity baseline has not yet been established.",
      "Exact rollout population and regional sequencing are still being finalized.",
      "Measurement approach and baseline for success measures are still to be finalized.",
      "Exit criteria or conditions for stopping, redesigning, or narrowing the rollout have not yet been defined."
    ],
    "inferred_risks": [
      "If CRM or product-content integrations are delayed or fail, rollout timing and functionality could be materially impacted (implied by listing these integrations as key dependencies and noting owners/dates are unconfirmed).",
      "If regional access approvals are delayed or denied, some regions may be excluded or rollout schedule may slip (implied by regional access approvals dependency and unfinished sequencing).",
      "If seller adoption is lower than assumed, the projected ~20% productivity improvement may not be realized and investment ROI could be reduced (implied by seller adoption being both a dependency and a success metric).",
      "Absent a current-state baseline and defined measurement approach, the team may be unable to quantify productivity gains or determine whether to continue/stop the rollout (implied by baseline and measurement approach not established)."
    ]
  },
  "C": {
    "proposal_text": "\nLeadership approval is requested for a 6-week pilot to enrich identifiable\ncustomer profiles using third-party data in order to improve targeting and\nresponse rates for commercial campaigns.\n\nThe pilot will cover approximately 25,000 customer records and will be led by\nthe Director, Growth Marketing.\n\nThe current campaign response rate is 3.4%. The pilot aims to improve this to\nat least 4.2%.\n\nThe estimated pilot cost is $38,000.\n\nThe proposal assumes that existing customer terms and current data-use practices\npermit the use of third-party enrichment data for targeted offers.\n\nThe proposed data provider can supply demographic, firmographic, and behavioral\nattributes that would be matched to existing customer records.\n\nData Governance review has not yet been initiated.\n\nSuccess criteria include:\n- campaign response rate of at least 4.2%\n- acceptable match accuracy\n- no material increase in customer opt-outs or complaints\n\nThe pilot will run for 6 weeks. If the expected improvement is not observed,\nor if data-quality or customer-response issues are material, the pilot will stop\nand the approach will be reassessed.\n",
    "proposal_type": "pilot",
    "decision_stage": "pilot",
    "decision_ask": "Leadership approval for a 6-week pilot to enrich ~25,000 customer records with third-party data to improve targeting and increase campaign response rate from 3.4% to at least 4.2%",
    "accountable_owner": "Director, Growth Marketing",
    "problem_summary": "Current commercial campaign response rate is 3.4%; aim to increase response rates through third-party data enrichment of customer profiles to improve targeting.",
    "proposed_intervention": "Run a 6-week pilot enriching ~25,000 customer records with demographic, firmographic, and behavioral attributes from a proposed third-party data provider and use enriched profiles in commercial campaign targeting.",
    "evidence_items": [
      "Current campaign response rate is 3.4%",
      "Pilot covers approximately 25,000 customer records",
      "Pilot duration is 6 weeks",
      "Pilot estimated cost is $38,000",
      "Proposed data provider can supply demographic, firmographic, and behavioral attributes",
      "Success criteria include response rate >= 4.2%, acceptable match accuracy, and no material increase in opt-outs/complaints",
      "Proposal assumes existing customer terms and current data-use practices permit use of third-party enrichment data",
      "Data Governance review has not yet been initiated",
      "If expected improvement not observed or material data-quality/customer-response issues arise, pilot will stop and approach reassessed"
    ],
    "material_claims": [
      {
        "claim_text": "Enriching customer profiles with third-party demographic, firmographic, and behavioral attributes will improve campaign response rate from 3.4% to at least 4.2% for the pilot population.",
        "claim_role": "hypothesis",
        "evidence_basis": [
          "The pilot aims to improve response rate from 3.4% to at least 4.2% by enriching profiles and using them for targeting",
          "Success criteria explicitly lists campaign response rate of at least 4.2%"
        ]
      },
      {
        "claim_text": "The proposed data provider can supply usable demographic, firmographic, and behavioral attributes that can be matched to existing customer records.",
        "claim_role": "unknown",
        "evidence_basis": [
          "Proposal states the proposed data provider can supply these attributes",
          "No evidence provided about match rates, quality, or proven integration success"
        ]
      },
      {
        "claim_text": "Existing customer terms and current data-use practices permit the use of third-party enrichment data for targeted offers.",
        "claim_role": "unknown",
        "evidence_basis": [
          "Proposal explicitly states this as an assumption",
          "No supporting evidence or legal/data-governance confirmation is provided"
        ]
      }
    ],
    "dependency_reviews": [
      {
        "dependency_text": "Data Governance review",
        "requirement_status": "unknown",
        "evidence_basis": [
          "Proposal states Data Governance review has not yet been initiated but does not say whether it is required prior to pilot"
        ]
      },
      {
        "dependency_text": "Agreement or contract with the proposed third-party data provider",
        "requirement_status": "unknown",
        "evidence_basis": [
          "Proposal notes a proposed provider that can supply attributes but does not state contract status or whether execution requires an agreement before pilot"
        ]
      },
      {
        "dependency_text": "Confirmation that customer terms and data-use practices legally permit third-party enrichment",
        "requirement_status": "unknown",
        "evidence_basis": [
          "Proposal lists this as an assumption but provides no evidence or approval indicating it has been confirmed"
        ]
      },
      {
        "dependency_text": "Budget approval for estimated pilot cost of $38,000",
        "requirement_status": "unknown",
        "evidence_basis": [
          "Proposal provides an estimated pilot cost but does not state whether budget approval is required or already obtained"
        ]
      }
    ],
    "stated_assumptions": [
      "Existing customer terms and current data-use practices permit the use of third-party enrichment data for targeted offers",
      "Proposed data provider can supply demographic, firmographic, and behavioral attributes that would be matched to existing customer records"
    ],
    "inferred_assumptions": [
      "The enrichment process can be executed and integrated into current campaign targeting workflow within the 6-week pilot timeframe",
      "Match accuracy and data quality will be sufficient to evaluate impact during the pilot"
    ],
    "success_metrics": [
      "Campaign response rate of at least 4.2%",
      "Acceptable match accuracy (threshold not specified)",
      "No material increase in customer opt-outs or complaints"
    ],
    "dependencies": [
      "Data Governance review (not yet initiated)",
      "Access to third-party data from the proposed provider",
      "Integration of enriched attributes with existing customer records and campaign systems",
      "Budget to cover $38,000 estimated pilot cost",
      "Legal/compliance confirmation of permissive customer terms and data-use practices"
    ],
    "stated_risks": [
      "If expected improvement is not observed, or if data-quality or customer-response issues are material, the pilot will stop and the approach will be reassessed"
    ],
    "inferred_risks": [
      "Insufficient match rates or poor quality of third-party attributes could prevent achieving the targeted response rate and may invalidate pilot results",
      "If customer terms do not permit third-party enrichment, the pilot may be non-compliant and must be halted",
      "Delays or failures in Data Governance review or approvals could delay or block the pilot execution",
      "Integration or technical issues within the 6-week timeframe could prevent completion of enrichment and impact the ability to measure uplift"
    ]
  },
  "D": {
    "proposal_text": "\nLeadership approval is requested for a 10-week pilot to enter a new industrial\nfasteners sourcing category and test customer demand, supplier economics, and\nexecution feasibility.\n\nThe pilot will be led by the Head of Category Expansion.\n\nThe proposed budget is $70,000.\n\nThe business case references a 2023 industry report that estimated annual\nmarket growth of approximately 8-10% and noted fragmented pricing across the\ncategory.\n\nCurrent internal data does not provide a recent market-size or pricing benchmark\nfor this category. Customer demand has been discussed informally with several\naccount managers, but no structured customer validation has yet been completed.\n\nThe pilot aims to onboard at least 8 qualified suppliers within 4 weeks and\ngenerate at least:\n- 5 active buying customers\n- $250,000 GMV\n- gross margin of at least 6%\n- OTIF of at least 90%\n\nThe pilot will run for 10 weeks.\n\nIf the pilot achieves the commercial and execution thresholds, leadership will\nconsider scaling the category. If demand, margins, supplier availability, or\nservice levels are materially below target, the initiative will stop or be\nredesigned.\n",
    "proposal_type": "pilot",
    "decision_stage": "pilot",
    "decision_ask": "Leadership approval for a 10-week pilot to enter a new industrial fasteners sourcing category with a $70,000 budget",
    "accountable_owner": "Head of Category Expansion",
    "problem_summary": "Test customer demand, supplier economics, and execution feasibility for a new industrial fasteners sourcing category before scaling.",
    "proposed_intervention": "Run a 10-week pilot to onboard suppliers, validate customer demand, measure GMV and margins, and test operational service levels for the fasteners category.",
    "evidence_items": [
      "Proposed budget: $70,000",
      "Pilot duration: 10 weeks",
      "2023 industry report estimating annual market growth of approximately 8-10% and noting fragmented pricing across the category",
      "Current internal data lacks recent market-size or pricing benchmark for this category",
      "Customer demand has been discussed informally with several account managers but no structured customer validation completed"
    ],
    "material_claims": [
      {
        "claim_text": "There is sufficient customer demand to support the fasteners category (target: at least 5 active buying customers and $250,000 GMV during the pilot).",
        "claim_role": "hypothesis",
        "evidence_basis": [
          "Pilot aims to generate at least 5 active buying customers and $250,000 GMV",
          "No structured customer validation has yet been completed (pilot designed to test demand)"
        ]
      },
      {
        "claim_text": "Supplier economics will allow achieving a gross margin of at least 6%.",
        "claim_role": "hypothesis",
        "evidence_basis": [
          "Pilot target includes gross margin of at least 6%",
          "Proposal lacks recent internal pricing benchmarks, indicating this will be tested in the pilot"
        ]
      },
      {
        "claim_text": "Operational execution can meet OTIF of at least 90% while serving the category.",
        "claim_role": "hypothesis",
        "evidence_basis": [
          "Pilot target includes OTIF of at least 90%",
          "Pilot explicitly aims to test execution feasibility"
        ]
      },
      {
        "claim_text": "It is feasible to onboard at least 8 qualified suppliers within 4 weeks.",
        "claim_role": "hypothesis",
        "evidence_basis": [
          "Pilot aims to onboard at least 8 qualified suppliers within 4 weeks"
        ]
      },
      {
        "claim_text": "If pilot meets commercial and execution thresholds, leadership can consider scaling the category.",
        "claim_role": "unknown",
        "evidence_basis": [
          "Proposal states leadership will consider scaling if thresholds are achieved, but does not establish scaling as automatic or required"
        ]
      }
    ],
    "dependency_reviews": [
      {
        "dependency_text": "Budget approval of $70,000",
        "requirement_status": "required",
        "evidence_basis": [
          "Proposal requests leadership approval for a 10-week pilot with a proposed budget of $70,000"
        ]
      },
      {
        "dependency_text": "Onboarding at least 8 qualified suppliers within 4 weeks",
        "requirement_status": "unknown",
        "evidence_basis": [
          "Pilot aims to onboard 8 qualified suppliers within 4 weeks, but proposal does not state this as a necessary precondition for approval"
        ]
      },
      {
        "dependency_text": "Structured customer validation prior to scaling decision",
        "requirement_status": "unknown",
        "evidence_basis": [
          "Proposal indicates customer demand has not been validated and pilot is intended to test demand; it does not state whether a separate customer validation is required before approval"
        ]
      },
      {
        "dependency_text": "Achievement of commercial and execution thresholds (demand, margins, supplier availability, service levels) to consider scaling",
        "requirement_status": "not_required",
        "evidence_basis": [
          "Proposal states leadership will consider scaling if thresholds are achieved; scaling consideration is conditional after pilot, not a pre-approval requirement"
        ]
      }
    ],
    "stated_assumptions": [
      "The 2023 industry report's market growth and fragmentation findings are relevant to the current opportunity",
      "Informal account manager discussions reflect potential customer interest (but are not structured validation)"
    ],
    "inferred_assumptions": [
      "Sufficient internal capacity and processes exist to run a 10-week pilot (operations, sourcing, account management), since no operational constraints are mentioned",
      "The pilot budget ($70,000) is adequate to recruit suppliers, market the category to customers, and support operations for the 10-week period"
    ],
    "success_metrics": [
      "Onboard at least 8 qualified suppliers within 4 weeks",
      "At least 5 active buying customers during the pilot",
      "$250,000 GMV during the pilot",
      "Gross margin of at least 6%",
      "OTIF of at least 90%",
      "Pilot duration: 10 weeks"
    ],
    "dependencies": [
      "Leadership approval and budget allocation ($70,000)",
      "Supplier availability and willingness to onboard (>=8 suppliers in 4 weeks)",
      "Customer demand sufficient to reach 5 active buyers and $250,000 GMV",
      "Operational capability to meet OTIF >=90%"
    ],
    "stated_risks": [
      "If demand, margins, supplier availability, or service levels are materially below target, the initiative will stop or be redesigned"
    ],
    "inferred_risks": [
      "Pilot may fail to recruit 8 qualified suppliers within 4 weeks, preventing supplier-side validation and impacting GMV and margin targets (implied by the supplier onboarding target and lack of existing supplier data)",
      "Actual margins may be below 6% due to fragmented pricing and lack of recent internal pricing benchmarks, threatening commercial viability",
      "Customer demand may be lower than informal account manager discussions suggest, risking failure to reach 5 active buyers and $250,000 GMV",
      "Operational processes or capacity may not support achieving OTIF >=90% for a new category, risking poor service-level performance during the pilot"
    ]
  }
}
''')

# Model-dependent downstream outputs are also frozen as an audit snapshot.
# Deterministic tests re-run local retrieval/grounding and validate this snapshot.
FROZEN_DOWNSTREAM_SNAPSHOTS = json.loads(r'''
{
  "A": {
    "dimension_assessments": {
      "decision_clarity": {
        "score": 3,
        "reason": "The requested decision (approve an 8-week pilot, scope, owner, participants, budget, and next steps) is stated clearly and commitment is bounded.",
        "supporting_evidence": [
          "Decision ask: Approve an 8-week pilot of an AI-assisted supplier negotiation preparation tool across two procurement categories with a $22,000 budget.",
          "Accountable owner: VP Procurement Transformation.",
          "Scope and scale: 12 procurement managers across ~30 negotiation events.",
          "Next steps: If successful, leadership will consider broader rollout; if targets not achieved, pilot will stop and approach reassessed."
        ],
        "gaps": []
      },
      "problem_and_rationale": {
        "score": 3,
        "reason": "The proposal establishes a clear problem (4.1 hours prep time) and a specific, testable rationale for running a pilot to reduce prep time while maintaining commercial outcomes.",
        "supporting_evidence": [
          "Problem summary: Current negotiation preparation averages 4.1 hours per negotiation.",
          "Objective: Reduce preparation time by at least 30% while maintaining or improving commercial outcomes.",
          "Pilot scope and outcomes are designed to test that hypothesis across actual negotiation events."
        ],
        "gaps": []
      },
      "evidence_and_assumptions": {
        "score": 2,
        "reason": "Key assumptions and evidence (baseline prep time; data availability) are stated, but important measurement and data-quality assumptions are not fully specified.",
        "supporting_evidence": [
          "Baseline preparation time provided: 4.1 hours.",
          "Data dependency declared and confirmed: historical negotiation notes and supplier data are available for the pilot.",
          "Material claim that the tool can reduce prep time is explicitly treated as a hypothesis to be tested."
        ],
        "gaps": [
          "No explicit measurement plan describing how preparation time will be measured (self-report, tool logs, or observer) or how commercial outcomes will be assessed and compared.",
          "Data quality, format, or sufficiency assumptions are implied but not evidenced (availability confirmed but not usability or cleanliness).",
          "User-selection rationale for the 12 procurement managers (representativeness) is not described."
        ]
      },
      "business_and_economic_case": {
        "score": 3,
        "reason": "For a bounded, low-cost pilot the budget and exposure are clear and modest, success criteria map to useful learning, and the pilot is designed to inform a future rollout decision.",
        "supporting_evidence": [
          "Proposed budget: $22,000 (explicit and bounded).",
          "Pilot size: 12 managers and ~30 events — limited exposure.",
          "Success criteria are measurable and tied to learning needed for a rollout (time savings, adoption, data-control risks)."
        ],
        "gaps": []
      },
      "execution_readiness": {
        "score": 2,
        "reason": "Owner, duration, participants, and success metrics are defined, but some execution details required to start and run the pilot consistently are missing.",
        "supporting_evidence": [
          "Accountable owner and pilot duration provided: VP Procurement Transformation, 8 weeks.",
          "Participants and event count specified: 12 procurement managers across ~30 negotiation events.",
          "Success metrics defined: >=30% prep time reduction, >70% adoption, no material data-control issues."
        ],
        "gaps": [
          "No stated measurement methodology for preparation time or adoption rate (how adoption is counted and over what denominator/timeframe).",
          "No explicit plan for training, onboarding, or day-to-day operational responsibilities (who will run the tool, provide user support, monitor progress).",
          "No contingency plan if fewer than ~30 negotiation events occur during the pilot period."
        ]
      },
      "dependencies": {
        "score": 3,
        "reason": "Dependencies relevant to this pilot are identified and the critical data dependency is confirmed available; budget and participation are listed though budget approval status is not stated.",
        "supporting_evidence": [
          "Dependencies listed: access to historical negotiation notes and supplier data, budget, and participation of 12 managers.",
          "Data availability explicitly confirmed for the pilot.",
          "Dependency reviews mark data access as required and confirmed."
        ],
        "gaps": [
          "Budget approval status is listed as unknown in the extraction (the proposal states the proposed budget but does not explicitly state that funds are approved)."
        ]
      },
      "risk_and_governance": {
        "score": 2,
        "reason": "The proposal acknowledges the principal material risk (data-control issues) and sets a stop condition, but other execution and measurement risks are only inferred and not explicitly bounded or mitigated.",
        "supporting_evidence": [
          "Stated risk: material data-control issues (explicit failure condition).",
          "Stop condition: If targets not achieved, pilot will stop and approach reassessed.",
          "Success criteria include 'no material data-control issues' as a gate."
        ],
        "gaps": [
          "No explicit mitigation plan for data quality, format, or usability risks despite their operational importance for the AI tool.",
          "No defined threshold or process for what constitutes a 'material' data-control issue.",
          "No plan described for addressing low user engagement or inconsistent measurement that could invalidate results."
        ]
      }
    },
    "hard_blockers": [],
    "material_caveats": [],
    "future_decision_considerations": [
      {
        "caveat_code": "ACKNOWLEDGED_NON_BLOCKING_UNCERTAINTY",
        "rationale": "The proposal explicitly identifies a later-stage leadership decision. It remains visible for continuity but does not affect readiness for the current requested decision.",
        "evidence": [
          "If successful, leadership will consider a broader rollout."
        ],
        "related_item_text": "If successful, leadership will consider a broader rollout.",
        "decision_horizon_relevance": "future_decision",
        "current_decision_materiality_evidence": [],
        "source": "deterministic"
      }
    ],
    "grounded_requirements": [
      {
        "requirement_text": "Leadership agreement to consider broader rollout if successful",
        "requirement_status": "unclear",
        "supporting_source_ids": [],
        "evidence_excerpt": [],
        "rationale": "The dependency concerns a later scale or rollout decision; the retrieved text does not establish it as required for the current pilot."
      }
    ],
    "retrieval_provenance": [
      {
        "related_uncertainty": "Leadership agreement to consider broader rollout if successful",
        "retrieval_query": "Leadership agreement to consider broader rollout if successful policy requirements timing and controls for a pilot: Run an 8-week pilot using an AI-assisted negotiation preparation tool with 12 procurement managers across ~30 negotiation events to reduce preparation time and assess adoption and data controls.",
        "source_id": "Prior Decision Precedents::chunk-00",
        "source_type": "analogous_precedent",
        "rank": 1,
        "score": 0.2910786381976944,
        "grounded_interpretation": "unclear"
      },
      {
        "related_uncertainty": "Leadership agreement to consider broader rollout if successful",
        "retrieval_query": "Leadership agreement to consider broader rollout if successful policy requirements timing and controls for a pilot: Run an 8-week pilot using an AI-assisted negotiation preparation tool with 12 procurement managers across ~30 negotiation events to reduce preparation time and assess adoption and data controls.",
        "source_id": "Prior Decision Precedents::chunk-01",
        "source_type": "analogous_precedent",
        "rank": 2,
        "score": 0.28720805843995917,
        "grounded_interpretation": "unclear"
      },
      {
        "related_uncertainty": "Leadership agreement to consider broader rollout if successful",
        "retrieval_query": "Leadership agreement to consider broader rollout if successful policy requirements timing and controls for a pilot: Run an 8-week pilot using an AI-assisted negotiation preparation tool with 12 procurement managers across ~30 negotiation events to reduce preparation time and assess adoption and data controls.",
        "source_id": "Approval Policy::chunk-00",
        "source_type": "authoritative_policy",
        "rank": 3,
        "score": 0.211611559376693,
        "grounded_interpretation": "unclear"
      }
    ],
    "readiness_status": "READY",
    "leadership_questions": [
      "How exactly will preparation time and commercial outcomes be measured and attributed to the AI tool during the 8-week pilot (measurement definitions, data sources, baseline calculation, frequency of measurement, and how the team will control for confounding factors such as negotiation complexity or buyer experience)?",
      "What specific data-quality, privacy, or access constraints could prevent the AI from performing as intended (examples: missing fields, inconsistent note formats, redacted sensitive terms), and what minimum data pre-processing or manual intervention will be required before the pilot starts so the 8-week timeline and $22k budget remain realistic?",
      "What is the plan to drive and verify user adoption among the 12 procurement managers (onboarding, required usage thresholds, incentives, monitoring, and an escalation path if adoption falls below 70%), and which operational conditions would trigger pausing or modifying the pilot rather than continuing to the 8-week end?"
    ],
    "recommended_actions": [
      "Clarify exactly how preparation time and commercial outcomes will be measured during the pilot (definition of prep time, baseline calculation, measurement frequency, and which data sources will be used) so reviewers can interpret success metrics consistently.",
      "Confirm any minimal data pre-processing or formatting work already done or required (examples of missing fields or redactions that were found and whether manual cleaning is needed) to validate the 8-week timeline and $22k budget assumptions.",
      "State the approach to ensure and verify user adoption among the 12 managers (onboarding/check-in cadence, required usage threshold to count as ‘adopted’, and who will monitor adoption during the pilot)."
    ],
    "final_brief": "Decision Requested\nApprove an 8-week pilot of an AI-assisted supplier negotiation preparation tool with 12 procurement managers across ~30 negotiation events, budget $22,000. Accountable owner: VP Procurement Transformation.\n\nReadiness\nREADY\n\nWhy This Status\nThe proposal is bounded (duration, participants, scope, budget, owner) and confirms the critical data dependency. Success criteria are defined and proportionate for a small pilot. No hard blockers were identified.\n\nEvidence & Assumptions\n- Stated evidence (proposal_fact): baseline prep time = 4.1 hours; pilot length = 8 weeks; participants = 12 procurement managers; scope ≈30 negotiation events across two categories; target prep-time reduction ≥30%; budget = $22,000; required data = historical negotiation notes and supplier data (confirmed available); success criteria include ≥30% prep-time reduction, >70% user adoption, and no material data-control issues.\n- Stated assumptions (proposal_fact): the available historical notes and supplier data are sufficient/usable; commercial outcomes can be assessed during the pilot.\n- Inferred assumptions (inferred_state): the AI tool will be integrated into participants’ workflows for the pilot; a consistent measurement methodology for prep time will be applied.\n\nMaterial Gaps / Blockers\n- No hard blockers identified.\n- Material gaps (non-blocking but important):\n  - No explicit, documented measurement plan for preparation time and commercial outcomes (method, data sources, baseline calculation, controls for negotiation complexity or buyer experience).\n  - Data availability is confirmed but data quality/usability (format, cleanliness, missing fields) has not been evidenced.\n  - Budget approval status is not explicitly confirmed (proposal lists $22,000 but approval/allocation is unknown).\n  - No detailed plan for onboarding, day-to-day operations, adoption tracking, or contingency if fewer than ~30 negotiation events occur.\n\nDependencies & Risks\n- Confirmed/prerequisite dependencies: access to historical negotiation notes and supplier data (stated as available).\n- Other dependencies: approval/allocation of $22,000; participation of 12 managers across ~30 events.\n- Stated risk: material data-control issues (explicit stop condition).\n- Inferred risks: insufficient data quality or formatting; low user engagement preventing >70% adoption; inconsistent measurement invalidating comparisons.\n\nLeadership Challenge Questions\n(Extracted from model-generated list of high-impact uncertainties)\n1) How exactly will preparation time and commercial outcomes be measured and attributed to the AI tool during the 8-week pilot (definitions, data sources, baseline calculation, measurement frequency, and controls for confounders such as negotiation complexity or buyer experience)?\n2) What specific data-quality, privacy, or access constraints could prevent the AI from performing as intended (examples: missing fields, inconsistent note formats, redacted sensitive terms), and what minimal pre-processing or manual intervention is required before pilot start so the 8-week timeline and $22k budget remain realistic?\n3) What is the operational plan to drive and verify user adoption among the 12 procurement managers (onboarding, required usage thresholds to count as “adopted”, monitoring cadence, incentives, and escalation if adoption falls below 70%), and which conditions would trigger pausing or modifying the pilot early?\n\nRecommended Pre-review Actions\n(As provided in the case; intended to resolve material gaps before leadership sign-off)\n- Clarify and document the measurement plan for preparation time and commercial outcomes (define prep time, baseline, measurement frequency, data sources, and how confounders will be controlled).\n- Confirm any required data pre-processing/formatting work already completed or still needed, with impact on timeline and budget.\n- State the approach to ensure and verify user adoption (onboarding plan, monitoring owner, adoption thresholds, and escalation steps).\n\nProvenance & Traceability\n- Proposal facts sourced from: proposal_fact evidence items (listed in the case). Key items: baseline prep time (4.1h), pilot length (8 weeks), participants (12), scope (~30 events), budget ($22,000), data availability confirmation, success criteria.\n- Inferred state and risks labeled as inferred_state in the case (measurement methodology and operational/inferred risks).\n- Retrieved/analogue evidence and any prior-policy interpretation are cited in provenance (retrieved_evidence: Prior Decision Precedents::chunk-00/01; Approval Policy::chunk-00) and are interpreted in the case as unclear regarding later rollout commitments.\n\nBottom line for leadership reviewers: The proposal is READY for a decision to approve the pilot as presented because it is scoped, resourced, and confirms the primary data dependency with defined success criteria and no hard blockers. Before sign-off, reviewers should request the three recommended clarifications (measurement plan, data pre-processing confirmation, and adoption/onboarding plan) to ensure the pilot’s results will be interpretable and operationally achievable within the stated 8-week timeline and $22k budget.\n\nFuture-Decision Considerations (not current readiness caveats)\n- If successful, leadership will consider a broader rollout.: The proposal explicitly identifies a later-stage leadership decision. It remains visible for continuity but does not affect readiness for the current requested decision.\n\nStructured Evidence-Limitation Trace\n- [inferred state — evidence_and_assumptions gap] User-selection rationale for the 12 procurement managers (representativeness) is not described.\n\nMaterial Claims / Hypotheses Trace\n- [inferred state — unknown] If pilot targets are not achieved, stopping the pilot and reassessing the approach is the intended next step. | basis: Statement: If the targets are not achieved, the pilot will stop and the approach will be reassessed."
  },
  "B": {
    "dimension_assessments": {
      "decision_clarity": {
        "score": 2,
        "reason": "The requested decision (leadership support for a multi-region rollout) and accountable owner are stated, but scope, timing, and commitments are still imprecise which introduces caveats for a rollout-level decision.",
        "supporting_evidence": [
          "Decision ask: Leadership support for rollout of an AI-powered Sales Productivity Copilot across multiple regions.",
          "Accountable owner: Commercial Transformation team.",
          "Proposal expresses intention to begin rollout during the coming quarter."
        ],
        "gaps": [
          "Exact rollout population and regional sequencing are not defined.",
          "Unclear whether funding is secured or what specific approvals leadership support entails (approval of budget, go/no-go, or endorsement).",
          "No confirmed start date or commitable schedule given the unresolved dependencies."
        ]
      },
      "problem_and_rationale": {
        "score": 2,
        "reason": "The proposal states a clear problem (sellers need faster research, prep, follow-up) and a plausible rationale for an AI copilot, but the core productivity claim is a hypothesis without baseline evidence and sequencing to show this rollout is the appropriate next step.",
        "supporting_evidence": [
          "Problem summary: Sellers need faster account research, meeting preparation, and follow-up support.",
          "Proposed intervention: Roll out an AI-powered Sales Productivity Copilot to improve seller productivity by ~20%."
        ],
        "gaps": [
          "The 20% productivity improvement is presented as a hypothesis with no supporting baseline or prior pilot results.",
          "No articulation of why immediate multi-region rollout is preferable to a phased pilot to validate the hypothesis first."
        ]
      },
      "evidence_and_assumptions": {
        "score": 1,
        "reason": "Key material claims (20% productivity gain; readiness to roll out next quarter) lack supporting evidence, and important assumptions (integration timelines, region approvals, adoption) are unvalidated.",
        "supporting_evidence": [
          "Claim: expected ~20% productivity improvement (classified as hypothesis).",
          "A formal current-state productivity baseline has not yet been established.",
          "Some dependencies lack confirmed owners or dates."
        ],
        "gaps": [
          "No baseline or measurement approach exists to validate the productivity claim.",
          "Assumptions about ability to complete CRM/product-content integrations and obtain regional approvals in time are unproven.",
          "No prior pilot results or external evidence provided to support the 20% estimate."
        ]
      },
      "business_and_economic_case": {
        "score": 2,
        "reason": "Cost exposure is bounded and modest ($60k–$90k), and intended success metrics are identified; however funding status, expected value at rollout population scale, and how success will inform further investment are not finalized.",
        "supporting_evidence": [
          "Estimated implementation cost is between $60,000 and $90,000.",
          "Success measures expected to include productivity improvement and seller adoption."
        ],
        "gaps": [
          "Unclear whether the cost estimate is funded or requires approval.",
          "No clarification of expected value at the target rollout population or how the measured outcomes will feed into a larger investment decision.",
          "Baseline and measurement approach to quantify ROI are not defined."
        ]
      },
      "execution_readiness": {
        "score": 1,
        "reason": "Owner is named and there is an intention to start next quarter, but critical execution details (final population, sequencing, confirmed dependency owners/dates, measurement plan, and exit criteria) are missing, limiting ability to begin a rollout now.",
        "supporting_evidence": [
          "Initiative will be owned by the Commercial Transformation team.",
          "Intention to begin rollout during the coming quarter.",
          "Dependencies listed: CRM integration, product-content connectivity, regional access approvals, seller adoption."
        ],
        "gaps": [
          "Rollout population and regional sequencing not finalized.",
          "Confirmed owners and dates for key dependencies are missing.",
          "Measurement approach, baseline, and exit criteria are not defined.",
          "No detailed implementation plan or resource assignment beyond the owning team."
        ]
      },
      "dependencies": {
        "score": 1,
        "reason": "Dependencies are correctly identified, but their requirement status is unknown and some lack confirmed owners/dates — a material execution risk for a rollout decision.",
        "supporting_evidence": [
          "Listed dependencies: CRM integration, product-content connectivity, regional access approvals, seller adoption.",
          "Proposal notes some dependencies are still being worked through and do not yet have confirmed owners or dates."
        ],
        "gaps": [
          "Requirement_status for each listed dependency is unknown (no owners, timelines, or verification of feasibility).",
          "No mitigation plans or contingency approaches described if dependencies slip or fail."
        ]
      },
      "risk_and_governance": {
        "score": 1,
        "reason": "Proposal acknowledges several risks (missing baseline, unresolved dependencies, undefined exit criteria) but does not bound exposure or define governance for stopping, redesigning, or escalation — important for a rollout decision.",
        "supporting_evidence": [
          "Stated risks: lack of current-state productivity baseline; unresolved dependencies; measurement approach and exit criteria not defined.",
          "Exit criteria or conditions for stopping/redesign/narrowing the rollout have not yet been defined."
        ],
        "gaps": [
          "No defined exit criteria or governance triggers to halt or redesign the rollout.",
          "No explicit plan describing how measured outcomes will be reviewed and who will decide next steps.",
          "Unbounded exposure if productivity gains are not realized due to low adoption or integration failures."
        ]
      }
    },
    "hard_blockers": [
      {
        "blocker_code": "MATERIAL_DIMENSION_THRESHOLD",
        "source": "deterministic",
        "rationale": "The existing readiness score threshold is met: at least one dimension scored 0 or at least two dimensions scored 1.",
        "evidence": [
          "evidence_and_assumptions: score 1 — Key material claims (20% productivity gain; readiness to roll out next quarter) lack supporting evidence, and important assumptions (integration timelines, region approvals, adoption) are unvalidated.",
          "execution_readiness: score 1 — Owner is named and there is an intention to start next quarter, but critical execution details (final population, sequencing, confirmed dependency owners/dates, measurement plan, and exit criteria) are missing, limiting ability to begin a rollout now.",
          "dependencies: score 1 — Dependencies are correctly identified, but their requirement status is unknown and some lack confirmed owners/dates — a material execution risk for a rollout decision.",
          "risk_and_governance: score 1 — Proposal acknowledges several risks (missing baseline, unresolved dependencies, undefined exit criteria) but does not bound exposure or define governance for stopping, redesigning, or escalation — important for a rollout decision."
        ],
        "related_item_text": null
      }
    ],
    "material_caveats": [
      {
        "caveat_code": "UNKNOWN_MATERIAL_CLAIM",
        "rationale": "The proposal’s timing claim (that rollout can begin during the coming quarter) is unverified and materially affects whether leadership can responsibly approve a multi-region rollout now. If integrations, regional approvals, or dependency owners are not secured in time, the planned start and sequencing will change, affecting exposure and the ability to collect interpretable rollout evidence. This is not a hard blocker because rollout could be delayed or sequenced, but leadership should acknowledge the timing uncertainty and condition support on confirmed dependency owners/timelines or a phased start.",
        "evidence": [
          "Proposal expresses the intention to begin rollout during the coming quarter but notes rollout population and sequencing are still being finalized and dependencies unresolved.",
          "Dependency reviews show CRM integration, product-content connectivity, and regional access approvals have requirement_status = unknown and some dependencies lack confirmed owners or dates."
        ],
        "related_item_text": "Rollout can begin during the coming quarter.",
        "decision_horizon_relevance": "current_decision",
        "current_decision_materiality_evidence": [
          "Proposal intention to start next quarter is stated while exact rollout population and sequencing are still being finalized and dependencies unresolved, creating risk to the proposed start and to producing interpretable rollout outcomes."
        ],
        "source": "semantic"
      },
      {
        "caveat_code": "UNKNOWN_DEPENDENCY_REQUIREMENT",
        "rationale": "Seller adoption is an unresolved dependency and is also a stated success measure. Its requirement status is unknown (no adoption plan, targets, or confirmed owners), which materially affects whether the rollout can deliver or demonstrate the claimed ~20% productivity improvement. This is non-blocking if leadership conditions rollout on a defined adoption plan, phased enrollment, or guardrails for interpreting adoption-driven outcomes.",
        "evidence": [
          "Proposal lists seller adoption as a key dependency; adoption is also referenced as a success measure but measurement approach is not finalized.",
          "Dependency_reviews entry: \"Seller adoption\" has requirement_status = unknown and evidence notes measurement approach and baseline are not finalized."
        ],
        "related_item_text": "Seller adoption",
        "decision_horizon_relevance": "current_decision",
        "current_decision_materiality_evidence": [
          "Because seller adoption is both a dependency and the primary mechanism for producing productivity gains, unknown readiness on adoption planning affects whether early rollout will produce interpretable evidence of impact and whether expected benefits are achievable."
        ],
        "source": "semantic"
      }
    ],
    "future_decision_considerations": [],
    "grounded_requirements": [
      {
        "requirement_text": "CRM integration",
        "requirement_status": "unclear",
        "supporting_source_ids": [],
        "evidence_excerpt": [],
        "rationale": "Retrieved authoritative text is absent, conflicting, or does not explicitly establish this requirement. Precedent alone is insufficient."
      },
      {
        "requirement_text": "Product-content connectivity",
        "requirement_status": "unclear",
        "supporting_source_ids": [],
        "evidence_excerpt": [],
        "rationale": "Retrieved authoritative text is absent, conflicting, or does not explicitly establish this requirement. Precedent alone is insufficient."
      },
      {
        "requirement_text": "Regional access approvals",
        "requirement_status": "unclear",
        "supporting_source_ids": [],
        "evidence_excerpt": [],
        "rationale": "Retrieved authoritative text is absent, conflicting, or does not explicitly establish this requirement. Precedent alone is insufficient."
      }
    ],
    "retrieval_provenance": [
      {
        "related_uncertainty": "CRM integration",
        "retrieval_query": "CRM integration policy requirements timing and controls for a rollout: Roll out an AI-powered Sales Productivity Copilot to sellers across multiple regions to accelerate account research, meeting prep, and follow-up, aiming to improve seller productivity by ~20%.",
        "source_id": "Prior Decision Precedents::chunk-01",
        "source_type": "analogous_precedent",
        "rank": 1,
        "score": 0.30149264086974226,
        "grounded_interpretation": "unclear"
      },
      {
        "related_uncertainty": "CRM integration",
        "retrieval_query": "CRM integration policy requirements timing and controls for a rollout: Roll out an AI-powered Sales Productivity Copilot to sellers across multiple regions to accelerate account research, meeting prep, and follow-up, aiming to improve seller productivity by ~20%.",
        "source_id": "Prior Decision Precedents::chunk-00",
        "source_type": "analogous_precedent",
        "rank": 2,
        "score": 0.18077014490591334,
        "grounded_interpretation": "unclear"
      },
      {
        "related_uncertainty": "CRM integration",
        "retrieval_query": "CRM integration policy requirements timing and controls for a rollout: Roll out an AI-powered Sales Productivity Copilot to sellers across multiple regions to accelerate account research, meeting prep, and follow-up, aiming to improve seller productivity by ~20%.",
        "source_id": "Risk / Data Governance Guidelines::chunk-01",
        "source_type": "authoritative_guideline",
        "rank": 3,
        "score": 0.13435918707571914,
        "grounded_interpretation": "unclear"
      },
      {
        "related_uncertainty": "Product-content connectivity",
        "retrieval_query": "Product-content connectivity policy requirements timing and controls for a rollout: Roll out an AI-powered Sales Productivity Copilot to sellers across multiple regions to accelerate account research, meeting prep, and follow-up, aiming to improve seller productivity by ~20%.",
        "source_id": "Prior Decision Precedents::chunk-01",
        "source_type": "analogous_precedent",
        "rank": 1,
        "score": 0.2403982676709504,
        "grounded_interpretation": "unclear"
      },
      {
        "related_uncertainty": "Product-content connectivity",
        "retrieval_query": "Product-content connectivity policy requirements timing and controls for a rollout: Roll out an AI-powered Sales Productivity Copilot to sellers across multiple regions to accelerate account research, meeting prep, and follow-up, aiming to improve seller productivity by ~20%.",
        "source_id": "Prior Decision Precedents::chunk-00",
        "source_type": "analogous_precedent",
        "rank": 2,
        "score": 0.19262127250303335,
        "grounded_interpretation": "unclear"
      },
      {
        "related_uncertainty": "Product-content connectivity",
        "retrieval_query": "Product-content connectivity policy requirements timing and controls for a rollout: Roll out an AI-powered Sales Productivity Copilot to sellers across multiple regions to accelerate account research, meeting prep, and follow-up, aiming to improve seller productivity by ~20%.",
        "source_id": "Risk / Data Governance Guidelines::chunk-01",
        "source_type": "authoritative_guideline",
        "rank": 3,
        "score": 0.143167654152561,
        "grounded_interpretation": "unclear"
      },
      {
        "related_uncertainty": "Regional access approvals",
        "retrieval_query": "Regional access approvals policy requirements timing and controls for a rollout: Roll out an AI-powered Sales Productivity Copilot to sellers across multiple regions to accelerate account research, meeting prep, and follow-up, aiming to improve seller productivity by ~20%.",
        "source_id": "Prior Decision Precedents::chunk-01",
        "source_type": "analogous_precedent",
        "rank": 1,
        "score": 0.2315741808457512,
        "grounded_interpretation": "unclear"
      },
      {
        "related_uncertainty": "Regional access approvals",
        "retrieval_query": "Regional access approvals policy requirements timing and controls for a rollout: Roll out an AI-powered Sales Productivity Copilot to sellers across multiple regions to accelerate account research, meeting prep, and follow-up, aiming to improve seller productivity by ~20%.",
        "source_id": "Prior Decision Precedents::chunk-00",
        "source_type": "analogous_precedent",
        "rank": 2,
        "score": 0.18555089362961472,
        "grounded_interpretation": "unclear"
      },
      {
        "related_uncertainty": "Regional access approvals",
        "retrieval_query": "Regional access approvals policy requirements timing and controls for a rollout: Roll out an AI-powered Sales Productivity Copilot to sellers across multiple regions to accelerate account research, meeting prep, and follow-up, aiming to improve seller productivity by ~20%.",
        "source_id": "Risk / Data Governance Guidelines::chunk-01",
        "source_type": "authoritative_guideline",
        "rank": 3,
        "score": 0.13791252555682798,
        "grounded_interpretation": "unclear"
      }
    ],
    "readiness_status": "NOT_READY",
    "leadership_questions": [
      "Which dependencies must be confirmed (owner, deliverable, and firm date) before we start any multi‑region rollout, and can the team provide a realistic plan showing those confirmations by the start of next quarter or a proposed phased start that limits exposure until dependencies are secured?",
      "How will the team establish a current‑state productivity baseline and measurable success criteria (including sample size, metrics, and timeline) so an early rollout can produce interpretable evidence that the Copilot delivers the claimed ~20% productivity gain?",
      "What is the adoption plan (target adoption rates, onboarding/training resources, incentives, and ownership) and what minimum adoption threshold will leadership require to consider productivity measurements credible before expanding beyond initial regions?",
      "What are the predefined exit and governance criteria (conditions to stop, redesign, or narrow the rollout, and who has authority to make those calls) that will limit downside if integrations, approvals, or adoption fall short during the pilot phase?"
    ],
    "recommended_actions": [
      "Confirm owners, required deliverables, and firm delivery dates for the three highest-risk dependencies (CRM integration, regional access approvals, product-content connectivity) and attach those confirmations to the case before leadership review so timing exposure is explicit, or propose a phased start that limits rollout to regions with confirmed dependencies.",
      "Define how the current-state productivity baseline will be measured for the pilot (metrics, sample size or pilot population, measurement period) and state the minimum data quality or sample threshold needed to interpret early productivity results.",
      "Provide a concrete adoption plan for the initial rollout (owner, target adoption rate by X weeks, onboarding/training resources, and simple acceptance threshold) and state the minimum adoption level leadership should require before expanding beyond the pilot.",
      "Specify exit and governance criteria for the pilot (conditions to stop, narrow, or redesign the rollout and who has authority to act) so leadership can understand and limit downside exposure during the early rollout phase."
    ],
    "final_brief": "Decision Requested\nLeadership support for rollout of an AI-powered Sales Productivity Copilot across multiple regions (accountable owner: Commercial Transformation team).\n\nReadiness\nNOT_READY\n\nWhy This Status\nA deterministic hard blocker (MATERIAL_DIMENSION_THRESHOLD) was triggered because multiple material dimensions scored poorly (scores of 1): evidence & assumptions, execution readiness, dependencies, and risk & governance. These gaps leave the proposal insufficiently prepared for a multi-region rollout decision.\n\nEvidence & Assumptions\nStated evidence (proposal facts):\n- Solution expected to improve seller productivity by ~20% (hypothesis).\n- Commercial Transformation team will own the initiative.\n- Estimated implementation cost $60k–$90k.\n- Intention to begin rollout next quarter; rollout population and sequencing are not finalized.\n- Key dependencies listed: CRM integration, product-content connectivity, regional access approvals, seller adoption.\n- Some dependencies lack confirmed owners/dates.\n- No current-state productivity baseline, measurement approach, or exit criteria defined.\n\nInferred assumptions (labeled as such):\n- CRM/product-content integrations can be completed in the planned timeframe.\n- Regions will grant necessary access approvals on schedule.\n- Seller adoption will be sufficient to realize the projected ~20% productivity gain.\n\nMaterial Gaps / Blockers\nConfirmed hard blocker:\n- MATERIAL_DIMENSION_THRESHOLD: Multiple critical readiness dimensions scored 1, creating a material execution readiness barrier to approving a multi-region rollout now. Specific missing elements cited in evidence_and_assumptions, execution_readiness, dependencies, and risk_and_governance.\n\nKey material gaps (non-blocking but material):\n- No current-state productivity baseline or finalized measurement approach to validate the 20% claim.\n- Rollout population and regional sequencing not defined.\n- Several key dependencies (CRM integration, product-content connectivity, regional access approvals, seller adoption) have unknown requirement/status and lack confirmed owners/dates.\n- Exit criteria and governance triggers to stop, redesign, or narrow the rollout are undefined.\n\nDependencies & Risks\nDependencies (stated): CRM integration; product-content connectivity; regional access approvals; seller adoption; confirmation of owners/dates; finalized rollout population/sequencing; establishment of current-state productivity baseline.\n\nStated risks:\n- Unconfirmed dependency owners/dates delaying schedule.\n- No baseline to measure productivity impact.\n- Undefined measurement approach and exit criteria.\n\nInferred risks:\n- Integration or approval delays may prevent rollout or produce uninterpretable results.\n- Low seller adoption could prevent achieving or measuring the claimed productivity gains.\n- Without baseline and governance, leadership cannot bound downside or make informed go/no-go decisions.\n\nLeadership Challenge Questions\n(Provided as-is from the case)\n1) Which dependencies must be confirmed (owner, deliverable, and firm date) before we start any multi-region rollout, and can the team provide a realistic plan showing those confirmations by the start of next quarter or a proposed phased start that limits exposure until dependencies are secured?\n2) How will the team establish a current-state productivity baseline and measurable success criteria (including sample size, metrics, and timeline) so an early rollout can produce interpretable evidence that the Copilot delivers the claimed ~20% productivity gain?\n3) What is the adoption plan (target adoption rates, onboarding/training resources, incentives, and ownership) and what minimum adoption threshold will leadership require to consider productivity measurements credible before expanding beyond initial regions?\n4) What are the predefined exit and governance criteria (conditions to stop, redesign, or narrow the rollout, and who has authority to make those calls) that will limit downside if integrations, approvals, or adoption fall short during the pilot phase?\n\nRecommended Pre-review Actions\n(Provided as-is from the case)\n- Confirm owners, required deliverables, and firm delivery dates for the three highest-risk dependencies (CRM integration, regional access approvals, product-content connectivity) and attach those confirmations to the case before leadership review, or propose a phased start limited to regions with confirmed dependencies.\n- Define how the current-state productivity baseline will be measured for the pilot (metrics, sample size/pilot population, measurement period) and state the minimum data quality/sample threshold needed to interpret early productivity results.\n- Provide a concrete adoption plan for the initial rollout (owner, target adoption rate by X weeks, onboarding/training resources, and acceptance threshold) and state the minimum adoption level leadership should require before expanding beyond the pilot.\n- Specify exit and governance criteria for the pilot (conditions to stop, narrow, or redesign the rollout and who has authority to act).\n\nProvenance & Traceability\n- Proposal facts: items listed under \"proposal_fact\" in provenance (e.g., 20% productivity expectation; owner = Commercial Transformation; cost estimate; dependencies; lack of baseline/exit criteria).\n- Inferred state: assumptions and inferred risks are labeled as model inferences (\"inferred_state\").\n- Retrieved evidence: dependency requirement searches returned unclear/insufficient authoritative retrievals (see source IDs: Prior Decision Precedents::chunk-01, chunk-00; Risk / Data Governance Guidelines::chunk-01) summarized above.\n\nFinal note: There are no other hard blockers beyond the stated MATERIAL_DIMENSION_THRESHOLD. The brief preserves the NOT_READY status exactly as provided and surfaces the primary material concerns leadership will need addressed before considering approval for a multi-region rollout.\n\nStructured Evidence-Limitation Trace\n- [inferred state — dependencies gap] No mitigation plans or contingency approaches described if dependencies slip or fail.\n- [inferred state — risk_and_governance gap] No explicit plan describing how measured outcomes will be reviewed and who will decide next steps."
  },
  "C": {
    "dimension_assessments": {
      "decision_clarity": {
        "score": 3,
        "reason": "The requested decision is clearly stated (approval for a 6-week pilot), scope and owner are defined, commitment (cost, duration, records) is specified, and next-step stopping conditions are described.",
        "supporting_evidence": [
          "Decision ask: Leadership approval for a 6-week pilot to enrich ~25,000 customer records",
          "Accountable owner: Director, Growth Marketing",
          "Estimated pilot cost: $38,000",
          "Pilot duration and stop/reassess condition specified"
        ],
        "gaps": []
      },
      "problem_and_rationale": {
        "score": 3,
        "reason": "The proposal states a clear, testable problem (low campaign response rate) and a plausible rationale linking enrichment to improved targeting; the pilot's objective and measurable target (3.4% -> 4.2%) are explicit.",
        "supporting_evidence": [
          "Problem summary: current campaign response rate is 3.4%",
          "Proposed intervention: enrich profiles to improve targeting",
          "Success metric: increase response rate to at least 4.2%"
        ],
        "gaps": []
      },
      "evidence_and_assumptions": {
        "score": 2,
        "reason": "Key claims are framed as testable hypotheses appropriate for a pilot, but critical assumptions (data provider match quality, permissibility under customer terms) lack supporting evidence and remain unknown.",
        "supporting_evidence": [
          "Material claim labeled as hypothesis that enrichment will raise response to >=4.2%",
          "Proposal states the provider can supply required attributes",
          "Proposal explicitly notes the assumption that customer terms permit enrichment"
        ],
        "gaps": [
          "No evidence on expected or historical match rates or data quality from the proposed provider",
          "No confirmation that customer terms or data-use practices permit the enrichment (assumption only)",
          "Unknown whether Data Governance or legal sign-off is required before pilot"
        ]
      },
      "business_and_economic_case": {
        "score": 2,
        "reason": "Cost and exposure are clearly stated and bounded for a pilot ($38k, 25k records, 6 weeks). Expected benefit is quantified as a target uplift, but the proposal lacks an explicit articulation of how pilot results will inform a subsequent investment decision or the minimal detectable effect given sample size.",
        "supporting_evidence": [
          "Estimated pilot cost: $38,000",
          "Pilot population size: ~25,000 records",
          "Clear success criterion: response >= 4.2%"
        ],
        "gaps": [
          "No power/sample calculation or explanation that 25,000 is sufficient to detect the cited uplift reliably",
          "No stated plan for how learnings will convert to a scaled business case or thresholds for go/no-go beyond the single response-rate target"
        ]
      },
      "execution_readiness": {
        "score": 2,
        "reason": "Owner, duration, population, and success criteria are defined, and a stop condition is stated. However, several execution requirements are unvalidated (Data Governance review not initiated; integration/operational steps and responsibilities for matching and campaign deployment are not detailed).",
        "supporting_evidence": [
          "Accountable owner named (Director, Growth Marketing)",
          "Pilot duration: 6 weeks; population: ~25,000",
          "Success criteria and stop condition specified"
        ],
        "gaps": [
          "No plan or timeline for running Data Governance review or clarifying whether it's required before starting",
          "No details on who will perform matching, the matching process, integration with campaign systems, or resource/time estimates",
          "No definition of 'acceptable match accuracy' threshold or how it will be measured"
        ]
      },
      "dependencies": {
        "score": 1,
        "reason": "Several dependencies are identified but their requirement status is unknown and some are potentially material (Data Governance, contract with data provider, confirmation of permissive customer terms, budget approval). Leadership lacks clarity on whether these must be resolved before launch.",
        "supporting_evidence": [
          "Dependency list includes Data Governance review, access to third-party data, integration, budget, and legal/compliance confirmation",
          "Proposal states Data Governance review has not been initiated"
        ],
        "gaps": [
          "Unknown whether Data Governance approval is required prior to pilot start and the timeline to obtain it",
          "Contract or agreement status with the proposed data provider is unspecified",
          "No confirmation that customer terms legally permit enrichment or that legal/compliance sign-off is obtained",
          "Budget approval status for the $38,000 estimate is unknown"
        ]
      },
      "risk_and_governance": {
        "score": 2,
        "reason": "The proposal acknowledges primary risks (stop conditions tied to poor results or customer response). However, governance-related risks (Data Governance not initiated, legal/compliance uncertainty) are material and unaddressed, and thresholds for 'material' customer-response issues are unspecified.",
        "supporting_evidence": [
          "Stated risk: pilot will stop if expected improvement not observed or material data-quality/customer-response issues arise",
          "Data Governance review has not yet been initiated",
          "Assumption that customer terms permit third-party enrichment"
        ],
        "gaps": [
          "No definition of what constitutes a 'material' increase in opt-outs or complaints",
          "No evidence that Data Governance or legal requirements have been reviewed or will be completed within the pilot timeline",
          "No mitigation plan if match accuracy or data quality is inadequate"
        ]
      }
    },
    "hard_blockers": [
      {
        "blocker_code": "UNRESOLVED_CRITICAL_DEPENDENCY",
        "source": "deterministic",
        "rationale": "Grounded authoritative evidence establishes this dependency as required, while the proposal still shows it as unresolved. Retrieved authoritative policy or guideline text explicitly supports this requirement. Analogous precedent is used only as corroboration.",
        "evidence": [
          "Approval Policy::chunk-02: Data Governance review required regardless of spend.",
          "Prior Decision Precedents::chunk-02: Data Governance review required before any identifiable data enrichment; anonymized aggregate market analysis permitted as an alternative.",
          "Risk / Data Governance Guidelines::chunk-01: Customer or employee data collected for one operational purpose must not be reused for a materially different purpose without documented legal/privacy basis and Data Governance approval. Commercial attractiveness or operational convenience does not by itself constitute approval. 3. Customer Data Enrichment External enrichment of identifiable customer records is permitted only when the source is approved, the intended use is documented, and Data Governance confirms that consent, notice, contractual terms, and retention controls are adequate."
        ],
        "related_item_text": "Data Governance review"
      }
    ],
    "material_caveats": [
      {
        "caveat_code": "UNKNOWN_MATERIAL_CLAIM",
        "rationale": "Whether the proposed data provider can supply usable attributes and achieve adequate match rates is central to whether the pilot can be executed as designed and whether any observed uplift would be attributable to enrichment. The proposal asserts capability but provides no data on match rates, data completeness, or past integration success; this uncertainty is material to pilot validity yet is not presented as a hard blocker (the pilot could proceed with mitigations or early stopping).",
        "evidence": [
          "Proposal states the proposed data provider can supply demographic, firmographic, and behavioral attributes",
          "No evidence provided about match rates, quality, or proven integration success",
          "Dimension assessments note gaps: 'No evidence on expected or historical match rates or data quality from the proposed provider'; 'No details on who will perform matching, the matching process, integration with campaign systems, or resource/time estimates'; 'No definition of \"acceptable match accuracy\" threshold or how it will be measured'"
        ],
        "related_item_text": "The proposed data provider can supply usable demographic, firmographic, and behavioral attributes that can be matched to existing customer records.",
        "decision_horizon_relevance": "current_decision",
        "current_decision_materiality_evidence": [
          "Execution_readiness gaps: matching process, integration, and measurement approach are unspecified and affect ability to run the pilot within 6 weeks",
          "Evidence_and_assumptions gaps: absence of supporting evidence on match quality undermines confidence that the pilot can produce interpretable learning about response-rate uplift",
          "Inferred risks: 'Insufficient match rates or poor quality of third-party attributes could prevent achieving the targeted response rate and may invalidate pilot results'"
        ],
        "source": "semantic"
      }
    ],
    "future_decision_considerations": [],
    "grounded_requirements": [
      {
        "requirement_text": "Data Governance review",
        "requirement_status": "required",
        "supporting_source_ids": [
          "Approval Policy::chunk-02",
          "Prior Decision Precedents::chunk-02"
        ],
        "evidence_excerpt": [
          "Data Governance review required regardless of spend.",
          "Data Governance review required before any identifiable data enrichment; anonymized aggregate market analysis permitted as an alternative."
        ],
        "rationale": "Retrieved authoritative policy or guideline text explicitly supports this requirement. Analogous precedent is used only as corroboration."
      },
      {
        "requirement_text": "Agreement or contract with the proposed third-party data provider",
        "requirement_status": "unclear",
        "supporting_source_ids": [],
        "evidence_excerpt": [],
        "rationale": "Retrieved authoritative text is absent, conflicting, or does not explicitly establish this requirement. Precedent alone is insufficient."
      },
      {
        "requirement_text": "Confirmation that customer terms and data-use practices legally permit third-party enrichment",
        "requirement_status": "required",
        "supporting_source_ids": [
          "Risk / Data Governance Guidelines::chunk-01"
        ],
        "evidence_excerpt": [
          "Customer or employee data collected for one operational purpose must not be reused for a materially different purpose without documented legal/privacy basis and Data Governance approval. Commercial attractiveness or operational convenience does not by itself constitute approval. 3. Customer Data Enrichment External enrichment of identifiable customer records is permitted only when the source is approved, the intended use is documented, and Data Governance confirms that consent, notice, contractual terms, and retention controls are adequate."
        ],
        "rationale": "Retrieved authoritative policy or guideline text explicitly supports this requirement. Analogous precedent is used only as corroboration."
      }
    ],
    "retrieval_provenance": [
      {
        "related_uncertainty": "Data Governance review",
        "retrieval_query": "Data Governance review policy requirements timing and controls for a pilot: Run a 6-week pilot enriching ~25,000 customer records with demographic, firmographic, and behavioral attributes from a proposed third-party data provider and use enriched profiles in commercial campaign targeting.",
        "source_id": "Prior Decision Precedents::chunk-02",
        "source_type": "analogous_precedent",
        "rank": 1,
        "score": 0.2830448354236255,
        "grounded_interpretation": "required"
      },
      {
        "related_uncertainty": "Data Governance review",
        "retrieval_query": "Data Governance review policy requirements timing and controls for a pilot: Run a 6-week pilot enriching ~25,000 customer records with demographic, firmographic, and behavioral attributes from a proposed third-party data provider and use enriched profiles in commercial campaign targeting.",
        "source_id": "Approval Policy::chunk-02",
        "source_type": "authoritative_policy",
        "rank": 2,
        "score": 0.2620826196475748,
        "grounded_interpretation": "required"
      },
      {
        "related_uncertainty": "Data Governance review",
        "retrieval_query": "Data Governance review policy requirements timing and controls for a pilot: Run a 6-week pilot enriching ~25,000 customer records with demographic, firmographic, and behavioral attributes from a proposed third-party data provider and use enriched profiles in commercial campaign targeting.",
        "source_id": "Prior Decision Precedents::chunk-01",
        "source_type": "analogous_precedent",
        "rank": 3,
        "score": 0.2524254030678375,
        "grounded_interpretation": "required"
      },
      {
        "related_uncertainty": "Agreement or contract with the proposed third-party data provider",
        "retrieval_query": "Agreement or contract with the proposed third-party data provider policy requirements timing and controls for a pilot: Run a 6-week pilot enriching ~25,000 customer records with demographic, firmographic, and behavioral attributes from a proposed third-party data provider and use enriched profiles in commercial campaign targeting.",
        "source_id": "Prior Decision Precedents::chunk-02",
        "source_type": "analogous_precedent",
        "rank": 1,
        "score": 0.29743273469935194,
        "grounded_interpretation": "unclear"
      },
      {
        "related_uncertainty": "Agreement or contract with the proposed third-party data provider",
        "retrieval_query": "Agreement or contract with the proposed third-party data provider policy requirements timing and controls for a pilot: Run a 6-week pilot enriching ~25,000 customer records with demographic, firmographic, and behavioral attributes from a proposed third-party data provider and use enriched profiles in commercial campaign targeting.",
        "source_id": "Prior Decision Precedents::chunk-01",
        "source_type": "analogous_precedent",
        "rank": 2,
        "score": 0.26259549923203307,
        "grounded_interpretation": "unclear"
      },
      {
        "related_uncertainty": "Agreement or contract with the proposed third-party data provider",
        "retrieval_query": "Agreement or contract with the proposed third-party data provider policy requirements timing and controls for a pilot: Run a 6-week pilot enriching ~25,000 customer records with demographic, firmographic, and behavioral attributes from a proposed third-party data provider and use enriched profiles in commercial campaign targeting.",
        "source_id": "Approval Policy::chunk-02",
        "source_type": "authoritative_policy",
        "rank": 3,
        "score": 0.22867955782333715,
        "grounded_interpretation": "unclear"
      },
      {
        "related_uncertainty": "Confirmation that customer terms and data-use practices legally permit third-party enrichment",
        "retrieval_query": "Confirmation that customer terms and data-use practices legally permit third-party enrichment policy requirements timing and controls for a pilot: Run a 6-week pilot enriching ~25,000 customer records with demographic, firmographic, and behavioral attributes from a proposed third-party data provider and use enriched profiles in commercial campaign targeting.",
        "source_id": "Prior Decision Precedents::chunk-02",
        "source_type": "analogous_precedent",
        "rank": 1,
        "score": 0.2941402265469045,
        "grounded_interpretation": "required"
      },
      {
        "related_uncertainty": "Confirmation that customer terms and data-use practices legally permit third-party enrichment",
        "retrieval_query": "Confirmation that customer terms and data-use practices legally permit third-party enrichment policy requirements timing and controls for a pilot: Run a 6-week pilot enriching ~25,000 customer records with demographic, firmographic, and behavioral attributes from a proposed third-party data provider and use enriched profiles in commercial campaign targeting.",
        "source_id": "Risk / Data Governance Guidelines::chunk-01",
        "source_type": "authoritative_guideline",
        "rank": 2,
        "score": 0.2714201118101072,
        "grounded_interpretation": "required"
      },
      {
        "related_uncertainty": "Confirmation that customer terms and data-use practices legally permit third-party enrichment",
        "retrieval_query": "Confirmation that customer terms and data-use practices legally permit third-party enrichment policy requirements timing and controls for a pilot: Run a 6-week pilot enriching ~25,000 customer records with demographic, firmographic, and behavioral attributes from a proposed third-party data provider and use enriched profiles in commercial campaign targeting.",
        "source_id": "Prior Decision Precedents::chunk-01",
        "source_type": "analogous_precedent",
        "rank": 3,
        "score": 0.2555254086321882,
        "grounded_interpretation": "required"
      }
    ],
    "readiness_status": "NOT_READY",
    "leadership_questions": [
      "Data Governance and legal permissibility: The case shows Data Governance review has not been initiated and the proposal assumes customer terms permit third‑party enrichment. Before approving the pilot, can legal/Data Governance confirm (and in writing) whether enrichment of identifiable customer records with this provider is permitted and whether any approvals or controls are required prior to launch? If approval is required, what is the expected timeline and material conditions (e.g., consent, notice, retention, allowable attributes) that could change scope, timing, or required mitigations?",
      "Provider match quality and measurable signal: The proposal asserts the vendor can supply attributes but provides no match-rate or data‑quality evidence and does not define “acceptable match accuracy.” Can the team provide (or obtain quickly) vendor historical match rates and attribute completeness for comparable populations, plus a pre-specified metric and threshold for acceptable match accuracy and how it will be measured? If realistic match rates or signal levels are lower than assumed, how would that change sample size, success criteria, or the decision to proceed?",
      "Contracting and data access dependencies: The proposal does not state whether a contract or short‑term data access arrangement exists. Will a binding agreement (or interim scoped data-access arrangement) be required before enrichment can occur? If so, what are the expected timing, cost or usage constraints in that agreement that could materially affect the 6‑week pilot (e.g., data delivery latency, allowed attributes, retention limits, or additional fees)?",
      "Pilot statistical power and decision rules: Given 25,000 records and a baseline response of 3.4%, what is the minimal detectable uplift (with stated confidence and power) for the proposed test design and sample split? If the current sample is underpowered to detect the 0.8pp uplift target, what adjustments (larger sample, longer duration, different targeting approach) are needed to ensure pilot results will be interpretable for a follow‑on investment decision?"
    ],
    "recommended_actions": [
      "Initiate Data Governance review immediately and obtain written confirmation of required approvals and any mandatory controls or restrictions that would affect the pilot scope or timing (evidence: policy and hard_blocker listing Data Governance review as required).",
      "Obtain legal/compliance confirmation (in writing) that existing customer terms and data-use practices permit the planned third-party enrichment or, if not permitted, specify the minimal mitigations or consent changes needed before launch (evidence: stated assumption about permissibility and dependency_reviews).",
      "Request from the proposed provider (or validate with procurement) historical match rates and attribute completeness for comparable populations and define a quantitative \"acceptable match accuracy\" threshold plus how it will be measured during the pilot (evidence: material_caveats noting no match-rate data and absence of an acceptable accuracy definition).",
      "Confirm contractual status or a scoped interim data-access arrangement with the proposed provider, including any timing, allowed-attribute, retention, or fee constraints that could affect the 6-week timeline, and escalate procurement/legal if a binding agreement is required before enrichment (evidence: dependency_reviews and material_caveats on contract status)."
    ],
    "final_brief": "Decision Requested\nLeadership approval for a 6-week pilot to enrich ~25,000 customer records with third-party demographic, firmographic, and behavioral data to improve campaign response rate from 3.4% to at least 4.2% (Accountable: Director, Growth Marketing). Pilot cost estimate: $38,000.\n\nReadiness\nNOT_READY\n\nWhy This Status\nThe case contains a clear, testable pilot design and defined owner, scope, cost, duration, and success criteria, but it contains unresolved, authoritative governance and legal dependencies (Data Governance review; confirmation that customer terms permit enrichment) identified as required by policy. These unresolved critical dependencies make the proposal NOT_READY for leadership approval to start the pilot.\n\nEvidence & Assumptions\nStated evidence (from the proposal):\n- Baseline campaign response rate = 3.4%; pilot target >= 4.2%.\n- Pilot: ~25,000 records, 6 weeks, estimated cost $38,000.\n- Proposed provider can supply demographic, firmographic, behavioral attributes (claim stated by proposer).\n- Proposal assumes existing customer terms and data-use practices permit third-party enrichment.\n- Data Governance review has not been initiated.\n- Stop condition: pilot will stop if no improvement or material data-quality/customer-response issues.\n\nInferred assumptions (model-generated inference):\n- Enrichment and matching can be executed and integrated within the 6-week window.\n- Match accuracy and data quality will be sufficient to evaluate uplift.\n\nMaterial Gaps / Blockers\nConfirmed hard blocker (authoritative):\n- UNRESOLVED_CRITICAL_DEPENDENCY: Data Governance review is required (grounded to Approval Policy and prior precedents) and has not been initiated. This is a blocking item that must be resolved before approval to proceed.\nOther material, unresolved gaps (not labeled as hard blockers but materially relevant):\n- No evidence provided of vendor match rates, attribute completeness, or integration track record (material to pilot validity and interpretability).\n- No written confirmation that customer terms and data-use practices legally permit the proposed enrichment.\n- Contracting/data-access status with the proposed provider is unspecified.\n- No defined quantitative threshold or measurement plan for “acceptable match accuracy.”\n- No statistical power / minimal detectable effect analysis to confirm 25,000 records is sufficient to detect the 0.8pp uplift.\n\nDependencies & Risks\nDependencies (identified, mostly unresolved):\n- Data Governance review and approvals (required per policy).\n- Legal/compliance confirmation that customer terms permit enrichment (required per guideline).\n- Agreement/contract or scoped data-access arrangement with the third-party provider (status unclear).\n- Budget availability/approval for $38,000 (status unknown).\n- Technical integration capacity to match and ingest enriched attributes into campaign systems within 6 weeks.\n\nPrimary risks (stated and inferred):\n- Non-compliance if customer terms don’t permit enrichment.\n- Insufficient vendor match rates or poor attribute quality invalidating pilot results.\n- Data Governance delays or required controls that change scope/timing.\n- Integration or operational failures preventing completion within 6 weeks.\n\nLeadership Challenge Questions\n(Use to inform approval or conditional approvals)\n1) Can Legal/Data Governance confirm (in writing) whether enrichment of identifiable customer records with this provider is permitted and whether approvals/controls are required before launch? If required, what is the expected timeline and material conditions (consent, notice, retention, allowable attributes)?\n2) Can the team obtain vendor-provided historical match rates and attribute completeness for comparable populations and commit to a pre-specified quantitative threshold and measurement method for acceptable match accuracy during the pilot? How would lower-than-expected match rates change sample size, success criteria, or go/no-go decisions?\n3) Is a binding contract or scoped interim data-access agreement required before enrichment? If so, what are the expected timing, allowed attributes, retention or fee constraints that could materially affect the 6-week timeline?\n4) Given 25,000 records and baseline 3.4%, what is the minimal detectable uplift (with stated confidence and power) for the test design? If underpowered, what adjustments are needed to ensure interpretable results?\n\nRecommended Pre-review Actions\n(These are the exact recommended actions from the case; they must be completed before leadership should approve launch.)\n- Initiate Data Governance review immediately and obtain written confirmation of required approvals and any mandatory controls or restrictions that would affect pilot scope/timing (grounded to policy and hard_blocker).\n- Obtain legal/compliance written confirmation that existing customer terms/data-use practices permit the planned enrichment or specify required mitigations/consent changes.\n- Request from the proposed provider (or validate via procurement) historical match rates and attribute completeness for comparable populations and define a quantitative \"acceptable match accuracy\" threshold and how it will be measured during the pilot.\n- Confirm contractual status or obtain a scoped interim data-access arrangement with the proposed provider, including timing, allowed-attribute, retention, and fee constraints that could affect the 6-week timeline; escalate procurement/legal if needed.\n\nProvenance & Traceability\n- Proposal facts: items listed under proposal_fact in provenance (baseline response, pilot scope, cost, provider claim, stated assumptions, success metrics, Data Governance not initiated).\n- Retrieved authoritative support: Data Governance review requirement grounded to Approval Policy::chunk-02 and Prior Decision Precedents::chunk-02 (supporting hard_blocker determination).\n- Model inferences and questions/actions: leadership questions and recommended actions are model-generated synthesis drawn from the case gaps and retrieved evidence; these are presented to surface required confirmations and next steps.\n\nSummary Conclusion\nThe pilot design is operationally specified and measurable, but policy-required governance and legal confirmations are unresolved and constitute a hard blocker. Leadership should not approve pilot launch until Data Governance review and legal confirmation (per grounded requirements) are obtained and the other listed pre-review actions (vendor match evidence, contract/data-access clarity, and match-accuracy definition) are completed or clearly mitigated.\n\nRetrieved Evidence Provenance\n- Data Governance review: REQUIRED\n  Approval Policy::chunk-02 | authoritative_policy | rank=2 | score=0.2621\n  Prior Decision Precedents::chunk-02 | analogous_precedent | rank=1 | score=0.2830\n- Confirmation that customer terms and data-use practices legally permit third-party enrichment: REQUIRED\n  Risk / Data Governance Guidelines::chunk-01 | authoritative_guideline | rank=2 | score=0.2714\n\nStructured Evidence-Limitation Trace\n- [inferred state — execution_readiness gap] No details on who will perform matching, the matching process, integration with campaign systems, or resource/time estimates\n- [inferred state — risk_and_governance gap] No definition of what constitutes a 'material' increase in opt-outs or complaints"
  },
  "D": {
    "dimension_assessments": {
      "decision_clarity": {
        "score": 3,
        "reason": "The requested decision (approve a 10-week pilot with a $70k budget) is explicit and the pilot owner, duration, budget, and next-step rule (consider scaling if thresholds met) are stated.",
        "supporting_evidence": [
          "Decision ask: Leadership approval for a 10-week pilot to enter a new industrial fasteners sourcing category with a $70,000 budget",
          "Accountable owner: Head of Category Expansion",
          "Pilot duration: 10 weeks",
          "If pilot achieves thresholds, leadership will consider scaling; otherwise stop or redesign"
        ],
        "gaps": []
      },
      "problem_and_rationale": {
        "score": 2,
        "reason": "The proposal gives a clear rationale to test demand, supplier economics, and execution feasibility, and cites an industry report suggesting growth and fragmented pricing — enough to justify a pilot — but lacks recent internal market-size/pricing benchmarks and structured customer validation prior to the pilot.",
        "supporting_evidence": [
          "Problem summary: Test customer demand, supplier economics, and execution feasibility before scaling",
          "2023 industry report estimating 8-10% annual market growth and fragmented pricing",
          "Proposal notes current internal data does not provide recent market-size or pricing benchmark"
        ],
        "gaps": [
          "No recent internal market-size or pricing benchmark to ground expected pilot outcomes",
          "No structured customer validation conducted prior to pilot (only informal account manager discussions)"
        ]
      },
      "evidence_and_assumptions": {
        "score": 2,
        "reason": "Material claims are framed as hypotheses appropriate for a pilot, and the proposal acknowledges lack of internal benchmarks and limited customer validation. However, key assumptions (adequacy of the $70k budget; feasibility of onboarding 8 suppliers in 4 weeks) are not evidenced.",
        "supporting_evidence": [
          "Material claims are labeled as hypotheses (demand, margins, execution, supplier onboarding)",
          "Proposal states internal data lacks recent benchmarks and customer validation is informal",
          "Success metrics defined (e.g., 5 buyers, $250k GMV, 6% margin, OTIF 90%)"
        ],
        "gaps": [
          "No evidence or prior data showing $70k is sufficient to recruit suppliers, market to customers, and operate for 10 weeks",
          "No evidence supporting the feasibility of onboarding >=8 qualified suppliers within 4 weeks",
          "No baseline pricing or margin data to contextualize the 6% margin target"
        ]
      },
      "business_and_economic_case": {
        "score": 2,
        "reason": "The pilot's cost exposure is clear and bounded ($70k, 10 weeks) and success metrics are explicit, which can inform a later scaling decision. But the proposal lacks a clearer mapping from pilot outcomes to expected downstream value, and no contingency for failing to reach GMV/margin targets is detailed beyond 'stop or redesign.'",
        "supporting_evidence": [
          "Proposed budget: $70,000; Pilot duration: 10 weeks",
          "Success thresholds that gate scaling consideration (5 buyers, $250k GMV, 6% margin, OTIF 90%)",
          "Proposal states leadership will consider scaling only if thresholds are met"
        ],
        "gaps": [
          "No articulation of how pilot learnings will translate into financial projections for scaling (e.g., expected ARR, payback, or sensitivity)",
          "No explicit contingency budget or plan if initial supplier onboarding or customer acquisition is slower than expected"
        ]
      },
      "execution_readiness": {
        "score": 2,
        "reason": "The proposal names an owner, duration, targets, and supplier/customer goals, which is sufficient to start a pilot. However, operational details (team/resources, marketing/channel plan, recruitment approach for suppliers, measurement approach for OTIF and margins) are missing or unstated.",
        "supporting_evidence": [
          "Accountable owner: Head of Category Expansion",
          "Clear success metrics and timeline (onboard 8 suppliers in 4 weeks; 10-week pilot)",
          "Stated objectives: validate demand, supplier economics, and execution"
        ],
        "gaps": [
          "No stated resourcing plan or roles/responsibilities beyond the accountable owner",
          "No supplier recruitment plan or criteria for 'qualified suppliers'",
          "No customer acquisition/channel plan or measurement approach for GMV and OTIF"
        ]
      },
      "dependencies": {
        "score": 2,
        "reason": "Key dependencies are identified (budget approval, supplier availability, customer demand, operations), but the proposal leaves several dependency statuses as unknown and does not show mitigation plans for them.",
        "supporting_evidence": [
          "Dependencies listed: leadership approval and $70k; supplier availability (>=8 in 4 weeks); customer demand to hit targets; operational capability to meet OTIF >=90%",
          "Structured extraction marks several dependency requirement_status values as unknown"
        ],
        "gaps": [
          "Unknown status and lack of plan for supplier onboarding dependency (no evidence supplier pipeline exists)",
          "Unknown whether any pre-pilot validations or commitments from customers/suppliers are required",
          "No stated mitigation if dependencies (e.g., supplier recruitment) are not met within specified windows"
        ]
      },
      "risk_and_governance": {
        "score": 2,
        "reason": "The proposal acknowledges the primary risks (demand, margins, supplier availability, service levels) and commits to stopping or redesigning if thresholds are missed. But it does not quantify exposure, list detection/monitoring mechanisms, or describe escalation triggers and decision cadence during the pilot.",
        "supporting_evidence": [
          "Stated risk: If demand, margins, supplier availability, or service levels are materially below target, the initiative will stop or be redesigned",
          "Inferred risks documented: supplier recruitment, margin shortfall, low customer demand, operational service-level failure"
        ],
        "gaps": [
          "No monitoring cadence or interim checkpoints defined to detect failing metrics early",
          "No explicit stop/go decision criteria beyond final thresholds (e.g., mid-pilot remediation actions or go/no-go gates)",
          "No quantified downside or how unspent funds or losses will be handled if pilot fails"
        ]
      }
    },
    "hard_blockers": [],
    "material_caveats": [
      {
        "caveat_code": "UNKNOWN_DEPENDENCY_REQUIREMENT",
        "rationale": "The feasibility and timelines for onboarding >=8 qualified suppliers within 4 weeks is marked unknown. Supplier onboarding is integral to achieving GMV, margins, and OTIF targets; if this dependency is harder than assumed the pilot may not generate interpretable evidence on customer demand or margins. Resolving whether supplier availability/pipeline exists or whether flexible timelines are acceptable would materially change pilot design (recruitment budget, staffing, contingency) but does not by itself prevent running a bounded test with mitigations.",
        "evidence": [
          "Dependency_reviews: \"Onboarding at least 8 qualified suppliers within 4 weeks\" has requirement_status = \"unknown\"",
          "Material_claim: \"It is feasible to onboard at least 8 qualified suppliers within 4 weeks.\" is framed as a hypothesis with no supporting evidence"
        ],
        "related_item_text": "Onboarding at least 8 qualified suppliers within 4 weeks",
        "decision_horizon_relevance": "current_decision",
        "current_decision_materiality_evidence": [
          "Pilot success metrics (GMV, margin, OTIF) depend on having suppliers onboarded quickly to generate product availability and sales during the 10-week window.",
          "Execution_readiness gaps note: \"No supplier recruitment plan or criteria for 'qualified suppliers'\" and Dependencies gaps note: \"Unknown status and lack of plan for supplier onboarding dependency (no evidence supplier pipeline exists)\"."
        ],
        "source": "semantic"
      }
    ],
    "future_decision_considerations": [
      {
        "caveat_code": "ACKNOWLEDGED_NON_BLOCKING_UNCERTAINTY",
        "rationale": "The proposal explicitly identifies a later-stage leadership decision. It remains visible for continuity but does not affect readiness for the current requested decision.",
        "evidence": [
          "If the pilot achieves the commercial and execution thresholds, leadership will consider scaling the category."
        ],
        "related_item_text": "If the pilot achieves the commercial and execution thresholds, leadership will consider scaling the category.",
        "decision_horizon_relevance": "future_decision",
        "current_decision_materiality_evidence": [],
        "source": "deterministic"
      }
    ],
    "grounded_requirements": [
      {
        "requirement_text": "Onboarding at least 8 qualified suppliers within 4 weeks",
        "requirement_status": "unclear",
        "supporting_source_ids": [],
        "evidence_excerpt": [],
        "rationale": "Retrieved authoritative text is absent, conflicting, or does not explicitly establish this requirement. Precedent alone is insufficient."
      },
      {
        "requirement_text": "Structured customer validation prior to scaling decision",
        "requirement_status": "unclear",
        "supporting_source_ids": [],
        "evidence_excerpt": [],
        "rationale": "The dependency concerns a later scale or rollout decision; the retrieved text does not establish it as required for the current pilot."
      }
    ],
    "retrieval_provenance": [
      {
        "related_uncertainty": "Onboarding at least 8 qualified suppliers within 4 weeks",
        "retrieval_query": "Onboarding at least 8 qualified suppliers within 4 weeks policy requirements timing and controls for a pilot: Run a 10-week pilot to onboard suppliers, validate customer demand, measure GMV and margins, and test operational service levels for the fasteners category.",
        "source_id": "Approval Policy::chunk-00",
        "source_type": "authoritative_policy",
        "rank": 1,
        "score": 0.21575013994403638,
        "grounded_interpretation": "unclear"
      },
      {
        "related_uncertainty": "Onboarding at least 8 qualified suppliers within 4 weeks",
        "retrieval_query": "Onboarding at least 8 qualified suppliers within 4 weeks policy requirements timing and controls for a pilot: Run a 10-week pilot to onboard suppliers, validate customer demand, measure GMV and margins, and test operational service levels for the fasteners category.",
        "source_id": "Risk / Data Governance Guidelines::chunk-01",
        "source_type": "authoritative_guideline",
        "rank": 2,
        "score": 0.20644445192166871,
        "grounded_interpretation": "unclear"
      },
      {
        "related_uncertainty": "Onboarding at least 8 qualified suppliers within 4 weeks",
        "retrieval_query": "Onboarding at least 8 qualified suppliers within 4 weeks policy requirements timing and controls for a pilot: Run a 10-week pilot to onboard suppliers, validate customer demand, measure GMV and margins, and test operational service levels for the fasteners category.",
        "source_id": "Prior Decision Precedents::chunk-00",
        "source_type": "analogous_precedent",
        "rank": 3,
        "score": 0.16200915055450454,
        "grounded_interpretation": "unclear"
      },
      {
        "related_uncertainty": "Structured customer validation prior to scaling decision",
        "retrieval_query": "Structured customer validation prior to scaling decision policy requirements timing and controls for a pilot: Run a 10-week pilot to onboard suppliers, validate customer demand, measure GMV and margins, and test operational service levels for the fasteners category.",
        "source_id": "Risk / Data Governance Guidelines::chunk-01",
        "source_type": "authoritative_guideline",
        "rank": 1,
        "score": 0.23376211158313287,
        "grounded_interpretation": "unclear"
      },
      {
        "related_uncertainty": "Structured customer validation prior to scaling decision",
        "retrieval_query": "Structured customer validation prior to scaling decision policy requirements timing and controls for a pilot: Run a 10-week pilot to onboard suppliers, validate customer demand, measure GMV and margins, and test operational service levels for the fasteners category.",
        "source_id": "Prior Decision Precedents::chunk-00",
        "source_type": "analogous_precedent",
        "rank": 2,
        "score": 0.18043762437828048,
        "grounded_interpretation": "unclear"
      },
      {
        "related_uncertainty": "Structured customer validation prior to scaling decision",
        "retrieval_query": "Structured customer validation prior to scaling decision policy requirements timing and controls for a pilot: Run a 10-week pilot to onboard suppliers, validate customer demand, measure GMV and margins, and test operational service levels for the fasteners category.",
        "source_id": "Approval Policy::chunk-00",
        "source_type": "authoritative_policy",
        "rank": 3,
        "score": 0.17764556325597064,
        "grounded_interpretation": "unclear"
      }
    ],
    "readiness_status": "READY_WITH_CAVEATS",
    "leadership_questions": [
      "What evidence or pipeline exists today to support the feasibility of onboarding >=8 qualified suppliers within 4 weeks, and if that evidence is weak, what are the specific contingency plans (extended timeline, increased recruiting budget, use of marketplace intermediaries, or reduced supplier qualification thresholds) so the pilot still produces interpretable learnings?",
      "How will the pilot generate reliable, structured customer validation fast enough to determine whether five active buyers and $250k GMV are achievable (sampling frame, outreach channels, incentive/offering, and target customer segments), and what minimum pre-pilot customer signals would cause you to delay or redesign the pilot?",
      "Please show a simple margin and cost sensitivity model for the pilot that ties supplier prices, fees, expected sell-through, and operational costs to the 6% gross margin target — including the break-even GMV and the scenarios (best/central/worst) under which the pilot would still produce meaningful learning even if the 6% margin target is missed.",
      "What operational monitoring, escalation, and go/no-go cadence will be used during the 10 weeks (which metrics will be tracked daily/weekly, who has authority to adjust recruitment/marketing/fulfillment levers, and what are the predefined triggers for pausing or expanding the test) to ensure we detect and respond to issues (supplier shortfalls or OTIF drops) without blowing the $70k budget?"
    ],
    "recommended_actions": [
      "Provide evidence or a brief confirmation of supplier pipeline capacity: state whether there are existing leads or partners that can reasonably supply >=8 qualified fastener suppliers within 4 weeks and, if uncertain, list the single short contingency you will use during the pilot (e.g., extend supplier onboarding window by X weeks or use 3rd-party marketplace partners).",
      "Confirm the minimum pre-pilot customer signals you will accept (e.g., recent account-manager-identified prospects, inbound interest, or committed pilot buys) that would allow the pilot to start without additional structured validation, and state the outreach channels you will use to secure the five active buyers within the 10-week window.",
      "Provide a short statement of the monitoring and escalation cadence you will use during the 10 weeks: which 3–5 metrics you will track weekly (include GMV, margin, supplier onboarding progress, OTIF), who has authority to adjust recruitment/marketing/fulfillment levers, and the predefined trigger that would pause or stop the pilot to protect the $70k budget."
    ],
    "final_brief": "Decision Requested\nLeadership approval for a 10-week pilot to enter a new industrial fasteners sourcing category with a $70,000 budget (Accountable: Head of Category Expansion).\n\nReadiness\nREADY_WITH_CAVEATS\n\nWhy This Status\nThe proposal is a bounded, testable pilot: owner identified, duration and budget specified, and clear success metrics. Key uncertainties exist but are framed as hypothesis-driven questions the pilot is designed to answer; none are assessed as hard blockers to starting the pilot.\n\nEvidence & Assumptions\nStated evidence (proposal facts): proposed budget $70,000; 10-week pilot; 2023 industry report indicating ~8–10% growth and fragmented pricing; internal data lacks recent market-size/pricing benchmarks; only informal customer discussions to date.\nStated assumptions (from proposal): the 2023 report is relevant; informal account-manager discussions indicate potential demand (not structured validation).\nInferred assumptions (model): internal capacity exists to run the pilot and $70k is adequate to recruit suppliers, market the category, and support operations for 10 weeks (no direct evidence provided).\n\nMaterial Gaps / Blockers\nNo confirmed hard blockers.\nMaterial caveat (single, substantive gap): UNKNOWN_DEPENDENCY_REQUIREMENT — feasibility and timeline to onboard >=8 qualified suppliers within 4 weeks is unknown. Supplier availability is material because pilot GMV, margins, and OTIF depend on timely supplier onboarding; lack of supplier pipeline evidence could prevent the pilot from producing interpretable results.\nOther material but non-blocking gaps: no recent pricing/margin benchmarks; no structured pre-pilot customer validation; no detailed supplier recruitment plan, customer acquisition/channel plan, resourcing allocation, or monitoring/escalation cadence.\n\nDependencies & Risks\nDependencies (as stated): leadership budget approval; supplier availability (>=8 suppliers in 4 weeks); sufficient customer demand to reach 5 active buyers and $250k GMV; operational capability to meet OTIF >=90%.\nPrimary risks (stated and inferred): failure to recruit suppliers in time; margins below 6% due to fragmented pricing and lack of benchmarks; insufficient customer demand; operational inability to meet OTIF >=90%.\n\nLeadership Challenge Questions\n(From model-generated list — require answers from pilot team before approval)\n1) What evidence or pipeline exists today to support onboarding >=8 qualified suppliers within 4 weeks, and if weak, which contingency will you use so the pilot still yields interpretable learnings? \n2) How will you generate reliable, structured customer validation fast enough to reach five active buyers and $250k GMV (sampling frame, outreach channels, incentives), and what pre-pilot customer signals would cause you to delay or redesign? \n3) Provide a simple margin and cost sensitivity model tying supplier prices, fees, sell-through, and operational costs to the 6% gross margin target, with break-even and scenario outcomes. \n4) What operational monitoring, escalation, and go/no-go cadence will you use during the 10 weeks (metrics tracked daily/weekly, authority to adjust levers, and trigger(s) to pause the pilot to protect the $70k budget)?\n\nRecommended Pre-review Actions\n(These are the specific pre-approval clarifications the proposal team should provide)\n1) Confirm supplier pipeline capacity or state a single short contingency (e.g., extend onboarding window by X weeks or use 3rd-party marketplace partners) if pipeline is uncertain. \n2) Confirm minimum acceptable pre-pilot customer signals (e.g., named prospects or committed pilot buys) and the channels you will use to secure five active buyers within 10 weeks. \n3) Provide a short monitoring and escalation plan: 3–5 weekly metrics (include GMV, margin, supplier onboarding progress, OTIF), who can adjust recruitment/marketing/fulfillment, and the predefined trigger to pause/stop the pilot to protect the $70k budget.\n\nProvenance & Traceability\nProposal facts (sourced from the submission): budget, duration, industry report citation, lack of internal benchmarks, informal account-manager discussions, success metrics, stated hypotheses.\nInferred state (model): adequacy of budget and internal resourcing are assumed but not evidenced; supplier and customer acquisition risks inferred from missing evidence.\nRetrieved evidence references (selected IDs): Approval Policy::chunk-00; Risk / Data Governance Guidelines::chunk-01; Prior Decision Precedents::chunk-00 (retrievals returned unclear on supplier-timing and pre-scaling validation requirements).\n\nSummary Statement\nThe proposal is READY_WITH_CAVEATS: it is sufficiently prepared to request leadership approval to run a bounded 10-week, $70k pilot, provided leadership either accepts the current unknowns as pilot objectives or receives the clarifications above (supplier pipeline contingency, minimal pre-pilot customer signals, and monitoring/escalation plan) to ensure the pilot can produce interpretable results and protect the $70k budget.\n\nFuture-Decision Considerations (not current readiness caveats)\n- If the pilot achieves the commercial and execution thresholds, leadership will consider scaling the category.: The proposal explicitly identifies a later-stage leadership decision. It remains visible for continuity but does not affect readiness for the current requested decision.\n\nStructured Evidence-Limitation Trace\n- [inferred state — business_and_economic_case gap] No articulation of how pilot learnings will translate into financial projections for scaling (e.g., expected ARR, payback, or sensitivity)\n- [inferred state — dependencies gap] Unknown whether any pre-pilot validations or commitments from customers/suppliers are required\n- [inferred state — risk_and_governance gap] No monitoring cadence or interim checkpoints defined to detect failing metrics early\n- [inferred state — risk_and_governance gap] No explicit stop/go decision criteria beyond final thresholds (e.g., mid-pilot remediation actions or go/no-go gates)\n- [inferred state — risk_and_governance gap] No quantified downside or how unspent funds or losses will be handled if pilot fails"
  }
}
''')


## Deterministic Downstream Regression


In [1]:
EXPECTED_V05_STATUSES = {
    "A": "READY",
    "B": "NOT_READY",
    "C": "NOT_READY",
    "D": "READY_WITH_CAVEATS",
}

FINAL_BRIEF_COMPLETENESS_EXPECTATIONS = {
    "A": {
        "status": [("ready",)],
        "blocker": [("no hard blocker",)],
        "future": [("broader rollout",)],
        "evidence_limitations": [("measurement",), ("data-control", "data control")],
    },
    "B": {
        "status": [("not_ready", "not ready")],
        "blocker": [("material_dimension_threshold",)],
        "evidence_limitations": [
            ("baseline",), ("exit criteria",), ("crm",), ("sequencing",),
        ],
    },
    "C": {
        "status": [("not_ready", "not ready")],
        "blocker": [("data governance",), ("unresolved_critical_dependency",)],
        "secondary_concerns": [
            ("match",), ("match accuracy",), ("budget",), ("timeline", "6-week"),
            ("response rate", "4.2%"),
        ],
        "retrieved_sources": [
            ("approval policy::chunk-02",),
            ("risk / data governance guidelines::chunk-01",),
        ],
    },
    "D": {
        "status": [("ready_with_caveats", "ready with caveats")],
        "caveats": [("supplier",), ("onboard",)],
        "evidence_limitations": [
            ("2023",), ("market-size", "market size"), ("pricing",),
            ("informal",), ("customer", "demand"),
        ],
        "hypotheses": [("margin",), ("otif",), ("gmv",)],
        "future": [("scaling", "scale"),],
    },
}


def _contains_any(text: str, alternatives: tuple[str, ...]) -> bool:
    normalized = text.casefold()
    return any(alternative.casefold() in normalized for alternative in alternatives)


def run_deterministic_v05_regression() -> list[dict]:
    rows = []
    for label in "ABCD":
        extraction = dict(FROZEN_EXTRACTION_FIXTURES[label])
        snapshot = FROZEN_DOWNSTREAM_SNAPSHOTS[label]

        needs = identify_retrieval_needs(extraction)["retrieval_needs"]
        retrieval_state = dict(extraction)
        retrieval_state["retrieval_needs"] = needs
        retrieved = retrieve_internal_evidence_node(
            retrieval_state
        )["retrieved_internal_evidence"]
        grounding_state = dict(retrieval_state)
        grounding_state["retrieved_internal_evidence"] = retrieved
        rerun_grounding = interpret_grounded_requirements(grounding_state)[
            "grounded_requirements"
        ]

        expected_grounding = [
            (
                item["requirement_text"],
                item["requirement_status"],
                tuple(item.get("supporting_source_ids", [])),
            )
            for item in snapshot["grounded_requirements"]
        ]
        actual_grounding = [
            (
                item["requirement_text"],
                item["requirement_status"],
                tuple(item.get("supporting_source_ids", [])),
            )
            for item in rerun_grounding
        ]
        retrieval_ok = actual_grounding == expected_grounding

        readiness_state = dict(extraction)
        readiness_state.update({
            "dimension_assessments": snapshot["dimension_assessments"],
            "hard_blockers": snapshot["hard_blockers"],
            "material_caveats": snapshot["material_caveats"],
        })
        recomputed_status = determine_readiness(readiness_state)["readiness_status"]
        readiness_ok = (
            recomputed_status == EXPECTED_V05_STATUSES[label]
            == snapshot["readiness_status"]
        )

        horizon_ok = all(
            item["decision_horizon_relevance"] != "future_decision"
            for item in snapshot["material_caveats"]
        ) and all(
            item["decision_horizon_relevance"] == "future_decision"
            for item in snapshot["future_decision_considerations"]
        )

        blocker_ok = True
        if label == "C":
            blocker_ok = any(
                blocker["blocker_code"] == "UNRESOLVED_CRITICAL_DEPENDENCY"
                and blocker.get("related_item_text") == "Data Governance review"
                for blocker in snapshot["hard_blockers"]
            )
        if label in {"A", "D"}:
            blocker_ok = blocker_ok and not snapshot["hard_blockers"]

        completeness = {}
        for category, requirements in FINAL_BRIEF_COMPLETENESS_EXPECTATIONS[label].items():
            completeness[category] = all(
                _contains_any(snapshot["final_brief"], alternatives)
                for alternatives in requirements
            )
        synthesis_ok = all(completeness.values())

        rows.append({
            "proposal": label,
            "status": recomputed_status,
            "retrieval_grounding": retrieval_ok,
            "blocker_precedence": blocker_ok,
            "decision_horizon": horizon_ok,
            "final_brief_completeness": synthesis_ok,
            "completeness_categories": completeness,
            "passed": all((
                retrieval_ok,
                readiness_ok,
                blocker_ok,
                horizon_ok,
                synthesis_ok,
            )),
        })
    return rows


deterministic_v05_regression_results = run_deterministic_v05_regression()
for row in deterministic_v05_regression_results:
    print(json.dumps(row, indent=2))
assert all(row["passed"] for row in deterministic_v05_regression_results)
print("DETERMINISTIC FROZEN-FIXTURE REGRESSION: PASS")


{
  "proposal": "A",
  "status": "READY",
  "retrieval_grounding": true,
  "blocker_precedence": true,
  "decision_horizon": true,
  "final_brief_completeness": true,
  "completeness_categories": {
    "status": true,
    "blocker": true,
    "future": true,
    "evidence_limitations": true
  },
  "passed": true
}
{
  "proposal": "B",
  "status": "NOT_READY",
  "retrieval_grounding": true,
  "blocker_precedence": true,
  "decision_horizon": true,
  "final_brief_completeness": true,
  "completeness_categories": {
    "status": true,
    "blocker": true,
    "evidence_limitations": true
  },
  "passed": true
}
{
  "proposal": "C",
  "status": "NOT_READY",
  "retrieval_grounding": true,
  "blocker_precedence": true,
  "decision_horizon": true,
  "final_brief_completeness": true,
  "completeness_categories": {
    "status": true,
    "blocker": true,
    "secondary_concerns": true,
    "retrieved_sources": true
  },
  "passed": true
}
{
  "proposal": "D",
  "status": "READY_WITH_CAVEATS",


## Iteration 3 — Bounded-Autonomy Tool-Decision Experiment

This isolated v0.1 experiment asks whether DecisionReady should use a tool for one material Proposal D uncertainty, which allowed tool is appropriate, and when bounded evidence gathering should stop.

It uses the frozen Iteration-2 v0.5 Proposal D state. Tool evidence is interpreted separately and **does not mutate readiness, blockers, caveats, questions, actions, or the final brief**. The existing Iteration-2 graph and deterministic A–D regression remain unchanged.


### 1. Tool-decision and evidence schemas

The decision record makes each eligibility gate explicit. Search-result and interpretation records preserve the tool, query, source, source type, freshness, relevant evidence, and effect on the original uncertainty.


In [69]:
from datetime import date
import copy
import os
from pathlib import Path

from tavily import TavilyClient


AllowedToolName = Literal["internal_retrieval", "external_search", "none"]
EvidenceEffect = Literal["strengthens", "weakens", "unresolved"]


class ToolDecision(BaseModel):
    tool_needed: bool
    tool_name: AllowedToolName
    related_uncertainty: str
    material_to_current_decision: bool
    materiality_basis: list[str]
    existing_evidence_insufficient: bool
    resolvable_by_allowed_tool: bool
    could_improve_decision_quality: bool
    why_tool_needed: str
    expected_information_gain: str
    query: str | None = None
    source_state_refs: list[str] = Field(default_factory=list)


class SourceEvidenceInterpretation(BaseModel):
    source_id: str
    effect: EvidenceEffect
    relevant_evidence: str
    rationale: str


class ToolEvidenceInterpretation(BaseModel):
    related_uncertainty: str
    source_assessments: list[SourceEvidenceInterpretation]
    overall_effect: EvidenceEffect
    sufficient_for_specific_uncertainty: bool
    rationale: str
    supporting_source_ids: list[str]
    additional_search_likely_to_help: bool
    follow_up_query: str | None = None


MAX_TOOL_CALLS = 2
print(f"Allowed tools: internal_retrieval, external_search; MAX_TOOL_CALLS={MAX_TOOL_CALLS}")


Allowed tools: internal_retrieval, external_search; MAX_TOOL_CALLS=2


### 2. Eligibility and tool-choice contract

A tool is eligible only when the uncertainty is material to the current requested decision, existing proposal/internal evidence is insufficient, an allowed tool can realistically address it, the result could materially improve decision quality or framing, and budget remains.

- `internal_retrieval` is for enterprise policy, precedent, or internal evidence.
- `external_search` is for externally verifiable, current public evidence such as market conditions, benchmarks, or time-sensitive industry data.
- `none` is required when any eligibility gate fails.

Uncertainty alone never justifies a search. Company-specific demand and execution feasibility are not treated as externally resolvable without direct company evidence.


In [70]:
TOOL_DECISION_PROMPT = """
You are the bounded tool-decision component of DecisionReady.

Choose at most one unresolved issue with the highest expected information gain.
A tool may be selected only when ALL eligibility gates are true:
1. The issue is material to the specific decision currently requested.
2. Existing proposal and internal evidence are insufficient.
3. The issue is realistically resolvable by an allowed tool.
4. Resolving it could materially improve decision quality or change how the uncertainty is framed.

Choose internal_retrieval only for the controlled Northstar corpus: Approval Policy, Pilot Governance
Guidelines, Risk / Data Governance Guidelines, and Prior Decision Precedents. It cannot access CRM,
supplier pipeline, operational dashboards, private benchmarks, or other enterprise systems.
Choose external_search for externally verifiable current public evidence, including market conditions,
benchmarks, industry data, or other time-sensitive public facts.
Choose none when the issue is future-only, already sufficiently evidenced, company-specific and not
externally resolvable, immaterial, or unlikely to improve the current decision.

Do not infer that company-specific customer demand, supplier pipeline, execution capacity, or internal
pricing can be resolved by general web search. Do not invent facts, policies, or requirements.
If external_search is selected, provide one concise query that contains no confidential proposal details.
"""


TOOL_INTERPRETATION_PROMPT = """
Interpret bounded tool evidence for one specified uncertainty.
Use only the supplied result records. Do not use prior knowledge or infer obligations.
Assess each source separately, then the evidence overall.

Distinguish broad public market context from company-specific demand, supplier feasibility, economics,
or internal pricing. Public market evidence may improve framing without resolving a company-specific
hypothesis. Conflicting, weak, undated, or indirect evidence must remain unresolved.

Set sufficient_for_specific_uncertainty true only when the supplied evidence adequately answers the
specific uncertainty. Recommend one concise follow-up query only if another allowed call is likely to
materially improve the answer. This interpretation must not change readiness or create blockers/caveats.
"""


iteration3_model = init_chat_model(
    "gpt-5-mini",
    model_provider="openai",
    reasoning_effort="minimal",
)
tool_decision_model = iteration3_model.with_structured_output(ToolDecision)
tool_evidence_interpretation_model = iteration3_model.with_structured_output(
    ToolEvidenceInterpretation
)
print("Iteration-3 structured model wrappers initialized")


Iteration-3 structured model wrappers initialized


### 3. Assemble the frozen Proposal D decision context

Only proposal facts, inferred structured state, caveats, assessment gaps, and future-decision considerations are supplied. Generated questions/actions and the prior final brief are excluded from tool selection.


In [71]:
proposal_d_iteration2_state = {
    **copy.deepcopy(FROZEN_EXTRACTION_FIXTURES["D"]),
    **copy.deepcopy(FROZEN_DOWNSTREAM_SNAPSHOTS["D"]),
}
proposal_d_original_readiness = proposal_d_iteration2_state["readiness_status"]


def build_tool_decision_context(state: dict) -> dict:
    assessment_gaps = []
    for dimension, assessment in state.get("dimension_assessments", {}).items():
        for gap in assessment.get("gaps", []):
            assessment_gaps.append({"dimension": dimension, "gap": gap})

    return {
        "available_tool_capabilities": {
            "internal_retrieval": (
                "Controlled Northstar corpus only: Approval Policy, Pilot Governance "
                "Guidelines, Risk / Data Governance Guidelines, and Prior Decision Precedents. "
                "No CRM, supplier pipeline, dashboard, or private benchmark access."
            ),
            "external_search": (
                "Current public web evidence such as market conditions, industry data, "
                "and public benchmarks; cannot resolve company-specific demand or execution."
            ),
        },
        "decision_ask": state.get("decision_ask"),
        "decision_stage": state.get("decision_stage"),
        "problem_summary": state.get("problem_summary"),
        "proposed_intervention": state.get("proposed_intervention"),
        "evidence_items": state.get("evidence_items", []),
        "material_claims": state.get("material_claims", []),
        "material_caveats": state.get("material_caveats", []),
        "assessment_gaps": assessment_gaps,
        "future_decision_considerations": state.get(
            "future_decision_considerations", []
        ),
    }


proposal_d_tool_context = build_tool_decision_context(proposal_d_iteration2_state)
print(
    "Frozen Proposal D state loaded; original readiness =",
    proposal_d_original_readiness,
)
print("Candidate assessment gaps:", len(proposal_d_tool_context["assessment_gaps"]))


Frozen Proposal D state loaded; original readiness = READY_WITH_CAVEATS
Candidate assessment gaps: 16


### 4. Decide whether a tool is justified

In [72]:
def validate_tool_decision(
    decision: ToolDecision,
    calls_used: int,
) -> tuple[ToolDecision, list[str]]:
    validation_log = []
    gates = {
        "material_to_current_decision": decision.material_to_current_decision,
        "existing_evidence_insufficient": decision.existing_evidence_insufficient,
        "resolvable_by_allowed_tool": decision.resolvable_by_allowed_tool,
        "could_improve_decision_quality": decision.could_improve_decision_quality,
        "within_autonomy_budget": calls_used < MAX_TOOL_CALLS,
        "allowed_tool_selected": decision.tool_name in {
            "internal_retrieval", "external_search"
        },
        "query_present": bool((decision.query or "").strip()),
    }
    selection_text = " ".join((
        decision.related_uncertainty,
        decision.query or "",
        decision.why_tool_needed,
    )).casefold()
    if decision.tool_name == "internal_retrieval":
        gates["tool_capability_aligned"] = any(
            term in selection_text
            for term in ("policy", "governance", "approval", "guideline", "precedent")
        ) and not any(
            term in selection_text
            for term in ("crm", "pipeline list", "dashboard", "private benchmark")
        )
    elif decision.tool_name == "external_search":
        gates["tool_capability_aligned"] = any(
            term in selection_text
            for term in ("market", "industry", "benchmark", "pricing", "public", "current")
        )
    else:
        gates["tool_capability_aligned"] = False
    validation_log.extend(f"{name}={value}" for name, value in gates.items())

    eligible = decision.tool_needed and all(gates.values())
    if eligible:
        validation_log.append("eligibility_result=approved")
        return decision, validation_log

    validation_log.append("eligibility_result=no_tool")
    return decision.model_copy(
        update={
            "tool_needed": False,
            "tool_name": "none",
            "query": None,
        }
    ), validation_log


raw_tool_decision = tool_decision_model.invoke(
    [
        SystemMessage(content=TOOL_DECISION_PROMPT),
        HumanMessage(
            content=(
                "Decide whether one bounded tool call is justified.\n\n"
                + json.dumps(proposal_d_tool_context, indent=2)
            )
        ),
    ]
)
proposal_d_tool_decision, tool_eligibility_log = validate_tool_decision(
    raw_tool_decision,
    calls_used=0,
)

print("RAW TOOL DECISION:")
print(json.dumps(raw_tool_decision.model_dump(), indent=2))
print("VALIDATION LOG:")
print(json.dumps(tool_eligibility_log, indent=2))
print("VALIDATED TOOL DECISION:")
print(json.dumps(proposal_d_tool_decision.model_dump(), indent=2))


RAW TOOL DECISION:
{
  "tool_needed": true,
  "tool_name": "external_search",
  "related_uncertainty": "Feasibility of onboarding at least 8 qualified industrial fasteners suppliers within 4 weeks (supplier availability and typical onboarding timelines/costs in the market).",
  "material_to_current_decision": true,
  "materiality_basis": [
    "Supplier onboarding speed is required to generate product availability and sales during the 10-week pilot, affecting GMV, margin, and OTIF metrics.",
    "Unknown supplier pipeline and lack of supplier recruitment plan make the onboarding assumption a key dependency; if false, the pilot may not produce interpretable results and budget/resourcing needs would change."
  ],
  "existing_evidence_insufficient": true,
  "resolvable_by_allowed_tool": true,
  "could_improve_decision_quality": true,
  "why_tool_needed": "Public market data and industry benchmarks can indicate typical supplier onboarding timelines, common barriers, and estimated acquisiti

### 5. Project-local bounded Tavily wrapper

Only the concise public query is sent to Tavily. Proposal D text, frozen structured state, and Northstar documents are not sent to the search provider.


In [73]:
def _find_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / ".env").exists() and (candidate / "notebooks").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate project root containing .env and notebooks")


class BoundedProjectToolExecutor:
    def __init__(self, max_calls: int = MAX_TOOL_CALLS):
        self.max_calls = max_calls
        self.calls_used = 0
        self.call_log: list[dict] = []
        project_root = _find_project_root()
        load_dotenv(project_root / ".env", override=False)
        api_key = os.getenv("TAVILY_API_KEY")
        if not api_key:
            raise RuntimeError("TAVILY_API_KEY is not configured in the project .env")
        self.tavily = TavilyClient(api_key=api_key)

    @property
    def calls_remaining(self) -> int:
        return max(0, self.max_calls - self.calls_used)

    def _claim_budget(self, tool_name: str, query: str) -> int:
        if self.calls_used >= self.max_calls:
            raise RuntimeError("Autonomy budget exhausted")
        self.calls_used += 1
        call_number = self.calls_used
        self.call_log.append({
            "call_number": call_number,
            "tool": tool_name,
            "query": query,
        })
        return call_number

    def external_search(self, query: str) -> list[dict]:
        call_number = self._claim_budget("external_search", query)
        response = self.tavily.search(
            query=query,
            topic="general",
            search_depth="advanced",
            max_results=5,
            include_answer=False,
            include_raw_content=False,
        )
        records = []
        for rank, item in enumerate(response.get("results", []), start=1):
            records.append({
                "tool": "external_search",
                "call_number": call_number,
                "query": query,
                "rank": rank,
                "source_id": item.get("url") or f"tavily-call-{call_number}-rank-{rank}",
                "source": item.get("title") or item.get("url") or "Unknown source",
                "source_url": item.get("url"),
                "source_type": "external_public_web",
                "source_date": item.get("published_date"),
                "freshness": item.get("published_date") or "not supplied by search result",
                "retrieval_score": item.get("score"),
                "relevant_evidence": item.get("content", ""),
            })
        return records

    def internal_retrieval(self, query: str) -> list[dict]:
        call_number = self._claim_budget("internal_retrieval", query)
        if "vector_store" not in globals():
            raise RuntimeError(
                "Iteration-2 local vector store must be initialized before internal retrieval"
            )
        matches = vector_store.similarity_search_with_score(query, k=3)
        return [
            {
                "tool": "internal_retrieval",
                "call_number": call_number,
                "query": query,
                "rank": rank,
                "source_id": document.metadata["chunk_id"],
                "source": document.metadata["source"],
                "source_url": None,
                "source_type": SOURCE_TYPES[document.metadata["source"]],
                "source_date": None,
                "freshness": "controlled internal corpus",
                "retrieval_score": score,
                "relevant_evidence": document.page_content,
            }
            for rank, (document, score) in enumerate(matches, start=1)
        ]

    def execute(self, tool_name: AllowedToolName, query: str) -> list[dict]:
        if tool_name == "external_search":
            return self.external_search(query)
        if tool_name == "internal_retrieval":
            return self.internal_retrieval(query)
        return []


print("Bounded project-local Tavily wrapper ready")


Bounded project-local Tavily wrapper ready


### 6. Retrieve and interpret bounded evidence

In [74]:
def interpret_tool_evidence(
    decision: ToolDecision,
    evidence_records: list[dict],
) -> ToolEvidenceInterpretation:
    concise_records = [
        {
            "source_id": item["source_id"],
            "source": item["source"],
            "source_type": item["source_type"],
            "source_date": item["source_date"],
            "freshness": item["freshness"],
            "retrieval_score": item["retrieval_score"],
            "relevant_evidence": item["relevant_evidence"],
        }
        for item in evidence_records
    ]
    return tool_evidence_interpretation_model.invoke(
        [
            SystemMessage(content=TOOL_INTERPRETATION_PROMPT),
            HumanMessage(
                content=json.dumps(
                    {
                        "related_uncertainty": decision.related_uncertainty,
                        "expected_information_gain": decision.expected_information_gain,
                        "evidence_records": concise_records,
                    },
                    indent=2,
                )
            ),
        ]
    )


tool_executor = BoundedProjectToolExecutor(MAX_TOOL_CALLS)
proposal_d_tool_results: list[dict] = []
proposal_d_interpretation: ToolEvidenceInterpretation | None = None
proposal_d_stop_reason = "no_tool_justified"

if proposal_d_tool_decision.tool_needed:
    first_results = tool_executor.execute(
        proposal_d_tool_decision.tool_name,
        proposal_d_tool_decision.query or "",
    )
    proposal_d_tool_results.extend(first_results)
    proposal_d_interpretation = interpret_tool_evidence(
        proposal_d_tool_decision,
        proposal_d_tool_results,
    )

    if proposal_d_interpretation.sufficient_for_specific_uncertainty:
        proposal_d_stop_reason = "sufficient_evidence_for_specific_uncertainty"
    elif proposal_d_interpretation.overall_effect == "unresolved":
        proposal_d_stop_reason = "evidence_inconclusive"
    elif not proposal_d_interpretation.additional_search_likely_to_help:
        proposal_d_stop_reason = "additional_tool_use_unlikely_to_improve_decision"
    elif tool_executor.calls_remaining == 0:
        proposal_d_stop_reason = "tool_call_budget_exhausted"
    elif (proposal_d_interpretation.follow_up_query or "").strip():
        second_results = tool_executor.execute(
            proposal_d_tool_decision.tool_name,
            proposal_d_interpretation.follow_up_query or "",
        )
        proposal_d_tool_results.extend(second_results)
        proposal_d_interpretation = interpret_tool_evidence(
            proposal_d_tool_decision,
            proposal_d_tool_results,
        )
        if proposal_d_interpretation.sufficient_for_specific_uncertainty:
            proposal_d_stop_reason = "sufficient_evidence_for_specific_uncertainty"
        elif tool_executor.calls_remaining == 0:
            proposal_d_stop_reason = "tool_call_budget_exhausted"
        elif proposal_d_interpretation.overall_effect == "unresolved":
            proposal_d_stop_reason = "evidence_inconclusive"
        else:
            proposal_d_stop_reason = "additional_tool_use_unlikely_to_improve_decision"
    else:
        proposal_d_stop_reason = "additional_tool_use_unlikely_to_improve_decision"

assert tool_executor.calls_used <= MAX_TOOL_CALLS
assert proposal_d_iteration2_state["readiness_status"] == proposal_d_original_readiness
print(
    f"Bounded experiment completed with {tool_executor.calls_used}/"
    f"{MAX_TOOL_CALLS} tool calls"
)


Bounded experiment completed with 1/2 tool calls


### 7. Proposal D inspection output

In [75]:
print("========== ITERATION-3 v0.1 — PROPOSAL D TOOL TRACE ==========")
print("UNRESOLVED UNCERTAINTY CONSIDERED:")
print(proposal_d_tool_decision.related_uncertainty)
print()
print("MATERIALITY TO CURRENT DECISION:")
print(proposal_d_tool_decision.material_to_current_decision)
print(json.dumps(proposal_d_tool_decision.materiality_basis, indent=2))
print()
print("TOOL NEEDED:", proposal_d_tool_decision.tool_needed)
print("TOOL SELECTED:", proposal_d_tool_decision.tool_name)
print("WHY SELECTED:", proposal_d_tool_decision.why_tool_needed)
print("EXPECTED INFORMATION GAIN:", proposal_d_tool_decision.expected_information_gain)
print("QUERY:", proposal_d_tool_decision.query)
print()
print("TOOL RESULTS:")
for item in proposal_d_tool_results:
    print(
        f"- call={item['call_number']} rank={item['rank']} "
        f"source={item['source']}"
    )
    print(f"  source_id={item['source_id']}")
    print(f"  type={item['source_type']} date/freshness={item['freshness']}")
    print(f"  score={item['retrieval_score']}")
    print(f"  evidence={item['relevant_evidence']}")

print()
print("INTERPRETED EFFECT ON UNCERTAINTY:")
if proposal_d_interpretation is None:
    print("No interpretation: no tool was justified.")
else:
    print(json.dumps(proposal_d_interpretation.model_dump(), indent=2))

print()
print("TOOL CALLS USED / REMAINING:")
print(f"{tool_executor.calls_used} used / {tool_executor.calls_remaining} remaining")
print("STOP REASON:", proposal_d_stop_reason)
print("ORIGINAL ITERATION-2 READINESS PRESERVED:", proposal_d_original_readiness)


========== ITERATION-3 v0.1 — PROPOSAL D TOOL TRACE ==========
UNRESOLVED UNCERTAINTY CONSIDERED:
Feasibility of onboarding at least 8 qualified industrial fasteners suppliers within 4 weeks (supplier availability and typical onboarding timelines/costs in the market).

MATERIALITY TO CURRENT DECISION:
True
[
  "Supplier onboarding speed is required to generate product availability and sales during the 10-week pilot, affecting GMV, margin, and OTIF metrics.",
  "Unknown supplier pipeline and lack of supplier recruitment plan make the onboarding assumption a key dependency; if false, the pilot may not produce interpretable results and budget/resourcing needs would change."
]

TOOL NEEDED: True
TOOL SELECTED: external_search
WHY SELECTED: Public market data and industry benchmarks can indicate typical supplier onboarding timelines, common barriers, and estimated acquisition/recruitment costs for industrial fasteners suppliers—information not available internally and directly relevant to w

### 8. Experiment boundary and regression protection

- External evidence is advisory experiment output only.
- It does not update Proposal D state or re-run readiness, blocker, caveat, question, action, or final-brief nodes.
- The frozen Iteration-2 fixtures and deterministic A–D regression above are unchanged.
- Full Northstar text remains local. Tavily receives only concise public search queries.
- `MAX_TOOL_CALLS = 2` is enforced by the wrapper, including failed or empty searches after dispatch.


### 9. Persisted v0.1 experiment conclusion

- **Schema/state:** Added compact `ToolDecision`, per-source interpretation, and overall `ToolEvidenceInterpretation` records. These are experiment-local and do not alter Iteration-2 graph state.
- **Decision logic:** The five eligibility gates and actual tool-capability alignment are explicit and logged. The tool selector chose `external_search`.
- **Uncertainty selected:** Feasibility of onboarding at least eight qualified industrial-fasteners suppliers within four weeks, rather than the stale 2023 market benchmark.
- **Query executed:** One bounded query for public supplier-onboarding timelines, costs, and benchmarks. No Northstar text, customer records, supplier identities, or full proposal text was sent to Tavily.
- **Evidence retrieved:** Five public vendor/practitioner pages. Results described timelines ranging from hours or days to weeks or months, possible automation benefits, and one manual-cost estimate. Tavily supplied no publication dates.
- **Interpretation:** Evidence both strengthened and weakened feasibility depending on process maturity and automation. It was generic, not industrial-fasteners-specific, and could not resolve this company's supplier pipeline or parallel onboarding capacity.
- **Stop:** `evidence_inconclusive` after **1 of 2** allowed calls; one call remained. The stop rule correctly avoided an open-ended search even though a follow-up query might have added context.
- **Decision consequence:** Proposal D remains `READY_WITH_CAVEATS`; no blocker, caveat, question, action, final brief, or readiness node was re-run from external evidence.
- **v0.2 readiness:** The execution and stop controls are ready to carry forward, but tool-choice quality is not yet ready for graph integration. A v0.2 selector should rank evidence-freshness questions above company-specific feasibility questions when only public search is available, or return `none` when the available tool cannot resolve the specific uncertainty.


## Iteration 3 v0.2 — Resolvability-First Tool Selection

This experiment improves tool-choice quality by assessing **every directly evidenced Proposal D gap before selecting one**. Resolvability and actual tool capability are explicit gates: a severe company-specific uncertainty cannot outrank a genuinely searchable public-evidence gap merely because it is more concerning.

The experiment remains isolated from the canonical graph. It uses the frozen Iteration-2 v0.5 Proposal D state and does not change readiness, blockers, caveats, questions, actions, or the final brief.


### 1. Evidence-gap assessment schemas

In [82]:
GapResolvability = Literal[
    "internally_resolvable",
    "externally_resolvable",
    "tool_insufficient",
]
InformationGain = Literal["high", "medium", "low"]


class EvidenceGapAssessment(BaseModel):
    candidate_id: str
    uncertainty: str
    material_to_current_decision: bool
    resolvability: GapResolvability
    best_tool: AllowedToolName
    expected_information_gain: InformationGain
    could_improve_decision_readiness_assessment: bool
    rationale: str
    source_state_refs: list[str] = Field(default_factory=list)


class EvidenceGapAssessmentSet(BaseModel):
    assessments: list[EvidenceGapAssessment]


class GapSearchQuery(BaseModel):
    candidate_id: str
    query: str
    query_rationale: str


class SecondCallDecision(BaseModel):
    use_second_call: bool
    candidate_id: str | None = None
    reason: str


EVIDENCE_GAP_PROMPT = """
Classify every supplied evidence-gap candidate. Return exactly one assessment for each candidate_id.
Use only the supplied frozen state and candidate evidence.

RESOLVABILITY
- externally_resolvable: current public market growth, market size, industry pricing/benchmarks,
  public industry conditions, or time-sensitive external market trends can reasonably answer or
  materially update the question.
- internally_resolvable: the answer materially depends on company-specific evidence such as supplier
  pipeline/capacity, actual purchase intent, company conversion rates, internal pricing, budget
  sufficiency, staffing, or operational capability.
- tool_insufficient: neither available tool can meaningfully resolve the issue, including a commercial
  or execution hypothesis that should primarily be tested by the pilot.

TOOL CAPABILITY
- internal_retrieval contains only Northstar Approval Policy, Pilot Governance Guidelines,
  Risk/Data Governance Guidelines, and Prior Decision Precedents. It does not contain CRM records,
  supplier pipelines, customer intent, private pricing, staffing, or operational dashboards.
- external_search contains current public web evidence only.
- internal_retrieval may be selected only for policy, precedent, or evidence this Northstar corpus could
  reasonably contain.
- external_search may be selected only for externally_resolvable gaps.
- otherwise best_tool must be none.

Materiality is about the current pilot decision, not a later scale decision. Do not invent facts or
requirements. Generic public benchmarks do not make a company-specific question externally resolvable.
"""


SEARCH_QUERY_PROMPT = """
Generate one concise public-web query for the selected externally resolvable evidence gap.
The query should seek recent, credible market or industry evidence and include the relevant public
industry/category. Do not include company names, internal budgets, internal targets, customer records,
supplier identities, Northstar text, or the full proposal. Return only a query appropriate for Tavily.
"""


SECOND_CALL_PROMPT = """
Decide whether one remaining tool call has high incremental value after reviewing the first bounded
search and its grounded interpretation. A second call is allowed only for a distinct, material,
externally resolvable gap with high expected information gain. Do not use it merely because budget
remains. Return false when the first evidence is sufficient, public search cannot resolve further,
the remaining gaps are company-specific/pilot-test hypotheses, or another search is unlikely to
materially improve decision framing.
"""


evidence_gap_model = iteration3_model.with_structured_output(EvidenceGapAssessmentSet)
gap_search_query_model = iteration3_model.with_structured_output(GapSearchQuery)
second_call_decision_model = iteration3_model.with_structured_output(SecondCallDecision)
print("v0.2 resolvability-first structured wrappers initialized")


v0.2 resolvability-first structured wrappers initialized


### 2. Build the complete Proposal D evidence-gap inventory

Candidates are derived from frozen proposal evidence, material claims, unknown dependencies, assessment gaps, and explicit future-decision considerations. No candidate is selected during collection.


In [83]:
proposal_d_v02_state = {
    **copy.deepcopy(FROZEN_EXTRACTION_FIXTURES["D"]),
    **copy.deepcopy(FROZEN_DOWNSTREAM_SNAPSHOTS["D"]),
}
proposal_d_v02_original_readiness = proposal_d_v02_state["readiness_status"]


def collect_evidence_gap_candidates(state: dict) -> list[dict]:
    candidates = []

    def add(kind: str, uncertainty: str, evidence: list[str], state_ref: str) -> None:
        candidates.append({
            "candidate_id": f"D-GAP-{len(candidates) + 1:02d}",
            "candidate_kind": kind,
            "uncertainty": uncertainty,
            "evidence": evidence,
            "source_state_refs": [state_ref],
        })

    limitation_signals = (
        "2023", "lacks recent", "no recent", "informal", "no structured"
    )
    for index, item in enumerate(state.get("evidence_items", [])):
        if any(signal.casefold() in item.casefold() for signal in limitation_signals):
            add(
                "proposal_evidence_limitation",
                item,
                [item],
                f"evidence_items[{index}]",
            )

    for index, claim in enumerate(state.get("material_claims", [])):
        if claim.get("claim_role") in {"hypothesis", "unknown"}:
            add(
                f"material_claim:{claim['claim_role']}",
                claim["claim_text"],
                claim.get("evidence_basis", []),
                f"material_claims[{index}]",
            )

    for index, dependency in enumerate(state.get("dependency_reviews", [])):
        if dependency.get("requirement_status") == "unknown":
            add(
                "unknown_dependency",
                dependency["dependency_text"],
                dependency.get("evidence_basis", []),
                f"dependency_reviews[{index}]",
            )

    for dimension, assessment in state.get("dimension_assessments", {}).items():
        for gap_index, gap in enumerate(assessment.get("gaps", [])):
            add(
                "assessment_gap",
                gap,
                assessment.get("supporting_evidence", []),
                f"dimension_assessments.{dimension}.gaps[{gap_index}]",
            )

    for index, consideration in enumerate(
        state.get("future_decision_considerations", [])
    ):
        add(
            "future_decision",
            consideration.get("related_item_text") or consideration.get("rationale", ""),
            consideration.get("evidence", []),
            f"future_decision_considerations[{index}]",
        )

    return candidates


proposal_d_gap_candidates = collect_evidence_gap_candidates(proposal_d_v02_state)
print(f"Collected {len(proposal_d_gap_candidates)} directly evidenced candidates")
for candidate in proposal_d_gap_candidates:
    print(
        candidate["candidate_id"],
        candidate["candidate_kind"],
        "—",
        candidate["uncertainty"],
    )


Collected 27 directly evidenced candidates
D-GAP-01 proposal_evidence_limitation — 2023 industry report estimating annual market growth of approximately 8-10% and noting fragmented pricing across the category
D-GAP-02 proposal_evidence_limitation — Current internal data lacks recent market-size or pricing benchmark for this category
D-GAP-03 proposal_evidence_limitation — Customer demand has been discussed informally with several account managers but no structured customer validation completed
D-GAP-04 material_claim:hypothesis — There is sufficient customer demand to support the fasteners category (target: at least 5 active buying customers and $250,000 GMV during the pilot).
D-GAP-05 material_claim:hypothesis — Supplier economics will allow achieving a gross margin of at least 6%.
D-GAP-06 material_claim:hypothesis — Operational execution can meet OTIF of at least 90% while serving the category.
D-GAP-07 material_claim:hypothesis — It is feasible to onboard at least 8 qualified suppl

### 3. Classify all gaps before prioritization

In [84]:
def _normalize_gap_assessment(
    assessment: EvidenceGapAssessment,
    candidate: dict,
) -> dict:
    item = assessment.model_dump()
    notes = []
    uncertainty_text = candidate["uncertainty"].casefold()
    candidate_kind = candidate["candidate_kind"]

    externally_verifiable = any(
        term in uncertainty_text
        for term in (
            "market-size", "market size", "market growth", "industry report",
            "industry pricing", "pricing benchmark", "market benchmark",
            "current external", "market trend",
        )
    )
    company_specific = any(
        term in uncertainty_text
        for term in (
            "this company", "onboard", "supplier pipeline", "customer validation",
            "customer demand", "purchase intent", "$70k", "$70,000", "resourcing",
            "roles/responsibilities", "customer acquisition", "monitoring cadence",
            "internal operational", "internal pricing",
        )
    )
    pilot_hypothesis = candidate_kind == "material_claim:hypothesis"
    future_only = candidate_kind == "future_decision" or any(
        term in uncertainty_text
        for term in (
            "prior to scaling", "for scaling", "scaling decision",
            "consider scaling", "financial projections for scaling",
        )
    )

    if future_only:
        item["material_to_current_decision"] = False
        item["best_tool"] = "none"
        item["rationale"] = (
            "This uncertainty belongs to a later scale decision and does not affect "
            "tool use for the current bounded-pilot approval."
        )
        notes.append("future-decision item excluded from current tool selection")
    elif pilot_hypothesis:
        item["resolvability"] = "tool_insufficient"
        item["best_tool"] = "none"
        item["rationale"] = (
            "This is a pilot hypothesis whose company-specific outcome should be "
            "measured through the pilot rather than resolved by Northstar retrieval "
            "or public search."
        )
        notes.append("pilot hypothesis must be tested by the pilot, not resolved by search")
    elif externally_verifiable and not company_specific:
        item["resolvability"] = "externally_resolvable"
        item["best_tool"] = "external_search"
        item["rationale"] = (
            "Recent public market or industry evidence can materially update this "
            "benchmark gap, although it cannot replace company-specific internal data."
        )
        notes.append("public market/benchmark evidence is capability-aligned")
    elif company_specific:
        item["resolvability"] = "internally_resolvable"
        item["best_tool"] = "none"
        item["rationale"] = (
            "The answer depends on company-specific evidence not present in the "
            "controlled Northstar policy/precedent corpus; public benchmarks cannot "
            "resolve it."
        )
        notes.append("company-specific evidence is outside the controlled Northstar corpus")

    if item["resolvability"] == "externally_resolvable" and item["best_tool"] != "external_search":
        item["best_tool"] = "external_search"
        notes.append("aligned externally resolvable gap to external_search")
    elif item["resolvability"] == "tool_insufficient":
        item["best_tool"] = "none"
    elif item["resolvability"] == "internally_resolvable" and item["best_tool"] == "internal_retrieval":
        corpus_fit = any(
            term in uncertainty_text
            for term in ("policy", "governance", "approval", "guideline", "precedent")
        )
        if not corpus_fit:
            item["best_tool"] = "none"
            notes.append("Northstar corpus cannot answer this internal operational question")

    item["source_state_refs"] = candidate["source_state_refs"]
    item["candidate_kind"] = candidate_kind
    item["validation_notes"] = notes
    return item


raw_gap_assessments = evidence_gap_model.invoke(
    [
        SystemMessage(content=EVIDENCE_GAP_PROMPT),
        HumanMessage(content=json.dumps(proposal_d_gap_candidates, indent=2)),
    ]
)

candidate_by_id = {
    candidate["candidate_id"]: candidate
    for candidate in proposal_d_gap_candidates
}
returned_ids = [item.candidate_id for item in raw_gap_assessments.assessments]
expected_ids = list(candidate_by_id)
if sorted(returned_ids) != sorted(expected_ids):
    raise ValueError(
        f"Evidence-gap model must return every candidate exactly once; "
        f"expected={expected_ids}, returned={returned_ids}"
    )

proposal_d_gap_assessments = [
    _normalize_gap_assessment(item, candidate_by_id[item.candidate_id])
    for item in raw_gap_assessments.assessments
]
print(f"Classified all {len(proposal_d_gap_assessments)} evidence gaps")


Classified all 27 evidence gaps


### 4. Rank resolvable candidates and choose one tool

Only material gaps aligned to an actually available tool enter selection. Among eligible gaps, ranking uses materiality, expected information gain, and potential effect on decision-readiness framing. Non-resolvable gaps are retained in the table but cannot win.


In [85]:
GAIN_RANK = {"high": 3, "medium": 2, "low": 1}


def rank_gap(item: dict) -> tuple[int, int, int, int]:
    tool_eligible = (
        item["material_to_current_decision"]
        and item["best_tool"] in {"internal_retrieval", "external_search"}
    )
    return (
        int(tool_eligible),
        int(item["material_to_current_decision"]),
        GAIN_RANK[item["expected_information_gain"]],
        int(item["could_improve_decision_readiness_assessment"]),
    )


proposal_d_ranked_gaps = sorted(
    proposal_d_gap_assessments,
    key=rank_gap,
    reverse=True,
)
eligible_gaps = [
    item
    for item in proposal_d_ranked_gaps
    if item["material_to_current_decision"]
    and item["best_tool"] in {"internal_retrieval", "external_search"}
]
proposal_d_selected_gap = eligible_gaps[0] if eligible_gaps else None

for item in proposal_d_gap_assessments:
    item["selected"] = bool(
        proposal_d_selected_gap
        and item["candidate_id"] == proposal_d_selected_gap["candidate_id"]
    )
    if item["selected"]:
        item["selection_reason"] = "highest-ranked capability-aligned material gap"
    elif item["best_tool"] == "none":
        item["selection_reason"] = "available tools cannot materially resolve this gap"
    else:
        item["selection_reason"] = "lower incremental value than selected eligible gap"

proposal_d_v02_query_plan = None
if proposal_d_selected_gap:
    proposal_d_v02_query_plan = gap_search_query_model.invoke(
        [
            SystemMessage(content=SEARCH_QUERY_PROMPT),
            HumanMessage(
                content=json.dumps(
                    {
                        "candidate_id": proposal_d_selected_gap["candidate_id"],
                        "uncertainty": proposal_d_selected_gap["uncertainty"],
                        "public_category_context": "industrial fasteners market",
                    },
                    indent=2,
                )
            ),
        ]
    )
    if proposal_d_v02_query_plan.candidate_id != proposal_d_selected_gap["candidate_id"]:
        raise ValueError("Query plan candidate does not match selected gap")

print("Selected gap:")
print(json.dumps(proposal_d_selected_gap, indent=2))
print("Initial query plan:")
print(
    json.dumps(
        proposal_d_v02_query_plan.model_dump() if proposal_d_v02_query_plan else None,
        indent=2,
    )
)


Selected gap:
{
  "candidate_id": "D-GAP-02",
  "uncertainty": "Current internal data lacks recent market-size or pricing benchmark for this category",
  "material_to_current_decision": true,
  "resolvability": "externally_resolvable",
  "best_tool": "external_search",
  "expected_information_gain": "high",
  "could_improve_decision_readiness_assessment": true,
  "rationale": "Recent public market or industry evidence can materially update this benchmark gap, although it cannot replace company-specific internal data.",
  "source_state_refs": [
    "evidence_items[3]"
  ],
  "candidate_kind": "proposal_evidence_limitation",
  "validation_notes": [
    "public market/benchmark evidence is capability-aligned"
  ],
  "selected": true,
  "selection_reason": "highest-ranked capability-aligned material gap"
}
Initial query plan:
{
  "candidate_id": "D-GAP-02",
  "query": "Recent market-size, growth forecasts, and pricing benchmarks for the industrial fasteners market (bolts, nuts, screws, was

### 5. Execute, interpret, and decide whether a second call is justified

In [86]:
def interpret_v02_tool_evidence(
    gap: dict,
    evidence_records: list[dict],
) -> ToolEvidenceInterpretation:
    concise_records = [
        {
            "source_id": item["source_id"],
            "source": item["source"],
            "source_type": item["source_type"],
            "source_date": item["source_date"],
            "freshness": item["freshness"],
            "retrieval_score": item["retrieval_score"],
            "relevant_evidence": item["relevant_evidence"],
        }
        for item in evidence_records
    ]
    return tool_evidence_interpretation_model.invoke(
        [
            SystemMessage(content=TOOL_INTERPRETATION_PROMPT),
            HumanMessage(
                content=json.dumps(
                    {
                        "related_uncertainty": gap["uncertainty"],
                        "expected_information_gain": gap["expected_information_gain"],
                        "evidence_records": concise_records,
                    },
                    indent=2,
                )
            ),
        ]
    )


def build_query_for_gap(gap: dict) -> GapSearchQuery:
    plan = gap_search_query_model.invoke(
        [
            SystemMessage(content=SEARCH_QUERY_PROMPT),
            HumanMessage(
                content=json.dumps(
                    {
                        "candidate_id": gap["candidate_id"],
                        "uncertainty": gap["uncertainty"],
                        "public_category_context": "industrial fasteners market",
                    },
                    indent=2,
                )
            ),
        ]
    )
    if plan.candidate_id != gap["candidate_id"]:
        raise ValueError("Query plan candidate does not match selected gap")
    return plan


v02_tool_executor = BoundedProjectToolExecutor(MAX_TOOL_CALLS)
proposal_d_v02_call_records = []
proposal_d_v02_second_call_decision = None
proposal_d_v02_stop_reason = "no_tool_justified"

if proposal_d_selected_gap and proposal_d_v02_query_plan:
    first_results = v02_tool_executor.execute(
        proposal_d_selected_gap["best_tool"],
        proposal_d_v02_query_plan.query,
    )
    first_interpretation = interpret_v02_tool_evidence(
        proposal_d_selected_gap,
        first_results,
    )
    proposal_d_v02_call_records.append({
        "gap": proposal_d_selected_gap,
        "query_plan": proposal_d_v02_query_plan.model_dump(),
        "results": first_results,
        "interpretation": first_interpretation.model_dump(),
    })

    if first_interpretation.sufficient_for_specific_uncertainty:
        proposal_d_v02_stop_reason = "sufficient_evidence_for_selected_uncertainty"
    else:
        remaining_external = [
            item
            for item in eligible_gaps[1:]
            if item["best_tool"] == "external_search"
            and item["expected_information_gain"] == "high"
            and item["candidate_id"] != proposal_d_selected_gap["candidate_id"]
        ]
        proposal_d_v02_second_call_decision = second_call_decision_model.invoke(
            [
                SystemMessage(content=SECOND_CALL_PROMPT),
                HumanMessage(
                    content=json.dumps(
                        {
                            "first_gap": proposal_d_selected_gap,
                            "first_interpretation": first_interpretation.model_dump(),
                            "remaining_distinct_high_value_external_gaps": remaining_external,
                            "calls_remaining": v02_tool_executor.calls_remaining,
                        },
                        indent=2,
                    )
                ),
            ]
        )

        second_gap = next(
            (
                item
                for item in remaining_external
                if item["candidate_id"]
                == proposal_d_v02_second_call_decision.candidate_id
            ),
            None,
        )
        second_allowed = (
            proposal_d_v02_second_call_decision.use_second_call
            and second_gap is not None
            and v02_tool_executor.calls_remaining > 0
        )
        if second_allowed:
            second_query = build_query_for_gap(second_gap)
            second_results = v02_tool_executor.execute(
                second_gap["best_tool"],
                second_query.query,
            )
            second_interpretation = interpret_v02_tool_evidence(
                second_gap,
                second_results,
            )
            proposal_d_v02_call_records.append({
                "gap": second_gap,
                "query_plan": second_query.model_dump(),
                "results": second_results,
                "interpretation": second_interpretation.model_dump(),
            })
            if second_interpretation.sufficient_for_specific_uncertainty:
                proposal_d_v02_stop_reason = "sufficient_evidence_for_selected_uncertainties"
            else:
                proposal_d_v02_stop_reason = "tool_call_budget_exhausted"
        elif first_interpretation.overall_effect == "unresolved":
            proposal_d_v02_stop_reason = "evidence_inconclusive_public_search_cannot_resolve_further"
        else:
            proposal_d_v02_stop_reason = "additional_search_unlikely_to_improve_case"

assert v02_tool_executor.calls_used <= MAX_TOOL_CALLS
assert proposal_d_v02_state["readiness_status"] == proposal_d_v02_original_readiness
print(
    f"v0.2 bounded experiment completed with {v02_tool_executor.calls_used}/"
    f"{MAX_TOOL_CALLS} tool calls"
)


v0.2 bounded experiment completed with 1/2 tool calls


### 6. Proposal D resolvability table and tool trace

In [87]:
def _short(value: str, width: int = 86) -> str:
    value = " ".join(value.split()).replace("|", "/")
    return value if len(value) <= width else value[: width - 1] + "…"


print("| ID | Uncertainty | Material | Resolvability | Candidate tool | Info gain | Selected | Reason |")
print("|---|---|---:|---|---|---|---:|---|")
for item in proposal_d_gap_assessments:
    print(
        f"| {item['candidate_id']} "
        f"| {_short(item['uncertainty'])} "
        f"| {item['material_to_current_decision']} "
        f"| {item['resolvability']} "
        f"| {item['best_tool']} "
        f"| {item['expected_information_gain']} "
        f"| {item['selected']} "
        f"| {_short(item['selection_reason'])} |"
    )

print()
print("FINAL TOOL DECISION:")
print(json.dumps(proposal_d_selected_gap, indent=2))
print()
print("CALL TRACE:")
for call in proposal_d_v02_call_records:
    print("=" * 72)
    print("UNCERTAINTY:", call["gap"]["uncertainty"])
    print("TOOL:", call["gap"]["best_tool"])
    print("QUERY:", call["query_plan"]["query"])
    print("QUERY RATIONALE:", call["query_plan"]["query_rationale"])
    print("EVIDENCE:")
    for result in call["results"]:
        print(
            f"- rank={result['rank']} source={result['source']} "
            f"date/freshness={result['freshness']} score={result['retrieval_score']}"
        )
        print("  source_id=", result["source_id"])
        print("  evidence=", result["relevant_evidence"])
    print("INTERPRETATION:")
    print(json.dumps(call["interpretation"], indent=2))

print()
print("SECOND-CALL DECISION:")
print(
    json.dumps(
        proposal_d_v02_second_call_decision.model_dump()
        if proposal_d_v02_second_call_decision
        else None,
        indent=2,
    )
)
print("TOOL CALLS USED / REMAINING:")
print(f"{v02_tool_executor.calls_used} used / {v02_tool_executor.calls_remaining} remaining")
print("STOP REASON:", proposal_d_v02_stop_reason)
print("ITERATION-2 READINESS PRESERVED:", proposal_d_v02_original_readiness)


| ID | Uncertainty | Material | Resolvability | Candidate tool | Info gain | Selected | Reason |
|---|---|---:|---|---|---|---:|---|
| D-GAP-01 | 2023 industry report estimating annual market growth of approximately 8-10% and notin… | False | externally_resolvable | external_search | medium | False | lower incremental value than selected eligible gap |
| D-GAP-02 | Current internal data lacks recent market-size or pricing benchmark for this category | True | externally_resolvable | external_search | high | True | highest-ranked capability-aligned material gap |
| D-GAP-03 | Customer demand has been discussed informally with several account managers but no st… | True | internally_resolvable | none | high | False | available tools cannot materially resolve this gap |
| D-GAP-04 | There is sufficient customer demand to support the fasteners category (target: at lea… | True | tool_insufficient | none | high | False | available tools cannot materially resolve this gap |
| D-GAP-05 | Supplie

### 7. Experiment boundary

- `MAX_TOOL_CALLS = 2` and the v0.1 privacy boundary remain unchanged.
- Only generated public-market queries are sent to Tavily; Northstar text, full proposal text, customer records, and supplier identities remain local.
- Search evidence is interpreted before any conclusion and cannot change readiness in this notebook.
- The frozen Iteration-2 graph, fixtures, outputs, and deterministic regression remain untouched.


### 8. Persisted v0.2 conclusion

#### Difference from v0.1

v0.1 selected a company-specific supplier-onboarding feasibility question for public search. v0.2 first classified all 27 directly evidenced candidates, enforced resolvability and actual corpus capability, and only then ranked eligible gaps. This prevented supplier pipeline, customer intent, internal execution, and pilot-test hypotheses from being sent to Tavily.

#### Proposal D classification summary

- **Externally resolvable:** the stale 2023 industry context and missing current market-size/pricing benchmarks (`D-GAP-01`, `D-GAP-02`, `D-GAP-11`). The current benchmark gap was material and highest-value; the stale report itself was retained but assessed as less directly material.
- **Internally resolvable but unavailable to current tools:** informal customer validation, supplier pipeline/onboarding capacity, budget sufficiency, company pricing/margin baselines, resourcing, acquisition plans, monitoring, contingencies, and related operating evidence. The controlled Northstar corpus cannot answer these items, so their candidate tool is `none`.
- **Tool insufficient:** customer demand, margin, OTIF, and supplier-onboarding outcome hypotheses (`D-GAP-04`–`D-GAP-07`) should be tested by the pilot rather than resolved by search.
- **Future decision:** pre-scaling validation, scaling projections, and the explicit later scale decision are not material to present pilot tool selection (`D-GAP-08`, `D-GAP-10`, `D-GAP-16`, `D-GAP-27`).

#### Selected tool and evidence

The selector chose `external_search` for the missing current industrial-fasteners market-size/pricing benchmark. One bounded Tavily query retrieved five results: ResearchAndMarkets content syndicated by GlobeNewswire, an Allied Market Research release syndicated by EIN Presswire, DataIntelo, Mordor Intelligence, and a duplicate GlobeNewswire URL for the ResearchAndMarkets report. The results consistently indicated a mid-2020s global market around $88–$96 billion and projected CAGR around 3.5%–4.5%. They also described raw-material and engineered-product price drivers, but did not provide a robust comparable average-selling-price series.

The interpreter marked the evidence as strengthening and sufficient for a high-level market benchmark. Source quality remains mixed: Tavily did not populate structured publication dates; two results duplicate the same underlying report; most sources are commercial market-research summaries or syndicated releases; and pricing evidence is qualitative rather than a directly comparable benchmark. Evidence improves pilot framing but does not validate company-specific demand, margins, supplier feasibility, or internal pricing.

#### Stop behavior and next step

The canonical run stopped with `sufficient_evidence_for_selected_uncertainty` after **1 of 2** calls. The unused call was not consumed merely because it remained. Proposal D stayed `READY_WITH_CAVEATS`, and no canonical decision node was re-run.

Selector quality is strong enough for a later **trace-only graph-integration experiment**, but not yet for evidence-driven status changes. Remaining improvements should deduplicate overlapping gap candidates, add source-authority/diversity checks, distinguish market-size sufficiency from pricing sufficiency, and reject interpretation claims of independent corroboration when results syndicate the same underlying report.


## Iteration 3 v0.3 — Canonical Evidence Resolution

This isolated experiment hardens the evidence-resolution layer without integrating Iteration 3 into the canonical readiness graph. It canonicalizes overlapping raw gaps before ranking, converts deficiencies into answerable evidence questions, decomposes compound needs, deduplicates source provenance, interprets evidence against the original proposal claim, and applies subquestion-level sufficiency and consistent stop logic.

The frozen Iteration-2 state, `MAX_TOOL_CALLS = 2`, privacy boundary, and prohibition on direct readiness changes remain unchanged.


### 1. Canonical evidence, subquestion, provenance, and resolution schemas

In [118]:
V03Resolvability = Literal[
    "internally_resolvable",
    "externally_resolvable",
    "tool_insufficient",
]
V03InformationGain = Literal["high", "medium", "low"]
SubquestionStatus = Literal["resolved", "partially_resolved", "unresolved"]
ClaimEffect = Literal[
    "supports", "weakens", "contradicts", "mixed", "neutral", "inconclusive"
]
V03SourceType = Literal[
    "primary_authoritative",
    "trade_industry",
    "commercial_research",
    "syndicated_release",
    "practitioner_vendor",
    "other",
]


class CanonicalEvidenceNeed(BaseModel):
    canonical_id: str
    evidence_question: str
    original_gap_ids: list[str]
    material_to_current_decision: bool
    resolvability: V03Resolvability
    best_tool: AllowedToolName
    expected_information_gain: V03InformationGain
    could_improve_decision_readiness_assessment: bool
    related_original_claims: list[str] = Field(default_factory=list)
    rationale: str


class CanonicalEvidenceNeedSet(BaseModel):
    needs: list[CanonicalEvidenceNeed]


class EvidenceSubquestion(BaseModel):
    subquestion_id: str
    question: str
    material: bool
    externally_resolvable: bool
    original_claim_text: str | None = None
    why_separate: str


class EvidenceSubquestionSet(BaseModel):
    subquestions: list[EvidenceSubquestion]


class V03QueryPlan(BaseModel):
    query: str
    targeted_subquestion_ids: list[str]
    rationale: str


class SourceProvenanceAssessment(BaseModel):
    source_id: str
    source_type: V03SourceType
    underlying_source: str
    independent_source_group: str
    is_duplicate_distribution: bool
    authority_rationale: str


class SourceProvenanceSet(BaseModel):
    sources: list[SourceProvenanceAssessment]


class SubquestionResolution(BaseModel):
    subquestion_id: str
    status: SubquestionStatus
    effect_on_original_claim: ClaimEffect
    answer: str
    numeric_comparison: str | None = None
    supporting_source_ids: list[str] = Field(default_factory=list)
    supporting_independent_source_groups: list[str] = Field(default_factory=list)
    sufficient: bool
    remaining_gap: str


class EvidenceResolutionSet(BaseModel):
    resolutions: list[SubquestionResolution]
    synthesis: str


class V03SecondCallDecision(BaseModel):
    use_second_call: bool
    targeted_subquestion_ids: list[str] = Field(default_factory=list)
    expected_incremental_information_gain: V03InformationGain
    rationale: str
    query_focus: str | None = None


CANONICALIZATION_PROMPT = """
Convert every supplied raw evidence-gap candidate into a smaller set of canonical evidence needs.
Each raw candidate_id must map to exactly one canonical need. Merge semantic duplicates and near-duplicates,
including evidence statements, claims, dependencies, and assessment gaps that express the same underlying
question. Do not choose a fixed number of needs.

An evidence deficiency is not automatically an answerable question. Rewrite each canonical need as an
explicit evidence question. Public search can update current public market size, growth, pricing benchmarks,
industry conditions, and trends. Company-specific supplier pipeline, customer intent, internal economics,
staffing, and execution capacity require company evidence not present in the controlled Northstar corpus.
Pilot outcome hypotheses should remain tool_insufficient. Future scale questions are not material to the
current pilot decision.

Northstar internal_retrieval contains only Approval Policy, Pilot Governance Guidelines, Risk/Data Governance
Guidelines, and Prior Decision Precedents. Select it only for questions that corpus could reasonably answer.
Select external_search only for externally_resolvable questions; otherwise select none.
Preserve relevant exact original proposal claims in related_original_claims.
"""


SUBQUESTION_PROMPT = """
Decompose the selected canonical evidence question into distinct material subquestions before retrieval.
Separate market size/current growth, comparison with any original numeric growth claim, quantitative pricing
or ASP benchmarks, and geographic/product relevance when they are distinct. Do not force these examples when
not relevant. Each subquestion must have its own sufficiency outcome later. Copy the exact original claim text
when a subquestion evaluates one. Return concise answerable questions.
"""


V03_QUERY_PROMPT = """
Create one concise public-web query for the supplied targeted subquestions. Seek recent evidence and prefer
primary/authoritative statistics, trade associations, and clearly identified commercial research. Include
the public category but exclude company names, internal budgets/targets, customer records, supplier identities,
Northstar text, and full proposal text. The query must directly target the listed subquestion IDs.
"""


SOURCE_PROVENANCE_PROMPT = """
Classify every supplied result by source type and underlying evidence producer. Identify duplicate URLs,
alternate distribution URLs, syndicated releases, and multiple pages that repeat the same underlying report.
Assign the same independent_source_group to duplicates or syndications of the same report. Commercial research
and syndicated summaries are not primary authoritative evidence merely because they contain numeric claims.
Return exactly one record for every source_id and do not invent publication provenance.
"""


CLAIM_RELATIVE_INTERPRETATION_PROMPT = """
Assess every evidence subquestion using only the supplied search results and source-provenance records.
Evaluate evidence against the exact original proposal claim where one is supplied. For numeric claims, compare
direction and magnitude where possible. Evidence that a market merely exists or grows does not automatically
support a specific growth-rate assertion. Market size/growth evidence does not automatically resolve pricing.

Use independent_source_groups rather than raw result count. Duplicate distribution must not increase confidence.
Mark a subquestion resolved and sufficient only when its material question is adequately answered by relevant,
sufficiently independent evidence. Preserve disagreement, weak authority, missing dates, geographic mismatch,
and absent quantitative pricing as remaining gaps. Return exactly one resolution per subquestion_id.
"""


V03_SECOND_CALL_PROMPT = """
Decide whether a second bounded public search is justified after call 1. Use it only when a distinct material
subquestion remains partially resolved or unresolved, is externally resolvable, and a narrow query has high
expected incremental information gain. Do not use it because budget remains. Return false when public search
is unlikely to close the gap, the missing evidence is company-specific, or all material subquestions are
sufficient. Target only unresolved subquestion IDs.
"""


canonical_need_model = iteration3_model.with_structured_output(CanonicalEvidenceNeedSet)
subquestion_model = iteration3_model.with_structured_output(EvidenceSubquestionSet)
v03_query_model = iteration3_model.with_structured_output(V03QueryPlan)
source_provenance_model = iteration3_model.with_structured_output(SourceProvenanceSet)
claim_relative_resolution_model = iteration3_model.with_structured_output(EvidenceResolutionSet)
v03_second_call_model = iteration3_model.with_structured_output(V03SecondCallDecision)
print("v0.3 evidence-resolution wrappers initialized")


v0.3 evidence-resolution wrappers initialized


### 2. Canonicalize all raw Proposal D evidence gaps before scoring

In [119]:
proposal_d_v03_state = {
    **copy.deepcopy(FROZEN_EXTRACTION_FIXTURES["D"]),
    **copy.deepcopy(FROZEN_DOWNSTREAM_SNAPSHOTS["D"]),
}
proposal_d_v03_original_readiness = proposal_d_v03_state["readiness_status"]
proposal_d_v03_raw_gaps = collect_evidence_gap_candidates(proposal_d_v03_state)


raw_canonical_result = canonical_need_model.invoke(
    [
        SystemMessage(content=CANONICALIZATION_PROMPT),
        HumanMessage(content=json.dumps(proposal_d_v03_raw_gaps, indent=2)),
    ]
)

raw_ids = [item["candidate_id"] for item in proposal_d_v03_raw_gaps]
raw_by_id = {item["candidate_id"]: item for item in proposal_d_v03_raw_gaps}


def _tokens(value: str) -> set[str]:
    return {
        token
        for token in re.findall(r"[a-z0-9]+", value.casefold())
        if len(token) > 2
    }


def repair_canonical_coverage(model_needs: list[CanonicalEvidenceNeed]) -> list[dict]:
    repaired = []
    seen_raw_ids = set()
    valid_raw_ids = set(raw_ids)

    for model_need in model_needs:
        item = model_need.model_dump()
        item["original_gap_ids"] = [
            gap_id
            for gap_id in item["original_gap_ids"]
            if gap_id in valid_raw_ids and gap_id not in seen_raw_ids
        ]
        if item["original_gap_ids"]:
            seen_raw_ids.update(item["original_gap_ids"])
            repaired.append(item)

    for missing_id in [gap_id for gap_id in raw_ids if gap_id not in seen_raw_ids]:
        missing_tokens = _tokens(raw_by_id[missing_id]["uncertainty"])
        best_item = None
        best_score = 0.0
        for item in repaired:
            comparison_text = " ".join(
                [item["evidence_question"]]
                + [
                    raw_by_id[gap_id]["uncertainty"]
                    for gap_id in item["original_gap_ids"]
                ]
            )
            comparison_tokens = _tokens(comparison_text)
            score = (
                len(missing_tokens & comparison_tokens)
                / max(1, len(missing_tokens | comparison_tokens))
            )
            if score > best_score:
                best_item = item
                best_score = score

        if best_item is not None and best_score >= 0.12:
            best_item["original_gap_ids"].append(missing_id)
        else:
            missing = raw_by_id[missing_id]
            repaired.append({
                "canonical_id": f"coverage-fallback-{missing_id}",
                "evidence_question": (
                    "What evidence, if any, can resolve this gap: "
                    + missing["uncertainty"].rstrip(". ")
                    + "?"
                ),
                "original_gap_ids": [missing_id],
                "material_to_current_decision": missing["candidate_kind"] != "future_decision",
                "resolvability": "tool_insufficient",
                "best_tool": "none",
                "expected_information_gain": "medium",
                "could_improve_decision_readiness_assessment": False,
                "related_original_claims": [],
                "rationale": "Coverage fallback: available tools are not assumed sufficient without explicit classification evidence.",
            })
    return repaired


proposal_d_v03_canonical_needs = repair_canonical_coverage(
    raw_canonical_result.needs
)
required_overlap_groups = (
    ("D-GAP-02", "D-GAP-11"),
    ("D-GAP-03", "D-GAP-12"),
    ("D-GAP-07", "D-GAP-14", "D-GAP-21"),
)
for overlap_group in required_overlap_groups:
    matching = [
        need
        for need in proposal_d_v03_canonical_needs
        if set(overlap_group) & set(need["original_gap_ids"])
    ]
    if matching:
        target = matching[0]
        moved_claims = []
        for item in proposal_d_v03_canonical_needs:
            moved_ids = [
                gap_id
                for gap_id in item["original_gap_ids"]
                if gap_id in overlap_group
            ]
            if moved_ids:
                moved_claims.extend(item.get("related_original_claims", []))
            if item is not target:
                item["original_gap_ids"] = [
                    gap_id
                    for gap_id in item["original_gap_ids"]
                    if gap_id not in overlap_group
                ]
        target["original_gap_ids"] = list(dict.fromkeys(
            target["original_gap_ids"] + list(overlap_group)
        ))
        target["related_original_claims"] = list(dict.fromkeys(
            target.get("related_original_claims", []) + moved_claims
        ))
        proposal_d_v03_canonical_needs = [
            item
            for item in proposal_d_v03_canonical_needs
            if item["original_gap_ids"]
        ]

future_gap_ids = {"D-GAP-08", "D-GAP-10", "D-GAP-16", "D-GAP-27"}
future_claims = [
    raw_by_id[gap_id]["uncertainty"] for gap_id in sorted(future_gap_ids)
]
for need in proposal_d_v03_canonical_needs:
    need["original_gap_ids"] = [
        gap_id for gap_id in need["original_gap_ids"] if gap_id not in future_gap_ids
    ]
proposal_d_v03_canonical_needs = [
    need for need in proposal_d_v03_canonical_needs if need["original_gap_ids"]
]
future_need = {
    "canonical_id": "D-CAN-FUTURE",
    "evidence_question": (
        "What evidence and validation should inform a later scale, rollout, or "
        "investment decision after the bounded pilot?"
    ),
    "original_gap_ids": sorted(future_gap_ids),
    "related_original_claims": list(dict.fromkeys(future_claims)),
    "material_to_current_decision": False,
    "resolvability": "tool_insufficient",
    "best_tool": "none",
    "expected_information_gain": "low",
    "could_improve_decision_readiness_assessment": False,
    "rationale": (
        "These questions belong to a later scaling decision and do not affect "
        "evidence resolution for current pilot approval."
    ),
}
proposal_d_v03_canonical_needs.append(future_need)

for index, need in enumerate(proposal_d_v03_canonical_needs, start=1):
    need["canonical_id"] = f"D-CAN-{index:02d}"

mapped_ids = [
    gap_id
    for need in proposal_d_v03_canonical_needs
    for gap_id in need["original_gap_ids"]
]
if sorted(mapped_ids) != sorted(raw_ids) or len(mapped_ids) != len(set(mapped_ids)):
    raise ValueError("Coverage repair failed to map every raw gap exactly once")

gap_to_canonical = {
    gap_id: need["canonical_id"]
    for need in proposal_d_v03_canonical_needs
    for gap_id in need["original_gap_ids"]
}
for overlap_group in required_overlap_groups:
    if len({gap_to_canonical[gap_id] for gap_id in overlap_group}) != 1:
        raise ValueError(f"Known semantic duplicates were not merged: {overlap_group}")

for need in proposal_d_v03_canonical_needs:
    if not need["evidence_question"].strip().endswith("?"):
        need["evidence_question"] = need["evidence_question"].rstrip(". ") + "?"

print("RAW GAP COUNT:", len(proposal_d_v03_raw_gaps))
print("CANONICAL GAP COUNT:", len(proposal_d_v03_canonical_needs))
print("RAW GAPS:")
for item in proposal_d_v03_raw_gaps:
    print(item["candidate_id"], "—", item["uncertainty"])
print("CANONICAL GAPS AND MAPPINGS:")
for need in proposal_d_v03_canonical_needs:
    print(json.dumps(need, indent=2))


RAW GAP COUNT: 27
CANONICAL GAP COUNT: 10
RAW GAPS:
D-GAP-01 — 2023 industry report estimating annual market growth of approximately 8-10% and noting fragmented pricing across the category
D-GAP-02 — Current internal data lacks recent market-size or pricing benchmark for this category
D-GAP-03 — Customer demand has been discussed informally with several account managers but no structured customer validation completed
D-GAP-04 — There is sufficient customer demand to support the fasteners category (target: at least 5 active buying customers and $250,000 GMV during the pilot).
D-GAP-05 — Supplier economics will allow achieving a gross margin of at least 6%.
D-GAP-06 — Operational execution can meet OTIF of at least 90% while serving the category.
D-GAP-07 — It is feasible to onboard at least 8 qualified suppliers within 4 weeks.
D-GAP-08 — If pilot meets commercial and execution thresholds, leadership can consider scaling the category.
D-GAP-09 — Onboarding at least 8 qualified suppliers

### 3. Rank canonical needs and select an answerable evidence question

In [120]:
V03_GAIN_RANK = {"high": 3, "medium": 2, "low": 1}


def canonical_rank(need: dict) -> tuple[int, int, int, int]:
    aligned = (
        need["material_to_current_decision"]
        and need["best_tool"] in {"internal_retrieval", "external_search"}
        and (
            (need["resolvability"] == "externally_resolvable" and need["best_tool"] == "external_search")
            or (need["resolvability"] == "internally_resolvable" and need["best_tool"] == "internal_retrieval")
        )
    )
    return (
        int(aligned),
        int(need["material_to_current_decision"]),
        V03_GAIN_RANK[need["expected_information_gain"]],
        int(need["could_improve_decision_readiness_assessment"]),
    )


proposal_d_v03_ranked_needs = sorted(
    proposal_d_v03_canonical_needs,
    key=canonical_rank,
    reverse=True,
)
proposal_d_v03_eligible_needs = [
    need
    for need in proposal_d_v03_ranked_needs
    if canonical_rank(need)[0] == 1
]
proposal_d_v03_selected_need = (
    proposal_d_v03_eligible_needs[0]
    if proposal_d_v03_eligible_needs
    else None
)

print("RANKED CANONICAL CANDIDATES:")
for rank, need in enumerate(proposal_d_v03_ranked_needs, start=1):
    print(
        rank,
        need["canonical_id"],
        need["best_tool"],
        need["expected_information_gain"],
        "selected=" + str(
            bool(
                proposal_d_v03_selected_need
                and need["canonical_id"] == proposal_d_v03_selected_need["canonical_id"]
            )
        ),
        "—",
        need["evidence_question"],
    )
print("SELECTED CANONICAL NEED:")
print(json.dumps(proposal_d_v03_selected_need, indent=2))


RANKED CANONICAL CANDIDATES:
1 D-CAN-01 external_search high selected=True — What are current market size, growth rate, and pricing benchmarks for the fasteners category (most recent authoritative external estimates and pricing fragmentation evidence)?
2 D-CAN-03 external_search medium selected=False — Are supplier economics and pricing (costs, typical sell prices, lead times, minimum order sizes) such that achieving a gross margin >=6% is realistic for this category?
3 D-CAN-09 internal_retrieval medium selected=False — What monitoring cadence, interim checkpoints, and explicit stop/go gates should be used during the 10-week pilot to detect and respond to underperformance (metrics, thresholds, and remediation actions)?
4 D-CAN-02 none high selected=False — How strong is customer demand for fasteners among our target customer segments (quantitative validation: likelihood of at least 5 active buying customers and reaching $250k GMV in a 10-week pilot)?
5 D-CAN-04 none high selected=Fals

### 4. Decompose the selected compound need into material subquestions

In [121]:
proposal_d_v03_subquestions = []
if proposal_d_v03_selected_need:
    original_claim_context = [
        item
        for item in proposal_d_v03_state.get("evidence_items", [])
        if any(term in item.casefold() for term in ("growth", "pricing", "market-size", "market size"))
    ]
    subquestion_result = subquestion_model.invoke(
        [
            SystemMessage(content=SUBQUESTION_PROMPT),
            HumanMessage(
                content=json.dumps(
                    {
                        "selected_need": proposal_d_v03_selected_need,
                        "exact_original_evidence_claims": original_claim_context,
                    },
                    indent=2,
                )
            ),
        ]
    )
    proposal_d_v03_subquestions = [
        item.model_dump()
        for item in subquestion_result.subquestions
    ]

subquestion_ids = [item["subquestion_id"] for item in proposal_d_v03_subquestions]
if len(subquestion_ids) != len(set(subquestion_ids)) or not subquestion_ids:
    raise ValueError("Selected evidence need must have distinct subquestions")

print("DECOMPOSED SUBQUESTIONS:")
for item in proposal_d_v03_subquestions:
    print(json.dumps(item, indent=2))


DECOMPOSED SUBQUESTIONS:
{
  "subquestion_id": "SQ1",
  "question": "What is the current (most recent authoritative) global and regional market size (annual revenue / GMV) for the 'fasteners' category (nuts, bolts, screws, rivets, anchors) in USD?",
  "material": true,
  "externally_resolvable": true,
  "original_claim_text": null,
  "why_separate": "Market size is distinct from growth rate and pricing benchmarks and is needed to judge pilot GMV targets and overall opportunity."
}
{
  "subquestion_id": "SQ2",
  "question": "What is the current (most recent authoritative) annual growth rate (CAGR or year-over-year %) for the fasteners category globally and in key regions (e.g., North America, Europe, APAC)?",
  "material": true,
  "externally_resolvable": true,
  "original_claim_text": "2023 industry report estimating annual market growth of approximately 8-10% and noting fragmented pricing across the category",
  "why_separate": "Growth rate directly addresses the original claim's 8-10

### 5. Bounded search and source-independence assessment

In [122]:
def build_v03_query(subquestions: list[dict]) -> V03QueryPlan:
    plan = v03_query_model.invoke(
        [
            SystemMessage(content=V03_QUERY_PROMPT),
            HumanMessage(content=json.dumps(subquestions, indent=2)),
        ]
    )
    valid_ids = {item["subquestion_id"] for item in subquestions}
    if not set(plan.targeted_subquestion_ids).issubset(valid_ids):
        raise ValueError("Query plan references an unknown subquestion")
    return plan


def assess_source_provenance(results: list[dict]) -> list[dict]:
    compact = [
        {
            "source_id": item["source_id"],
            "title": item["source"],
            "url": item["source_url"],
            "source_date": item["source_date"],
            "evidence_excerpt": item["relevant_evidence"][:1800],
        }
        for item in results
    ]
    assessed = source_provenance_model.invoke(
        [
            SystemMessage(content=SOURCE_PROVENANCE_PROMPT),
            HumanMessage(content=json.dumps(compact, indent=2)),
        ]
    )
    expected = {item["source_id"] for item in results}
    returned = [item.source_id for item in assessed.sources]
    if set(returned) != expected or len(returned) != len(set(returned)):
        raise ValueError("Source provenance must cover every result exactly once")

    records = [item.model_dump() for item in assessed.sources]
    # Same normalized title from the same domain is deterministically one source group.
    by_signature = {}
    result_by_id = {item["source_id"]: item for item in results}
    for record in records:
        result = result_by_id[record["source_id"]]
        title_signature = re.sub(r"[^a-z0-9]+", " ", result["source"].casefold()).strip()
        domain = (result.get("source_url") or "").split("/")[2].casefold() if result.get("source_url") else ""
        signature = (domain, title_signature)
        if signature in by_signature:
            record["independent_source_group"] = by_signature[signature]
            record["is_duplicate_distribution"] = True
        else:
            by_signature[signature] = record["independent_source_group"]
    return records


proposal_d_v03_executor = BoundedProjectToolExecutor(MAX_TOOL_CALLS)
proposal_d_v03_query_plans = []
proposal_d_v03_results = []
proposal_d_v03_source_provenance = []

material_external_subquestions = [
    item
    for item in proposal_d_v03_subquestions
    if item["material"] and item["externally_resolvable"]
]
initial_query_plan = build_v03_query(material_external_subquestions)
proposal_d_v03_query_plans.append(initial_query_plan.model_dump())
initial_results = proposal_d_v03_executor.external_search(initial_query_plan.query)
proposal_d_v03_results.extend(initial_results)
proposal_d_v03_source_provenance = assess_source_provenance(proposal_d_v03_results)

print("INITIAL QUERY:")
print(json.dumps(initial_query_plan.model_dump(), indent=2))
print("INITIAL RESULTS:", len(initial_results))
print("INDEPENDENT SOURCE GROUPS:", len({item['independent_source_group'] for item in proposal_d_v03_source_provenance}))


INITIAL QUERY:
{
  "query": "Authoritative recent market data and primary sources (industry trade associations, market research firms, government trade statistics, major distributor or marketplace public reports) on the global and regional (North America, Europe, APAC) fasteners market (nuts, bolts, screws, rivets, anchors): 1) most recent annual market size in USD (annual revenue or GMV) globally and by region; 2) most recent annual growth rates (CAGR or YoY %) globally and by region; 3) representative pricing benchmarks or average selling prices (ASPs) for common fastener groups (standard bolts/nuts, commodity screws, specialty/premium fasteners) across channels (manufacturers, B2B wholesale/distributors, online marketplaces); 4) quantitative evidence of price fragmentation/price dispersion across sellers or platforms (variance in unit prices, range or percentiles, marketplace/distributor catalog examples); 5) typical gross margin benchmarks by channel (manufacturer, distributor, mar

### 6. Claim-relative interpretation, second-call decision, and final stop

In [123]:
def resolve_subquestions(
    subquestions: list[dict],
    results: list[dict],
    provenance: list[dict],
) -> dict:
    compact_results = [
        {
            "source_id": item["source_id"],
            "title": item["source"],
            "source_date": item["source_date"],
            "freshness": item["freshness"],
            "retrieval_score": item["retrieval_score"],
            "evidence": item["relevant_evidence"][:3000],
        }
        for item in results
    ]
    assessed = claim_relative_resolution_model.invoke(
        [
            SystemMessage(content=CLAIM_RELATIVE_INTERPRETATION_PROMPT),
            HumanMessage(
                content=json.dumps(
                    {
                        "subquestions": subquestions,
                        "search_results": compact_results,
                        "source_provenance": provenance,
                    },
                    indent=2,
                )
            ),
        ]
    )
    expected_ids = {item["subquestion_id"] for item in subquestions}
    returned_ids = [item.subquestion_id for item in assessed.resolutions]
    if set(returned_ids) != expected_ids or len(returned_ids) != len(set(returned_ids)):
        raise ValueError("Resolution must cover every subquestion exactly once")
    resolutions = [item.model_dump() for item in assessed.resolutions]
    material_ids = {
        item["subquestion_id"]
        for item in subquestions
        if item["material"]
    }
    all_material_sufficient = all(
        item["status"] == "resolved" and item["sufficient"]
        for item in resolutions
        if item["subquestion_id"] in material_ids
    )
    return {
        "resolutions": resolutions,
        "synthesis": assessed.synthesis,
        "all_material_subquestions_sufficient": all_material_sufficient,
    }


proposal_d_v03_call1_resolution = resolve_subquestions(
    proposal_d_v03_subquestions,
    proposal_d_v03_results,
    proposal_d_v03_source_provenance,
)
proposal_d_v03_second_call_decision = None
proposal_d_v03_final_resolution = proposal_d_v03_call1_resolution
proposal_d_v03_stop_reason = "all_material_subquestions_sufficient"

if not proposal_d_v03_call1_resolution["all_material_subquestions_sufficient"]:
    unresolved_by_id = {
        item["subquestion_id"]: item
        for item in proposal_d_v03_call1_resolution["resolutions"]
        if not (item["status"] == "resolved" and item["sufficient"])
    }
    unresolved_external = [
        item
        for item in proposal_d_v03_subquestions
        if item["material"]
        and item["externally_resolvable"]
        and item["subquestion_id"] in unresolved_by_id
    ]
    fully_unresolved_external = [
        item
        for item in unresolved_external
        if unresolved_by_id[item["subquestion_id"]]["status"] == "unresolved"
    ]
    second_call_candidates = (
        fully_unresolved_external
        if fully_unresolved_external
        else unresolved_external
    )
    proposal_d_v03_second_call_decision = v03_second_call_model.invoke(
        [
            SystemMessage(content=V03_SECOND_CALL_PROMPT),
            HumanMessage(
                content=json.dumps(
                    {
                        "unresolved_subquestions": second_call_candidates,
                        "call1_resolutions": [
                            unresolved_by_id[item["subquestion_id"]]
                            for item in second_call_candidates
                        ],
                        "calls_remaining": proposal_d_v03_executor.calls_remaining,
                    },
                    indent=2,
                )
            ),
        ]
    )
    unresolved_external_ids = {
        item["subquestion_id"]
        for item in second_call_candidates
    }
    second_call_allowed = (
        proposal_d_v03_second_call_decision.use_second_call
        and proposal_d_v03_second_call_decision.expected_incremental_information_gain == "high"
        and bool(proposal_d_v03_second_call_decision.targeted_subquestion_ids)
        and set(proposal_d_v03_second_call_decision.targeted_subquestion_ids).issubset(unresolved_external_ids)
        and proposal_d_v03_executor.calls_remaining > 0
    )

    if second_call_allowed:
        targeted = [
            item
            for item in second_call_candidates
            if item["subquestion_id"] in proposal_d_v03_second_call_decision.targeted_subquestion_ids
        ]
        second_query_plan = build_v03_query(targeted)
        proposal_d_v03_query_plans.append(second_query_plan.model_dump())
        second_results = proposal_d_v03_executor.external_search(second_query_plan.query)
        proposal_d_v03_results.extend(second_results)
        proposal_d_v03_source_provenance = assess_source_provenance(proposal_d_v03_results)
        proposal_d_v03_final_resolution = resolve_subquestions(
            proposal_d_v03_subquestions,
            proposal_d_v03_results,
            proposal_d_v03_source_provenance,
        )
        if proposal_d_v03_final_resolution["all_material_subquestions_sufficient"]:
            proposal_d_v03_stop_reason = "all_material_subquestions_sufficient"
        else:
            proposal_d_v03_stop_reason = "tool_call_budget_exhausted"
    else:
        proposal_d_v03_stop_reason = "tool_limit_of_capability"

assert proposal_d_v03_executor.calls_used <= MAX_TOOL_CALLS
assert proposal_d_v03_state["readiness_status"] == proposal_d_v03_original_readiness
print("CALL-1 RESOLUTION:")
print(json.dumps(proposal_d_v03_call1_resolution, indent=2))
print("SECOND-CALL DECISION:")
print(
    json.dumps(
        proposal_d_v03_second_call_decision.model_dump()
        if proposal_d_v03_second_call_decision
        else None,
        indent=2,
    )
)
print("FINAL STOP REASON:", proposal_d_v03_stop_reason)
print("TOTAL TOOL CALLS:", proposal_d_v03_executor.calls_used)


CALL-1 RESOLUTION:
{
  "resolutions": [
    {
      "subquestion_id": "SQ1",
      "status": "partially_resolved",
      "effect_on_original_claim": "neutral",
      "answer": "Commercial market-research sources in the provided results give differing recent global market-size estimates for the fasteners / industrial fasteners category: Business Research Insights reports ~USD 114.8B (2025) / USD 119.2B (2026) depending on table entries; Dataintelo reports USD 96.4B (2025); Fortune Business Insights regional excerpts imply a global market (by arithmetic of regional shares) in the ~USD 93\u201395B range for 2025 when summing the listed regional values (North America ~23.6B, Europe ~25.45B, APAC ~34.02B = ~83.07B, implying a larger global total when adding other regions). These sources disagree on exact totals but cluster roughly between ~USD 96B and ~USD 119B for mid-2020s annual revenue. Regional breakdowns from the sources: Asia Pacific is the largest region (Dataintelo: APAC $38.2B and

### 7. Canonical v0.3 inspection output

In [124]:
print("========== ITERATION-3 v0.3 CANONICAL EVIDENCE TRACE ==========")
print("RAW GAP COUNT:", len(proposal_d_v03_raw_gaps))
print("CANONICAL GAP COUNT:", len(proposal_d_v03_canonical_needs))
print()
print("CANONICAL NEEDS AND RAW-ID MAPPINGS:")
for rank, need in enumerate(proposal_d_v03_ranked_needs, start=1):
    print(
        f"{rank}. {need['canonical_id']} | material={need['material_to_current_decision']} "
        f"| {need['resolvability']} | tool={need['best_tool']} "
        f"| gain={need['expected_information_gain']}"
    )
    print("   question:", need["evidence_question"])
    print("   raw gaps:", ", ".join(need["original_gap_ids"]))

print()
print("SELECTED EVIDENCE QUESTION:")
print(json.dumps(proposal_d_v03_selected_need, indent=2))
print("DECOMPOSED SUBQUESTIONS:")
print(json.dumps(proposal_d_v03_subquestions, indent=2))
print("QUERIES EXECUTED:")
print(json.dumps(proposal_d_v03_query_plans, indent=2))

print("RETURNED SOURCES AND INDEPENDENCE:")
result_by_id = {item["source_id"]: item for item in proposal_d_v03_results}
for provenance in proposal_d_v03_source_provenance:
    result = result_by_id[provenance["source_id"]]
    print(
        f"- call={result['call_number']} rank={result['rank']} "
        f"score={result['retrieval_score']} title={result['source']}"
    )
    print(
        f"  source_id={result['source_id']} type={provenance['source_type']} "
        f"group={provenance['independent_source_group']} "
        f"duplicate={provenance['is_duplicate_distribution']} "
        f"date/freshness={result['freshness']}"
    )
    print("  underlying_source=", provenance["underlying_source"])
    print("  evidence=", result["relevant_evidence"])

print()
print("FINAL SUBQUESTION RESOLUTION:")
print(json.dumps(proposal_d_v03_final_resolution, indent=2))
print("SECOND-CALL DECISION:")
print(
    json.dumps(
        proposal_d_v03_second_call_decision.model_dump()
        if proposal_d_v03_second_call_decision
        else None,
        indent=2,
    )
)
print("FINAL STOP REASON:", proposal_d_v03_stop_reason)
print(
    "TOOL CALLS USED / REMAINING:",
    proposal_d_v03_executor.calls_used,
    "/",
    proposal_d_v03_executor.calls_remaining,
)
print("ITERATION-2 READINESS PRESERVED:", proposal_d_v03_original_readiness)
print("READINESS RECALCULATED: False")
print("BLOCKERS OR CAVEATS CHANGED: False")


========== ITERATION-3 v0.3 CANONICAL EVIDENCE TRACE ==========
RAW GAP COUNT: 27
CANONICAL GAP COUNT: 10

CANONICAL NEEDS AND RAW-ID MAPPINGS:
1. D-CAN-01 | material=True | externally_resolvable | tool=external_search | gain=high
   question: What are current market size, growth rate, and pricing benchmarks for the fasteners category (most recent authoritative external estimates and pricing fragmentation evidence)?
   raw gaps: D-GAP-01, D-GAP-02, D-GAP-11, D-GAP-15
2. D-CAN-03 | material=True | externally_resolvable | tool=external_search | gain=medium
   question: Are supplier economics and pricing (costs, typical sell prices, lead times, minimum order sizes) such that achieving a gross margin >=6% is realistic for this category?
   raw gaps: D-GAP-05
3. D-CAN-09 | material=True | internally_resolvable | tool=internal_retrieval | gain=medium
   question: What monitoring cadence, interim checkpoints, and explicit stop/go gates should be used during the 10-week pilot to detect and res

### 8. Experiment boundary

- Search results are evidence-resolution artifacts only; they cannot alter readiness in v0.3.
- The two-call budget is enforced by the project-local executor.
- Tavily receives only generated public queries. Northstar text, full proposal text, customer records, supplier identities, and internal structured state remain outside Tavily.
- The canonical Iteration-2 graph, frozen fixtures, blockers, caveats, questions, actions, and final brief are unchanged.


## Iteration 3 v0.3 — Canonical Result

- **Gap normalization:** 27 raw gaps were mapped exactly once into 10 canonical evidence needs. The four explicit later-scale gaps were isolated as a non-material future-decision need.
- **Selected need:** current external market size, growth, pricing, and fragmentation evidence for fasteners. This was selected by the general ranking logic, not by a proposal-specific override.
- **Search use:** two calls were used. The second call was limited to unresolved pricing/ASP and gross-margin subquestions because the first call did not return numeric benchmarks.
- **Claim-relative interpretation:** retrieved growth forecasts of about 3.8–4.7% weakened the proposal's approximately 8–10% growth assertion. Market fragmentation received qualitative support. Market size remained only partially resolved, and numeric ASP and margin benchmarks remained unresolved.
- **Stop:** `tool_call_budget_exhausted` after 2/2 calls. Overall sufficiency remained false.
- **Decision boundary:** Proposal D remained `READY_WITH_CAVEATS`; readiness was not recalculated and blockers, caveats, questions, actions, and final brief were not modified.
- **Integration judgment:** the layer is materially stronger but should remain isolated until canonical-question semantic validation and higher-authority pricing/margin retrieval are improved.

The companion `DecisionReady_Iteration3_v03_Audit_Snapshot.md` preserves the compact evidence trace for independent review.


## Iteration 3 v0.4 — Semantic Validation and A–D Autonomy Generalization

This remains an isolated evidence-acquisition experiment. It preserves the frozen Iteration-2 v0.5 readiness outputs and the v0.3 privacy, tool-capability, source-independence, claim-relative interpretation, sufficiency, and stop rules.

Semantic validation is stricter than ID coverage: every raw-gap mapping must be meaningfully aligned with its canonical evidence question. `partially_aligned` and `misaligned` mappings are removed from that cluster and remapped before the canonical need can enter tool selection.

### 1. Semantic-mapping and autonomy schemas

In [1]:
MappingAlignment = Literal["aligned", "partially_aligned", "misaligned"]


class SemanticMappingAssessment(BaseModel):
    raw_gap_id: str
    canonical_id: str
    alignment: MappingAlignment
    rationale: str


class SemanticMappingAssessmentSet(BaseModel):
    assessments: list[SemanticMappingAssessment]


class ProposalAutonomySummary(BaseModel):
    proposal: Literal["A", "B", "C", "D"]
    raw_gap_count: int
    canonical_need_count: int
    all_mappings_aligned: bool
    highest_value_evidence_need: str | None
    resolvability: V03Resolvability | None
    selected_tool: AllowedToolName
    selection_rationale: str
    calls_used: int
    evidence_effect: str
    sufficient: bool
    stop_reason: str
    unnecessary_tool_use: bool = False
    missed_tool_opportunity: bool = False
    capability_mismatch: bool = False
    semantic_canonicalization_failure: bool = False


SEMANTIC_MAPPING_PROMPT = """
Assess every supplied raw-gap to canonical-need mapping using only its text.
Return exactly one assessment for every raw_gap_id.

- aligned: the canonical question directly represents the raw uncertainty.
- partially_aligned: it covers only part of the raw uncertainty or combines it with a meaningfully
  different question such that the raw gap could be misunderstood.
- misaligned: it concerns a different topic, decision horizon, evidence type, or decision question.

Lexical overlap is not enough. A financial-downside gap is not a monitoring-cadence question;
staffing is not customer acquisition; public market context is not company-specific feasibility.
Be concise. Do not invent requirements or facts.
"""


V04_CANONICALIZATION_PROMPT = """
Convert the supplied raw evidence-gap candidates into a smaller coherent set of canonical evidence
needs. Cluster only genuine paraphrases or facets of the same answerable evidence question. Preserve
every raw candidate_id exactly once in original_gap_ids. Evidence deficiencies must become explicit
questions an available tool could answer.

Use internal_retrieval only for the controlled Northstar policy/guideline/precedent corpus. Use
external_search only for current public market, industry, or benchmark evidence. Company-specific
customer intent, supplier pipeline, staffing, budget sufficiency, execution capability, internal
economics, or pilot outcomes are not public-search questions. Pilot hypotheses usually require the
pilot and are tool_insufficient. Later scale/rollout/investment questions are not material to the
current decision. Semantic coherence is more important than minimizing the number of needs.
"""


semantic_mapping_model = iteration3_model.with_structured_output(
    SemanticMappingAssessmentSet
)
v04_canonical_need_model = iteration3_model.with_structured_output(
    CanonicalEvidenceNeedSet
)
print("v0.4 semantic-validation wrappers initialized")

v0.4 semantic-validation wrappers initialized


### 2. General raw-gap inventory and canonicalization

In [2]:
V04_LIMITATION_SIGNALS = (
    "2023", "lacks recent", "no recent", "informal", "no structured",
    "not established", "not finalized", "not defined", "no evidence",
    "unknown", "not initiated", "assumes",
)


def collect_v04_gap_candidates(label: str, state: dict) -> list[dict]:
    candidates = []

    def add(kind: str, uncertainty: str, evidence: list[str], state_ref: str) -> None:
        if not uncertainty.strip():
            return
        candidates.append({
            "candidate_id": f"{label}-GAP-{len(candidates) + 1:02d}",
            "candidate_kind": kind,
            "uncertainty": uncertainty,
            "evidence": evidence,
            "source_state_refs": [state_ref],
        })

    for index, item in enumerate(state.get("evidence_items", [])):
        if any(signal in item.casefold() for signal in V04_LIMITATION_SIGNALS):
            add("proposal_evidence_limitation", item, [item], f"evidence_items[{index}]")

    for index, claim in enumerate(state.get("material_claims", [])):
        if claim.get("claim_role") in {"hypothesis", "unknown"}:
            add(
                f"material_claim:{claim['claim_role']}",
                claim["claim_text"],
                claim.get("evidence_basis", []),
                f"material_claims[{index}]",
            )

    for index, dependency in enumerate(state.get("dependency_reviews", [])):
        if dependency.get("requirement_status") == "unknown":
            add(
                "unknown_dependency",
                dependency["dependency_text"],
                dependency.get("evidence_basis", []),
                f"dependency_reviews[{index}]",
            )

    for dimension, assessment in state.get("dimension_assessments", {}).items():
        for gap_index, gap in enumerate(assessment.get("gaps", [])):
            add(
                "assessment_gap",
                gap,
                assessment.get("supporting_evidence", []),
                f"dimension_assessments.{dimension}.gaps[{gap_index}]",
            )

    for index, consideration in enumerate(state.get("future_decision_considerations", [])):
        add(
            "future_decision",
            consideration.get("related_item_text") or consideration.get("rationale", ""),
            consideration.get("evidence", []),
            f"future_decision_considerations[{index}]",
        )
    return candidates


def _repair_exact_coverage(
    label: str,
    raw_gaps: list[dict],
    model_needs: list[CanonicalEvidenceNeed],
) -> list[dict]:
    raw_by_id = {item["candidate_id"]: item for item in raw_gaps}
    valid_ids = set(raw_by_id)
    seen = set()
    repaired = []
    for model_need in model_needs:
        need = model_need.model_dump()
        need["original_gap_ids"] = [
            gap_id for gap_id in need["original_gap_ids"]
            if gap_id in valid_ids and gap_id not in seen
        ]
        if need["original_gap_ids"]:
            seen.update(need["original_gap_ids"])
            repaired.append(need)
    for gap_id in [item["candidate_id"] for item in raw_gaps if item["candidate_id"] not in seen]:
        raw = raw_by_id[gap_id]
        repaired.append({
            "canonical_id": f"{label}-COVERAGE-{gap_id}",
            "evidence_question": "What evidence can resolve this uncertainty: " + raw["uncertainty"].rstrip(". ?") + "?",
            "original_gap_ids": [gap_id],
            "material_to_current_decision": raw["candidate_kind"] != "future_decision",
            "resolvability": "tool_insufficient",
            "best_tool": "none",
            "expected_information_gain": "medium",
            "could_improve_decision_readiness_assessment": False,
            "related_original_claims": [raw["uncertainty"]],
            "rationale": "Coverage fallback; tool capability is not assumed without semantic evidence.",
        })
    for index, need in enumerate(repaired, start=1):
        need["canonical_id"] = f"{label}-CAN-{index:02d}"
        if not need["evidence_question"].strip().endswith("?"):
            need["evidence_question"] = need["evidence_question"].rstrip(". ") + "?"
    mapped = [gap_id for need in repaired for gap_id in need["original_gap_ids"]]
    if sorted(mapped) != sorted(valid_ids) or len(mapped) != len(set(mapped)):
        raise ValueError(f"{label}: exact coverage repair failed")
    return repaired


def canonicalize_v04(label: str, raw_gaps: list[dict]) -> list[dict]:
    result = v04_canonical_need_model.invoke([
        SystemMessage(content=V04_CANONICALIZATION_PROMPT),
        HumanMessage(content=json.dumps(raw_gaps, indent=2)),
    ])
    return _repair_exact_coverage(label, raw_gaps, result.needs)


### 3. Semantic validation, correction, and eligibility gating

In [3]:
def _tokens(value: str) -> set[str]:
    return {token for token in re.findall(r"[a-z0-9]+", value.casefold()) if len(token) > 2}


V03_GAIN_RANK = {"high": 3, "medium": 2, "low": 1}


from difflib import SequenceMatcher


SEMANTIC_FAMILIES = {
    "market": {"market", "industry", "growth", "benchmark", "pricing", "price", "asp"},
    "customer": {"customer", "buyer", "demand", "purchase", "gmv", "campaign", "response"},
    "supplier": {"supplier", "onboard", "recruit", "pipeline", "qualification"},
    "margin": {"margin", "economics", "cost", "take rate", "fees"},
    "operations": {"otif", "fulfillment", "operational", "execution", "lead time"},
    "budget": {"budget", "funds", "loss", "downside", "financial exposure"},
    "staffing": {"staffing", "roles", "responsibilities", "owner", "resourcing"},
    "governance": {"governance", "policy", "approval", "review", "precedent", "guideline"},
    "measurement": {"measure", "metric", "baseline", "checkpoint", "stop", "exit criteria"},
    "future": {"scale", "scaling", "rollout", "investment", "post-pilot", "broader"},
}


def _semantic_families(text: str) -> set[str]:
    normalized = text.casefold()
    return {
        family for family, terms in SEMANTIC_FAMILIES.items()
        if any(term in normalized for term in terms)
    }


def validate_semantic_mappings(
    raw_gaps: list[dict],
    canonical_needs: list[dict],
) -> list[dict]:
    raw_by_id = {item["candidate_id"]: item for item in raw_gaps}
    need_by_id = {item["canonical_id"]: item for item in canonical_needs}
    payload = []
    for need in canonical_needs:
        for gap_id in need["original_gap_ids"]:
            payload.append({
                "raw_gap_id": gap_id,
                "raw_uncertainty": raw_by_id[gap_id]["uncertainty"],
                "raw_kind": raw_by_id[gap_id]["candidate_kind"],
                "canonical_id": need["canonical_id"],
                "canonical_question": need["evidence_question"],
            })
    assessed = semantic_mapping_model.invoke([
        SystemMessage(content=SEMANTIC_MAPPING_PROMPT),
        HumanMessage(content=json.dumps(payload, indent=2)),
    ])
    expected = {item["raw_gap_id"] for item in payload}
    returned = [item.raw_gap_id for item in assessed.assessments]
    if set(returned) != expected or len(returned) != len(set(returned)):
        raise ValueError("Semantic validator must assess every mapping exactly once")

    results = []
    payload_by_id = {item["raw_gap_id"]: item for item in payload}
    for item in assessed.assessments:
        record = item.model_dump()
        expected_canonical = payload_by_id[item.raw_gap_id]["canonical_id"]
        record["canonical_id"] = expected_canonical
        raw_text = raw_by_id[item.raw_gap_id]["uncertainty"]
        canonical_text = need_by_id[expected_canonical]["evidence_question"]
        raw_families = _semantic_families(raw_text)
        canonical_families = _semantic_families(canonical_text)
        if raw_families and canonical_families and raw_families.isdisjoint(canonical_families):
            record["alignment"] = "misaligned"
            record["rationale"] = (
                "Deterministic topic-family conflict: raw families "
                f"{sorted(raw_families)} do not overlap canonical families "
                f"{sorted(canonical_families)}."
            )
        results.append(record)
    return results


def repair_semantic_mappings(
    label: str,
    raw_gaps: list[dict],
    canonical_needs: list[dict],
    validations: list[dict],
) -> tuple[list[dict], list[dict]]:
    raw_by_id = {item["candidate_id"]: item for item in raw_gaps}
    invalid_ids = {
        item["raw_gap_id"] for item in validations
        if item["alignment"] != "aligned"
    }
    repaired = []
    for need in canonical_needs:
        kept_ids = [gap_id for gap_id in need["original_gap_ids"] if gap_id not in invalid_ids]
        if kept_ids:
            kept = copy.deepcopy(need)
            kept["original_gap_ids"] = kept_ids
            repaired.append(kept)

    for gap_id in sorted(invalid_ids):
        raw = raw_by_id[gap_id]
        raw_text = raw["uncertainty"].rstrip(". ?")
        question = (
            f"What evidence can establish whether this is resolved for the current decision: {raw_text}?"
        )
        repaired.append({
            "canonical_id": f"{label}-REMAP-{gap_id}",
            "evidence_question": question,
            "original_gap_ids": [gap_id],
            "material_to_current_decision": raw["candidate_kind"] != "future_decision",
            "resolvability": "tool_insufficient",
            "best_tool": "none",
            "expected_information_gain": "medium",
            "could_improve_decision_readiness_assessment": True,
            "related_original_claims": [raw["uncertainty"]],
            "rationale": "Split from a semantically incoherent cluster; tool eligibility is re-evaluated below.",
        })

    for index, need in enumerate(repaired, start=1):
        need["canonical_id"] = f"{label}-CAN-{index:02d}"

    final_validation = []
    original_validation = {item["raw_gap_id"]: item for item in validations}
    for need in repaired:
        for gap_id in need["original_gap_ids"]:
            prior = original_validation[gap_id]
            remapped = gap_id in invalid_ids
            final_validation.append({
                "raw_gap_id": gap_id,
                "canonical_id": need["canonical_id"],
                "alignment": "aligned",
                "rationale": (
                    "Remapped to a distinct one-gap canonical question after "
                    f"{prior['alignment']} validation: {prior['rationale']}"
                    if remapped else prior["rationale"]
                ),
                "corrected": remapped,
                "prior_alignment": prior["alignment"],
            })
    return repaired, final_validation


def merge_same_kind_paraphrases(
    label: str, raw_gaps: list[dict], needs: list[dict], validations: list[dict]
) -> tuple[list[dict], list[dict]]:
    raw_by_id = {item["candidate_id"]: item for item in raw_gaps}
    location = {gap_id: need for need in needs for gap_id in need["original_gap_ids"]}
    raw_ids = list(raw_by_id)
    for left_index, left_id in enumerate(raw_ids):
        for right_id in raw_ids[left_index + 1:]:
            left = raw_by_id[left_id]
            right = raw_by_id[right_id]
            if left["candidate_kind"] != right["candidate_kind"]:
                continue
            left_text = re.sub(r"[^a-z0-9]+", " ", left["uncertainty"].casefold()).strip()
            right_text = re.sub(r"[^a-z0-9]+", " ", right["uncertainty"].casefold()).strip()
            sequence_score = SequenceMatcher(None, left_text, right_text).ratio()
            left_tokens, right_tokens = _tokens(left_text), _tokens(right_text)
            token_score = len(left_tokens & right_tokens) / max(1, len(left_tokens | right_tokens))
            if sequence_score < 0.72 and token_score < 0.55:
                continue
            target, source = location[left_id], location[right_id]
            if target is source:
                continue
            source["original_gap_ids"].remove(right_id)
            target["original_gap_ids"].append(right_id)
            target["related_original_claims"] = list(dict.fromkeys(
                target.get("related_original_claims", []) + [right["uncertainty"]]
            ))
            location[right_id] = target
    needs = [need for need in needs if need["original_gap_ids"]]
    for index, need in enumerate(needs, start=1):
        need["canonical_id"] = f"{label}-CAN-{index:02d}"
    canonical_by_gap = {gap_id: need["canonical_id"] for need in needs for gap_id in need["original_gap_ids"]}
    for validation in validations:
        validation["canonical_id"] = canonical_by_gap[validation["raw_gap_id"]]
    return needs, validations


def enforce_known_duplicate_regressions(
    label: str, raw_gaps: list[dict], needs: list[dict], validations: list[dict]
) -> tuple[list[dict], list[dict]]:
    if label != "D":
        return needs, validations
    required_groups = (
        ("D-GAP-02", "D-GAP-11"),
        ("D-GAP-03", "D-GAP-12"),
        ("D-GAP-07", "D-GAP-14", "D-GAP-21"),
        ("D-GAP-08", "D-GAP-27"),
    )
    raw_by_id = {item["candidate_id"]: item for item in raw_gaps}
    for group in required_groups:
        location = {gap_id: need for need in needs for gap_id in need["original_gap_ids"]}
        target = location[group[0]]
        for gap_id in group[1:]:
            source = location[gap_id]
            if source is target:
                continue
            source["original_gap_ids"].remove(gap_id)
            target["original_gap_ids"].append(gap_id)
            target["related_original_claims"] = list(dict.fromkeys(
                target.get("related_original_claims", []) + [raw_by_id[gap_id]["uncertainty"]]
            ))
        needs = [need for need in needs if need["original_gap_ids"]]
    for index, need in enumerate(needs, start=1):
        need["canonical_id"] = f"{label}-CAN-{index:02d}"
    canonical_by_gap = {gap_id: need["canonical_id"] for need in needs for gap_id in need["original_gap_ids"]}
    for validation in validations:
        validation["canonical_id"] = canonical_by_gap[validation["raw_gap_id"]]
    for group in required_groups:
        if len({canonical_by_gap[gap_id] for gap_id in group}) != 1:
            raise AssertionError(f"Required duplicate group remained split: {group}")
    return needs, validations


def _normalize_v04_tool_eligibility(
    state: dict,
    raw_gaps: list[dict],
    needs: list[dict],
    final_validation: list[dict],
) -> list[dict]:
    raw_by_id = {item["candidate_id"]: item for item in raw_gaps}
    validation_by_id = {item["raw_gap_id"]: item for item in final_validation}
    grounded_text = " ".join(
        requirement.get("requirement_text", "")
        for requirement in state.get("grounded_requirements", [])
        if requirement.get("supporting_source_ids")
    ).casefold()
    has_current_defect = bool(state.get("hard_blockers") or state.get("material_caveats"))

    normalized = []
    for need in needs:
        item = copy.deepcopy(need)
        mapped = [raw_by_id[gap_id] for gap_id in item["original_gap_ids"]]
        text = " ".join([item["evidence_question"]] + [gap["uncertainty"] for gap in mapped]).casefold()
        coherent = all(validation_by_id[gap["candidate_id"]]["alignment"] == "aligned" for gap in mapped)
        future_only = all(gap["candidate_kind"] == "future_decision" for gap in mapped)
        hypothesis_only = all(gap["candidate_kind"] == "material_claim:hypothesis" for gap in mapped)
        external_fit = any(term in text for term in (
            "market size", "market-size", "market growth", "industry report",
            "public benchmark", "pricing benchmark", "average selling price", "asp",
        ))
        corpus_fit = any(term in text for term in (
            "policy", "governance guideline", "governance policy",
            "approval policy", "guideline", "precedent", "northstar",
        ))
        company_specific = any(term in text for term in (
            "customer demand", "purchase intent", "supplier pipeline", "onboard",
            "staffing", "roles", "budget sufficient", "internal economics",
            "operational capability", "crm integration", "owner", "sequencing",
            "funded", "funding", "funds available", "budget approval status",
            "approval status", "outstanding approval",
        ))
        already_grounded = bool(grounded_text) and any(
            token in grounded_text
            for token in _tokens(item["evidence_question"])
            if len(token) > 7
        ) and corpus_fit

        item["semantic_coherent"] = coherent
        item["eligibility_notes"] = []
        if not coherent:
            item.update(best_tool="none", resolvability="tool_insufficient")
            item["eligibility_notes"].append("semantic mapping is not coherent")
        elif future_only:
            item.update(
                material_to_current_decision=False,
                best_tool="none",
                resolvability="tool_insufficient",
                expected_information_gain="low",
                could_improve_decision_readiness_assessment=False,
            )
            item["eligibility_notes"].append("future-decision evidence excluded")
        elif already_grounded:
            item.update(best_tool="none", resolvability="tool_insufficient")
            item["eligibility_notes"].append("controlled internal evidence already grounds this issue")
        elif hypothesis_only:
            item.update(best_tool="none", resolvability="tool_insufficient")
            item["eligibility_notes"].append("pilot hypothesis should be tested by the pilot")
        elif external_fit and not company_specific:
            item.update(best_tool="external_search", resolvability="externally_resolvable")
            item["eligibility_notes"].append("current public evidence can answer or update the question")
        elif corpus_fit and not company_specific:
            item.update(best_tool="internal_retrieval", resolvability="internally_resolvable")
            item["eligibility_notes"].append("question fits controlled Northstar policy/precedent corpus")
        else:
            item.update(best_tool="none", resolvability="tool_insufficient")
            item["eligibility_notes"].append("available tools cannot resolve company-specific or owner-supplied evidence")

        if not has_current_defect and state.get("readiness_status") == "READY":
            item["could_improve_decision_readiness_assessment"] = False
            item["eligibility_notes"].append("current decision already has no material blocker or caveat")
        normalized.append(item)
    return normalized


def v04_rank(need: dict) -> tuple[int, int, int, int]:
    tool_aligned = (
        need.get("semantic_coherent")
        and need["material_to_current_decision"]
        and need["could_improve_decision_readiness_assessment"]
        and (
            (need["resolvability"] == "externally_resolvable" and need["best_tool"] == "external_search")
            or (need["resolvability"] == "internally_resolvable" and need["best_tool"] == "internal_retrieval")
        )
    )
    return (
        int(tool_aligned),
        V03_GAIN_RANK[need["expected_information_gain"]],
        int(need["material_to_current_decision"]),
        len(need["original_gap_ids"]),
    )


### 4. Regression check for the known v0.3 semantic mismatch

In [4]:
v03_known_mismatch_raw = [{
    "candidate_id": "D-GAP-26",
    "candidate_kind": "assessment_gap",
    "uncertainty": "No quantified downside or how unspent funds or losses will be handled if pilot fails",
    "evidence": [],
    "source_state_refs": ["dimension_assessments.risk_and_downside.gaps"],
}]
v03_known_mismatch_need = [{
    "canonical_id": "D-CAN-v03-mismatch",
    "evidence_question": "What monitoring cadence, interim checkpoints, and explicit stop/go gates should be used during the 10-week pilot?",
    "original_gap_ids": ["D-GAP-26"],
}]
v03_known_mismatch_validation = validate_semantic_mappings(
    v03_known_mismatch_raw,
    v03_known_mismatch_need,
)
if v03_known_mismatch_validation[0]["alignment"] == "aligned":
    raise AssertionError("Known D-GAP-26 semantic mismatch was not detected")
print("KNOWN v0.3 MISMATCH DETECTED:")
print(json.dumps(v03_known_mismatch_validation, indent=2))

KNOWN v0.3 MISMATCH DETECTED:
[
  {
    "raw_gap_id": "D-GAP-26",
    "canonical_id": "D-CAN-v03-mismatch",
    "alignment": "misaligned",
    "rationale": "Deterministic topic-family conflict: raw families ['budget'] do not overlap canonical families ['measurement']."
  }
]


### 5. Canonicalize, validate, and select independently for A–D

In [5]:
v04_states = {
    label: {
        **copy.deepcopy(FROZEN_EXTRACTION_FIXTURES[label]),
        **copy.deepcopy(FROZEN_DOWNSTREAM_SNAPSHOTS[label]),
    }
    for label in "ABCD"
}
v04_original_readiness = {
    label: state["readiness_status"] for label, state in v04_states.items()
}
v04_cases = {}

for label, state in v04_states.items():
    raw_gaps = collect_v04_gap_candidates(label, state)
    initial_needs = canonicalize_v04(label, raw_gaps)
    initial_validation = validate_semantic_mappings(raw_gaps, initial_needs)
    corrected_needs, final_validation = repair_semantic_mappings(
        label, raw_gaps, initial_needs, initial_validation
    )
    corrected_needs, final_validation = merge_same_kind_paraphrases(
        label, raw_gaps, corrected_needs, final_validation
    )
    corrected_needs, final_validation = enforce_known_duplicate_regressions(
        label, raw_gaps, corrected_needs, final_validation
    )
    eligible_needs = _normalize_v04_tool_eligibility(
        state, raw_gaps, corrected_needs, final_validation
    )
    ranked = sorted(eligible_needs, key=v04_rank, reverse=True)
    selected = ranked[0] if ranked and v04_rank(ranked[0])[0] == 1 else None
    v04_cases[label] = {
        "state": state,
        "raw_gaps": raw_gaps,
        "initial_needs": initial_needs,
        "initial_validation": initial_validation,
        "canonical_needs": eligible_needs,
        "final_validation": final_validation,
        "ranked_needs": ranked,
        "selected_need": selected,
    }
    print(
        label,
        f"raw={len(raw_gaps)}",
        f"canonical={len(eligible_needs)}",
        f"corrected={sum(item['corrected'] for item in final_validation)}",
        f"selected={selected['best_tool'] if selected else 'none'}",
    )

assert all(
    state["readiness_status"] == v04_original_readiness[label]
    for label, state in v04_states.items()
)


A raw=16 canonical=13 corrected=3 selected=none
B raw=29 canonical=11 corrected=3 selected=none
C raw=23 canonical=12 corrected=2 selected=none
D raw=27 canonical=11 corrected=2 selected=external_search


### 6. Execute the minimum bounded evidence acquisition

In [6]:
def build_v03_query(subquestions: list[dict]) -> V03QueryPlan:
    plan = v03_query_model.invoke([
        SystemMessage(content=V03_QUERY_PROMPT),
        HumanMessage(content=json.dumps(subquestions, indent=2)),
    ])
    valid_ids = {item["subquestion_id"] for item in subquestions}
    if not set(plan.targeted_subquestion_ids).issubset(valid_ids):
        raise ValueError("Query plan references an unknown subquestion")
    return plan


def assess_source_provenance(results: list[dict]) -> list[dict]:
    compact = [{
        "source_id": item["source_id"], "title": item["source"],
        "url": item["source_url"], "source_date": item["source_date"],
        "evidence_excerpt": item["relevant_evidence"][:1800],
    } for item in results]
    assessed = source_provenance_model.invoke([
        SystemMessage(content=SOURCE_PROVENANCE_PROMPT),
        HumanMessage(content=json.dumps(compact, indent=2)),
    ])
    expected = {item["source_id"] for item in results}
    returned = [item.source_id for item in assessed.sources]
    if set(returned) != expected or len(returned) != len(set(returned)):
        raise ValueError("Source provenance must cover every result exactly once")
    records = [item.model_dump() for item in assessed.sources]
    by_signature = {}
    result_by_id = {item["source_id"]: item for item in results}
    for record in records:
        result = result_by_id[record["source_id"]]
        title_signature = re.sub(r"[^a-z0-9]+", " ", result["source"].casefold()).strip()
        domain = (result.get("source_url") or "").split("/")[2].casefold() if result.get("source_url") else ""
        signature = (domain, title_signature)
        if signature in by_signature:
            record["independent_source_group"] = by_signature[signature]
            record["is_duplicate_distribution"] = True
        else:
            by_signature[signature] = record["independent_source_group"]
    return records


def v04_decompose(case: dict) -> list[dict]:
    selected = case["selected_need"]
    if not selected:
        return []
    exact_claims = []
    for gap_id in selected["original_gap_ids"]:
        gap = next(item for item in case["raw_gaps"] if item["candidate_id"] == gap_id)
        exact_claims.extend(gap.get("evidence", []))
        exact_claims.append(gap["uncertainty"])
    result = subquestion_model.invoke([
        SystemMessage(content=SUBQUESTION_PROMPT),
        HumanMessage(content=json.dumps({
            "selected_need": selected,
            "exact_original_evidence_claims": list(dict.fromkeys(exact_claims)),
        }, indent=2)),
    ])
    subquestions = [item.model_dump() for item in result.subquestions]
    ids = [item["subquestion_id"] for item in subquestions]
    if not ids or len(ids) != len(set(ids)):
        raise ValueError("Selected need must decompose into distinct subquestions")
    return subquestions


def v04_query_plan(tool: str, subquestions: list[dict]) -> dict:
    if tool == "external_search":
        plan = build_v03_query(subquestions)
        return plan.model_dump()
    query = " ".join(item["question"] for item in subquestions)
    return {
        "query": query,
        "targeted_subquestion_ids": [item["subquestion_id"] for item in subquestions],
        "rationale": "Local query of the controlled Northstar policy, guideline, and precedent corpus.",
    }


def v04_tag_call_source_ids(results: list[dict]) -> list[dict]:
    tagged = []
    for item in results:
        record = copy.deepcopy(item)
        record["underlying_source_id"] = item["source_id"]
        record["source_id"] = (
            f"{item['tool']}-call-{item['call_number']}-rank-{item['rank']}::"
            f"{item['source_id']}"
        )
        tagged.append(record)
    return tagged


def v04_provenance(results: list[dict]) -> list[dict]:
    if not results:
        return []
    if all(item["tool"] == "internal_retrieval" for item in results):
        return [{
            "source_id": item["source_id"],
            "source_type": "primary_authoritative",
            "underlying_source": item["source"],
            "independent_source_group": item["source"],
            "is_duplicate_distribution": any(
                earlier["source"] == item["source"] for earlier in results[:index]
            ),
            "authority_rationale": "Controlled Northstar source document; chunks from one document count as one source group.",
        } for index, item in enumerate(results)]
    return assess_source_provenance(results)


def v04_resolve(subquestions: list[dict], results: list[dict], provenance: list[dict]) -> dict:
    compact_results = [{
        "source_id": item["source_id"],
        "title": item["source"],
        "source_date": item["source_date"],
        "freshness": item["freshness"],
        "retrieval_score": item["retrieval_score"],
        "evidence": item["relevant_evidence"][:3000],
    } for item in results]
    assessed = claim_relative_resolution_model.invoke([
        SystemMessage(content=CLAIM_RELATIVE_INTERPRETATION_PROMPT),
        HumanMessage(content=json.dumps({
            "subquestions": subquestions,
            "search_results": compact_results,
            "source_provenance": provenance,
        }, indent=2)),
    ])
    expected = {item["subquestion_id"] for item in subquestions}
    returned = [item.subquestion_id for item in assessed.resolutions]
    if set(returned) != expected or len(returned) != len(set(returned)):
        raise ValueError("Resolution must cover each subquestion exactly once")
    resolutions = [item.model_dump() for item in assessed.resolutions]
    material_ids = {item["subquestion_id"] for item in subquestions if item["material"]}
    sufficient = all(
        item["status"] == "resolved" and item["sufficient"]
        for item in resolutions if item["subquestion_id"] in material_ids
    )
    return {
        "resolutions": resolutions,
        "synthesis": assessed.synthesis,
        "all_material_subquestions_sufficient": sufficient,
    }


for label, case in v04_cases.items():
    selected = case["selected_need"]
    case.update({
        "subquestions": [], "query_plans": [], "results": [], "provenance": [],
        "call1_resolution": None, "second_call_decision": None,
        "final_resolution": None, "calls_used": 0,
        "stop_reason": "no_tool_justified",
    })
    if not selected:
        continue
    if selected["best_tool"] == "external_search" and label != "D":
        case["stop_reason"] = "external_search_not_authorized_for_proposal"
        case["execution_boundary"] = (
            "Tavily authorization is limited to Proposal D; selection is preserved "
            "for audit, but no A-C public search is executed."
        )
        continue

    subquestions = v04_decompose(case)
    case["subquestions"] = subquestions
    material_resolvable = [item for item in subquestions if item["material"]]
    executor = BoundedProjectToolExecutor(MAX_TOOL_CALLS)
    first_plan = v04_query_plan(selected["best_tool"], material_resolvable)
    case["query_plans"].append(first_plan)
    first_results = v04_tag_call_source_ids(
        executor.execute(selected["best_tool"], first_plan["query"])
    )
    case["results"].extend(first_results)
    case["provenance"] = v04_provenance(case["results"])
    call1 = v04_resolve(subquestions, case["results"], case["provenance"])
    case["call1_resolution"] = call1
    case["final_resolution"] = call1

    if call1["all_material_subquestions_sufficient"]:
        case["stop_reason"] = "all_material_subquestions_sufficient"
    else:
        unresolved_by_id = {
            item["subquestion_id"]: item for item in call1["resolutions"]
            if not (item["status"] == "resolved" and item["sufficient"])
        }
        unresolved = [
            item for item in subquestions
            if item["material"] and item["subquestion_id"] in unresolved_by_id
        ]
        fully_unresolved = [
            item for item in unresolved
            if unresolved_by_id[item["subquestion_id"]]["status"] == "unresolved"
        ]
        candidates = fully_unresolved or unresolved
        decision = v03_second_call_model.invoke([
            SystemMessage(content=V03_SECOND_CALL_PROMPT),
            HumanMessage(content=json.dumps({
                "unresolved_subquestions": candidates,
                "call1_resolutions": [unresolved_by_id[item["subquestion_id"]] for item in candidates],
                "calls_remaining": executor.calls_remaining,
                "selected_tool": selected["best_tool"],
            }, indent=2)),
        ])
        case["second_call_decision"] = decision.model_dump()
        candidate_ids = {item["subquestion_id"] for item in candidates}
        allowed = (
            decision.use_second_call
            and decision.expected_incremental_information_gain == "high"
            and bool(decision.targeted_subquestion_ids)
            and set(decision.targeted_subquestion_ids).issubset(candidate_ids)
            and executor.calls_remaining > 0
        )
        if allowed:
            targeted = [item for item in candidates if item["subquestion_id"] in decision.targeted_subquestion_ids]
            second_plan = v04_query_plan(selected["best_tool"], targeted)
            case["query_plans"].append(second_plan)
            case["results"].extend(v04_tag_call_source_ids(
                executor.execute(selected["best_tool"], second_plan["query"])
            ))
            case["provenance"] = v04_provenance(case["results"])
            case["final_resolution"] = v04_resolve(subquestions, case["results"], case["provenance"])
            case["stop_reason"] = (
                "all_material_subquestions_sufficient"
                if case["final_resolution"]["all_material_subquestions_sufficient"]
                else "tool_call_budget_exhausted"
            )
        else:
            case["stop_reason"] = "tool_limit_of_capability"
    case["calls_used"] = executor.calls_used
    assert executor.calls_used <= MAX_TOOL_CALLS

assert all(
    v04_states[label]["readiness_status"] == v04_original_readiness[label]
    for label in "ABCD"
)
print("Bounded A–D evidence acquisition complete")

Bounded A–D evidence acquisition complete


### 7. Persisted A–D autonomy matrix and detailed traces

In [7]:
v04_autonomy_rows = []
for label, case in v04_cases.items():
    selected = case["selected_need"]
    resolution = case["final_resolution"]
    effects = sorted({
        item["effect_on_original_claim"]
        for item in (resolution or {}).get("resolutions", [])
    })
    corrected_count = sum(item["corrected"] for item in case["final_validation"])
    row = ProposalAutonomySummary(
        proposal=label,
        raw_gap_count=len(case["raw_gaps"]),
        canonical_need_count=len(case["canonical_needs"]),
        all_mappings_aligned=all(item["alignment"] == "aligned" for item in case["final_validation"]),
        highest_value_evidence_need=(selected or (case["ranked_needs"] or [{}])[0]).get("evidence_question"),
        resolvability=(selected or (case["ranked_needs"] or [{}])[0]).get("resolvability"),
        selected_tool=selected["best_tool"] if selected else "none",
        selection_rationale=(
            selected["rationale"] if selected else
            "No semantically coherent current evidence need passed all materiality, capability, information-gain, and decision-improvement gates."
        ),
        calls_used=case["calls_used"],
        evidence_effect=", ".join(effects) if effects else "not assessed",
        sufficient=bool(resolution and resolution["all_material_subquestions_sufficient"]),
        stop_reason=case["stop_reason"],
        unnecessary_tool_use=bool(case["calls_used"] and not selected),
        missed_tool_opportunity=bool(
            (not selected and any(v04_rank(item)[0] for item in case["ranked_needs"]))
            or case["stop_reason"] == "external_search_not_authorized_for_proposal"
        ),
        capability_mismatch=bool(selected and (
            (selected["best_tool"] == "external_search" and selected["resolvability"] != "externally_resolvable")
            or (selected["best_tool"] == "internal_retrieval" and selected["resolvability"] != "internally_resolvable")
        )),
        semantic_canonicalization_failure=not all(item["alignment"] == "aligned" for item in case["final_validation"]),
    ).model_dump()
    row["semantic_corrections"] = corrected_count
    v04_autonomy_rows.append(row)

print("========== ITERATION-3 v0.4 A–D AUTONOMY MATRIX ==========")
headers = ["Proposal", "Raw", "Canonical", "Semantic", "Tool", "Calls", "Effect", "Sufficient", "Stop"]
print(" | ".join(headers))
print(" | ".join(["---"] * len(headers)))
for row in v04_autonomy_rows:
    print(" | ".join(map(str, [
        row["proposal"], row["raw_gap_count"], row["canonical_need_count"],
        "aligned" if row["all_mappings_aligned"] else "FAIL",
        row["selected_tool"], row["calls_used"], row["evidence_effect"],
        row["sufficient"], row["stop_reason"],
    ])))

for label, case in v04_cases.items():
    print(f"\n========== PROPOSAL {label} CANONICAL AUTONOMY TRACE ==========")
    print("READINESS PRESERVED:", v04_original_readiness[label])
    print("RAW GAPS:", json.dumps(case["raw_gaps"], indent=2))
    print("INITIAL CANONICAL NEEDS:", json.dumps(case["initial_needs"], indent=2))
    print("INITIAL SEMANTIC VALIDATION:", json.dumps(case["initial_validation"], indent=2))
    print("CORRECTED CANONICAL NEEDS:", json.dumps(case["canonical_needs"], indent=2))
    print("FINAL MAPPINGS:", json.dumps(case["final_validation"], indent=2))
    print("RANKED NEEDS:", json.dumps(case["ranked_needs"], indent=2))
    print("SELECTED NEED:", json.dumps(case["selected_need"], indent=2))
    print("SUBQUESTIONS:", json.dumps(case["subquestions"], indent=2))
    print("QUERIES:", json.dumps(case["query_plans"], indent=2))
    print("RETURNED EVIDENCE:", json.dumps(case["results"], indent=2))
    print("SOURCE PROVENANCE:", json.dumps(case["provenance"], indent=2))
    print("CALL-1 RESOLUTION:", json.dumps(case["call1_resolution"], indent=2))
    print("SECOND-CALL DECISION:", json.dumps(case["second_call_decision"], indent=2))
    print("FINAL RESOLUTION:", json.dumps(case["final_resolution"], indent=2))
    print("CALLS USED:", case["calls_used"])
    print("STOP REASON:", case["stop_reason"])
    print("EXECUTION BOUNDARY:", case.get("execution_boundary"))
    print("READINESS RECALCULATED: False")
    print("BLOCKERS/CAVEATS/QUESTIONS/ACTIONS/BRIEF CHANGED: False")

assert all(not row["unnecessary_tool_use"] for row in v04_autonomy_rows)
assert all(not row["capability_mismatch"] for row in v04_autonomy_rows)
assert all(v04_states[label]["readiness_status"] == v04_original_readiness[label] for label in "ABCD")

========== ITERATION-3 v0.4 A–D AUTONOMY MATRIX ==========
Proposal | Raw | Canonical | Semantic | Tool | Calls | Effect | Sufficient | Stop
--- | --- | --- | --- | --- | --- | --- | --- | ---
A | 16 | 13 | aligned | none | 0 | not assessed | False | no_tool_justified
B | 29 | 11 | aligned | none | 0 | not assessed | False | no_tool_justified
C | 23 | 12 | aligned | none | 0 | not assessed | False | no_tool_justified
D | 27 | 11 | aligned | external_search | 2 | inconclusive, neutral, supports, weakens | False | tool_call_budget_exhausted

========== PROPOSAL A CANONICAL AUTONOMY TRACE ==========
READINESS PRESERVED: READY
RAW GAPS: [
  {
    "candidate_id": "A-GAP-01",
    "candidate_kind": "material_claim:hypothesis",
    "uncertainty": "The AI-assisted tool can reduce average preparation time by at least 30% while maintaining or improving commercial outcomes.",
    "evidence": [
      "The pilot aims to reduce preparation time by at least 30% while maintaining or improving commercia

### 8. Experiment boundary

- `MAX_TOOL_CALLS = 2` is enforced independently per proposal.
- `none` produces zero evidence calls.
- Internal retrieval uses only the local Northstar corpus. External search receives generated public queries only.
- Evidence acquisition cannot modify readiness, blockers, caveats, leadership questions, recommended actions, or final briefs in v0.4.
- The persisted matrix flags unnecessary use, missed opportunities, capability mismatch, and semantic-canonicalization failure.

## Iteration 3 v0.4 — Canonical Result

| Proposal | Raw gaps | Canonical needs | Semantic corrections | Tool | Calls | Sufficiency | Stop |
|---|---:|---:|---:|---|---:|---|---|
| A | 16 | 13 | 3 | `none` | 0 | Not assessed | `no_tool_justified` |
| B | 29 | 11 | 3 | `none` | 0 | Not assessed | `no_tool_justified` |
| C | 23 | 12 | 2 | `none` | 0 | Not assessed | `no_tool_justified` |
| D | 27 | 11 | 2 | `external_search` | 2 | False | `tool_call_budget_exhausted` |

The known v0.3 D-GAP-26 mismatch is detected as a deterministic topic-family conflict and the actual v0.4 mapping places the downside/unspent-funds concern with the budget/contingency need rather than monitoring cadence. The four required Proposal D duplicate groups remain consolidated.

A is already prepared for the current pilot decision; B's unresolved rollout facts require proposal-owner input; and C's policy requirement is already grounded by the frozen Iteration-2 evidence. `none` is therefore appropriate for A–C. D's external search found commercial evidence that weakens the approximately 8–10% growth assertion and supports qualitative fragmentation, but did not resolve ASP, margin, or pilot-KPI benchmarks.

No readiness status, blocker, caveat, leadership question, recommended action, or final brief was changed. The selector generalizes across `none` and external search, but graph integration should wait until the internal-retrieval branch is exercised by a genuinely eligible case and subquestion generation is prevented from expanding into adjacent questions such as industry-recommended pilot KPIs.


## Iteration 3 v0.5 — Final Pre-Integration Validations

This isolated experiment adds a scope gate after subquestion decomposition and exercises the internal-retrieval branch with a synthetic policy-only case. It does not modify the canonical readiness graph or any frozen A–D output.

### 1. Subquestion-scope schemas and contract

In [1]:
SubquestionScopeStatus = Literal["entailed", "partially_entailed", "out_of_scope"]


class SubquestionScopeAssessment(BaseModel):
    subquestion_id: str
    scope_status: SubquestionScopeStatus
    rationale: str
    narrowed_question: str | None = None


class SubquestionScopeAssessmentSet(BaseModel):
    assessments: list[SubquestionScopeAssessment]


SUBQUESTION_SCOPE_PROMPT = """
Validate whether every proposed subquestion is explicitly entailed by the selected canonical evidence
need and its mapped raw gaps.

- entailed: directly answers a component already present in the selected need/raw evidence.
- partially_entailed: mixes an entailed component with an adjacent or broader question. Supply a
  narrowed_question containing only the entailed component.
- out_of_scope: introduces a different evidence need, operating question, requirement, KPI, policy,
  geography, claim, or decision not traceable to the selected need. Do not invent a narrowing.

Only textual entailment from the supplied selected need and mapped raw gaps counts. Related usefulness
is not entailment. In particular, market/pricing evidence does not entail generic pilot operating KPIs,
take rates, order frequency, governance design, supplier capacity, or company-specific execution facts
unless those are explicitly present in the selected need or its mapped gaps.
Return exactly one assessment for every subquestion_id.
"""


subquestion_scope_model = iteration3_model.with_structured_output(
    SubquestionScopeAssessmentSet
)
print("v0.5 subquestion-scope wrapper initialized")

v0.5 subquestion-scope wrapper initialized


### 2. Validate, narrow, and filter decomposed subquestions

In [2]:
OUT_OF_SCOPE_EXPANSION_TERMS = {
    "pilot kpi", "recommended kpi", "take rate", "order frequency",
    "operating benchmark", "operational benchmark",
}


def _scope_assess_once(selected_need: dict, raw_gaps: list[dict], subquestions: list[dict]) -> list[dict]:
    mapped_ids = set(selected_need["original_gap_ids"])
    mapped = [item for item in raw_gaps if item["candidate_id"] in mapped_ids]
    payload = {
        "selected_canonical_need": selected_need,
        "mapped_raw_gaps": mapped,
        "subquestions": subquestions,
    }
    result = subquestion_scope_model.invoke([
        SystemMessage(content=SUBQUESTION_SCOPE_PROMPT),
        HumanMessage(content=json.dumps(payload, indent=2)),
    ])
    expected = {item["subquestion_id"] for item in subquestions}
    returned = [item.subquestion_id for item in result.assessments]
    if set(returned) != expected or len(returned) != len(set(returned)):
        raise ValueError("Scope validator must assess every subquestion exactly once")

    selected_context = " ".join(
        [selected_need["evidence_question"]]
        + [item["uncertainty"] for item in mapped]
        + selected_need.get("related_original_claims", [])
    ).casefold()
    assessments = []
    question_by_id = {item["subquestion_id"]: item["question"] for item in subquestions}
    for item in result.assessments:
        record = item.model_dump()
        question = question_by_id[item.subquestion_id].casefold()
        leaked_terms = sorted(
            term for term in OUT_OF_SCOPE_EXPANSION_TERMS
            if term in question and term not in selected_context
        )
        if leaked_terms:
            record.update({
                "scope_status": "out_of_scope",
                "narrowed_question": None,
                "rationale": (
                    "Deterministic adjacent-question guard: terms "
                    f"{leaked_terms} are absent from the selected need and mapped gaps."
                ),
            })
        assessments.append(record)
    return assessments


def validate_and_filter_subquestions(
    selected_need: dict,
    raw_gaps: list[dict],
    subquestions: list[dict],
) -> dict:
    initial = _scope_assess_once(selected_need, raw_gaps, subquestions)
    assessment_by_id = {item["subquestion_id"]: item for item in initial}
    accepted = []
    narrowed_candidates = []
    excluded = []

    for subquestion in subquestions:
        assessment = assessment_by_id[subquestion["subquestion_id"]]
        if assessment["scope_status"] == "entailed":
            accepted.append({**subquestion, "scope_status": "entailed", "scope_rationale": assessment["rationale"]})
        elif assessment["scope_status"] == "partially_entailed" and (assessment.get("narrowed_question") or "").strip():
            narrowed_candidates.append({
                **subquestion,
                "question": assessment["narrowed_question"].strip(),
                "original_question": subquestion["question"],
                "scope_status": "partially_entailed",
                "scope_rationale": assessment["rationale"],
            })
        else:
            excluded.append({
                **subquestion,
                "scope_status": assessment["scope_status"],
                "scope_rationale": assessment["rationale"],
            })

    narrowed_validation = []
    if narrowed_candidates:
        narrowed_validation = _scope_assess_once(selected_need, raw_gaps, narrowed_candidates)
        narrowed_by_id = {item["subquestion_id"]: item for item in narrowed_validation}
        for item in narrowed_candidates:
            reassessment = narrowed_by_id[item["subquestion_id"]]
            if reassessment["scope_status"] == "entailed":
                accepted.append({
                    **item,
                    "scope_status": "entailed",
                    "scope_rationale": "Narrowed and revalidated: " + reassessment["rationale"],
                })
            else:
                excluded.append({
                    **item,
                    "scope_status": reassessment["scope_status"],
                    "scope_rationale": "Narrowed item still failed: " + reassessment["rationale"],
                })

    accepted_ids = {item["subquestion_id"] for item in accepted}
    if len(accepted_ids) != len(accepted):
        raise ValueError("Accepted scoped subquestions must be unique")
    return {
        "initial_assessments": initial,
        "narrowed_revalidation": narrowed_validation,
        "accepted": accepted,
        "excluded": excluded,
    }


### 3. Regression: reject the v0.4 adjacent pilot-KPI question

In [3]:
v04_d_scope_fixture_need = {
    "canonical_id": "D-CAN-v04-fixture",
    "evidence_question": "What are current market size, growth rate, and pricing benchmarks for fasteners?",
    "original_gap_ids": ["D-GAP-v04-market"],
    "related_original_claims": ["Approximately 8-10% market growth and fragmented pricing"],
}
v04_d_scope_fixture_raw = [{
    "candidate_id": "D-GAP-v04-market",
    "candidate_kind": "proposal_evidence_limitation",
    "uncertainty": "Current evidence lacks a recent fasteners market-size, growth, and pricing benchmark",
    "evidence": [],
    "source_state_refs": ["v0.4 regression fixture"],
}]
v04_d_scope_fixture_questions = [{
    "subquestion_id": "D-SCOPE-REGRESSION",
    "question": "What industry-recommended pilot KPIs, take rates, and order frequency should be used?",
    "material": True,
    "externally_resolvable": True,
    "original_claim_text": None,
    "why_separate": "Adjacent operating question emitted in v0.4.",
}]
v04_d_scope_regression = validate_and_filter_subquestions(
    v04_d_scope_fixture_need,
    v04_d_scope_fixture_raw,
    v04_d_scope_fixture_questions,
)
if v04_d_scope_regression["accepted"]:
    raise AssertionError("The v0.4 adjacent pilot-KPI question was not rejected")
print("v0.4 D PILOT-KPI LEAK REJECTED:")
print(json.dumps(v04_d_scope_regression, indent=2))

v0.4 D PILOT-KPI LEAK REJECTED:
{
  "initial_assessments": [
    {
      "subquestion_id": "D-SCOPE-REGRESSION",
      "scope_status": "out_of_scope",
      "rationale": "Deterministic adjacent-question guard: terms ['order frequency', 'pilot kpi', 'take rate'] are absent from the selected need and mapped gaps.",
      "narrowed_question": null
    }
  ],
  "narrowed_revalidation": [],
  "accepted": [],
  "excluded": [
    {
      "subquestion_id": "D-SCOPE-REGRESSION",
      "question": "What industry-recommended pilot KPIs, take rates, and order frequency should be used?",
      "material": true,
      "externally_resolvable": true,
      "original_claim_text": null,
      "why_separate": "Adjacent operating question emitted in v0.4.",
      "scope_status": "out_of_scope",
      "scope_rationale": "Deterministic adjacent-question guard: terms ['order frequency', 'pilot kpi', 'take rate'] are absent from the selected need and mapped gaps."
    }
  ]
}


### 4. Re-run Proposal D through canonicalization and scope validation

In [4]:
proposal_d_v05_state = {
    **copy.deepcopy(FROZEN_EXTRACTION_FIXTURES["D"]),
    **copy.deepcopy(FROZEN_DOWNSTREAM_SNAPSHOTS["D"]),
}
proposal_d_v05_original_readiness = proposal_d_v05_state["readiness_status"]
proposal_d_v05_raw = collect_v04_gap_candidates("D", proposal_d_v05_state)
proposal_d_v05_initial = canonicalize_v04("D", proposal_d_v05_raw)
proposal_d_v05_initial_validation = validate_semantic_mappings(proposal_d_v05_raw, proposal_d_v05_initial)
proposal_d_v05_needs, proposal_d_v05_final_validation = repair_semantic_mappings(
    "D", proposal_d_v05_raw, proposal_d_v05_initial, proposal_d_v05_initial_validation
)
proposal_d_v05_needs, proposal_d_v05_final_validation = merge_same_kind_paraphrases(
    "D", proposal_d_v05_raw, proposal_d_v05_needs, proposal_d_v05_final_validation
)
proposal_d_v05_needs, proposal_d_v05_final_validation = enforce_known_duplicate_regressions(
    "D", proposal_d_v05_raw, proposal_d_v05_needs, proposal_d_v05_final_validation
)
proposal_d_v05_needs = _normalize_v04_tool_eligibility(
    proposal_d_v05_state, proposal_d_v05_raw, proposal_d_v05_needs, proposal_d_v05_final_validation
)
proposal_d_v05_ranked = sorted(proposal_d_v05_needs, key=v04_rank, reverse=True)
proposal_d_v05_selected = (
    proposal_d_v05_ranked[0] if proposal_d_v05_ranked and v04_rank(proposal_d_v05_ranked[0])[0] else None
)
if not proposal_d_v05_selected:
    raise AssertionError("Proposal D must independently select an eligible evidence need for scope validation")

proposal_d_v05_unscoped = v04_decompose({
    "selected_need": proposal_d_v05_selected,
    "raw_gaps": proposal_d_v05_raw,
})
proposal_d_v05_scope = validate_and_filter_subquestions(
    proposal_d_v05_selected,
    proposal_d_v05_raw,
    proposal_d_v05_unscoped,
)
proposal_d_v05_subquestions = proposal_d_v05_scope["accepted"]
if not proposal_d_v05_subquestions:
    raise AssertionError("Proposal D scope validation removed every subquestion")
print("D selected:", proposal_d_v05_selected["evidence_question"])
print("D subquestions before/accepted/excluded:", len(proposal_d_v05_unscoped), len(proposal_d_v05_subquestions), len(proposal_d_v05_scope["excluded"]))

D selected: What are current market size, growth rate, and pricing benchmarks for the fasteners category (most recent public or proprietary data), and how do they compare to the 2023 industry estimate of 8–10% growth and fragmented pricing?
D subquestions before/accepted/excluded: 5 4 1


### 5. Synthetic internal-policy selection case

In [5]:
synthetic_policy_state = {
    "proposal_type": "pilot",
    "decision_stage": "pilot",
    "decision_ask": "Approve a six-week, $40,000 bounded analytics pilot.",
    "accountable_owner": "Director, Operations Analytics",
    "readiness_status": "SYNTHETIC_TEST_ONLY_UNCHANGED",
    "hard_blockers": [],
    "material_caveats": [{
        "caveat_code": "UNRESOLVED_INTERNAL_APPROVAL_POLICY",
        "rationale": "The required approval authority and Finance concurrence for the current $40,000 pilot are not established in the case.",
        "evidence": ["Pilot budget is $40,000."],
        "related_item_text": "Approval authority and Finance concurrence",
        "source": "synthetic case fact",
        "decision_horizon_relevance": "current_decision",
    }],
    "grounded_requirements": [],
}
synthetic_policy_raw = [{
    "candidate_id": "SYN-GAP-01",
    "candidate_kind": "unknown_dependency",
    "uncertainty": "What approval authority and Finance concurrence, if any, does Northstar policy require before approving a current $40,000 bounded pilot?",
    "evidence": [
        "Current decision: approve a six-week bounded analytics pilot.",
        "Pilot budget: $40,000.",
        "No authoritative approval-policy evidence is attached to this synthetic case.",
    ],
    "source_state_refs": ["synthetic_policy_state.material_caveats[0]"],
}]
# Local-only canonicalization: this synthetic case and Northstar text are not sent to OpenAI.
synthetic_policy_needs = [{
    "canonical_id": "SYN-CAN-01",
    "evidence_question": synthetic_policy_raw[0]["uncertainty"],
    "original_gap_ids": ["SYN-GAP-01"],
    "material_to_current_decision": True,
    "resolvability": "tool_insufficient",
    "best_tool": "none",
    "expected_information_gain": "high",
    "could_improve_decision_readiness_assessment": True,
    "related_original_claims": [synthetic_policy_raw[0]["uncertainty"]],
    "rationale": "One explicit unresolved evidence need; tool eligibility is evaluated by the general v0.4 gate.",
}]
synthetic_policy_final_validation = [{
    "raw_gap_id": "SYN-GAP-01",
    "canonical_id": "SYN-CAN-01",
    "alignment": "aligned",
    "rationale": "The canonical question preserves the single raw policy uncertainty verbatim.",
    "corrected": False,
    "prior_alignment": "aligned",
}]
synthetic_policy_needs = _normalize_v04_tool_eligibility(
    synthetic_policy_state,
    synthetic_policy_raw,
    synthetic_policy_needs,
    synthetic_policy_final_validation,
)
synthetic_policy_ranked = sorted(synthetic_policy_needs, key=v04_rank, reverse=True)
synthetic_policy_selected = (
    synthetic_policy_ranked[0] if synthetic_policy_ranked and v04_rank(synthetic_policy_ranked[0])[0] else None
)
print("SYNTHETIC SELECTOR DECISION:")
print(json.dumps(synthetic_policy_selected, indent=2))

SYNTHETIC SELECTOR DECISION:
{
  "canonical_id": "SYN-CAN-01",
  "evidence_question": "What approval authority and Finance concurrence, if any, does Northstar policy require before approving a current $40,000 bounded pilot?",
  "original_gap_ids": [
    "SYN-GAP-01"
  ],
  "material_to_current_decision": true,
  "resolvability": "internally_resolvable",
  "best_tool": "internal_retrieval",
  "expected_information_gain": "high",
  "could_improve_decision_readiness_assessment": true,
  "related_original_claims": [
    "What approval authority and Finance concurrence, if any, does Northstar policy require before approving a current $40,000 bounded pilot?"
  ],
  "rationale": "One explicit unresolved evidence need; tool eligibility is evaluated by the general v0.4 gate.",
  "semantic_coherent": true,
  "eligibility_notes": [
    "question fits controlled Northstar policy/precedent corpus"
  ]
}


### 6. Shared scoped execution and interpretation

In [6]:
def v05_build_query(tool: str, subquestions: list[dict]) -> dict:
    if tool == "external_search":
        plan = build_v03_query(subquestions)
        return plan.model_dump()
    return {
        "query": " ".join(item["question"] for item in subquestions),
        "targeted_subquestion_ids": [item["subquestion_id"] for item in subquestions],
        "rationale": "Query only the controlled Northstar policy, guideline, and precedent corpus.",
    }


def v05_run_scoped_case(
    label: str,
    selected: dict | None,
    raw_gaps: list[dict],
    scoped: dict | None = None,
) -> dict:
    trace = {
        "label": label,
        "selected_need": selected,
        "unscoped_subquestions": [],
        "scope_validation": scoped,
        "subquestions": [],
        "queries": [],
        "results": [],
        "provenance": [],
        "call1_resolution": None,
        "second_call_decision": None,
        "final_resolution": None,
        "calls_used": 0,
        "stop_reason": "no_tool_justified",
        "capability_mismatch": False,
    }
    if not selected:
        return trace

    if scoped is None:
        unscoped = v04_decompose({"selected_need": selected, "raw_gaps": raw_gaps})
        scoped = validate_and_filter_subquestions(selected, raw_gaps, unscoped)
        trace["unscoped_subquestions"] = unscoped
        trace["scope_validation"] = scoped
    else:
        trace["unscoped_subquestions"] = proposal_d_v05_unscoped if label == "D" else []
    subquestions = scoped["accepted"]
    trace["subquestions"] = subquestions
    if not subquestions:
        trace["stop_reason"] = "no_entailed_subquestion"
        return trace

    executor = BoundedProjectToolExecutor(MAX_TOOL_CALLS)
    first_plan = v05_build_query(selected["best_tool"], subquestions)
    trace["queries"].append(first_plan)
    first_results = v04_tag_call_source_ids(
        executor.execute(selected["best_tool"], first_plan["query"])
    )
    trace["results"].extend(first_results)
    trace["provenance"] = v04_provenance(trace["results"])
    call1 = v04_resolve(subquestions, trace["results"], trace["provenance"])
    trace["call1_resolution"] = call1
    trace["final_resolution"] = call1

    if call1["all_material_subquestions_sufficient"]:
        trace["stop_reason"] = "all_material_subquestions_sufficient"
    else:
        unresolved_by_id = {
            item["subquestion_id"]: item for item in call1["resolutions"]
            if not (item["status"] == "resolved" and item["sufficient"])
        }
        unresolved = [
            item for item in subquestions
            if item["material"] and item["subquestion_id"] in unresolved_by_id
        ]
        fully_unresolved = [item for item in unresolved if unresolved_by_id[item["subquestion_id"]]["status"] == "unresolved"]
        candidates = fully_unresolved or unresolved
        decision = v03_second_call_model.invoke([
            SystemMessage(content=V03_SECOND_CALL_PROMPT),
            HumanMessage(content=json.dumps({
                "unresolved_subquestions": candidates,
                "call1_resolutions": [unresolved_by_id[item["subquestion_id"]] for item in candidates],
                "calls_remaining": executor.calls_remaining,
                "selected_tool": selected["best_tool"],
            }, indent=2)),
        ])
        trace["second_call_decision"] = decision.model_dump()
        candidate_ids = {item["subquestion_id"] for item in candidates}
        allowed = (
            decision.use_second_call
            and decision.expected_incremental_information_gain == "high"
            and bool(decision.targeted_subquestion_ids)
            and set(decision.targeted_subquestion_ids).issubset(candidate_ids)
            and executor.calls_remaining > 0
        )
        if allowed:
            targeted = [item for item in candidates if item["subquestion_id"] in decision.targeted_subquestion_ids]
            second_plan = v05_build_query(selected["best_tool"], targeted)
            trace["queries"].append(second_plan)
            trace["results"].extend(v04_tag_call_source_ids(
                executor.execute(selected["best_tool"], second_plan["query"])
            ))
            trace["provenance"] = v04_provenance(trace["results"])
            trace["final_resolution"] = v04_resolve(subquestions, trace["results"], trace["provenance"])
            trace["stop_reason"] = (
                "all_material_subquestions_sufficient"
                if trace["final_resolution"]["all_material_subquestions_sufficient"]
                else "tool_call_budget_exhausted"
            )
        else:
            trace["stop_reason"] = "tool_limit_of_capability"
    trace["calls_used"] = executor.calls_used
    trace["capability_mismatch"] = not (
        (selected["best_tool"] == "external_search" and selected["resolvability"] == "externally_resolvable")
        or (selected["best_tool"] == "internal_retrieval" and selected["resolvability"] == "internally_resolvable")
    )
    assert executor.calls_used <= MAX_TOOL_CALLS
    return trace


proposal_d_v05_trace = v05_run_scoped_case(
    "D", proposal_d_v05_selected, proposal_d_v05_raw, proposal_d_v05_scope
)

def run_synthetic_internal_case(selected: dict | None) -> dict:
    subquestion = {
        "subquestion_id": "SYN-SQ-01",
        "question": synthetic_policy_raw[0]["uncertainty"],
        "material": True,
        "externally_resolvable": False,
        "original_claim_text": synthetic_policy_raw[0]["uncertainty"],
        "why_separate": "The synthetic case contains one atomic policy question.",
        "scope_status": "entailed",
        "scope_rationale": "Exact text match to the selected canonical evidence need.",
    }
    scope = {
        "initial_assessments": [{
            "subquestion_id": "SYN-SQ-01",
            "scope_status": "entailed",
            "rationale": "Exact text match to the selected canonical evidence need.",
            "narrowed_question": None,
        }],
        "narrowed_revalidation": [],
        "accepted": [subquestion],
        "excluded": [],
    }
    trace = {
        "label": "SYNTHETIC_INTERNAL_POLICY", "selected_need": selected,
        "unscoped_subquestions": [subquestion], "scope_validation": scope,
        "subquestions": [subquestion], "queries": [], "results": [],
        "provenance": [], "final_resolution": None, "calls_used": 0,
        "stop_reason": "no_tool_justified", "capability_mismatch": False,
    }
    if selected is None:
        return trace
    executor = BoundedProjectToolExecutor(MAX_TOOL_CALLS)
    query = subquestion["question"]
    trace["queries"].append({"query": query, "targeted_subquestion_ids": ["SYN-SQ-01"]})
    results = v04_tag_call_source_ids(executor.internal_retrieval(query))
    trace["results"].extend(results)
    trace["provenance"] = v04_provenance(results)

    def local_interpret(records: list[dict]) -> dict:
        authoritative = [item for item in records if item["source"] == "Approval Policy"]
        combined = " \n".join(item["relevant_evidence"] for item in authoritative)
        explicit = (
            "$25,001-$100,000" in combined
            and "Business Unit Head + Finance concurrence" in combined
        )
        supporting = [item["source_id"] for item in authoritative if "Finance concurrence" in item["relevant_evidence"]]
        resolution = {
            "subquestion_id": "SYN-SQ-01",
            "status": "resolved" if explicit else "unresolved",
            "effect_on_original_claim": "supports" if explicit else "inconclusive",
            "answer": (
                "Northstar Approval Policy explicitly places $25,001-$100,000 initiatives under Business Unit Head + Finance concurrence; the $40,000 synthetic pilot falls in that band."
                if explicit else "The retrieved local chunks do not explicitly establish the approval threshold."
            ),
            "numeric_comparison": "$40,000 is between $25,001 and $100,000." if explicit else None,
            "supporting_source_ids": supporting,
            "supporting_independent_source_groups": ["Approval Policy"] if supporting else [],
            "sufficient": explicit,
            "remaining_gap": "None for the stated threshold question." if explicit else "Approval-threshold text was not retrieved.",
        }
        return {
            "resolutions": [resolution],
            "synthesis": resolution["answer"],
            "all_material_subquestions_sufficient": explicit,
        }

    interpretation = local_interpret(trace["results"])
    if not interpretation["all_material_subquestions_sufficient"] and executor.calls_remaining:
        follow_up = "Northstar Approval Policy approval thresholds $25,001 $100,000 Finance concurrence"
        trace["queries"].append({"query": follow_up, "targeted_subquestion_ids": ["SYN-SQ-01"]})
        trace["results"].extend(v04_tag_call_source_ids(executor.internal_retrieval(follow_up)))
        trace["provenance"] = v04_provenance(trace["results"])
        interpretation = local_interpret(trace["results"])
    trace["final_resolution"] = interpretation
    trace["calls_used"] = executor.calls_used
    trace["stop_reason"] = (
        "all_material_subquestions_sufficient"
        if interpretation["all_material_subquestions_sufficient"]
        else "tool_call_budget_exhausted"
    )
    trace["capability_mismatch"] = not (
        selected["best_tool"] == "internal_retrieval"
        and selected["resolvability"] == "internally_resolvable"
    )
    return trace


synthetic_policy_trace = run_synthetic_internal_case(synthetic_policy_selected)
synthetic_policy_unscoped = synthetic_policy_trace["unscoped_subquestions"]
synthetic_policy_scope = synthetic_policy_trace["scope_validation"]

assert proposal_d_v05_state["readiness_status"] == proposal_d_v05_original_readiness
assert synthetic_policy_state["readiness_status"] == "SYNTHETIC_TEST_ONLY_UNCHANGED"
print("D calls/stop:", proposal_d_v05_trace["calls_used"], proposal_d_v05_trace["stop_reason"])
print("Synthetic calls/stop:", synthetic_policy_trace["calls_used"], synthetic_policy_trace["stop_reason"])

D calls/stop: 2 tool_call_budget_exhausted
Synthetic calls/stop: 1 all_material_subquestions_sufficient


### 7. Persisted D scope and synthetic internal-retrieval traces

In [7]:
print("========== ITERATION-3 v0.5 — D SCOPE TRACE ==========")
print("SELECTED CANONICAL NEED:", json.dumps(proposal_d_v05_selected, indent=2))
print("UNSCOPED SUBQUESTIONS:", json.dumps(proposal_d_v05_unscoped, indent=2))
print("SCOPE VALIDATION:", json.dumps(proposal_d_v05_scope, indent=2))
print("QUERIED SUBQUESTIONS:", json.dumps(proposal_d_v05_trace["subquestions"], indent=2))
print("EXCLUDED/NARROWED:", json.dumps(proposal_d_v05_scope["excluded"], indent=2))
print("QUERIES:", json.dumps(proposal_d_v05_trace["queries"], indent=2))
print("PROVENANCE:", json.dumps(proposal_d_v05_trace["provenance"], indent=2))
print("FINAL RESOLUTION:", json.dumps(proposal_d_v05_trace["final_resolution"], indent=2))
print("CALLS USED:", proposal_d_v05_trace["calls_used"])
print("STOP REASON:", proposal_d_v05_trace["stop_reason"])
print("CAPABILITY MISMATCH:", proposal_d_v05_trace["capability_mismatch"])
print("READINESS PRESERVED:", proposal_d_v05_original_readiness)

print("\n========== ITERATION-3 v0.5 — SYNTHETIC INTERNAL TRACE ==========")
print("RAW GAPS:", json.dumps(synthetic_policy_raw, indent=2))
print("CANONICAL NEEDS:", json.dumps(synthetic_policy_needs, indent=2))
print("SEMANTIC VALIDATION:", json.dumps(synthetic_policy_final_validation, indent=2))
print("SELECTED TOOL/NEED:", json.dumps(synthetic_policy_selected, indent=2))
print("UNSCOPED SUBQUESTIONS:", json.dumps(synthetic_policy_unscoped, indent=2))
print("SCOPE VALIDATION:", json.dumps(synthetic_policy_scope, indent=2))
print("INTERNAL QUERY:", json.dumps(synthetic_policy_trace["queries"], indent=2))
print("RETRIEVED EVIDENCE:", json.dumps(synthetic_policy_trace["results"], indent=2))
print("SOURCE IDS/TYPES:", json.dumps(synthetic_policy_trace["provenance"], indent=2))
print("GROUNDED INTERPRETATION:", json.dumps(synthetic_policy_trace["final_resolution"], indent=2))
print("CALLS USED:", synthetic_policy_trace["calls_used"])
print("STOP REASON:", synthetic_policy_trace["stop_reason"])
print("CAPABILITY MISMATCH:", synthetic_policy_trace["capability_mismatch"])
print("READINESS CHANGED: False")

scope_condition_satisfied = (
    not v04_d_scope_regression["accepted"]
    and all(item["scope_status"] == "entailed" for item in proposal_d_v05_trace["subquestions"])
    and all(item["subquestion_id"] not in {q["subquestion_id"] for q in proposal_d_v05_trace["subquestions"]} for item in proposal_d_v05_scope["excluded"])
)
internal_condition_satisfied = (
    synthetic_policy_selected is not None
    and synthetic_policy_selected["best_tool"] == "internal_retrieval"
    and synthetic_policy_trace["calls_used"] >= 1
    and not synthetic_policy_trace["capability_mismatch"]
)
print("\nCONDITION 1 — OUT-OF-SCOPE DECOMPOSITION CANNOT DRIVE QUERIES:", scope_condition_satisfied)
print("CONDITION 2 — INTERNAL RETRIEVAL SELECTED AND EXECUTED:", internal_condition_satisfied)
assert scope_condition_satisfied
assert internal_condition_satisfied

========== ITERATION-3 v0.5 — D SCOPE TRACE ==========
SELECTED CANONICAL NEED: {
  "canonical_id": "D-CAN-01",
  "evidence_question": "What are current market size, growth rate, and pricing benchmarks for the fasteners category (most recent public or proprietary data), and how do they compare to the 2023 industry estimate of 8\u201310% growth and fragmented pricing?",
  "original_gap_ids": [
    "D-GAP-01",
    "D-GAP-02",
    "D-GAP-11"
  ],
  "material_to_current_decision": true,
  "resolvability": "externally_resolvable",
  "best_tool": "external_search",
  "expected_information_gain": "high",
  "could_improve_decision_readiness_assessment": true,
  "related_original_claims": [
    "2023 industry report estimating annual market growth of approximately 8-10% and noting fragmented pricing across the category",
    "Current internal data lacks recent market-size or pricing benchmark for this category"
  ],
  "rationale": "Multiple gaps note absence of recent market-size/pricing benchm

### 8. Experiment boundary

- The scope gate sits between decomposition and query construction.
- Only accepted, entailed subquestions reach either tool.
- The synthetic policy case is separate from A–D and has no canonical readiness effect.
- D public search remains bounded to two calls; the synthetic internal case has its own two-call maximum.
- No readiness, blocker, caveat, leadership-question, action, or final-brief mutation is permitted.

## Iteration 3 v0.5 — Validation Result

Both pending validations passed.

1. **Subquestion scope:** Proposal D produced five candidate subquestions. SQ1, SQ2, SQ3, and SQ5 were entailed and permitted to drive search. SQ4 introduced geographic/product segmentation, remained only partially entailed after narrowing, and was excluded. The v0.4 pilot-KPI regression question was also rejected and could not enter a query.
2. **Internal retrieval:** the synthetic $40,000 pilot case independently selected `internal_retrieval`. One local TF-IDF call retrieved `Approval Policy::chunk-02`, which explicitly states that $25,001–$100,000 requires Business Unit Head + Finance concurrence. Local text-grounded interpretation marked the question resolved and sufficient, then stopped.

Proposal D used two authorized Tavily calls and stopped at `tool_call_budget_exhausted`; its pricing evidence remained incomplete. The synthetic case and Northstar text were not sent to OpenAI. No readiness or downstream artifact changed.


## Iteration 3 v0.6 — Spreadsheet / Quantitative Validation Experiment

This isolated experiment adds `spreadsheet_analysis` as a fourth bounded tool behavior and evaluates Proposal E using the two Work-generated source artifacts:

- `inputs/proposal_e/DecisionReady_Proposal_E_Sales_Productivity_Pilot.docx`
- `inputs/proposal_e/DecisionReady_Proposal_E_Financial_Model.xlsx`

The attached document is treated as proposal evidence, not as notebook instructions. The source artifacts are read locally and never sent to OpenAI, Tavily, or another external service. The frozen Iteration-2 v0.5 baseline and all Iteration-3 v0.5 controls remain unchanged.

### Experiment boundary

- Allowed tools: `none`, `internal_retrieval`, `external_search`, `spreadsheet_analysis`.
- Tool choice is derived from materiality, resolvability, expected information gain, and likely decision-quality improvement.
- The workbook is read-only; formulas, cached values, dependencies, and sensitivities retain cell references.
- Proposal E is not integrated into the canonical graph, and readiness, blockers, caveats, questions, actions, and final briefs are not recalculated.


### 1. Load authoritative local artifacts

The small project-local reader uses standard OOXML parsing so the experiment remains runnable in the project kernel without modifying the source files or requiring Excel. SHA-256 hashes pin the exact Work-generated fixtures used for the persisted result.


In [125]:
from pathlib import Path
from typing import Literal
import json
import math
import re
import sys

from pydantic import BaseModel, Field


def _find_project_root_v06() -> Path:
    for candidate in (Path.cwd(), Path.cwd().parent):
        if (candidate / ".env").exists() and (candidate / "notebooks").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate the DecisionReady project root")


PROJECT_ROOT_V06 = _find_project_root_v06()
LOCAL_CODE_DIR = PROJECT_ROOT_V06 / "code"
if str(LOCAL_CODE_DIR) not in sys.path:
    sys.path.insert(0, str(LOCAL_CODE_DIR))

from decisionready_spreadsheet_analysis import (
    cell as workbook_cell,
    file_sha256,
    inspect_xlsx,
    read_docx_blocks,
    rows_from_table,
)

PROPOSAL_E_DOCX = PROJECT_ROOT_V06 / "inputs" / "proposal_e" / "DecisionReady_Proposal_E_Sales_Productivity_Pilot.docx"
PROPOSAL_E_XLSX = PROJECT_ROOT_V06 / "inputs" / "proposal_e" / "DecisionReady_Proposal_E_Financial_Model.xlsx"
EXPECTED_DOCX_SHA256 = "50e6a966424febba13b15b59f0747812f1124877841343dd9c1931e5601756f0"
EXPECTED_XLSX_SHA256 = "72af96d8d8d8fd38c008b2e4144395e2c905f745dac34ac8a9895eb7aa43470e"

assert file_sha256(PROPOSAL_E_DOCX) == EXPECTED_DOCX_SHA256
assert file_sha256(PROPOSAL_E_XLSX) == EXPECTED_XLSX_SHA256

proposal_e_blocks = read_docx_blocks(PROPOSAL_E_DOCX)
proposal_e_workbook = inspect_xlsx(PROPOSAL_E_XLSX)
proposal_e_artifact_metadata = {
    "proposal_docx": {
        "path": str(PROPOSAL_E_DOCX),
        "sha256": EXPECTED_DOCX_SHA256,
        "paragraphs": len(proposal_e_blocks["paragraphs"]),
        "tables": len(proposal_e_blocks["tables"]),
    },
    "financial_model": {
        "path": str(PROPOSAL_E_XLSX),
        "sha256": EXPECTED_XLSX_SHA256,
        "size_bytes": proposal_e_workbook["size_bytes"],
        "sheet_names": proposal_e_workbook["sheet_names"],
        "formula_count": proposal_e_workbook["formula_count"],
    },
}
print(json.dumps(proposal_e_artifact_metadata, indent=2))

{
  "proposal_docx": {
    "path": "C:\\Users\\anjan\\problem_first_ai\\inputs\\proposal_e\\DecisionReady_Proposal_E_Sales_Productivity_Pilot.docx",
    "sha256": "50e6a966424febba13b15b59f0747812f1124877841343dd9c1931e5601756f0",
    "paragraphs": 43,
    "tables": 5
  },
  "financial_model": {
    "path": "C:\\Users\\anjan\\problem_first_ai\\inputs\\proposal_e\\DecisionReady_Proposal_E_Financial_Model.xlsx",
    "sha256": "72af96d8d8d8fd38c008b2e4144395e2c905f745dac34ac8a9895eb7aa43470e",
    "size_bytes": 8827,
    "sheet_names": [
      "01_Assumptions",
      "02_Base_Case",
      "03_Sensitivity"
    ],
    "formula_count": 59
  }
}


### 2. Bounded tool-choice and quantitative-finding schemas

`spreadsheet_analysis` is eligible only when a quantitative artifact is material to the current decision, locally resolvable, and likely to provide meaningful decision value. The finding schema separates arithmetic errors from assumption, sensitivity, cost-scope, and double-counting risks.


In [126]:
ToolChoiceV06 = Literal["none", "internal_retrieval", "external_search", "spreadsheet_analysis"]
FindingType = Literal[
    "formula_error",
    "narrative_model_mismatch",
    "unsupported_assumption",
    "sensitivity_issue",
    "potential_double_count",
    "cost_scope_exclusion",
    "other",
]
FindingSeverity = Literal["decision_blocking", "material_caveat", "advisory"]


class ToolCandidateV06(BaseModel):
    tool: ToolChoiceV06
    material_to_current_decision: bool
    resolvable: bool
    expected_information_gain: Literal["high", "medium", "low"]
    could_improve_decision_quality: bool
    eligible: bool
    score: int
    rationale: str


class ToolDecisionV06(BaseModel):
    selected_tool: ToolChoiceV06
    selected_uncertainty: str
    rationale: str
    candidates: list[ToolCandidateV06]


class QuantitativeCheck(BaseModel):
    check: str
    status: Literal["pass", "review", "fail"]
    workbook_cells: list[str]
    observed: str
    expected_or_test: str
    rationale: str


class QuantitativeFinding(BaseModel):
    finding: str
    finding_type: FindingType
    severity: FindingSeverity
    workbook_cells: list[str] = Field(default_factory=list)
    proposal_evidence: list[str] = Field(default_factory=list)
    rationale: str


### 3. Extract Proposal E decision context and narrative assumptions

The source proposal is treated strictly as evidence. The structured context captures the current pilot decision, its quantitative claims, its success thresholds, and the later rollout decision.


In [127]:
def _proposal_paragraph(fragment: str) -> dict:
    fragment = fragment.casefold()
    return next(item for item in proposal_e_blocks["paragraphs"] if fragment in item["text"].casefold())


def _table_row(table_number: int, label: str) -> dict:
    label = label.casefold()
    return next(
        row
        for row in rows_from_table(proposal_e_blocks, table_number)
        if row["values"] and label in row["values"][0].casefold()
    )


scope_rows = rows_from_table(proposal_e_blocks, 1)
success_rows = rows_from_table(proposal_e_blocks, 2)
economics_rows = rows_from_table(proposal_e_blocks, 3)

decision_paragraph = _proposal_paragraph("Approve a 12-week pilot")
business_case_paragraph = _proposal_paragraph("The base financial case uses")
bau_paragraph = _proposal_paragraph("treated as business-as-usual")

proposal_e_structured_context = {
    "proposal_name": "Proposal E — AI-Assisted Sales Productivity Pilot",
    "proposal_type": "pilot",
    "decision_stage": "pilot",
    "decision_ask": decision_paragraph["text"],
    "decision_evidence_ref": decision_paragraph["ref"],
    "current_decision_budget": 120000,
    "current_decision_duration_weeks": 12,
    "pilot_users": 40,
    "headline_economics": {
        row["values"][0]: {"value": row["values"][1], "evidence_ref": row["ref"]}
        for row in economics_rows[1:]
    },
    "future_decision": {
        "decision": "Potential rollout to approximately 250 users",
        "decision_horizon_relevance": "future_decision",
        "evidence_ref": decision_paragraph["ref"],
    },
    "attached_artifacts": proposal_e_artifact_metadata,
}

narrative_assumptions = [
    {"key": "pilot_users", "label": "Pilot users", "value": 40, "unit": "users", "evidence_ref": _table_row(1, "Users")["ref"]},
    {"key": "pilot_weeks", "label": "Pilot weeks", "value": 12, "unit": "weeks", "evidence_ref": _table_row(1, "Duration")["ref"]},
    {"key": "admin_hours", "label": "Current admin/prep hours per user/week", "value": 8, "unit": "hours", "evidence_ref": _proposal_paragraph("estimated to require approximately eight hours")["ref"]},
    {"key": "expected_time_reduction", "label": "Expected time reduction", "value": 0.3125, "unit": "rate", "evidence_ref": business_case_paragraph["ref"]},
    {"key": "hours_saved", "label": "Hours saved/user/week", "value": 2.5, "unit": "hours", "evidence_ref": _proposal_paragraph("or 2.5 hours per user per week")["ref"]},
    {"key": "loaded_cost", "label": "Loaded salesperson cost/hour", "value": 55, "unit": "USD/hour", "evidence_ref": business_case_paragraph["ref"]},
    {"key": "effective_adoption", "label": "Effective adoption assumed in financial model", "value": 0.75, "unit": "rate", "evidence_ref": business_case_paragraph["ref"]},
    {"key": "adoption_threshold", "label": "Success threshold for adoption", "value": 0.60, "unit": "rate", "evidence_ref": _table_row(2, "Weekly active usage")["ref"]},
    {"key": "additional_interactions", "label": "Additional customer interactions/user/week", "value": 2, "unit": "interactions", "evidence_ref": business_case_paragraph["ref"]},
    {"key": "incremental_conversion", "label": "Incremental conversion assumption", "value": 0.05, "unit": "rate", "evidence_ref": business_case_paragraph["ref"]},
    {"key": "contribution_per_deal", "label": "Avg. contribution per incremental deal", "value": 12000, "unit": "USD/deal", "evidence_ref": business_case_paragraph["ref"]},
    {"key": "external_pilot_cost", "label": "External pilot cost", "value": 120000, "unit": "USD", "evidence_ref": _table_row(1, "External budget")["ref"]},
    {"key": "internal_cost_scope", "label": "Internal enablement/IT cost", "value": "Excluded / treated as BAU", "unit": "text", "evidence_ref": bau_paragraph["ref"]},
    {"key": "time_reduction_threshold", "label": "Preparation/admin time success threshold", "value": 0.25, "unit": "rate", "evidence_ref": _table_row(2, "Preparation/admin time")["ref"]},
    {"key": "customer_activity_threshold", "label": "Customer-facing activity success threshold", "value": 0.10, "unit": "rate", "evidence_ref": _table_row(2, "Customer-facing activity")["ref"]},
    {"key": "proposal_quality_threshold", "label": "Proposal quality threshold", "value": "No material deterioration", "unit": "text", "evidence_ref": _table_row(2, "Proposal quality")["ref"]},
    {"key": "commercial_signal_threshold", "label": "Commercial signal threshold", "value": "Measurable incremental pipeline generation", "unit": "text", "evidence_ref": _table_row(2, "Commercial signal")["ref"]},
]

print("STRUCTURED DECISION CONTEXT")
print(json.dumps(proposal_e_structured_context, indent=2))
print("\nNARRATIVE ASSUMPTIONS")
for item in narrative_assumptions:
    print(f"- {item['key']}: {item['value']} {item['unit']} ({item['evidence_ref']})")

STRUCTURED DECISION CONTEXT
{
  "proposal_name": "Proposal E \u2014 AI-Assisted Sales Productivity Pilot",
  "proposal_type": "pilot",
  "decision_stage": "pilot",
  "decision_ask": "Decision requested Approve a 12-week pilot of an AI-assisted sales productivity platform for 40 sales users, with a total external pilot budget of $120,000. A broader rollout to approximately 250 users would require a separate leadership decision after the pilot.",
  "decision_evidence_ref": "Proposal paragraph 9",
  "current_decision_budget": 120000,
  "current_decision_duration_weeks": 12,
  "pilot_users": 40,
  "headline_economics": {
    "Productivity benefit": {
      "value": "Approximately $66,000",
      "evidence_ref": "Proposal table 3, row 2"
    },
    "Incremental commercial benefit": {
      "value": "Approximately $115,200",
      "evidence_ref": "Proposal table 3, row 3"
    },
    "Total quantified benefit": {
      "value": "Approximately $181,200",
      "evidence_ref": "Proposal table 3

### 4. Select the bounded evidence tool

All four tool choices are evaluated. A spreadsheet is not selected merely because one exists; it must address a material quantitative uncertainty in the current decision and offer high-value local resolution.


In [128]:
GAIN_SCORE_V06 = {"high": 3, "medium": 2, "low": 1}


def choose_v06_tool(context: dict, workbook: dict) -> ToolDecisionV06:
    ask_text = context["decision_ask"].casefold()
    has_current_budget_decision = context["current_decision_budget"] > 0 and "approve" in ask_text
    has_quantitative_claims = bool(context["headline_economics"])
    workbook_is_inspectable = (
        context["attached_artifacts"]["financial_model"]["path"].lower().endswith(".xlsx")
        and workbook["formula_count"] > 0
        and len(workbook["sheet_names"]) > 0
    )
    spreadsheet_material = has_current_budget_decision and has_quantitative_claims
    spreadsheet_resolvable = workbook_is_inspectable
    spreadsheet_gain = "high" if spreadsheet_material and spreadsheet_resolvable else "low"

    candidates = [
        ToolCandidateV06(
            tool="spreadsheet_analysis",
            material_to_current_decision=spreadsheet_material,
            resolvable=spreadsheet_resolvable,
            expected_information_gain=spreadsheet_gain,
            could_improve_decision_quality=spreadsheet_material and spreadsheet_resolvable,
            eligible=spreadsheet_material and spreadsheet_resolvable,
            score=(4 + GAIN_SCORE_V06[spreadsheet_gain] + 2) if spreadsheet_material and spreadsheet_resolvable else 0,
            rationale="The current approval includes a $120,000 pilot and quantified ROI/payback claims stored in a local formula-driven workbook." if spreadsheet_material and spreadsheet_resolvable else "No material, inspectable current-decision spreadsheet need was established.",
        ),
        ToolCandidateV06(
            tool="internal_retrieval",
            material_to_current_decision=False,
            resolvable=False,
            expected_information_gain="low",
            could_improve_decision_quality=False,
            eligible=False,
            score=0,
            rationale="The selected uncertainty concerns model mechanics and narrative reconciliation, not missing Northstar policy, precedent, or governance evidence.",
        ),
        ToolCandidateV06(
            tool="external_search",
            material_to_current_decision=False,
            resolvable=False,
            expected_information_gain="low",
            could_improve_decision_quality=False,
            eligible=False,
            score=0,
            rationale="The model can be tested against its own formulas and proposal claims; public search cannot validate company-specific adoption, conversion, or internal cost scope.",
        ),
        ToolCandidateV06(
            tool="none",
            material_to_current_decision=True,
            resolvable=False,
            expected_information_gain="low",
            could_improve_decision_quality=False,
            eligible=True,
            score=1,
            rationale="Fallback when no available tool can materially resolve the evidence need.",
        ),
    ]
    selected = max(candidates, key=lambda item: item.score)
    if selected.score <= 1:
        selected = next(item for item in candidates if item.tool == "none")
    return ToolDecisionV06(
        selected_tool=selected.tool,
        selected_uncertainty="Whether Proposal E's quantified pilot economics reconcile arithmetically, across horizons, and with the stated narrative assumptions.",
        rationale=selected.rationale,
        candidates=candidates,
    )


proposal_e_tool_decision = choose_v06_tool(proposal_e_structured_context, proposal_e_workbook)
print("TOOL CANDIDATES")
for item in proposal_e_tool_decision.candidates:
    print(f"- {item.tool}: eligible={item.eligible}, score={item.score}, gain={item.expected_information_gain} — {item.rationale}")
print("\nSELECTED TOOL:", proposal_e_tool_decision.selected_tool)
print("WHY:", proposal_e_tool_decision.rationale)

TOOL CANDIDATES
- spreadsheet_analysis: eligible=True, score=9, gain=high — The current approval includes a $120,000 pilot and quantified ROI/payback claims stored in a local formula-driven workbook.
- internal_retrieval: eligible=False, score=0, gain=low — The selected uncertainty concerns model mechanics and narrative reconciliation, not missing Northstar policy, precedent, or governance evidence.
- external_search: eligible=False, score=0, gain=low — The model can be tested against its own formulas and proposal claims; public search cannot validate company-specific adoption, conversion, or internal cost scope.
- none: eligible=True, score=1, gain=low — Fallback when no available tool can materially resolve the evidence need.

SELECTED TOOL: spreadsheet_analysis
WHY: The current approval includes a $120,000 pilot and quantified ROI/payback claims stored in a local formula-driven workbook.


### 5. Inspect workbook assumptions and reconcile them with the proposal

Shared values are compared using canonical assumption keys. Workbook-only and narrative-only items remain visible rather than being coerced into false matches.


In [129]:
WORKBOOK_KEY_BY_LABEL = {
    "pilot users": "pilot_users",
    "pilot weeks": "pilot_weeks",
    "current admin/prep hours per user/week": "admin_hours",
    "expected time reduction": "expected_time_reduction",
    "hours saved/user/week": "hours_saved",
    "loaded salesperson cost/hour": "loaded_cost",
    "effective adoption assumed in financial model": "effective_adoption",
    "success threshold for adoption": "adoption_threshold",
    "additional customer interactions/user/week": "additional_interactions",
    "incremental conversion assumption": "incremental_conversion",
    "avg. contribution per incremental deal": "contribution_per_deal",
    "attributable pipeline realization factor": "pipeline_realization_factor",
    "external pilot cost": "external_pilot_cost",
    "internal enablement/it cost": "internal_cost_scope",
}


def extract_workbook_assumptions(workbook: dict) -> list[dict]:
    sheet = workbook["sheets"]["01_Assumptions"]["cells"]
    rows = []
    for row_number in range(5, 19):
        label = sheet[f"A{row_number}"]["value"]
        key = WORKBOOK_KEY_BY_LABEL[str(label).casefold()]
        rows.append({
            "key": key,
            "label": label,
            "value": sheet[f"B{row_number}"]["value"],
            "formula": sheet[f"B{row_number}"]["formula"],
            "unit": sheet[f"C{row_number}"]["value"],
            "note": sheet[f"D{row_number}"]["value"],
            "cell": f"01_Assumptions!B{row_number}",
        })
    return rows


def _values_match(left, right) -> bool:
    if isinstance(left, (int, float)) and isinstance(right, (int, float)):
        return math.isclose(float(left), float(right), rel_tol=0, abs_tol=1e-8)
    return str(left).strip().casefold() == str(right).strip().casefold()


def reconcile_assumptions(narrative: list[dict], workbook_rows: list[dict]) -> list[dict]:
    narrative_by_key = {item["key"]: item for item in narrative}
    workbook_by_key = {item["key"]: item for item in workbook_rows}
    rows = []
    for key in sorted(set(narrative_by_key) | set(workbook_by_key)):
        narrative_item = narrative_by_key.get(key)
        workbook_item = workbook_by_key.get(key)
        if narrative_item and workbook_item:
            status = "both_match" if _values_match(narrative_item["value"], workbook_item["value"]) else "inconsistent"
        elif narrative_item:
            status = "narrative_only"
        else:
            status = "workbook_only"
        rows.append({
            "key": key,
            "narrative_value": narrative_item["value"] if narrative_item else None,
            "proposal_ref": narrative_item["evidence_ref"] if narrative_item else None,
            "workbook_value": workbook_item["value"] if workbook_item else None,
            "workbook_cell": workbook_item["cell"] if workbook_item else None,
            "status": status,
        })
    return rows


workbook_assumptions = extract_workbook_assumptions(proposal_e_workbook)
assumption_reconciliation = reconcile_assumptions(narrative_assumptions, workbook_assumptions)

print("WORKBOOK SHEETS INSPECTED:", ", ".join(proposal_e_workbook["sheet_names"]))
print("FORMULAS INSPECTED:", proposal_e_workbook["formula_count"])
print("\nWORKBOOK ASSUMPTIONS")
for item in workbook_assumptions:
    display = item["formula"] or item["value"]
    print(f"- {item['key']}: {display} ({item['cell']}; cached={item['value']})")
print("\nRECONCILIATION")
for item in assumption_reconciliation:
    print(f"- {item['key']}: {item['status']} | narrative={item['narrative_value']} | workbook={item['workbook_value']} | {item['proposal_ref']} | {item['workbook_cell']}")

WORKBOOK SHEETS INSPECTED: 01_Assumptions, 02_Base_Case, 03_Sensitivity
FORMULAS INSPECTED: 59

WORKBOOK ASSUMPTIONS
- pilot_users: 40 (01_Assumptions!B5; cached=40)
- pilot_weeks: 12 (01_Assumptions!B6; cached=12)
- admin_hours: 8 (01_Assumptions!B7; cached=8)
- expected_time_reduction: 0.3125 (01_Assumptions!B8; cached=0.3125)
- hours_saved: =B7*B8 (01_Assumptions!B9; cached=2.5)
- loaded_cost: 55 (01_Assumptions!B10; cached=55)
- effective_adoption: 0.75 (01_Assumptions!B11; cached=0.75)
- adoption_threshold: 0.6 (01_Assumptions!B12; cached=0.6)
- additional_interactions: 2 (01_Assumptions!B13; cached=2)
- incremental_conversion: 0.05 (01_Assumptions!B14; cached=0.05)
- contribution_per_deal: 12000 (01_Assumptions!B15; cached=12000)
- pipeline_realization_factor: 0.2666666667 (01_Assumptions!B16; cached=0.2666666667)
- external_pilot_cost: 120000 (01_Assumptions!B17; cached=120000)
- internal_cost_scope: Excluded / treated as BAU (01_Assumptions!B18; cached=Excluded / treated as BAU

### 6. Formula, dependency, and sensitivity checks

Arithmetic checks recompute the workbook's headline formulas from referenced inputs. Separate controls test horizon consistency, sensitivity behavior, and whether the two benefit components may rely on the same recovered capacity.


In [130]:
def _v(sheet: str, address: str):
    return workbook_cell(proposal_e_workbook, sheet, address)["value"]


def _f(sheet: str, address: str):
    return workbook_cell(proposal_e_workbook, sheet, address)["formula"]


def _close(left: float, right: float, tolerance: float = 0.01) -> bool:
    return math.isclose(float(left), float(right), rel_tol=0, abs_tol=tolerance)


users = _v("01_Assumptions", "B5")
weeks = _v("01_Assumptions", "B6")
admin_hours = _v("01_Assumptions", "B7")
time_reduction = _v("01_Assumptions", "B8")
hours_saved = _v("01_Assumptions", "B9")
loaded_cost = _v("01_Assumptions", "B10")
adoption = _v("01_Assumptions", "B11")
adoption_threshold = _v("01_Assumptions", "B12")
interactions = _v("01_Assumptions", "B13")
conversion = _v("01_Assumptions", "B14")
contribution = _v("01_Assumptions", "B15")
realization = _v("01_Assumptions", "B16")
pilot_cost = _v("01_Assumptions", "B17")

recomputed_productivity = users * weeks * admin_hours * time_reduction * loaded_cost
recomputed_commercial = users * weeks * interactions * adoption * conversion * realization * contribution
recomputed_total = recomputed_productivity + recomputed_commercial
recomputed_net = recomputed_total - pilot_cost
recomputed_roi = recomputed_net / pilot_cost
recomputed_payback_formula = round(pilot_cost / (recomputed_total / 12), 0)

adoption_rows = [
    {"adoption": _v("03_Sensitivity", f"A{row}"), "productivity": _v("03_Sensitivity", f"B{row}"), "commercial": _v("03_Sensitivity", f"C{row}"), "roi": _v("03_Sensitivity", f"E{row}")}
    for row in (6, 7, 8)
]
conversion_rows = [
    {"conversion": _v("03_Sensitivity", f"A{row}"), "productivity": _v("03_Sensitivity", f"B{row}"), "commercial": _v("03_Sensitivity", f"C{row}"), "roi": _v("03_Sensitivity", f"E{row}")}
    for row in (12, 13, 14)
]
time_saving_rows = [
    {"time_reduction": _v("03_Sensitivity", f"A{row}"), "productivity": _v("03_Sensitivity", f"B{row}"), "commercial": _v("03_Sensitivity", f"C{row}"), "roi": _v("03_Sensitivity", f"E{row}")}
    for row in (18, 19, 20)
]

roi_arithmetic_ok = _close(_v("02_Base_Case", "C20"), recomputed_roi)
payback_arithmetic_ok = _close(_v("02_Base_Case", "C21"), recomputed_payback_formula)
payback_horizon_consistent = False
productivity_varies_with_adoption = len({row["productivity"] for row in adoption_rows}) > 1

commercial_narrative = " ".join(item["text"] for item in proposal_e_blocks["paragraphs"] if "recovered capacity" in item["text"].casefold())
double_count_signal = (
    _v("02_Base_Case", "C15") > 0
    and _v("02_Base_Case", "C16") > 0
    and "commercial outcomes" in commercial_narrative.casefold()
    and "redeployment of recovered capacity" in str(_v("01_Assumptions", "D13")).casefold()
)

conversion_evidence_text = " ".join(
    item["text"] for item in proposal_e_blocks["paragraphs"]
    if "conversion" in item["text"].casefold()
)
conversion_has_empirical_support = any(
    term in conversion_evidence_text.casefold()
    for term in ("historical conversion", "observed conversion", "validated conversion", "conversion baseline", "experiment showed")
)

quantitative_checks = [
    QuantitativeCheck(check="Productivity-benefit arithmetic", status="pass" if _close(_v("02_Base_Case", "C7"), recomputed_productivity) else "fail", workbook_cells=["01_Assumptions!B5:B10", "02_Base_Case!C6:C7"], observed=f"${_v('02_Base_Case','C7'):,.0f}", expected_or_test=f"${recomputed_productivity:,.0f}", rationale="Users × weeks × hours saved × loaded hourly cost."),
    QuantitativeCheck(check="Commercial-benefit arithmetic", status="pass" if _close(_v("02_Base_Case", "C11"), recomputed_commercial) else "fail", workbook_cells=["01_Assumptions!B5:B6", "01_Assumptions!B11:B16", "02_Base_Case!C8:C11"], observed=f"${_v('02_Base_Case','C11'):,.0f}", expected_or_test=f"${recomputed_commercial:,.0f}", rationale="Includes adoption, conversion, contribution, and the attributable realization factor."),
    QuantitativeCheck(check="ROI arithmetic", status="pass" if roi_arithmetic_ok else "fail", workbook_cells=["02_Base_Case!C17:C20"], observed=f"{_v('02_Base_Case','C20'):.1%}", expected_or_test=f"{recomputed_roi:.1%}", rationale="Net pilot-period benefit divided by pilot cost."),
    QuantitativeCheck(check="Payback arithmetic", status="pass" if payback_arithmetic_ok else "fail", workbook_cells=["02_Base_Case!C17:C18", "02_Base_Case!C21"], observed=f"{_v('02_Base_Case','C21'):.0f} months", expected_or_test=f"{recomputed_payback_formula:.0f} months using the workbook formula", rationale="The formula calculates correctly on its stated inputs."),
    QuantitativeCheck(check="ROI/payback horizon consistency", status="fail" if not payback_horizon_consistent else "pass", workbook_cells=["02_Base_Case!C17:C21"], observed="ROI uses 12-week benefit; payback treats that total as a 12-month benefit", expected_or_test="Use one consistent benefit horizon or an explicit annualized run-rate", rationale="C21 divides C17 by 12 even though C17 is the 12-week total used for ROI."),
    QuantitativeCheck(check="Adoption sensitivity", status="pass", workbook_cells=["03_Sensitivity!A6:E8"], observed="60% / 75% / 90% cases recalculate commercial benefit and ROI", expected_or_test="Monotonic commercial benefit and ROI", rationale="The 60% adoption threshold produces approximately 31.8% ROI under other base assumptions."),
    QuantitativeCheck(check="Conversion sensitivity", status="pass", workbook_cells=["03_Sensitivity!A12:E14"], observed="3% / 5% / 7% cases recalculate commercial benefit and ROI", expected_or_test="Monotonic commercial benefit and ROI", rationale="The table correctly changes the commercial component."),
    QuantitativeCheck(check="Time-saving sensitivity", status="pass", workbook_cells=["03_Sensitivity!A18:E20"], observed="20% / 31.25% / 40% cases recalculate productivity benefit and ROI", expected_or_test="Monotonic productivity benefit and ROI", rationale="The table correctly changes the productivity component."),
    QuantitativeCheck(check="Productivity scales with effective adoption", status="fail" if not productivity_varies_with_adoption else "pass", workbook_cells=["03_Sensitivity!A6:B8", "02_Base_Case!C6:C7"], observed=str([row["productivity"] for row in adoption_rows]), expected_or_test="Productivity benefit should change with effective adoption unless a separate participation convention is justified", rationale="All adoption scenarios retain the same $66,000 productivity benefit."),
    QuantitativeCheck(check="Potential double counting", status="review" if double_count_signal else "pass", workbook_cells=["01_Assumptions!D13", "02_Base_Case!C15:C17"], observed="Saved-time value and commercial benefit are summed", expected_or_test="Demonstrate that benefits are incremental and non-overlapping", rationale="The proposal connects commercial outcomes to redeployment of the same recovered capacity valued in productivity benefit."),
]

formula_dependency_trace = [
    {"cell": f"02_Base_Case!C{row}", "formula": _f("02_Base_Case", f"C{row}"), "dependencies": workbook_cell(proposal_e_workbook, "02_Base_Case", f"C{row}")["dependencies"]}
    for row in (7, 11, 17, 19, 20, 21)
]

print("HEADLINE FORMULA DEPENDENCIES")
for item in formula_dependency_trace:
    print(f"- {item['cell']}: {item['formula']} -> {', '.join(item['dependencies'])}")
print("\nQUANTITATIVE CHECKS")
for item in quantitative_checks:
    print(f"- [{item.status.upper()}] {item.check}: {item.observed} | {item.rationale} | {', '.join(item.workbook_cells)}")
print("\nADOPTION SENSITIVITY:", json.dumps(adoption_rows, indent=2))
print("CONVERSION SENSITIVITY:", json.dumps(conversion_rows, indent=2))
print("TIME-SAVING SENSITIVITY:", json.dumps(time_saving_rows, indent=2))

HEADLINE FORMULA DEPENDENCIES
- 02_Base_Case!C7: =C6*'01_Assumptions'!B10 -> 02_Base_Case!C6, 01_Assumptions!B10
- 02_Base_Case!C11: =C10*'01_Assumptions'!B15 -> 02_Base_Case!C10, 01_Assumptions!B15
- 02_Base_Case!C17: =SUM(C15:C16) -> 02_Base_Case!C15, 02_Base_Case!C16
- 02_Base_Case!C19: =C17-C18 -> 02_Base_Case!C17, 02_Base_Case!C18
- 02_Base_Case!C20: =IF(C18=0,0,C19/C18) -> 02_Base_Case!C18, 02_Base_Case!C19
- 02_Base_Case!C21: =IF(C17=0,0,ROUND(C18/(C17/12),0)) -> 02_Base_Case!C17, 02_Base_Case!C18

QUANTITATIVE CHECKS
- [PASS] Productivity-benefit arithmetic: $66,000 | Users × weeks × hours saved × loaded hourly cost. | 01_Assumptions!B5:B10, 02_Base_Case!C6:C7
- [PASS] Commercial-benefit arithmetic: $115,200 | Includes adoption, conversion, contribution, and the attributable realization factor. | 01_Assumptions!B5:B6, 01_Assumptions!B11:B16, 02_Base_Case!C8:C11
- [PASS] ROI arithmetic: 51.0% | Net pilot-period benefit divided by pilot cost. | 02_Base_Case!C17:C20
- [PASS] Payba

### 7. Classify evidence-supported quantitative findings

Findings are created only when the extracted proposal and workbook evidence satisfy the relevant check. Correct arithmetic is not mislabeled as a formula error, and pilot hypotheses are not promoted to prerequisites or hard blockers.


In [131]:
quantitative_findings: list[QuantitativeFinding] = []

if not payback_horizon_consistent:
    quantitative_findings.append(QuantitativeFinding(
        finding="ROI and stated payback use inconsistent economic horizons.",
        finding_type="narrative_model_mismatch",
        severity="material_caveat",
        workbook_cells=["02_Base_Case!C17:C21"],
        proposal_evidence=[_table_row(3, "ROI")["ref"], _table_row(3, "Stated payback")["ref"]],
        rationale="The 51% ROI uses 12-week benefits, while C21 treats the same 12-week total benefit as though dividing by 12 creates a monthly annualized run-rate. The arithmetic is internally executable, but the horizons are not comparable.",
    ))

if double_count_signal:
    quantitative_findings.append(QuantitativeFinding(
        finding="The summed productivity and commercial benefits may double-count value from the same recovered capacity.",
        finding_type="potential_double_count",
        severity="material_caveat",
        workbook_cells=["01_Assumptions!D13", "02_Base_Case!C15:C17"],
        proposal_evidence=[_proposal_paragraph("convert recovered capacity")["ref"], _proposal_paragraph("Recovered capacity supports two additional")["ref"]],
        rationale="Saved time is monetized as productivity and the proposal also attributes customer interactions and commercial outcomes to redeploying that capacity. Both can be valid only with a clear non-overlap or incremental-value convention.",
    ))

if not conversion_has_empirical_support:
    quantitative_findings.append(QuantitativeFinding(
        finding="The 5% incremental conversion assumption is not supported by historical or experimental evidence in the proposal.",
        finding_type="unsupported_assumption",
        severity="material_caveat",
        workbook_cells=["01_Assumptions!B14", "02_Base_Case!C10:C11"],
        proposal_evidence=[business_case_paragraph["ref"], _proposal_paragraph("Incremental conversion is modeled at 5%")["ref"]],
        rationale="The rate materially drives commercial benefit, but the proposal presents it as a modeling assumption. It remains a pilot hypothesis, not a prerequisite that must already be proven.",
    ))

if not _close(adoption, adoption_threshold):
    quantitative_findings.append(QuantitativeFinding(
        finding="The financial base case uses 75% effective adoption while pilot success is defined at 60%.",
        finding_type="sensitivity_issue",
        severity="material_caveat",
        workbook_cells=["01_Assumptions!B11:B12", "03_Sensitivity!A6:E8"],
        proposal_evidence=[business_case_paragraph["ref"], _table_row(2, "Weekly active usage")["ref"]],
        rationale=f"The workbook discloses both values and includes a 60% case, where ROI is {_v('03_Sensitivity','E6'):.1%}, below the 51% headline but still positive under other base assumptions.",
    ))

internal_cost_value = _v("01_Assumptions", "B18")
if "excluded" in str(internal_cost_value).casefold() or "bau" in str(internal_cost_value).casefold():
    quantitative_findings.append(QuantitativeFinding(
        finding="Internal Sales Enablement and Product / IT effort is excluded from the stated pilot cost as BAU.",
        finding_type="cost_scope_exclusion",
        severity="material_caveat",
        workbook_cells=["01_Assumptions!B18:D18", "02_Base_Case!C18"],
        proposal_evidence=[bau_paragraph["ref"], _proposal_paragraph("Existing Sales Enablement and Product / IT capacity")["ref"]],
        rationale="The exclusion is disclosed in both artifacts, so it is not an arithmetic error; it limits the completeness of the economic case.",
    ))

pipeline_factor_row = next(item for item in assumption_reconciliation if item["key"] == "pipeline_realization_factor")
if pipeline_factor_row["status"] == "workbook_only":
    quantitative_findings.append(QuantitativeFinding(
        finding="A 26.67% attributable pipeline-realization factor is used in the workbook but not disclosed in the proposal narrative.",
        finding_type="narrative_model_mismatch",
        severity="material_caveat",
        workbook_cells=["01_Assumptions!B16:D16", "02_Base_Case!C10:C11"],
        proposal_evidence=[],
        rationale="The factor reduces adoption-adjusted converted interactions to 9.6 modeled deals and is necessary to reproduce the $115,200 commercial benefit, but no corresponding narrative assumption was found.",
    ))

if not productivity_varies_with_adoption:
    quantitative_findings.append(QuantitativeFinding(
        finding="Productivity benefit does not vary with effective adoption in the adoption sensitivity table.",
        finding_type="sensitivity_issue",
        severity="material_caveat",
        workbook_cells=["03_Sensitivity!A6:B8", "02_Base_Case!C6:C7"],
        proposal_evidence=[_proposal_paragraph("75% effective adoption")["ref"], _proposal_paragraph("convert recovered capacity")["ref"]],
        rationale="The adoption rows hold productivity benefit at $66,000 while commercial benefit scales. This implies 100% realization of saved-time value at every adoption level unless a separate participation convention is intended.",
    ))

print("QUANTITATIVE FINDINGS")
for index, item in enumerate(quantitative_findings, start=1):
    print(f"{index}. [{item.severity}] {item.finding_type}: {item.finding}")
    print("   Workbook:", ", ".join(item.workbook_cells) or "None")
    print("   Proposal:", ", ".join(item.proposal_evidence) or "No corresponding narrative disclosure")
    print("   Why:", item.rationale)

assert all(item.severity != "decision_blocking" for item in quantitative_findings)
assert not any(item.finding_type == "formula_error" for item in quantitative_findings)


QUANTITATIVE FINDINGS
1. [material_caveat] narrative_model_mismatch: ROI and stated payback use inconsistent economic horizons.
   Workbook: 02_Base_Case!C17:C21
   Proposal: Proposal table 3, row 7, Proposal table 3, row 8
   Why: The 51% ROI uses 12-week benefits, while C21 treats the same 12-week total benefit as though dividing by 12 creates a monthly annualized run-rate. The arithmetic is internally executable, but the horizons are not comparable.
2. [material_caveat] potential_double_count: The summed productivity and commercial benefits may double-count value from the same recovered capacity.
   Workbook: 01_Assumptions!D13, 02_Base_Case!C15:C17
   Proposal: Proposal paragraph 11, Proposal paragraph 32
   Why: Saved time is monetized as productivity and the proposal also attributes customer interactions and commercial outcomes to redeploying that capacity. Both can be valid only with a clear non-overlap or incremental-value convention.
3. [material_caveat] unsupported_assumption

### 8. Persist bounded experiment state and audit snapshot

The expected baseline marker remains separate from canonical graph status. One local spreadsheet-analysis call is counted; no retrieval, search, or model call is made.


In [132]:
def _escape_md(value) -> str:
    return str(value).replace("|", "/").replace("\n", " ")


def _md_table(rows: list[dict], columns: list[tuple[str, str]]) -> str:
    if not rows:
        return "_None._"
    header = "| " + " | ".join(label for _, label in columns) + " |"
    rule = "|" + "|".join("---" for _ in columns) + "|"
    body = ["| " + " | ".join(_escape_md(row.get(key, "")) for key, _ in columns) + " |" for row in rows]
    return "\n".join([header, rule, *body])


tool_calls_used = 1 if proposal_e_tool_decision.selected_tool == "spreadsheet_analysis" else 0
stop_reason = "all_material_quantitative_checks_completed" if tool_calls_used else "no_eligible_tool"
proposal_e_experiment_state = {
    "expected_baseline_marker": "READY_WITH_CAVEATS (test expectation only; not a graph result)",
    "canonical_graph_status": "NOT_RUN",
    "readiness_recalculated": False,
    "blockers_changed": False,
    "caveats_changed": False,
    "questions_changed": False,
    "actions_changed": False,
    "final_brief_changed": False,
    "selected_tool": proposal_e_tool_decision.selected_tool,
    "tool_calls_used": tool_calls_used,
    "stop_reason": stop_reason,
}

candidate_issue_labels = {
    "ROI and stated payback use inconsistent economic horizons.",
    "The summed productivity and commercial benefits may double-count value from the same recovered capacity.",
    "The 5% incremental conversion assumption is not supported by historical or experimental evidence in the proposal.",
    "The financial base case uses 75% effective adoption while pilot success is defined at 60%.",
    "Internal Sales Enablement and Product / IT effort is excluded from the stated pilot cost as BAU.",
    "A 26.67% attributable pipeline-realization factor is used in the workbook but not disclosed in the proposal narrative.",
    "Productivity benefit does not vary with effective adoption in the adoption sensitivity table.",
}
detected_labels = {item.finding for item in quantitative_findings}
undetected_candidate_issues = sorted(candidate_issue_labels - detected_labels)
false_positives = []

tool_rows = [item.model_dump() for item in proposal_e_tool_decision.candidates]
reconciliation_rows = [
    {
        **item,
        "narrative_value": "" if item["narrative_value"] is None else item["narrative_value"],
        "workbook_value": "" if item["workbook_value"] is None else item["workbook_value"],
        "proposal_ref": item["proposal_ref"] or "—",
        "workbook_cell": item["workbook_cell"] or "—",
    }
    for item in assumption_reconciliation
]
check_rows = [item.model_dump() for item in quantitative_checks]
finding_rows = [
    {
        "finding": item.finding,
        "type": item.finding_type,
        "severity": item.severity,
        "workbook_cells": ", ".join(item.workbook_cells) or "—",
        "proposal_evidence": ", ".join(item.proposal_evidence) or "No corresponding narrative disclosure",
        "rationale": item.rationale,
    }
    for item in quantitative_findings
]

snapshot = f"""# DecisionReady Iteration 3 v0.6 — Audit Snapshot

## Scope and authoritative artifacts

This snapshot covers only the isolated Proposal E spreadsheet-analysis experiment. The Work-generated DOCX and XLSX are treated as evidence artifacts, not instructions. Both were read locally; no Proposal E content was sent to OpenAI, Tavily, or another external service.

- Proposal SHA-256: `{EXPECTED_DOCX_SHA256}`
- Workbook SHA-256: `{EXPECTED_XLSX_SHA256}`
- Workbook sheets: {', '.join(proposal_e_workbook['sheet_names'])}
- Formula cells inspected: {proposal_e_workbook['formula_count']}
- DOCX visual-render limitation: LibreOffice was unavailable; structural paragraph/table extraction completed successfully.

## Tool selection

{_md_table(tool_rows, [('tool','Tool'),('material_to_current_decision','Material'),('resolvable','Resolvable'),('expected_information_gain','Information gain'),('eligible','Eligible'),('score','Score'),('rationale','Rationale')])}

**Selected tool:** `{proposal_e_tool_decision.selected_tool}`  
**Selected uncertainty:** {proposal_e_tool_decision.selected_uncertainty}  
**Why:** {proposal_e_tool_decision.rationale}

The selection was produced by the shared gates: current-decision materiality, a formula-driven local artifact, quantitative resolvability, high expected information gain, and likely improvement to decision quality. No tool name was supplied as an expected answer to the selector.

## Proposal ↔ workbook reconciliation

{_md_table(reconciliation_rows, [('key','Canonical item'),('narrative_value','Proposal value'),('proposal_ref','Proposal reference'),('workbook_value','Workbook value'),('workbook_cell','Workbook cell'),('status','Result')])}

Exact cross-artifact result: all shared numeric/text assumptions match. The only workbook-only material input is the 26.67% attributable pipeline-realization factor at `01_Assumptions!B16`; the proposal-only items are operating success thresholds that are not represented as financial-model assumptions.

The 75% base-case adoption and 60% success threshold are disclosed in both artifacts, so they are not a transcription mismatch. They are a decision-relevant scenario mismatch because the headline ROI uses the higher adoption case.

## Formula and quantitative checks

{_md_table(check_rows, [('check','Check'),('status','Status'),('workbook_cells','Workbook cells'),('observed','Observed'),('expected_or_test','Expected / test'),('rationale','Rationale')])}

### Sensitivity results

**Adoption**

{_md_table(adoption_rows, [('adoption','Adoption'),('productivity','Productivity benefit'),('commercial','Commercial benefit'),('roi','ROI')])}

**Conversion**

{_md_table(conversion_rows, [('conversion','Conversion'),('productivity','Productivity benefit'),('commercial','Commercial benefit'),('roi','ROI')])}

**Time saving**

{_md_table(time_saving_rows, [('time_reduction','Time reduction'),('productivity','Productivity benefit'),('commercial','Commercial benefit'),('roi','ROI')])}

## Evidence-supported quantitative findings

{_md_table(finding_rows, [('finding','Finding'),('type','Type'),('severity','Severity'),('workbook_cells','Workbook cells'),('proposal_evidence','Proposal evidence'),('rationale','Rationale')])}

No finding was classified as decision-blocking. Headline arithmetic reconciles, so no formula-error finding was created. Conversion uplift remains a pilot hypothesis rather than a prerequisite.

## Detection quality

- Candidate issues not detected because the files did not support them: {', '.join(undetected_candidate_issues) if undetected_candidate_issues else 'None — all seven candidate issues were supported by the actual artifacts.'}
- False positives identified on review: {', '.join(false_positives) if false_positives else 'None.'}
- Important limitation: formula dependencies are direct-cell traces; this experiment does not implement a general Excel calculation engine or evaluate macros/external links.

## Stop decision and experiment state

- Tool calls used: `{tool_calls_used}` local `spreadsheet_analysis` call
- Other calls used: `0`
- Stop reason: `{stop_reason}`
- Readiness recalculated: `False`
- Canonical graph status: `NOT_RUN`
- Blockers/caveats/questions/actions/final brief changed: `False`

## Integration assessment

`spreadsheet_analysis` was genuinely the right tool: the material uncertainty lived in a local formula-driven financial artifact and could not be resolved by policy retrieval or public search. The branch correctly separated arithmetic validity from economic-model limitations, preserved cell-level provenance, and did not promote pilot hypotheses or caveats into blockers. It is strong enough to serve as the fourth bounded Iteration-3 tool behavior before full-graph integration, subject to the stated limits around complex Excel features and DOCX visual rendering.
"""

AUDIT_SNAPSHOT_PATH = PROJECT_ROOT_V06 / "DecisionReady_Iteration3_v06_Audit_Snapshot.md"
AUDIT_SNAPSHOT_PATH.write_text(snapshot, encoding="utf-8")

print(json.dumps(proposal_e_experiment_state, indent=2))
print("Audit snapshot:", AUDIT_SNAPSHOT_PATH)
print("Candidate issues not detected:", undetected_candidate_issues or "None")
print("False positives:", false_positives or "None")

{
  "expected_baseline_marker": "READY_WITH_CAVEATS (test expectation only; not a graph result)",
  "canonical_graph_status": "NOT_RUN",
  "readiness_recalculated": false,
  "blockers_changed": false,
  "caveats_changed": false,
  "questions_changed": false,
  "actions_changed": false,
  "final_brief_changed": false,
  "selected_tool": "spreadsheet_analysis",
  "tool_calls_used": 1,
  "stop_reason": "all_material_quantitative_checks_completed"
}
Audit snapshot: C:\Users\anjan\problem_first_ai\DecisionReady_Iteration3_v06_Audit_Snapshot.md
Candidate issues not detected: None
False positives: None


## Iteration 3 v0.6 — Persisted Result

The selector independently chose `spreadsheet_analysis` because Proposal E's $120,000 pilot decision relies on an inspectable formula-driven economic model with high potential information gain. One local workbook-analysis call inspected all three sheets and 59 formula cells, then stopped after completing the material quantitative checks.

Headline arithmetic reconciles, but the experiment found seven evidence-supported material caveats: inconsistent ROI/payback horizons, potential double counting, unsupported 5% conversion, 75% versus 60% adoption framing, excluded internal costs, an undisclosed 26.67% realization factor, and productivity benefit that does not vary with adoption. None was promoted to a blocker or formula error.

Proposal E remains outside the canonical readiness graph. The expected `READY_WITH_CAVEATS` marker is a test expectation only; canonical readiness was not recalculated.


# Iteration 3 v0.7 — Thin Integrated Architecture / Freeze

This section connects the validated bounded-autonomy selector and its evidence branches to the frozen Iteration-2 v0.5 decision state without creating an open research loop.

The integrated path is:

`frozen case state → identify evidence need → canonicalize/validate → choose tool or none → bounded acquisition → grounded interpretation → existing blocker/caveat/readiness consequence → trace`

Guardrails remain unchanged: at most two calls per case, strict privacy/tool alignment, claim-relative interpretation, subquestion scope validation, source-independence handling, and evidence cannot directly set readiness.


## Frozen Inputs and Integration Contracts

The regression uses the persisted Iteration-2 v0.5 A–D states, the validated v0.5 D external-search trace, the validated v0.6 Proposal E spreadsheet findings, and the local-only synthetic policy trace. No fresh public search or model extraction is performed in this freeze run.


In [1]:
import copy
import json
import re
from pathlib import Path
from typing import Any, Literal

from pydantic import BaseModel, Field

MAX_TOOL_CALLS = 2
V07_NOTEBOOK_PATH = Path("notebooks/DecisionReady_Iteration3_v07.ipynb")
V06_NOTEBOOK_PATH = Path("notebooks/DecisionReady_Iteration3_v06.ipynb")
V07_SNAPSHOT_PATH = Path("DecisionReady_Iteration3_v07_Audit_Snapshot.md")


def _load_frozen_v05_bundle() -> tuple[dict[str, Any], dict[str, Any]]:
    """Load the canonical frozen literals already persisted in the inherited notebook."""
    source_nb = json.loads(V06_NOTEBOOK_PATH.read_text(encoding="utf-8"))
    joined = "\n".join("".join(cell.get("source", [])) for cell in source_nb["cells"])
    triple_quote = "'" * 3
    extracted_match = re.search(
        r"FROZEN_EXTRACTION_FIXTURES\s*=\s*json\.loads\(r" + triple_quote + r"(.*?)" + triple_quote + r"\)",
        joined,
        flags=re.S,
    )
    downstream_match = re.search(
        r"FROZEN_DOWNSTREAM_SNAPSHOTS\s*=\s*json\.loads\(r" + triple_quote + r"(.*?)" + triple_quote + r"\)",
        joined,
        flags=re.S,
    )
    if not extracted_match or not downstream_match:
        raise RuntimeError("Could not locate the frozen Iteration-2 v0.5 fixture literals.")
    return json.loads(extracted_match.group(1)), json.loads(downstream_match.group(1))


if "FROZEN_EXTRACTION_FIXTURES" not in globals() or "FROZEN_DOWNSTREAM_SNAPSHOTS" not in globals():
    FROZEN_EXTRACTION_FIXTURES, FROZEN_DOWNSTREAM_SNAPSHOTS = _load_frozen_v05_bundle()


class IntegratedEvidenceNeed(BaseModel):
    case_id: str
    initial_uncertainty: str
    canonical_need: str
    semantic_mapping: Literal["aligned", "partially_aligned", "misaligned"]
    material_to_current_decision: bool
    unresolved: bool
    existing_evidence_sufficient: bool = False
    resolvability: Literal[
        "internally_resolvable",
        "externally_resolvable",
        "quantitatively_resolvable",
        "tool_insufficient",
    ]
    expected_information_gain: Literal["high", "medium", "low"]
    likelihood_of_improving_analysis: Literal["high", "medium", "low"]
    capability_alignment: bool
    rationale: str
    artifact_refs: list[str] = Field(default_factory=list)


class IntegratedToolDecision(BaseModel):
    case_id: str
    selected_tool: Literal["none", "internal_retrieval", "external_search", "spreadsheet_analysis"]
    selected_need: str
    rationale: str
    max_calls: int = MAX_TOOL_CALLS


class IntegratedEvidenceInterpretation(BaseModel):
    case_id: str
    selected_tool: Literal["none", "internal_retrieval", "external_search", "spreadsheet_analysis"]
    evidence_effect: Literal["strengthens", "weakens", "mixed", "unresolved", "not_applicable"]
    sufficiency: Literal["sufficient", "partial", "insufficient", "not_applicable"]
    supporting_source_ids: list[str] = Field(default_factory=list)
    evidence_summary: list[str] = Field(default_factory=list)
    unresolved_points: list[str] = Field(default_factory=list)
    grounded_consequence: str
    calls_used: int = 0
    stop_reason: str


class IntegratedCaseTrace(BaseModel):
    case_id: str
    initial_evidence_need: str
    canonical_evidence_need: str
    materiality: bool
    resolvability: str
    tool_selected: str
    calls_used: int
    evidence_interpretation: IntegratedEvidenceInterpretation
    unresolved_points: list[str]
    blockers: list[dict[str, Any]]
    caveats: list[dict[str, Any]]
    final_readiness: str
    baseline_readiness: str
    readiness_changed: bool
    consequence_rationale: str
    downstream_brief: str
    brief_evidence_addendum: list[str] = Field(default_factory=list)


## Evidence-Need Inventory and Conditional Routing

Each case supplies a canonical, semantically validated evidence need. The selector applies one shared eligibility rule; it does not map proposal names directly to tools. A need must be material, unresolved, aligned to a tool, likely to add useful information, and not already sufficiently answered.


In [2]:
CASE_EVIDENCE_NEEDS: dict[str, IntegratedEvidenceNeed] = {
    "A": IntegratedEvidenceNeed(
        case_id="A",
        initial_uncertainty="Later rollout economics and operating design remain to be established.",
        canonical_need="Evidence for a later scale/rollout decision",
        semantic_mapping="aligned",
        material_to_current_decision=False,
        unresolved=True,
        resolvability="tool_insufficient",
        expected_information_gain="low",
        likelihood_of_improving_analysis="low",
        capability_alignment=False,
        rationale="The uncertainty belongs to a future decision and does not affect approval of the bounded pilot now.",
    ),
    "B": IntegratedEvidenceNeed(
        case_id="B",
        initial_uncertainty="Decision scope, ownership, sequencing, economics, and execution facts are incomplete.",
        canonical_need="Proposal-owner clarification and company-specific execution evidence",
        semantic_mapping="aligned",
        material_to_current_decision=True,
        unresolved=True,
        resolvability="tool_insufficient",
        expected_information_gain="low",
        likelihood_of_improving_analysis="low",
        capability_alignment=False,
        rationale="The missing facts must come from the proposal owner or internal operating teams; public search and the controlled policy corpus cannot supply them.",
    ),
    "C": IntegratedEvidenceNeed(
        case_id="C",
        initial_uncertainty="Whether Data Governance review is mandatory before approval.",
        canonical_need="Authoritative internal governance requirement for the proposed pilot",
        semantic_mapping="aligned",
        material_to_current_decision=True,
        unresolved=False,
        existing_evidence_sufficient=True,
        resolvability="internally_resolvable",
        expected_information_gain="low",
        likelihood_of_improving_analysis="low",
        capability_alignment=True,
        rationale="The frozen state already contains authoritative Northstar policy evidence confirming the requirement; repeat retrieval would add little value.",
    ),
    "D": IntegratedEvidenceNeed(
        case_id="D",
        initial_uncertainty="The proposal relies on stale 2023 market-growth evidence and lacks a current pricing benchmark.",
        canonical_need="Current public fasteners market size, growth, and pricing evidence relevant to the pilot hypothesis",
        semantic_mapping="aligned",
        material_to_current_decision=True,
        unresolved=True,
        resolvability="externally_resolvable",
        expected_information_gain="high",
        likelihood_of_improving_analysis="high",
        capability_alignment=True,
        rationale="Current market conditions and public benchmarks can update the dated external assumptions, while company-specific supplier feasibility remains outside public-search scope.",
    ),
    "E": IntegratedEvidenceNeed(
        case_id="E",
        initial_uncertainty="The proposal's pilot economics depend on a supplied workbook whose formulas, assumptions, horizons, and sensitivities require reconciliation.",
        canonical_need="Local quantitative validation of the attached Proposal E financial model against the proposal narrative",
        semantic_mapping="aligned",
        material_to_current_decision=True,
        unresolved=True,
        resolvability="quantitatively_resolvable",
        expected_information_gain="high",
        likelihood_of_improving_analysis="high",
        capability_alignment=True,
        rationale="The attached workbook is decision-material and can be inspected locally without disclosing it externally.",
        artifact_refs=[
            "inputs/proposal_e/DecisionReady_Proposal_E_Financial_Model.xlsx",
            "inputs/proposal_e/DecisionReady_Proposal_E_Sales_Productivity_Pilot.docx",
        ],
    ),
    "SYNTHETIC_POLICY": IntegratedEvidenceNeed(
        case_id="SYNTHETIC_POLICY",
        initial_uncertainty="Which internal approval authority applies to a $40,000 pilot?",
        canonical_need="Northstar approval-policy authority for a $40,000 pilot",
        semantic_mapping="aligned",
        material_to_current_decision=True,
        unresolved=True,
        resolvability="internally_resolvable",
        expected_information_gain="high",
        likelihood_of_improving_analysis="high",
        capability_alignment=True,
        rationale="The question is explicitly about internal policy, is not already answered in the synthetic case, and the controlled Northstar corpus can resolve it.",
    ),
}


def choose_integrated_tool(need: IntegratedEvidenceNeed) -> IntegratedToolDecision:
    eligible = (
        need.semantic_mapping == "aligned"
        and need.material_to_current_decision
        and need.unresolved
        and not need.existing_evidence_sufficient
        and need.capability_alignment
        and need.expected_information_gain in {"high", "medium"}
        and need.likelihood_of_improving_analysis in {"high", "medium"}
    )
    if not eligible:
        reason = (
            "No call: the need is future-stage, tool-insufficient, semantically ineligible, "
            "or already sufficiently grounded."
        )
        return IntegratedToolDecision(
            case_id=need.case_id,
            selected_tool="none",
            selected_need=need.canonical_need,
            rationale=f"{reason} {need.rationale}",
        )

    tool_by_resolvability = {
        "internally_resolvable": "internal_retrieval",
        "externally_resolvable": "external_search",
        "quantitatively_resolvable": "spreadsheet_analysis",
    }
    selected = tool_by_resolvability.get(need.resolvability, "none")
    return IntegratedToolDecision(
        case_id=need.case_id,
        selected_tool=selected,
        selected_need=need.canonical_need,
        rationale=need.rationale,
    )


TOOL_DECISIONS = {
    case_id: choose_integrated_tool(need)
    for case_id, need in CASE_EVIDENCE_NEEDS.items()
}


## Bounded Acquisition and Grounded Interpretation

This freeze regression replays only evidence already acquired and validated in v0.5/v0.6. `none` performs zero calls. The external, internal, and spreadsheet branches retain their original call counts, scope gates, source hierarchy, and stop reasons.


In [3]:
D_FROZEN_EXTERNAL_INTERPRETATION = IntegratedEvidenceInterpretation(
    case_id="D",
    selected_tool="external_search",
    evidence_effect="mixed",
    sufficiency="partial",
    supporting_source_ids=[
        "Business Research Insights",
        "DataIntelo",
        "MarketsandMarkets",
        "Grand View Research",
        "Transparency Market Research",
        "IBISWorld",
    ],
    evidence_summary=[
        "Recent commercial market reports generally indicate roughly 3–5% CAGR, weakening the proposal's dated 8–10% growth assumption.",
        "Public evidence qualitatively supports fragmented market/pricing conditions.",
        "The retrieved sources are predominantly commercial market-research pages; no primary government or trade dataset was found in the bounded search.",
    ],
    unresolved_points=[
        "Comparable numeric ASP/channel pricing remains unverified.",
        "The proposal's margin assumptions remain company-specific and unresolved.",
        "Supplier onboarding feasibility remains an internal operating question, not a public-search question.",
    ],
    grounded_consequence=(
        "The evidence updates the strength of the market hypothesis but does not prove or disprove the pilot. "
        "It remains a non-blocking evidence limitation; no blocker is created directly from search."
    ),
    calls_used=2,
    stop_reason="tool_call_budget_exhausted",
)


E_VALIDATED_FINDINGS = [
    {
        "finding": "ROI and stated payback use inconsistent benefit horizons.",
        "finding_type": "narrative_model_mismatch",
        "severity": "material_caveat",
        "workbook_cells": ["02_Base_Case!B19", "02_Base_Case!B21", "02_Base_Case!B22"],
        "rationale": "Pilot-period benefit drives ROI, while payback annualizes the benefit run-rate.",
    },
    {
        "finding": "Productivity value and commercial benefit may double-count recovered selling capacity.",
        "finding_type": "potential_double_count",
        "severity": "material_caveat",
        "workbook_cells": ["02_Base_Case!B13", "02_Base_Case!B17", "02_Base_Case!B18"],
        "rationale": "Both benefit streams rely on the same recovered time without an explicit incremental-value bridge.",
    },
    {
        "finding": "The 5% incremental conversion assumption lacks supporting evidence in the proposal.",
        "finding_type": "unsupported_assumption",
        "severity": "material_caveat",
        "workbook_cells": ["01_Assumptions!B12", "02_Base_Case!B16"],
        "rationale": "It is a pilot hypothesis, not a prerequisite, but materially influences modeled benefit.",
    },
    {
        "finding": "The base case assumes 75% effective adoption while the success threshold is 60%.",
        "finding_type": "narrative_model_mismatch",
        "severity": "material_caveat",
        "workbook_cells": ["01_Assumptions!B9", "01_Assumptions!B10", "02_Base_Case!B7"],
        "rationale": "Decision review should examine economics at the stated success threshold, not only the base case.",
    },
    {
        "finding": "Internal enablement and IT effort are excluded from pilot cost as BAU.",
        "finding_type": "cost_scope_exclusion",
        "severity": "material_caveat",
        "workbook_cells": ["01_Assumptions!B15", "02_Base_Case!B20"],
        "rationale": "This narrows the cost scope without constituting a formula error.",
    },
    {
        "finding": "A 26.67% attributable pipeline-realization factor appears only in the workbook.",
        "finding_type": "narrative_model_mismatch",
        "severity": "material_caveat",
        "workbook_cells": ["01_Assumptions!B16", "02_Base_Case!B17"],
        "rationale": "The factor materially reduces modeled commercial benefit but is not disclosed in the narrative.",
    },
    {
        "finding": "Productivity benefit does not vary with effective adoption in the sensitivity model.",
        "finding_type": "sensitivity_issue",
        "severity": "material_caveat",
        "workbook_cells": ["03_Sensitivity!E5", "03_Sensitivity!F5", "03_Sensitivity!G5", "02_Base_Case!B13"],
        "rationale": "Commercial benefit changes with adoption, but the productivity component remains fixed across adoption scenarios.",
    },
]


E_FROZEN_SPREADSHEET_INTERPRETATION = IntegratedEvidenceInterpretation(
    case_id="E",
    selected_tool="spreadsheet_analysis",
    evidence_effect="mixed",
    sufficiency="sufficient",
    supporting_source_ids=[
        "DecisionReady_Proposal_E_Sales_Productivity_Pilot.docx",
        "DecisionReady_Proposal_E_Financial_Model.xlsx",
    ],
    evidence_summary=[finding["finding"] for finding in E_VALIDATED_FINDINGS],
    unresolved_points=[
        "Actual conversion uplift remains a pilot hypothesis.",
        "Incremental internal enablement/IT cost remains unquantified.",
        "The economic interpretation needs an explicit treatment of recovered-capacity double counting.",
    ],
    grounded_consequence=(
        "The local workbook analysis produced seven material caveats and no arithmetic formula error or decision-blocking finding. "
        "Those structured findings flow to the existing caveat/readiness precedence; the tool does not set readiness directly."
    ),
    calls_used=1,
    stop_reason="all_material_spreadsheet_checks_completed",
)


SYNTHETIC_FROZEN_INTERNAL_INTERPRETATION = IntegratedEvidenceInterpretation(
    case_id="SYNTHETIC_POLICY",
    selected_tool="internal_retrieval",
    evidence_effect="strengthens",
    sufficiency="sufficient",
    supporting_source_ids=["Approval Policy::chunk-02"],
    evidence_summary=[
        "Northstar Approval Policy states that $25,001–$100,000 commitments require Business Unit Head approval plus Finance concurrence."
    ],
    unresolved_points=[],
    grounded_consequence="The retrieved authoritative policy resolves the synthetic approval-authority question.",
    calls_used=1,
    stop_reason="all_material_subquestions_sufficient",
)


def acquire_and_interpret(decision: IntegratedToolDecision) -> IntegratedEvidenceInterpretation:
    if decision.selected_tool == "none":
        return IntegratedEvidenceInterpretation(
            case_id=decision.case_id,
            selected_tool="none",
            evidence_effect="not_applicable",
            sufficiency="not_applicable",
            grounded_consequence="No evidence acquisition was justified; existing proposal/state evidence remains authoritative.",
            calls_used=0,
            stop_reason="no_eligible_tool_call",
        )
    frozen = {
        "D": D_FROZEN_EXTERNAL_INTERPRETATION,
        "E": E_FROZEN_SPREADSHEET_INTERPRETATION,
        "SYNTHETIC_POLICY": SYNTHETIC_FROZEN_INTERNAL_INTERPRETATION,
    }.get(decision.case_id)
    if frozen is None:
        raise RuntimeError(f"No validated bounded evidence trace exists for {decision.case_id}.")
    if frozen.selected_tool != decision.selected_tool:
        raise AssertionError("Selector result does not match the validated branch trace.")
    if frozen.calls_used > decision.max_calls:
        raise AssertionError("Validated trace exceeds the bounded call budget.")
    return frozen.model_copy(deep=True)


## Existing Decision Consequence Layer

For A–D, blocker, caveat, and readiness outputs are copied from the frozen Iteration-2 v0.5 state. D's new public evidence remains a structured interpretation and does not bypass that state. For E, the already validated spreadsheet findings are passed through the unchanged precedence rule: blockers → `NOT_READY`; otherwise material caveats → `READY_WITH_CAVEATS`; otherwise `READY`.


In [4]:
def _normalize_items(items: Any) -> list[dict[str, Any]]:
    normalized: list[dict[str, Any]] = []
    for item in items or []:
        if hasattr(item, "model_dump"):
            normalized.append(item.model_dump())
        elif isinstance(item, dict):
            normalized.append(copy.deepcopy(item))
        else:
            normalized.append({"value": str(item)})
    return normalized


def existing_readiness_precedence(blockers: list[dict[str, Any]], caveats: list[dict[str, Any]]) -> str:
    if blockers:
        return "NOT_READY"
    if caveats:
        return "READY_WITH_CAVEATS"
    return "READY"


def apply_existing_consequence(
    case_id: str,
    interpretation: IntegratedEvidenceInterpretation,
) -> tuple[str, str, list[dict[str, Any]], list[dict[str, Any]], str]:
    if case_id in {"A", "B", "C", "D"}:
        frozen = copy.deepcopy(FROZEN_DOWNSTREAM_SNAPSHOTS[case_id])
        baseline = frozen["readiness_status"]
        blockers = _normalize_items(frozen.get("hard_blockers"))
        caveats = _normalize_items(frozen.get("material_caveats"))
        final = existing_readiness_precedence(blockers, caveats)
        if final != baseline:
            raise AssertionError(f"Frozen {case_id} precedence no longer reproduces its status.")
        rationale = (
            "Frozen Iteration-2 v0.5 blockers and caveats determine readiness. "
            "The integrated evidence trace is additive and cannot directly set severity."
        )
        return baseline, final, blockers, caveats, rationale

    if case_id == "E":
        blockers: list[dict[str, Any]] = []
        caveats = [
            {
                "caveat_code": finding["finding_type"].upper(),
                "rationale": finding["rationale"],
                "evidence": finding["workbook_cells"],
                "related_item_text": finding["finding"],
                "source": "retrieved_evidence:local_spreadsheet_analysis",
            }
            for finding in E_VALIDATED_FINDINGS
            if finding["severity"] == "material_caveat"
        ]
        final = existing_readiness_precedence(blockers, caveats)
        return (
            "READY_WITH_CAVEATS",
            final,
            blockers,
            caveats,
            "Seven validated quantitative findings enter as material caveats; no finding is a formula error or prerequisite blocker.",
        )

    return "NOT_APPLICABLE", "NOT_APPLICABLE", [], [], "Selector/tool reachability test; canonical readiness is intentionally not run."


def build_integrated_downstream_brief(
    case_id: str,
    final_readiness: str,
    blockers: list[dict[str, Any]],
    caveats: list[dict[str, Any]],
    interpretation: IntegratedEvidenceInterpretation,
) -> tuple[str, list[str]]:
    """Carry concise interpreted evidence into synthesis without changing severity."""
    if case_id in {"A", "B", "C", "D"}:
        base_brief = str(FROZEN_DOWNSTREAM_SNAPSHOTS[case_id].get("final_brief", ""))
    elif case_id == "E":
        caveat_lines = "\n".join(
            f"- {item['related_item_text']} Evidence: {', '.join(item['evidence'])}."
            for item in caveats
        )
        base_brief = (
            f"Decision Readiness: {final_readiness}\n\n"
            "Decision: approve a bounded 12-week AI sales-productivity pilot for 40 users.\n\n"
            "Hard Blockers\n- None identified.\n\n"
            "Material Quantitative Caveats\n"
            f"{caveat_lines}\n\n"
            "The modeled commercial outcomes remain pilot hypotheses rather than prerequisites."
        )
    else:
        return "Not applicable: selector/tool reachability test only.", []

    if interpretation.selected_tool == "none":
        return base_brief, []

    addendum = [
        f"Evidence effect: {interpretation.evidence_effect}; sufficiency: {interpretation.sufficiency}.",
        *interpretation.evidence_summary,
        *[f"Still unresolved: {item}" for item in interpretation.unresolved_points],
        f"Sources: {', '.join(interpretation.supporting_source_ids)}.",
        f"Stop reason: {interpretation.stop_reason}; calls used: {interpretation.calls_used}.",
    ]
    rendered = (
        base_brief
        + "\n\nIntegrated Evidence Interpretation (does not independently set severity)\n"
        + "\n".join(f"- {line}" for line in addendum)
    )
    return rendered, addendum


def run_integrated_case(case_id: str) -> IntegratedCaseTrace:
    need = CASE_EVIDENCE_NEEDS[case_id]
    decision = TOOL_DECISIONS[case_id]
    interpretation = acquire_and_interpret(decision)
    baseline, final, blockers, caveats, consequence = apply_existing_consequence(case_id, interpretation)
    downstream_brief, brief_addendum = build_integrated_downstream_brief(
        case_id,
        final,
        blockers,
        caveats,
        interpretation,
    )
    return IntegratedCaseTrace(
        case_id=case_id,
        initial_evidence_need=need.initial_uncertainty,
        canonical_evidence_need=need.canonical_need,
        materiality=need.material_to_current_decision,
        resolvability=need.resolvability,
        tool_selected=decision.selected_tool,
        calls_used=interpretation.calls_used,
        evidence_interpretation=interpretation,
        unresolved_points=interpretation.unresolved_points,
        blockers=blockers,
        caveats=caveats,
        final_readiness=final,
        baseline_readiness=baseline,
        readiness_changed=final != baseline,
        consequence_rationale=consequence,
        downstream_brief=downstream_brief,
        brief_evidence_addendum=brief_addendum,
    )


INTEGRATED_TRACES = {
    case_id: run_integrated_case(case_id)
    for case_id in CASE_EVIDENCE_NEEDS
}


## Integrated A–E Trace

The table below is the canonical v0.7 routing and readiness trace. The synthetic policy case is included only to demonstrate the internal-retrieval branch and has no readiness status.


In [5]:
def _short(value: str, limit: int = 118) -> str:
    return value if len(value) <= limit else value[: limit - 1] + "…"


print("| Case | Canonical evidence need | Material | Resolvability | Tool | Calls | Effect / sufficiency | Blockers | Caveats | Final readiness | Changed? | Stop reason |")
print("|---|---|---:|---|---|---:|---|---:|---:|---|---:|---|")
for case_id, trace in INTEGRATED_TRACES.items():
    interp = trace.evidence_interpretation
    print(
        f"| {case_id} | {_short(trace.canonical_evidence_need)} | {trace.materiality} | "
        f"{trace.resolvability} | {trace.tool_selected} | {trace.calls_used} | "
        f"{interp.evidence_effect} / {interp.sufficiency} | {len(trace.blockers)} | "
        f"{len(trace.caveats)} | {trace.final_readiness} | {trace.readiness_changed} | {interp.stop_reason} |"
    )


print("\nProposal D evidence-to-consequence trace")
print(json.dumps(INTEGRATED_TRACES["D"].model_dump(), indent=2))

print("\nProposal E evidence-to-consequence trace")
print(json.dumps(INTEGRATED_TRACES["E"].model_dump(), indent=2))


| Case | Canonical evidence need | Material | Resolvability | Tool | Calls | Effect / sufficiency | Blockers | Caveats | Final readiness | Changed? | Stop reason |
|---|---|---:|---|---|---:|---|---:|---:|---|---:|---|
| A | Evidence for a later scale/rollout decision | False | tool_insufficient | none | 0 | not_applicable / not_applicable | 0 | 0 | READY | False | no_eligible_tool_call |
| B | Proposal-owner clarification and company-specific execution evidence | True | tool_insufficient | none | 0 | not_applicable / not_applicable | 1 | 2 | NOT_READY | False | no_eligible_tool_call |
| C | Authoritative internal governance requirement for the proposed pilot | True | internally_resolvable | none | 0 | not_applicable / not_applicable | 1 | 1 | NOT_READY | False | no_eligible_tool_call |
| D | Current public fasteners market size, growth, and pricing evidence relevant to the pilot hypothesis | True | externally_resolvable | external_search | 2 | mixed / partial | 0 | 1 | READY_WITH_CAVE

## Frozen Regression and Outcome Reachability

These assertions are the final regression. They verify the A–D frozen baseline, Proposal E's quantitative-caveat result, preservation of C's policy-grounded blocker, non-escalation for D/E, bounded calls, and reachability of all four tool outcomes.


In [6]:
TARGET_STATUSES = {
    "A": "READY",
    "B": "NOT_READY",
    "C": "NOT_READY",
    "D": "READY_WITH_CAVEATS",
    "E": "READY_WITH_CAVEATS",
}

REGRESSION_RESULTS: list[dict[str, Any]] = []
for case_id, expected in TARGET_STATUSES.items():
    trace = INTEGRATED_TRACES[case_id]
    passed = trace.final_readiness == expected and not trace.readiness_changed
    REGRESSION_RESULTS.append(
        {
            "case": case_id,
            "baseline": trace.baseline_readiness,
            "expected": expected,
            "actual": trace.final_readiness,
            "passed": passed,
        }
    )
    assert passed, f"{case_id} regression failed: {trace.final_readiness} != {expected}"

assert INTEGRATED_TRACES["C"].tool_selected == "none"
assert any(
    blocker.get("blocker_code") == "UNRESOLVED_CRITICAL_DEPENDENCY"
    for blocker in INTEGRATED_TRACES["C"].blockers
)
assert any(
    source_id == "Approval Policy::chunk-02"
    for item in FROZEN_DOWNSTREAM_SNAPSHOTS["C"].get("grounded_requirements", [])
    for source_id in item.get("supporting_source_ids", [])
)
assert INTEGRATED_TRACES["D"].tool_selected == "external_search"
assert INTEGRATED_TRACES["D"].calls_used == 2
assert not INTEGRATED_TRACES["D"].blockers
assert INTEGRATED_TRACES["E"].tool_selected == "spreadsheet_analysis"
assert INTEGRATED_TRACES["E"].calls_used == 1
assert len(INTEGRATED_TRACES["E"].caveats) == 7
assert not INTEGRATED_TRACES["E"].blockers
assert INTEGRATED_TRACES["D"].brief_evidence_addendum
assert "3–5% CAGR" in INTEGRATED_TRACES["D"].downstream_brief
assert len(INTEGRATED_TRACES["E"].brief_evidence_addendum) >= 7
assert "02_Base_Case!B19" in INTEGRATED_TRACES["E"].downstream_brief
assert INTEGRATED_TRACES["SYNTHETIC_POLICY"].tool_selected == "internal_retrieval"
assert all(trace.calls_used <= MAX_TOOL_CALLS for trace in INTEGRATED_TRACES.values())

REACHABLE_TOOL_OUTCOMES = {trace.tool_selected for trace in INTEGRATED_TRACES.values()}
assert REACHABLE_TOOL_OUTCOMES == {
    "none",
    "internal_retrieval",
    "external_search",
    "spreadsheet_analysis",
}

print("| Case | Frozen baseline | Expected | Actual | Pass |")
print("|---|---|---|---|---:|")
for row in REGRESSION_RESULTS:
    print(f"| {row['case']} | {row['baseline']} | {row['expected']} | {row['actual']} | {row['passed']} |")
print(f"\nReachable tool outcomes: {sorted(REACHABLE_TOOL_OUTCOMES)}")
print("Final regression: PASS — no post-regression tuning performed.")


| Case | Frozen baseline | Expected | Actual | Pass |
|---|---|---|---|---:|
| A | READY | READY | READY | True |
| B | NOT_READY | NOT_READY | NOT_READY | True |
| C | NOT_READY | NOT_READY | NOT_READY | True |
| D | READY_WITH_CAVEATS | READY_WITH_CAVEATS | READY_WITH_CAVEATS | True |
| E | READY_WITH_CAVEATS | READY_WITH_CAVEATS | READY_WITH_CAVEATS | True |

Reachable tool outcomes: ['external_search', 'internal_retrieval', 'none', 'spreadsheet_analysis']
Final regression: PASS — no post-regression tuning performed.


## Persisted v0.7 Audit Snapshot

The next cell writes the compact independent-review artifact from the exact in-memory traces asserted above.


In [7]:
def _bullet_list(items: list[str], empty: str = "None") -> str:
    return "\n".join(f"- {item}" for item in items) if items else f"- {empty}"


def _codes(items: list[dict[str, Any]], key: str) -> str:
    values = [str(item.get(key) or item.get("related_item_text") or item.get("value")) for item in items]
    return ", ".join(values) if values else "None"


matrix_rows = []
for case_id in ["A", "B", "C", "D", "E", "SYNTHETIC_POLICY"]:
    trace = INTEGRATED_TRACES[case_id]
    matrix_rows.append(
        "| " + " | ".join(
            [
                case_id,
                trace.resolvability,
                trace.tool_selected,
                str(trace.calls_used),
                f"{trace.evidence_interpretation.evidence_effect} / {trace.evidence_interpretation.sufficiency}",
                trace.evidence_interpretation.stop_reason,
                trace.final_readiness,
                "Yes" if trace.readiness_changed else "No",
            ]
        ) + " |"
    )

d_trace = INTEGRATED_TRACES["D"]
e_trace = INTEGRATED_TRACES["E"]
c_trace = INTEGRATED_TRACES["C"]

e_findings_lines = []
for finding in E_VALIDATED_FINDINGS:
    e_findings_lines.append(
        f"- **{finding['finding']}** — `{finding['finding_type']}`, `{finding['severity']}`; "
        f"cells: {', '.join(finding['workbook_cells'])}. {finding['rationale']}"
    )

snapshot_text = f"""# DecisionReady Iteration 3 v0.7 — Audit Snapshot

## Scope and freeze boundary

This is the canonical thin-integration snapshot. It connects the validated evidence selector and the `none`, internal retrieval, external search, and spreadsheet branches to the existing DecisionReady consequence layer. It does not create an open research loop, re-run extraction, or allow a tool result to set readiness directly.

The regression replays persisted, previously validated evidence traces. No fresh network call or OpenAI call was made. The call counts below preserve the bounded acquisition counts of the validated branch executions.

## Integrated control flow

`frozen case state → identify evidence need → canonicalize/validate → choose tool or none → bounded acquisition → grounded interpretation → existing blocker/caveat/readiness consequence → trace`

Shared controls:

- `MAX_TOOL_CALLS = {MAX_TOOL_CALLS}` per case.
- Only semantically aligned, current-decision-material, unresolved, tool-resolvable needs with meaningful expected information gain are eligible.
- Tool interpretation is claim-relative and preserves unresolved subquestions.
- Evidence first becomes a structured interpretation. The existing blocker/caveat/readiness precedence determines severity.
- `none` is valid and performs zero calls.

## A–E routing and status matrix

| Case | Resolvability | Selected tool | Calls | Evidence effect / sufficiency | Stop reason | Final readiness | Changed from baseline? |
|---|---|---:|---:|---|---|---|---:|
{chr(10).join(matrix_rows)}

All four tool outcomes are reachable: `{', '.join(sorted(REACHABLE_TOOL_OUTCOMES))}`.

## Why each route was selected

- **A — none:** the unresolved economics/operating design belongs to a later rollout decision, not the current bounded pilot decision.
- **B — none:** the important gaps require proposal-owner or company-specific execution facts; neither public search nor the controlled policy corpus can supply them.
- **C — none:** authoritative Northstar evidence is already present in the frozen state, so repeat retrieval has low incremental value. The policy-grounded unresolved dependency remains intact.
- **D — external_search:** dated public market-growth evidence and the missing current pricing benchmark are externally verifiable and material to the present pilot hypothesis.
- **E — spreadsheet_analysis:** the supplied local workbook is decision-material, quantitatively resolvable, and inspectable without external disclosure.
- **Synthetic policy — internal_retrieval:** the unresolved approval-authority question is explicitly about internal policy and is resolvable from the controlled Northstar corpus.

## Proposal D: external evidence to consequence trace

**Initial need:** {d_trace.initial_evidence_need}

**Canonical need:** {d_trace.canonical_evidence_need}

**Sources retained:** {', '.join(d_trace.evidence_interpretation.supporting_source_ids)}.

**Grounded interpretation:**

{_bullet_list(d_trace.evidence_interpretation.evidence_summary)}

**Still unresolved:**

{_bullet_list(d_trace.unresolved_points)}

**Consequence:** {d_trace.evidence_interpretation.grounded_consequence} Readiness remains `{d_trace.final_readiness}` with blockers `{_codes(d_trace.blockers, 'blocker_code')}` and frozen caveats `{_codes(d_trace.caveats, 'caveat_code')}`. Stop reason: `{d_trace.evidence_interpretation.stop_reason}` after {d_trace.calls_used} calls.

**Downstream synthesis:** the frozen final brief is retained and receives a labeled integrated-evidence addendum containing the effect, sufficiency, sources, unresolved points, and stop reason. This addendum does not alter severity.

## Proposal E: spreadsheet evidence to consequence trace

**Initial need:** {e_trace.initial_evidence_need}

**Canonical need:** {e_trace.canonical_evidence_need}

**Validated quantitative findings:**

{chr(10).join(e_findings_lines)}

**Consequence discipline:** no arithmetic formula error or prerequisite blocker was established. The seven findings enter the existing consequence layer as material caveats. Pilot conversion/commercial outcomes remain hypotheses. Result: `{e_trace.final_readiness}`, blockers `{_codes(e_trace.blockers, 'blocker_code')}`, caveats `{len(e_trace.caveats)}`. Stop reason: `{e_trace.evidence_interpretation.stop_reason}` after {e_trace.calls_used} local call.

**Downstream synthesis:** the Proposal E brief is built from the resulting blockers/caveats and receives the same labeled evidence addendum with workbook/source provenance and unresolved hypotheses.

## Proposal C blocker preservation

Proposal C remains `{c_trace.final_readiness}`. No new retrieval was performed because the frozen state already contains authoritative support, including `Approval Policy::chunk-02`. Its `UNRESOLVED_CRITICAL_DEPENDENCY` blocker remains a consequence of the existing blocker layer, not a direct retrieval-side status assignment.

## Deterministic regression

| Case | Frozen baseline | Target | Actual | Result |
|---|---|---|---|---|
{chr(10).join(f"| {row['case']} | {row['baseline']} | {row['expected']} | {row['actual']} | {'PASS' if row['passed'] else 'FAIL'} |" for row in REGRESSION_RESULTS)}

Additional assertions passed:

- C's policy-grounded blocker and source trace are preserved.
- D and E receive no unsupported blocker escalation.
- D and E interpreted evidence is present in a labeled downstream-brief addendum without independently setting severity.
- D uses two bounded external calls; E uses one local spreadsheet call; the synthetic policy case uses one local internal retrieval call.
- A, B, and C use zero calls.
- Every branch respects `MAX_TOOL_CALLS = 2`.
- No post-regression tuning was performed.

## Regressions, limitations, and freeze judgment

No A–E status regression occurred. External-search evidence for D remains commercially sourced and only partially sufficient; numeric pricing/margin evidence is unresolved. Proposal E's findings validate model consistency and limitations but do not provide actual realized pilot performance. A–D continue to inherit the frozen extraction fixtures, so this test isolates integration behavior rather than live extraction variability.

**Freeze judgment:** the bounded-autonomy architecture is sufficiently thin, traceable, and regression-stable to freeze as the Iteration-3 integration baseline. Any future expansion should preserve the evidence-first consequence boundary and add new tools only through the same eligibility and trace contracts.
"""

V07_SNAPSHOT_PATH.write_text(snapshot_text, encoding="utf-8")
print(f"Wrote {V07_SNAPSHOT_PATH.resolve()}")


Wrote C:\Users\anjan\problem_first_ai\DecisionReady_Iteration3_v07_Audit_Snapshot.md


## Iteration 3 v0.7 Result

- The bounded selector is now connected to a single auditable consequence path.
- A, B, and C select `none`; D selects `external_search`; E selects `spreadsheet_analysis`; the synthetic policy case selects `internal_retrieval`.
- Frozen A–D results remain `READY`, `NOT_READY`, `NOT_READY`, and `READY_WITH_CAVEATS`.
- Proposal E resolves to `READY_WITH_CAVEATS` from seven validated spreadsheet findings and no hard blocker.
- All four tool outcomes are reachable within the two-call budget.
- No fresh research or model calls were made, and no status was set directly by a tool result.
